In [ ]:
# -------------------------------------------------------------------------------------------------
# 0. INSTALLATION & SETUP
# -------------------------------------------------------------------------------------------------
!pip -q install -U "pandas<2.4.0,>=2.0" "scikit-learn" "scipy" "transformer-lens"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.5.10 — STRICT OUT-OF-SAMPLE TEMPORAL IDENTIFIABILITY BENCHMARK
#
# Target: Mathematically proving whether precursor dynamics possess genuine out-of-sample
#         predictive power regarding future degradation without label/window contamination.
# Model : Qwen/Qwen2.5-0.5B via HookedTransformer (blocks.6.hook_resid_post)
# =================================================================================================

import json
import math
import random
import hashlib
import warnings

import numpy as np
import torch
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mutual_info_score

warnings.filterwarnings("ignore")
from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. REPRODUCIBILITY & SETUP
# -------------------------------------------------------------------------------------------------
SEED = 205930
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 115)
print(f"🔬 MILESTONE 20.5.10: STRICT OUT-OF-SAMPLE IDENTIFIABILITY AUDIT ({DEVICE.upper()})")
print(f"Seed: {SEED} | Model: {MODEL_NAME} | Hook: blocks.{PREFERRED_LAYER}.hook_resid_post")
print("=" * 115)

# -------------------------------------------------------------------------------------------------
# 2. GENERATION OF INDEPENDENT TRAIN & TEST PRECURSOR STREAMS
# -------------------------------------------------------------------------------------------------
def generate_stream(stream_id, seed_offset, n_steps=100, drift_start=35):
    rng = np.random.RandomState(SEED + seed_offset)
    stream_data = []

    for t in range(n_steps):
        a = rng.uniform(-1.0, 1.0)
        while abs(a) < 0.10:
            a = rng.uniform(-1.0, 1.0)
        b = rng.uniform(-1.0, 1.0)

        precursor = bool(20 <= t < drift_start)
        drift = bool(t >= drift_start)

        if precursor:
            # Subtle latent zero-mean variance expansion
            step_offset = t - 20
            sigma_a = 0.20 + 0.015 * min(15, step_offset)
            sigma_b = 0.16 + 0.015 * min(15, step_offset)
            eta_a = rng.normal(0.0, sigma_a)
            eta_b = rng.normal(0.0, sigma_b)
            mechanism = float((0.8 + eta_a) * a - (0.55 + eta_b) * b)
            alpha = 0.0
        elif drift:
            # Deterministic causal shift
            alpha = float(min(1.0, 0.10 + 0.90 * ((t - drift_start) / 30.0)))
            mechanism = float((1.0 - 2.0 * alpha) * (0.8 * a) - 0.55 * b)
        else:
            alpha = 0.0
            mechanism = float(0.8 * a - 0.55 * b)

        raw_id = f"{stream_id}_{t}_{a:.6f}_{b:.6f}_{alpha:.4f}_{precursor}"
        fingerprint = hashlib.sha256(raw_id.encode()).hexdigest()

        stream_data.append({
            "stream_id": stream_id,
            "t_step": t,
            "a": float(a),
            "b": float(b),
            "alpha": float(alpha),
            "precursor": precursor,
            "drift": drift,
            "mechanism": mechanism,
            "fingerprint": fingerprint
        })
    return stream_data

# Generation of clean calibration baseline (N=160)
cal_meta = [generate_stream("CAL", 100, n_steps=1)[0] for _ in range(160)]

# Dedicated Independent Training Stream for Temporal Predictors (Out-of-sample fit)
train_stream_meta = generate_stream("TRAIN_STREAM", 1)

# Dedicated Evaluation Stream
test_stream_meta = generate_stream("TEST_STREAM", 2)

print("🔐 Asserting zero fingerprint collisions between partitions...")
cal_fps = set(x["fingerprint"] for x in cal_meta)
train_fps = set(x["fingerprint"] for x in train_stream_meta)
test_fps = set(x["fingerprint"] for x in test_stream_meta)
assert len(train_fps.intersection(test_fps)) == 0, "Train and Test stream overlap!"
assert len(cal_fps.intersection(test_fps)) == 0, "Cal and Test stream overlap!"
print("🔐 Partition Independence: VERIFIED")

# -------------------------------------------------------------------------------------------------
# 3. HOOKED TRANSFORMER ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------
print(f"⏳ Loading {MODEL_NAME}...")
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

def make_prompt(row):
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=16):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from internal representation...")
cal_X = extract_features(cal_meta)
cal_y = np.array([x["mechanism"] for x in cal_meta])

train_X = extract_features(train_stream_meta)
train_y = np.array([x["mechanism"] for x in train_stream_meta])

test_X = extract_features(test_stream_meta)
test_y = np.array([x["mechanism"] for x in test_stream_meta])

# -------------------------------------------------------------------------------------------------
# 4. FROZEN SELF-MODEL & RESIDUAL COMPUTATION
# -------------------------------------------------------------------------------------------------
class FrozenSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.reg = RidgeCV(alphas=np.logspace(-3, 4, 20))

    def fit(self, X, y):
        X_s = self.scaler.fit_transform(X)
        self.reg.fit(X_s, y)

    def predict(self, X):
        return self.reg.predict(self.scaler.transform(X))

self_model = FrozenSelfModel()
self_model.fit(cal_X, cal_y)

# Residuals
cal_residuals = np.abs(cal_y - self_model.predict(cal_X))
train_residuals = np.abs(train_y - self_model.predict(train_X))
test_residuals = np.abs(test_y - self_model.predict(test_X))

cal_med = float(np.median(cal_residuals))
cal_mad = float(np.median(np.abs(cal_residuals - cal_med)))
cal_mad_scale = float(1.4826 * cal_mad + EPS)

# -------------------------------------------------------------------------------------------------
# 5. STRICT TARGET DEFINITION: UNCONTAMINATED FUTURE DEGRADATION
# -------------------------------------------------------------------------------------------------
# The future degradation metric is strictly the mean residual in the post-changepoint window (steps 35..45)
# This eliminates overlap between precursor observations (t=20..34) and the target.
TARGET_HORIZON_START = 35
TARGET_HORIZON_END = 46

train_future_degradation = float(np.mean(train_residuals[TARGET_HORIZON_START:TARGET_HORIZON_END]))
test_future_degradation = float(np.mean(test_residuals[TARGET_HORIZON_START:TARGET_HORIZON_END]))

# Pointwise dynamic target: Expected mean error across the first 6 steps of actual drift (t=35..40)
# Target is fixed as the true onset impact, evaluated at each precursor timestep t in [20, 34]
target_drift_severity_train = np.array([
    float(np.mean(train_residuals[35:41])) - cal_med for _ in range(20, 35)
])
target_drift_severity_test = np.array([
    float(np.mean(test_residuals[35:41])) - cal_med for _ in range(20, 35)
])

# -------------------------------------------------------------------------------------------------
# 6. FEATURE ENGINEERING: TEMPORAL PREDICTORS
# -------------------------------------------------------------------------------------------------
def extract_temporal_features(res_stream, act_stream, window_size=6):
    pca = PCA(n_components=4, random_state=SEED)
    act_reduced = pca.fit_transform(act_stream)

    ar_feats = []
    act_feats = []

    # Extract only across precursor window t in [20..34]
    for t in range(20, 35):
        # 1. Autoregressive features over past W steps
        res_window = res_stream[t - window_size : t]
        ar_feats.append([
            float(np.median(res_window)),
            float(1.4826 * np.median(np.abs(res_window - np.median(res_window)))), # Local MAD
            float(np.var(res_window)),
            float(res_window[-1] - res_window[0]) # Raw trajectory slope
        ])

        # 2. Latent activation manifold features (Window pooled mean and variance)
        act_window = act_reduced[t - window_size : t]
        act_feats.append(np.hstack([
            np.mean(act_window, axis=0),
            np.var(act_window, axis=0)
        ]))

    return np.array(ar_feats), np.array(act_feats)

train_ar_X, train_act_X = extract_temporal_features(train_residuals, train_X)
test_ar_X, test_act_X = extract_temporal_features(test_residuals, test_X)

# -------------------------------------------------------------------------------------------------
# 7. MODEL TRAINING & OUT-OF-SAMPLE INFERENCE
# -------------------------------------------------------------------------------------------------
# Baseline 1: Autoregressive Residual Predictor
scaler_ar = StandardScaler()
X_tr_ar = scaler_ar.fit_transform(train_ar_X)
X_te_ar = scaler_ar.transform(test_ar_X)

model_b1 = RidgeCV(alphas=np.logspace(-2, 3, 10))
model_b1.fit(X_tr_ar, target_drift_severity_train)
b1_preds = model_b1.predict(X_te_ar)

# Baseline 2: Latent Representation Predictor
scaler_act = StandardScaler()
X_tr_act = scaler_act.fit_transform(train_act_X)
X_te_act = scaler_act.transform(test_act_X)

model_b2 = RidgeCV(alphas=np.logspace(-2, 3, 10))
model_b2.fit(X_tr_act, target_drift_severity_train)
b2_preds = model_b2.predict(X_te_act)

# Baseline 3: M20.5.8 Handcrafted State Machine Emulation (Dispersion-Energy Accumulator)
b3_precursor_energy = []
curr_energy = 0.0
for t in range(20, 35):
    res_window = test_residuals[t - 6 : t]
    local_mad = 1.4826 * np.median(np.abs(res_window - np.median(res_window)))
    disp_ratio = local_mad / cal_mad_scale
    if disp_ratio > 1.20:
        curr_energy = 0.88 * curr_energy + min(1.5, disp_ratio - 1.0)
    else:
        curr_energy = max(0.0, 0.72 * curr_energy - 0.20)
    b3_precursor_energy.append(curr_energy)
b3_preds = np.array(b3_precursor_energy)

# Baseline 0: Permutation Null Model (Monte Carlo block shuffle of test predictions)
N_SHUFFLE = 1000
null_rhos = []
for _ in range(N_SHUFFLE):
    shuff_idx = np.random.permutation(len(target_drift_severity_test))
    r_null, _ = spearmanr(b1_preds, target_drift_severity_test[shuff_idx])
    null_rhos.append(r_null if not np.isnan(r_null) else 0.0)
b0_p95_null_rho = float(np.quantile(null_rhos, 0.95))

# -------------------------------------------------------------------------------------------------
# 8. EMPIRICAL IDENTIFIABILITY METRICS
# -------------------------------------------------------------------------------------------------
# Rank Correlation with the target progression
# Target progression reflects true cumulative degradation severity over steps 35..45
true_degradation_profile = np.array([
    float(np.mean(test_residuals[35 : 35 + (k - 19)])) for k in range(20, 35)
])

rho_b1, p_b1 = spearmanr(b1_preds, true_degradation_profile)
rho_b2, p_b2 = spearmanr(b2_preds, true_degradation_profile)
rho_b3, p_b3 = spearmanr(b3_preds, true_degradation_profile)

r2_b1 = float(r2_score(true_degradation_profile, b1_preds))
r2_b2 = float(r2_score(true_degradation_profile, b2_preds))

# Mutual Information Proxies
def compute_mi(pred_arr, target_arr, n_bins=3):
    q = np.linspace(0, 1, n_bins + 1)
    b_p = np.digitize(pred_arr, np.quantile(pred_arr, q)[:-1])
    b_t = np.digitize(target_arr, np.quantile(target_arr, q)[:-1])
    return float(mutual_info_score(b_p, b_t))

mi_b1 = compute_mi(b1_preds, true_degradation_profile)
mi_b2 = compute_mi(b2_preds, true_degradation_profile)
mi_b3 = compute_mi(b3_preds, true_degradation_profile)

# -------------------------------------------------------------------------------------------------
# 9. RIGID SCIENTIFIC VERDICT EVALUATION
# -------------------------------------------------------------------------------------------------
print("\n" + "=" * 115)
print("🏁 MILESTONE 20.5.10 AUDIT REPORT: CAUSAL IDENTIFIABILITY & STATISTICAL SUFFICIENCY")
print("=" * 115)
print(f"Precursor Window Evaluated   : Steps 20 to 34 (Total: {len(true_degradation_profile)} steps)")
print(f"Target Drift Horizon Window  : Steps 35 to 45 (Completely disjoint from evaluation)")
print("-" * 115)
print(f"Baseline 0 (Null P95 Limit)  : Rank Correlation Rho = {b0_p95_null_rho:.4f}")
print(f"Baseline 1 (AR Residual)     : Rho = {rho_b1:.4f} (p={p_b1:.4f}) | R² = {r2_b1:.4f} | MI = {mi_b1:.4f}")
print(f"Baseline 2 (Latent Act)      : Rho = {rho_b2:.4f} (p={p_b2:.4f}) | R² = {r2_b2:.4f} | MI = {mi_b2:.4f}")
print(f"Baseline 3 (Handcrafted SM)  : Rho = {rho_b3:.4f} (p={p_b3:.4f}) | MI = {mi_b3:.4f}")
print("-" * 115)

# Scientific Criteria
crit_out_of_sample_valid = bool(rho_b1 > b0_p95_null_rho or rho_b2 > b0_p95_null_rho)
crit_architecture_gap = bool(rho_b1 > rho_b3 and rho_b2 > rho_b3)
crit_representation_gain = bool(rho_b2 >= rho_b1)

print("\nDIAGNOSTIC VERDICT MATRIX:")
if not crit_out_of_sample_valid:
    verdict = "INSUFFICIENT_TEMPORAL_INFORMATION"
    diagnosis = "Precursor dynamics show no statistically significant relationship with future drift severity when evaluated out-of-sample. The signal in M20.5.9 was an artifact of in-sample training leakage."
elif crit_architecture_gap:
    verdict = "ARCHITECTURE_BOTTLENECK_CONFIRMED"
    diagnosis = "True out-of-sample temporal models (B1/B2) decisively outperform the handcrafted state machine (B3). Information exists, but the threshold-based state machine cannot exploit it."
else:
    verdict = "MARGINAL_SIGNIFICANCE"
    diagnosis = "Temporal correlation is statistically borderline; representation-level modeling does not significantly outperform scalar residual tracking."

print(f"Status Verdict : 🏷️ {verdict}")
print(f"Diagnosis      : {diagnosis}")

# -------------------------------------------------------------------------------------------------
# 10. TELEMETRY EXPORT
# -------------------------------------------------------------------------------------------------
telemetry = {
    "milestone": "M20.5.10",
    "verdict": verdict,
    "diagnosis": diagnosis,
    "criteria": {
        "out_of_sample_information_valid": crit_out_of_sample_valid,
        "architecture_gap_proven": crit_architecture_gap,
        "representation_gain_proven": crit_representation_gain
    },
    "metrics": {
        "b0_null_p95": float(b0_p95_null_rho),
        "b1_residual_rho": float(rho_b1),
        "b1_residual_p": float(p_b1),
        "b1_residual_mi": float(mi_b1),
        "b2_latent_rho": float(rho_b2),
        "b2_latent_p": float(p_b2),
        "b2_latent_mi": float(mi_b2),
        "b3_handcrafted_rho": float(rho_b3),
        "b3_handcrafted_p": float(p_b3),
        "b3_handcrafted_mi": float(mi_b3)
    }
}

with open("m20_5_10_identifiability_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry written to 'm20_5_10_identifiability_results.json'")
print("=" * 115)

🔬 MILESTONE 20.5.10: STRICT OUT-OF-SAMPLE IDENTIFIABILITY AUDIT (CUDA)
Seed: 205930 | Model: Qwen/Qwen2.5-0.5B | Hook: blocks.6.hook_resid_post
🔐 Asserting zero fingerprint collisions between partitions...
🔐 Partition Independence: VERIFIED
⏳ Loading Qwen/Qwen2.5-0.5B...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🔬 Extracting activations from internal representation...

🏁 MILESTONE 20.5.10 AUDIT REPORT: CAUSAL IDENTIFIABILITY & STATISTICAL SUFFICIENCY
Precursor Window Evaluated   : Steps 20 to 34 (Total: 15 steps)
Target Drift Horizon Window  : Steps 35 to 45 (Completely disjoint from evaluation)
-------------------------------------------------------------------------------------------------------------------
Baseline 0 (Null P95 Limit)  : Rank Correlation Rho = 0.0000
Baseline 1 (AR Residual)     : Rho = nan (p=nan) | R² = -2.4744 | MI = 0.0000
Baseline 2 (Latent Act)      : Rho = nan (p=nan) | R² = -2.4744 | MI = 0.0000
Baseline 3 (Handcrafted SM)  : Rho = 0.7464 (p=0.0014) | MI = 0.6499
-------------------------------------------------------------------------------------------------------------------

DIAGNOSTIC VERDICT MATRIX:
Status Verdict : 🏷️ INSUFFICIENT_TEMPORAL_INFORMATION
Diagnosis      : Precursor dynamics show no st

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.5.11 — PREDICTION IDENTIFIABILITY & FEATURE DISSECTION AUDIT
#
# Target: Eliminating zero-variance degenerate targets, detrending time confounds,
#         and isolating which structural feature carries genuine out-of-sample predictability.
# Model : Qwen/Qwen2.5-0.5B via HookedTransformer (blocks.6.hook_resid_post)
# =================================================================================================

import json
import warnings
from collections import deque

import numpy as np
import torch
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
from transformer_lens import HookedTransformer

SEED = 205931
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 115)
print(f"🔬 MILESTONE 20.5.11: PREDICTIVE FEATURE DISSECTION AUDIT ({DEVICE.upper()})")
print("=" * 115)

# -------------------------------------------------------------------------------------------------
# 1. STREAM GENERATION (MULTIPLE INDEPENDENT REALIZATIONS)
# -------------------------------------------------------------------------------------------------
def generate_stream_batch(n_streams=20, n_steps=90, drift_start=35):
    all_streams = []
    for s_idx in range(n_streams):
        stream = []
        rng = np.random.RandomState(SEED + s_idx * 17)
        for t in range(n_steps):
            a = rng.uniform(-1.0, 1.0)
            while abs(a) < 0.10: a = rng.uniform(-1.0, 1.0)
            b = rng.uniform(-1.0, 1.0)

            precursor = bool(20 <= t < drift_start)
            drift = bool(t >= drift_start)

            if precursor:
                step_offset = t - 20
                eta_a = rng.normal(0.0, 0.20 + 0.015 * step_offset)
                eta_b = rng.normal(0.0, 0.16 + 0.015 * step_offset)
                mechanism = float((0.8 + eta_a) * a - (0.55 + eta_b) * b)
                alpha = 0.0
            elif drift:
                alpha = float(min(1.0, 0.10 + 0.90 * ((t - drift_start) / 30.0)))
                mechanism = float((1.0 - 2.0 * alpha) * (0.8 * a) - 0.55 * b)
            else:
                alpha = 0.0
                mechanism = float(0.8 * a - 0.55 * b)

            stream.append({"a": float(a), "b": float(b), "mechanism": mechanism, "t": t})
        all_streams.append(stream)
    return all_streams

# 10 streams for training feature regressors, 10 streams for out-of-sample evaluation
train_streams = generate_stream_batch(n_streams=10, drift_start=35)
test_streams  = generate_stream_batch(n_streams=10, drift_start=35)

cal_meta = generate_stream_batch(n_streams=2, n_steps=80, drift_start=999)[0]

# -------------------------------------------------------------------------------------------------
# 2. MODEL EXTRACTION & INFERENCE
# -------------------------------------------------------------------------------------------------
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break

def extract_stream_data(streams):
    extracted = []
    for stream in streams:
        prompts = [
            f"You are solving a numerical reasoning task.\nInput A = {x['a']:.5f}\nInput B = {x['b']:.5f}\nInfer the relationship between the two quantities.\nProvide the predicted resulting value."
            for x in stream
        ]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
            act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        y = np.array([x["mechanism"] for x in stream])
        extracted.append((act, y))
    return extracted

print("🔬 Extracting activations for training and testing streams...")
cal_X_batch = extract_stream_data([cal_meta])[0]
train_extracted = extract_stream_data(train_streams)
test_extracted  = extract_stream_data(test_streams)

# Fit Baseline Nominal Predictor on Calibration stream
cal_X, cal_y = cal_X_batch
scaler = StandardScaler()
nominal_reg = RidgeCV(alphas=np.logspace(-3, 4, 20))
nominal_reg.fit(scaler.fit_transform(cal_X), cal_y)

cal_res = np.abs(cal_y - nominal_reg.predict(scaler.transform(cal_X)))
base_mad = float(1.4826 * np.median(np.abs(cal_res - np.median(cal_res)))) + EPS

# -------------------------------------------------------------------------------------------------
# 3. FEATURE EXTRACTION PIPELINE (POINTWISE PER TIMESTEP)
# -------------------------------------------------------------------------------------------------
def extract_pointwise_features(stream_X, stream_y):
    preds = nominal_reg.predict(scaler.transform(stream_X))
    residuals = np.abs(stream_y - preds)

    feats = {
        "residual": [],
        "local_mad": [],
        "dispersion_ratio": [],
        "precursor_energy": [],
        "drift_evidence": [],
        "slope": []
    }

    history = deque(maxlen=40)
    p_energy = 0.0
    d_evidence = 0.0

    for t in range(len(residuals)):
        r = residuals[t]
        history.append(r)
        h_list = list(history)

        # Dispersion
        w8 = h_list[-8:] if len(h_list) >= 8 else h_list
        med_w8 = np.median(w8)
        loc_mad = float(1.4826 * np.median(np.abs(w8 - med_w8)) + EPS)
        disp = loc_mad / base_mad

        # Energy accumulator
        if disp > 1.20:
            p_energy = 0.88 * p_energy + min(1.5, disp - 1.0)
        else:
            p_energy = max(0.0, 0.72 * p_energy - 0.20)

        # Drift evidence
        if r > np.quantile(cal_res, 0.90):
            d_evidence = 0.92 * d_evidence + min(2.5, (r - np.quantile(cal_res, 0.90)) / base_mad)
        else:
            d_evidence = max(0.0, 0.70 * d_evidence - 0.3)

        # Slope
        if len(h_list) >= 5:
            x_ax = np.arange(5).reshape(-1, 1)
            reg = LinearRegression().fit(x_ax, h_list[-5:])
            slope = float(reg.coef_[0])
        else:
            slope = 0.0

        feats["residual"].append(r)
        feats["local_mad"].append(loc_mad)
        feats["dispersion_ratio"].append(disp)
        feats["precursor_energy"].append(p_energy)
        feats["drift_evidence"].append(d_evidence)
        feats["slope"].append(slope)

    return {k: np.array(v) for k, v in feats.items()}, residuals

# -------------------------------------------------------------------------------------------------
# 4. HORIZON PREDICTION AUDIT (HORIZON H = 10 STEPS AHEAD)
# -------------------------------------------------------------------------------------------------
# Evaluate at precursor steps t in [20, 30] predicting residual at t + H (which lands in [30, 40])
HORIZON = 10
EVAL_STEPS = list(range(20, 31))

def collect_evaluation_pairs(extracted_data):
    X_dict = {k: [] for k in ["residual", "local_mad", "dispersion_ratio", "precursor_energy", "drift_evidence", "slope"]}
    y_target = []

    for stream_X, stream_y in extracted_data:
        feats, resids = extract_pointwise_features(stream_X, stream_y)
        for t in EVAL_STEPS:
            if t + HORIZON < len(resids):
                for k in X_dict.keys():
                    X_dict[k].append(feats[k][t])
                # Target is genuine non-constant pointwise error at t+H
                y_target.append(resids[t + HORIZON])

    return {k: np.array(v) for k, v in X_dict.items()}, np.array(y_target)

train_feats, train_targets = collect_evaluation_pairs(train_extracted)
test_feats, test_targets   = collect_evaluation_pairs(test_extracted)

print("\n📊 DATASET VARIANCE SANITY CHECK:")
print(f"  Target Samples Count : {len(test_targets)}")
print(f"  Target Variance      : {np.var(test_targets):.5f} (Must be > 0)")
print(f"  Target Mean / Std    : {np.mean(test_targets):.4f} / {np.std(test_targets):.4f}")

# -------------------------------------------------------------------------------------------------
# 5. FEATURE ABLATION & SCIENTIFIC EVALUATION
# -------------------------------------------------------------------------------------------------
feature_results = {}

for f_name in train_feats.keys():
    x_tr = train_feats[f_name].reshape(-1, 1)
    x_te = test_feats[f_name].reshape(-1, 1)

    # Fit simple univariate Ridge
    rg = RidgeCV()
    rg.fit(x_tr, train_targets)
    preds = rg.predict(x_te)

    rho, p_val = spearmanr(preds, test_targets)
    feature_results[f_name] = {
        "rho": float(rho) if not np.isnan(rho) else 0.0,
        "p_val": float(p_val) if not np.isnan(p_val) else 1.0,
        "raw_x_rho": float(spearmanr(test_feats[f_name], test_targets)[0])
    }

# Detrended Control: Measure correlation with time step 't' itself
t_indices = np.array([t for _ in range(len(test_extracted)) for t in EVAL_STEPS])
rho_time_confound, _ = spearmanr(t_indices, test_targets)

# -------------------------------------------------------------------------------------------------
# 6. REPORT
# -------------------------------------------------------------------------------------------------
print("\n" + "=" * 115)
print(f"🏁 MILESTONE 20.5.11 AUDIT RESULTS: PREDICTIVE HORIZON DISSECTION (H = {HORIZON} Steps)")
print("=" * 115)
print(f"Time Index Confound (t vs Target) : Spearman Rho = {rho_time_confound:.4f}")
print("-" * 115)
print(f"{'Feature Name':<25} | {'Out-of-Sample Rho':<20} | {'p-value':<15} | {'Direct Signal'}")
print("-" * 85)
for f_name, res in feature_results.items():
    print(f"{f_name:<25} | {res['rho']:<20.4f} | {res['p_val']:<15.4e} | {res['raw_x_rho']:<10.4f}")

# -------------------------------------------------------------------------------------------------
# 7. TELEMETRY EXPORT
# -------------------------------------------------------------------------------------------------
telemetry = {
    "milestone": "M20.5.11",
    "horizon_steps": HORIZON,
    "target_variance": float(np.var(test_targets)),
    "time_confound_rho": float(rho_time_confound),
    "feature_ablation": feature_results
}

with open("m20_5_11_dissection_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry written to 'm20_5_11_dissection_results.json'")
print("=" * 115)

🔬 MILESTONE 20.5.11: PREDICTIVE FEATURE DISSECTION AUDIT (CUDA)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🔬 Extracting activations for training and testing streams...

📊 DATASET VARIANCE SANITY CHECK:
  Target Samples Count : 110
  Target Variance      : 0.04282 (Must be > 0)
  Target Mean / Std    : 0.2594 / 0.2069

🏁 MILESTONE 20.5.11 AUDIT RESULTS: PREDICTIVE HORIZON DISSECTION (H = 10 Steps)
Time Index Confound (t vs Target) : Spearman Rho = -0.0358
-------------------------------------------------------------------------------------------------------------------
Feature Name              | Out-of-Sample Rho    | p-value         | Direct Signal
-------------------------------------------------------------------------------------
residual                  | -0.0242              | 8.0196e-01      | -0.0242   
local_mad                 | 0.0498               | 6.0562e-01      | -0.0498   
dispersion_ratio          | 0.0498               | 6.0562e-01      | -0.0498   
precursor_energy          | 0.1135               | 2.3756e

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.5.12 — ENVIRONMENTAL PREDICTABILITY & ORACLE IDENTIFIABILITY AUDIT
#
# Target: Determining the mathematical upper bound of predictability across 4 oracle tiers
#         (Null, Current Observables, Full History, Ground-Truth Latents) across horizons H in {1, 3, 6, 10}.
# Model : Qwen/Qwen2.5-0.5B via HookedTransformer (blocks.6.hook_resid_post)
# =================================================================================================

import json
import warnings
from collections import deque

import numpy as np
import torch
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
from transformer_lens import HookedTransformer

SEED = 205932
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 115)
print(f"🔬 MILESTONE 20.5.12: ENVIRONMENTAL PREDICTABILITY & ORACLE AUDIT ({DEVICE.upper()})")
print("=" * 115)

# -------------------------------------------------------------------------------------------------
# 1. STREAM REALIZATIONS WITH GROUND-TRUTH TELEMETRY
# -------------------------------------------------------------------------------------------------
def generate_instrumented_stream_batch(n_streams=20, n_steps=90, drift_start=35):
    all_streams = []
    for s_idx in range(n_streams):
        stream = []
        rng = np.random.RandomState(SEED + s_idx * 31)
        for t in range(n_steps):
            a = rng.uniform(-1.0, 1.0)
            while abs(a) < 0.10: a = rng.uniform(-1.0, 1.0)
            b = rng.uniform(-1.0, 1.0)

            precursor = bool(20 <= t < drift_start)
            drift = bool(t >= drift_start)

            if precursor:
                step_offset = t - 20
                sigma_a = 0.20 + 0.015 * step_offset
                sigma_b = 0.16 + 0.015 * step_offset
                eta_a = rng.normal(0.0, sigma_a)
                eta_b = rng.normal(0.0, sigma_b)
                mechanism = float((0.8 + eta_a) * a - (0.55 + eta_b) * b)
                alpha = 0.0
            elif drift:
                alpha = float(min(1.0, 0.10 + 0.90 * ((t - drift_start) / 30.0)))
                mechanism = float((1.0 - 2.0 * alpha) * (0.8 * a) - 0.55 * b)
                eta_a, eta_b, sigma_a, sigma_b = 0.0, 0.0, 0.0, 0.0
            else:
                alpha = 0.0
                mechanism = float(0.8 * a - 0.55 * b)
                eta_a, eta_b, sigma_a, sigma_b = 0.0, 0.0, 0.0, 0.0

            stream.append({
                "t": t,
                "a": float(a),
                "b": float(b),
                "mechanism": mechanism,
                # Ground-truth generator latents for Oracle O3
                "eta_a": float(eta_a),
                "eta_b": float(eta_b),
                "sigma_instability": float(sigma_a + sigma_b),
                "alpha_true": float(alpha),
                "time_step": float(t)
            })
        all_streams.append(stream)
    return all_streams

train_stream_meta = generate_instrumented_stream_batch(n_streams=10, drift_start=35)
test_stream_meta  = generate_instrumented_stream_batch(n_streams=10, drift_start=35)
cal_stream_meta   = generate_instrumented_stream_batch(n_streams=2, n_steps=80, drift_start=999)[0]

# -------------------------------------------------------------------------------------------------
# 2. ACTIVATION EXTRACTION & NOMINAL SELF-MODEL
# -------------------------------------------------------------------------------------------------
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break

def extract_stream_activations(streams):
    extracted = []
    for stream in streams:
        prompts = [
            f"You are solving a numerical reasoning task.\nInput A = {x['a']:.5f}\nInput B = {x['b']:.5f}\nInfer the relationship between the two quantities.\nProvide the predicted resulting value."
            for x in stream
        ]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
            act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        y = np.array([x["mechanism"] for x in stream])
        extracted.append((act, y, stream))
    return extracted

print("🔬 Extracting activations from internal representation...")
cal_X, cal_y, _ = extract_stream_activations([cal_stream_meta])[0]
train_extracted = extract_stream_activations(train_stream_meta)
test_extracted  = extract_stream_activations(test_stream_meta)

scaler_nominal = StandardScaler()
nominal_model = RidgeCV(alphas=np.logspace(-3, 4, 20))
nominal_model.fit(scaler_nominal.fit_transform(cal_X), cal_y)

# -------------------------------------------------------------------------------------------------
# 3. MULTI-TIER FEATURE COMPILATION
# -------------------------------------------------------------------------------------------------
def compile_stream_features(extracted_batch, window_k=5):
    dataset_records = []

    for act, y, stream_meta in extracted_batch:
        preds = nominal_model.predict(scaler_nominal.transform(act))
        residuals = np.abs(y - preds)

        # PCA for activation representation summary
        act_summary = act[:, :8] # First 8 dominant dimensions

        for t in range(window_k, len(residuals)):
            # 1. Tier O1: Current observables at t
            res_w = residuals[t-window_k : t+1]
            o1_feats = [
                stream_meta[t]["a"],
                stream_meta[t]["b"],
                residuals[t],
                float(np.var(res_w)),
                float(res_w[-1] - res_w[0])
            ]

            # 2. Tier O2: Full temporal history [t-k .. t]
            o2_feats = list(residuals[t-window_k : t+1]) + list(act_summary[t])

            # 3. Tier O3: Generator latent parameters
            o3_feats = [
                stream_meta[t]["eta_a"],
                stream_meta[t]["eta_b"],
                stream_meta[t]["sigma_instability"],
                stream_meta[t]["alpha_true"],
                stream_meta[t]["time_step"]
            ]

            dataset_records.append({
                "stream_t": t,
                "residual_t": residuals[t],
                "all_residuals": residuals,
                "o1": np.array(o1_feats),
                "o2": np.array(o2_feats),
                "o3": np.array(o3_feats)
            })

    return dataset_records

train_records = compile_stream_features(train_extracted)
test_records  = compile_stream_features(test_extracted)

# -------------------------------------------------------------------------------------------------
# 4. ORACLE IDENTIFIABILITY EVALUATION ACROSS MULTIPLE HORIZONS
# -------------------------------------------------------------------------------------------------
HORIZONS = [1, 3, 6, 10]
oracle_results = {}

for H in HORIZONS:
    # Build paired datasets for horizon H (evaluated in precursor phase t in [20, 34])
    def build_xy(records):
        X_o1, X_o2, X_o3, y = [], [], [], []
        for r in records:
            t = r["stream_t"]
            if 20 <= t <= 34 and (t + H) < len(r["all_residuals"]):
                X_o1.append(r["o1"])
                X_o2.append(r["o2"])
                X_o3.append(r["o3"])
                # Target: Actual future residual at t + H
                y.append(r["all_residuals"][t + H])
        return np.array(X_o1), np.array(X_o2), np.array(X_o3), np.array(y)

    tr_o1, tr_o2, tr_o3, tr_y = build_xy(train_records)
    te_o1, te_o2, te_o3, te_y = build_xy(test_records)

    h_res = {}

    # Tier 0: Null Limit (P95 of Permutations)
    null_rhos = []
    for _ in range(300):
        shuff_y = np.random.permutation(te_y)
        null_rhos.append(spearmanr(te_y, shuff_y)[0])
    h_res["O0_null_p95"] = float(np.quantile([r for r in null_rhos if not np.isnan(r)], 0.95))

    # Helper to train and evaluate Oracle models
    def evaluate_oracle(X_train, X_test, y_train, y_test, use_nonlinear=False):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)

        if use_nonlinear:
            model = GradientBoostingRegressor(n_estimators=40, max_depth=3, random_state=SEED)
        else:
            model = RidgeCV(alphas=np.logspace(-2, 3, 10))

        model.fit(X_tr, y_train)
        preds = model.predict(X_te)

        rho, p = spearmanr(preds, y_test)
        r2 = r2_score(y_test, preds)
        return {
            "rho": float(rho) if not np.isnan(rho) else 0.0,
            "p_val": float(p) if not np.isnan(p) else 1.0,
            "r2": float(r2)
        }

    # Tier O1: Current Observables Oracle
    h_res["O1_current"] = evaluate_oracle(tr_o1, te_o1, tr_y, te_y)

    # Tier O2: Full Temporal History Oracle (Nonlinear Gradient Boosting)
    h_res["O2_full_history"] = evaluate_oracle(tr_o2, te_o2, tr_y, te_y, use_nonlinear=True)

    # Tier O3: Ground-Truth Generator Latents Oracle (Nonlinear Upper Bound)
    h_res["O3_generator_latent"] = evaluate_oracle(tr_o3, te_o3, tr_y, te_y, use_nonlinear=True)

    oracle_results[f"H_{H}"] = h_res

# -------------------------------------------------------------------------------------------------
# 5. SCIENTIFIC AUDIT REPORT
# -------------------------------------------------------------------------------------------------
print("\n" + "=" * 115)
print("🏁 MILESTONE 20.5.12 AUDIT RESULTS: ENVIRONMENTAL PREDICTABILITY & ORACLE TIERS")
print("=" * 115)
print(f"{'Horizon':<10} | {'O0 Null (P95)':<15} | {'O1 Current (Rho)':<18} | {'O2 History (Rho)':<18} | {'O3 Latent (Rho)':<18} | {'O3 R²'}")
print("-" * 95)
for h_key, metrics in oracle_results.items():
    print(f"{h_key:<10} | {metrics['O0_null_p95']:<15.4f} | {metrics['O1_current']['rho']:<18.4f} | {metrics['O2_full_history']['rho']:<18.4f} | {metrics['O3_generator_latent']['rho']:<18.4f} | {metrics['O3_generator_latent']['r2']:.4f}")

# Synthesizing Verdict for H=6 & H=10
o3_valid = any(oracle_results[h]["O3_generator_latent"]["rho"] > oracle_results[h]["O0_null_p95"] + 0.15 for h in ["H_6", "H_10"])
o2_valid = any(oracle_results[h]["O2_full_history"]["rho"] > oracle_results[h]["O0_null_p95"] + 0.15 for h in ["H_6", "H_10"])

print("\n" + "=" * 115)
print("📊 CAUSAL BOTTLENECK DIAGNOSIS:")
if not o3_valid:
    verdict = "ENVIRONMENTAL_UNPREDICTABILITY"
    diagnosis = "Even the Ground-Truth Latents Oracle (O3) with access to eta_a, eta_b, and sigma cannot predict future error. The synthetic generator is fundamentally i.i.d. noise; no future signal exists."
elif o3_valid and not o2_valid:
    verdict = "REPRESENTATION_EXTRACTION_BOTTLENECK"
    diagnosis = "The environment contains causal predictive information (O3 succeeds), but this information is not recoverable from empirical activations or past residuals (O2 fails)."
else:
    verdict = "TEMPORAL_LEARNING_OPPORTUNITY"
    diagnosis = "Both environment latents (O3) and empirical history (O2) predict future degradation. A temporal forecasting layer is mathematically viable."

print(f"Status Verdict : 🏷️ {verdict}")
print(f"Interpretation : {diagnosis}")

telemetry = {
    "milestone": "M20.5.12",
    "verdict": verdict,
    "diagnosis": diagnosis,
    "horizons": oracle_results
}

with open("m20_5_12_oracle_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry written to 'm20_5_12_oracle_results.json'")
print("=" * 115)

🔬 MILESTONE 20.5.12: ENVIRONMENTAL PREDICTABILITY & ORACLE AUDIT (CUDA)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.5.13 — ONLINE TEMPORAL SELF-MODEL LEARNING & CAUSAL FORECASTING
#
# Target: Validating whether an endogenous temporal layer can learn future degradation
#         online, in a strictly causal setting, across horizons H in {1, 3, 6, 10}.
# Model : Qwen/Qwen2.5-0.5B via HookedTransformer (blocks.6.hook_resid_post)
# =================================================================================================

import json
import warnings
from collections import deque

import numpy as np
import torch
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import RidgeCV, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
from transformer_lens import HookedTransformer

SEED = 205933
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 115)
print(f"🔬 MILESTONE 20.5.13: ONLINE TEMPORAL FORECASTING BENCHMARK ({DEVICE.upper()})")
print("=" * 115)

# -------------------------------------------------------------------------------------------------
# 1. INDEPENDENT STREAM GENERATION
# -------------------------------------------------------------------------------------------------
def generate_stream_batch(n_streams=20, n_steps=90, drift_start=35):
    all_streams = []
    for s_idx in range(n_streams):
        stream = []
        rng = np.random.RandomState(SEED + s_idx * 43)
        for t in range(n_steps):
            a = rng.uniform(-1.0, 1.0)
            while abs(a) < 0.10: a = rng.uniform(-1.0, 1.0)
            b = rng.uniform(-1.0, 1.0)

            precursor = bool(20 <= t < drift_start)
            drift = bool(t >= drift_start)

            if precursor:
                step_offset = t - 20
                sigma_a = 0.20 + 0.015 * step_offset
                sigma_b = 0.16 + 0.015 * step_offset
                eta_a = rng.normal(0.0, sigma_a)
                eta_b = rng.normal(0.0, sigma_b)
                mechanism = float((0.8 + eta_a) * a - (0.55 + eta_b) * b)
            elif drift:
                alpha = float(min(1.0, 0.10 + 0.90 * ((t - drift_start) / 30.0)))
                mechanism = float((1.0 - 2.0 * alpha) * (0.8 * a) - 0.55 * b)
            else:
                mechanism = float(0.8 * a - 0.55 * b)

            stream.append({"t": t, "a": float(a), "b": float(b), "mechanism": mechanism})
        all_streams.append(stream)
    return all_streams

train_stream_meta = generate_stream_batch(n_streams=10, drift_start=35)
test_stream_meta  = generate_stream_batch(n_streams=10, drift_start=35)
cal_stream_meta   = generate_stream_batch(n_streams=2, n_steps=80, drift_start=999)[0]

# -------------------------------------------------------------------------------------------------
# 2. ACTIVATION EXTRACTION & NOMINAL SELF-MODEL
# -------------------------------------------------------------------------------------------------
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break

def extract_stream_activations(streams):
    extracted = []
    for stream in streams:
        prompts = [
            f"You are solving a numerical reasoning task.\nInput A = {x['a']:.5f}\nInput B = {x['b']:.5f}\nInfer the relationship between the two quantities.\nProvide the predicted resulting value."
            for x in stream
        ]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
            act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        y = np.array([x["mechanism"] for x in stream])
        extracted.append((act, y))
    return extracted

print("🔬 Extracting activations from internal representations...")
cal_X, cal_y = extract_stream_activations([cal_stream_meta])[0]
train_extracted = extract_stream_activations(train_stream_meta)
test_extracted  = extract_stream_activations(test_stream_meta)

scaler_nominal = StandardScaler()
nominal_model = RidgeCV(alphas=np.logspace(-3, 4, 20))
nominal_model.fit(scaler_nominal.fit_transform(cal_X), cal_y)

# Fit PCA on calibration representations for temporal encoding (Latent Dim = 4)
pca_dim = 4
pca_encoder = PCA(n_components=pca_dim, random_state=SEED)
pca_encoder.fit(cal_X)

# -------------------------------------------------------------------------------------------------
# 3. ENDOGENOUS TEMPORAL FEATURE PIPELINE
# -------------------------------------------------------------------------------------------------
def extract_temporal_trajectories(extracted_batch, window_k=5):
    processed_streams = []
    for act, y in extracted_batch:
        preds = nominal_model.predict(scaler_nominal.transform(act))
        residuals = np.abs(y - preds)
        z_act = pca_encoder.transform(act)

        feature_sequence = []
        for t in range(len(residuals)):
            if t < window_k:
                # Pad early steps
                pad_res = [residuals[0]] * (window_k - t) + list(residuals[:t+1])
                pad_act = [z_act[0]] * (window_k - t) + list(z_act[:t+1])
            else:
                pad_res = list(residuals[t - window_k : t + 1])
                pad_act = list(z_act[t - window_k : t + 1])

            res_history = np.array(pad_res) # Shape: (K+1,)
            act_history = np.array(pad_act).flatten() # Shape: ((K+1)*pca_dim,)

            # Combine temporal features
            temporal_vec = np.hstack([res_history, act_history])
            res_only_vec = res_history

            feature_sequence.append({
                "t": t,
                "residual": residuals[t],
                "temporal_full": temporal_vec,
                "temporal_res_only": res_only_vec
            })
        processed_streams.append({"features": feature_sequence, "residuals": residuals})
    return processed_streams

train_processed = extract_temporal_trajectories(train_extracted, window_k=5)
test_processed  = extract_temporal_trajectories(test_extracted, window_k=5)

# -------------------------------------------------------------------------------------------------
# 4. STRICT CAUSAL TRAINING & EVALUATION (HORIZONS H in {1, 3, 6, 10})
# -------------------------------------------------------------------------------------------------
HORIZONS = [1, 3, 6, 10]
benchmark_results = {}

for H in HORIZONS:
    # Compile training pairs (strictly t -> t+H within each stream)
    X_tr_full, X_tr_res, y_tr = [], [], []
    for stream in train_processed:
        feats = stream["features"]
        resids = stream["residuals"]
        for t in range(20, 35): # Precursor window
            if t + H < len(resids):
                X_tr_full.append(feats[t]["temporal_full"])
                X_tr_res.append(feats[t]["temporal_res_only"])
                y_tr.append(resids[t + H])

    X_tr_full, X_tr_res, y_tr = np.array(X_tr_full), np.array(X_tr_res), np.array(y_tr)

    # Fit Online/Adaptive Temporal Forecasters
    scaler_full = StandardScaler()
    X_tr_full_s = scaler_full.fit_transform(X_tr_full)
    forecaster_full = RidgeCV(alphas=np.logspace(-2, 3, 10))
    forecaster_full.fit(X_tr_full_s, y_tr)

    scaler_res = StandardScaler()
    X_tr_res_s = scaler_res.fit_transform(X_tr_res)
    forecaster_res = RidgeCV(alphas=np.logspace(-2, 3, 10))
    forecaster_res.fit(X_tr_res_s, y_tr)

    # Fit Offline Oracle (O2 Gradient Boosting)
    oracle_o2 = GradientBoostingRegressor(n_estimators=40, max_depth=3, random_state=SEED)
    oracle_o2.fit(X_tr_full_s, y_tr)

    # Compile test pairs
    X_te_full, X_te_res, y_te, te_persistence = [], [], [], []
    X_te_shuffled = [] # For time-shuffle ablation

    for stream in test_processed:
        feats = stream["features"]
        resids = stream["residuals"]
        for t in range(20, 35):
            if t + H < len(resids):
                X_te_full.append(feats[t]["temporal_full"])
                X_te_res.append(feats[t]["temporal_res_only"])
                y_te.append(resids[t + H])
                te_persistence.append(feats[t]["residual"])

                # Create time-shuffled counterpart of temporal vector
                shuff_vec = feats[t]["temporal_full"].copy()
                # Permute the history slots (first 6 values are residuals)
                shuff_vec[:6] = np.random.permutation(shuff_vec[:6])
                X_te_shuffled.append(shuff_vec)

    X_te_full_s = scaler_full.transform(np.array(X_te_full))
    X_te_res_s  = scaler_res.transform(np.array(X_te_res))
    X_te_shuff_s = scaler_full.transform(np.array(X_te_shuffled))
    y_te = np.array(y_te)
    te_persistence = np.array(te_persistence)

    # Inferences
    pred_full = forecaster_full.predict(X_te_full_s)
    pred_res  = forecaster_res.predict(X_te_res_s)
    pred_shuff = forecaster_full.predict(X_te_shuff_s)
    pred_o2   = oracle_o2.predict(X_te_full_s)

    # Metrics computation
    rho_full, p_full = spearmanr(pred_full, y_te)
    rho_res, _       = spearmanr(pred_res, y_te)
    rho_shuff, _     = spearmanr(pred_shuff, y_te)
    rho_pers, _      = spearmanr(te_persistence, y_te)
    rho_o2, _        = spearmanr(pred_o2, y_te)

    # Null Limit via Monte Carlo
    null_rhos = [spearmanr(y_te, np.random.permutation(y_te))[0] for _ in range(300)]
    null_p95 = float(np.quantile([r for r in null_rhos if not np.isnan(r)], 0.95))

    benchmark_results[f"H_{H}"] = {
        "null_p95": null_p95,
        "persistence_rho": float(rho_pers),
        "ar_only_rho": float(rho_res),
        "temporal_learned_rho": float(rho_full),
        "temporal_p_val": float(p_full),
        "shuffled_ablation_rho": float(rho_shuff),
        "oracle_o2_rho": float(rho_o2),
        "r2_score": float(r2_score(y_te, pred_full))
    }

# -------------------------------------------------------------------------------------------------
# 5. AUDIT REPORT & DECISION MATRIX
# -------------------------------------------------------------------------------------------------
print("\n" + "=" * 115)
print("🏁 MILESTONE 20.5.13 AUDIT RESULTS: ENDOGENOUS TEMPORAL FORECASTING")
print("=" * 115)
print(f"{'Horizon':<8} | {'Null P95':<10} | {'Persist':<10} | {'AR-Only':<10} | {'Learned (Rho)':<15} | {'Shuffled':<10} | {'O2 Oracle':<10} | {'R²'}")
print("-" * 95)
for h_key, r in benchmark_results.items():
    print(f"{h_key:<8} | {r['null_p95']:<10.4f} | {r['persistence_rho']:<10.4f} | {r['ar_only_rho']:<10.4f} | {r['temporal_learned_rho']:<15.4f} | {r['shuffled_ablation_rho']:<10.4f} | {r['oracle_o2_rho']:<10.4f} | {r['r2_score']:.4f}")

# Rigorous Success Checks
crit_beats_null = all(benchmark_results[h]["temporal_learned_rho"] > benchmark_results[h]["null_p95"] for h in benchmark_results)
crit_beats_persist = all(benchmark_results[h]["temporal_learned_rho"] > benchmark_results[h]["persistence_rho"] for h in benchmark_results)
crit_causal_temporal = all(benchmark_results[h]["temporal_learned_rho"] > benchmark_results[h]["shuffled_ablation_rho"] for h in benchmark_results)

print("\n" + "=" * 115)
print("📊 CAUSAL INTEGRITY & LEARNING DIAGNOSTICS:")
print(f"  1. Beats Null Limit (P95) Across All Horizons        : {'✅ YES' if crit_beats_null else '❌ NO'}")
print(f"  2. Beats Naive Persistence Across All Horizons       : {'✅ YES' if crit_beats_persist else '❌ NO'}")
print(f"  3. Temporal Collapse on Shuffled Input (Order-Aware) : {'✅ YES' if crit_causal_temporal else '❌ NO'}")

if crit_beats_null and crit_beats_persist and crit_causal_temporal:
    verdict = "AUTONOMOUS_FORECASTING_VALIDATED"
    diagnosis = "The endogenous temporal layer successfully learns forward degradation causality out-of-sample. Performance collapses under temporal shuffling, proving dependency on causal time order."
else:
    verdict = "PARTIAL_TEMPORAL_LEARNING"
    diagnosis = "The model captures marginal statistical correlations, but fails strict causal ordering or long-horizon outperformance."

print(f"\nFinal Verdict  : 🏷️ {verdict}")
print(f"Interpretation : {diagnosis}")

telemetry = {
    "milestone": "M20.5.13",
    "verdict": verdict,
    "diagnosis": diagnosis,
    "horizons": benchmark_results
}

with open("m20_5_13_forecasting_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry written to 'm20_5_13_forecasting_results.json'")
print("=" * 115)

🔬 MILESTONE 20.5.13: ONLINE TEMPORAL FORECASTING BENCHMARK (CUDA)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🔬 Extracting activations from internal representations...

🏁 MILESTONE 20.5.13 AUDIT RESULTS: ENDOGENOUS TEMPORAL FORECASTING
Horizon  | Null P95   | Persist    | AR-Only    | Learned (Rho)   | Shuffled   | O2 Oracle  | R²
-----------------------------------------------------------------------------------------------
H_1      | 0.1426     | -0.0149    | 0.1022     | 0.3453          | 0.3373     | 0.8955     | 0.0330
H_3      | 0.1381     | -0.0273    | 0.1717     | 0.2879          | 0.2646     | 0.9247     | 0.0332
H_6      | 0.1444     | 0.0038     | 0.0802     | 0.4104          | 0.3757     | 0.9270     | 0.0487
H_10     | 0.1372     | 0.0002     | 0.0711     | 0.3050          | 0.2781     | 0.9152     | 0.0335

📊 CAUSAL INTEGRITY & LEARNING DIAGNOSTICS:
  1. Beats Null Limit (P95) Across All Horizons        : ✅ YES
  2. Beats Naive Persistence Across All Horizons       : ✅ YES
  3. Temporal Collapse on Shuffled Input (

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.6 — DUAL-COLUMN ARCHITECTURE & CROSS-REGIME EVALUATION
# Model : Qwen/Qwen2.5-0.5B via HookedTransformer (blocks.6.hook_resid_post)
# =================================================================================================

import json
import math
import random
import hashlib
import warnings
from collections import deque

import numpy as np
import torch
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. SETUP & REPRODUCIBILITY
# -------------------------------------------------------------------------------------------------
SEED = 206001
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 115)
print(f"🔬 MILESTONE 20.6: DUAL-COLUMN EPISTEMIC ARCHITECTURE ({DEVICE.upper()})")
print(f"Seed: {SEED} | Model: {MODEL_NAME} | Hook: blocks.{PREFERRED_LAYER}.hook_resid_post")
print("=" * 115)

# -------------------------------------------------------------------------------------------------
# 2. CROSS-REGIME BENCHMARK GENERATION
# -------------------------------------------------------------------------------------------------
def generate_sample(split, idx, a=None, b=None, alpha=0.0, context_type="STANDARD", t_step=None,
                    non_linear=False, precursor=False):
    if a is None:
        a = np.random.uniform(-1.0, 1.0)
        while abs(a) < 0.10: a = np.random.uniform(-1.0, 1.0)
    if b is None:
        b = np.random.uniform(-1.0, 1.0)

    if non_linear:
        mechanism = float(np.sin(3.0 * a) + (b ** 2) + 1.5 * a * b)
    elif precursor:
        step_offset = max(0, t_step - 20) if t_step is not None else 0
        eta_a = np.random.normal(0.0, 0.20 + 0.015 * min(15, step_offset))
        eta_b = np.random.normal(0.0, 0.16 + 0.015 * min(15, step_offset))
        mechanism = float((0.8 + eta_a) * a - (0.55 + eta_b) * b)
    else:
        mechanism = float((1.0 - 2.0 * alpha) * (0.8 * a) - 0.55 * b)

    raw_id = f"{split}_{idx}_{t_step}_{a:.6f}_{b:.6f}_{alpha:.4f}_{context_type}_{non_linear}_{precursor}"
    fingerprint = hashlib.sha256(raw_id.encode()).hexdigest()

    return {
        "split": split, "idx": idx, "t_step": t_step, "a": float(a), "b": float(b),
        "alpha": float(alpha), "context_type": context_type, "non_linear": non_linear,
        "precursor": precursor, "mechanism": mechanism, "fingerprint": fingerprint
    }

# Baseline partitions
N_TRAIN, N_CALIBRATION = 220, 160
train_meta = [generate_sample("TRAIN", i, alpha=0.0, context_type="STANDARD") for i in range(N_TRAIN)]
cal_meta   = [generate_sample("CALIBRATION", i, alpha=0.0, context_type="STANDARD") for i in range(N_CALIBRATION)]

# Cross-Regime 1: Gradual Ramp (Canonical Precursor + Smooth Ramp) - T=100
stream_R1_meta = []
for t in range(100):
    if t < 20: s = generate_sample("R1_GRADUAL", t, alpha=0.0, t_step=t)
    elif t < 35: s = generate_sample("R1_GRADUAL", t, alpha=0.0, t_step=t, precursor=True)
    else:
        a_t = float(min(1.0, 0.05 + 0.95 * ((t - 35) / 50.0)))
        s = generate_sample("R1_GRADUAL", t, alpha=a_t, t_step=t)
    stream_R1_meta.append(s)

# Cross-Regime 2: Abrupt Shock Step (Zero precursor, instant drop) - T=100
stream_R2_meta = []
for t in range(100):
    a_t = 0.0 if t < 40 else 0.85
    stream_R2_meta.append(generate_sample("R2_ABRUPT", t, alpha=a_t, t_step=t))

# Cross-Regime 3: Oscillatory Drift (Rhythmic degradation and partial recovery) - T=100
stream_R3_meta = []
for t in range(100):
    if t < 25: a_t = 0.0
    else: a_t = float(0.45 * (1.0 + math.sin(0.25 * (t - 25))))
    stream_R3_meta.append(generate_sample("R3_OSCILLATORY", t, alpha=a_t, t_step=t))

# Cross-Regime 4: Compound Novelty + Gradual Drift - T=100
stream_R4_meta = []
for t in range(100):
    ctx = "STANDARD" if t < 30 else "NOVEL_FORMAT"
    if t < 20: s = generate_sample("R4_COMPOUND", t, alpha=0.0, context_type=ctx, t_step=t)
    elif t < 35: s = generate_sample("R4_COMPOUND", t, alpha=0.0, context_type=ctx, t_step=t, precursor=True)
    else:
        a_t = float(min(1.0, 0.10 + 0.90 * ((t - 35) / 50.0)))
        s = generate_sample("R4_COMPOUND", t, alpha=a_t, context_type=ctx, t_step=t)
    stream_R4_meta.append(s)

# Training streams for Causal Forecaster (5 independent streams)
train_forecaster_streams = []
for s_idx in range(5):
    st = []
    for t in range(70):
        if t < 20: s = generate_sample(f"TR_FC_{s_idx}", t, alpha=0.0, t_step=t)
        elif t < 35: s = generate_sample(f"TR_FC_{s_idx}", t, alpha=0.0, t_step=t, precursor=True)
        else:
            a_t = float(min(1.0, 0.10 + 0.90 * ((t - 35) / 30.0)))
            s = generate_sample(f"TR_FC_{s_idx}", t, alpha=a_t, t_step=t)
        st.append(s)
    train_forecaster_streams.append(st)

print("🔐 Asserting partition independence and collision bounds...")
assert len(set(x["fingerprint"] for x in train_meta).intersection(set(x["fingerprint"] for x in cal_meta))) == 0
print("🔐 Partition Orthogonality: VERIFIED")

# -------------------------------------------------------------------------------------------------
# 3. HOOKED TRANSFORMER ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------
print(f"⏳ Loading {MODEL_NAME}...")
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break

def make_prompt(row):
    if row["context_type"] == "NOVEL_FORMAT":
        return (
            "<<SYSTEM EXPERIMENTAL QUERY PROTOCOL>>\n"
            f"Coordinate Alpha: {row['a']:.6f}\nCoordinate Beta:  {row['b']:.6f}\n"
            "Synthesize latent transformation outcome value."
        )
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\nInput B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\nProvide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=16):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda": torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from internal representation...")
train_X = extract_features(train_meta)
train_y = np.array([x["mechanism"] for x in train_meta])

cal_X = extract_features(cal_meta)
cal_y = np.array([x["mechanism"] for x in cal_meta])

R1_X = extract_features(stream_R1_meta); R1_y = np.array([x["mechanism"] for x in stream_R1_meta])
R2_X = extract_features(stream_R2_meta); R2_y = np.array([x["mechanism"] for x in stream_R2_meta])
R3_X = extract_features(stream_R3_meta); R3_y = np.array([x["mechanism"] for x in stream_R3_meta])
R4_X = extract_features(stream_R4_meta); R4_y = np.array([x["mechanism"] for x in stream_R4_meta])

# -------------------------------------------------------------------------------------------------
# 4. FROZEN SELF-MODEL & NOVELTY SENSOR
# -------------------------------------------------------------------------------------------------
class FrozenSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.reg = RidgeCV(alphas=np.logspace(-3, 4, 20))

    def fit(self, X, y):
        self.reg.fit(self.scaler.fit_transform(X), y)

    def predict(self, X):
        return self.reg.predict(self.scaler.transform(X))

self_model = FrozenSelfModel()
self_model.fit(train_X, train_y)

class GeometricNoveltySensor:
    def __init__(self, n_components=12):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()

    def fit(self, X):
        Z = self.pca.fit_transform(self.scaler.fit_transform(X))
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1))
        self.tau_novel = float(np.quantile(d_mah, 0.95))

    def score(self, X):
        Z = self.pca.transform(self.scaler.transform(X))
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1))
        return np.clip(1.0 / (1.0 + np.exp(-1.5 * (d_mah - self.tau_novel))), 0.01, 0.99)

novelty_sensor = GeometricNoveltySensor()
novelty_sensor.fit(train_X)

cal_residuals = np.abs(cal_y - self_model.predict(cal_X))
cal_med = float(np.median(cal_residuals))
cal_mad = float(np.median(np.abs(cal_residuals - cal_med)))
cal_mad_scale = float(1.4826 * cal_mad + EPS)

cal_profile = {
    "median": cal_med, "mad": cal_mad, "mad_scale": cal_mad_scale,
    "q75": float(np.quantile(cal_residuals, 0.75)),
    "q90": float(np.quantile(cal_residuals, 0.90)),
    "q99": float(np.quantile(cal_residuals, 0.99))
}

pca_temporal = PCA(n_components=4, random_state=SEED)
pca_temporal.fit(cal_X)

# -------------------------------------------------------------------------------------------------
# 5. CAUSAL TEMPORAL FORECASTER TRAINING (OFFLINE FIT ON INDEPENDENT STREAMS)
# -------------------------------------------------------------------------------------------------
WINDOW_K = 5
HORIZON_H = 6

X_fc_train, y_fc_train = [], []
for st in train_forecaster_streams:
    st_X = extract_features(st)
    st_y = np.array([x["mechanism"] for x in st])
    resids = np.abs(st_y - self_model.predict(st_X))
    z_act = pca_temporal.transform(st_X)

    for t in range(20, 35):
        if t + HORIZON_H < len(resids):
            pad_res = list(resids[t - WINDOW_K : t + 1])
            pad_act = list(z_act[t - WINDOW_K : t + 1].flatten())
            X_fc_train.append(np.hstack([pad_res, pad_act]))
            y_fc_train.append(resids[t + HORIZON_H])

scaler_fc = StandardScaler()
X_fc_train_s = scaler_fc.fit_transform(np.array(X_fc_train))
temporal_regressor = RidgeCV(alphas=np.logspace(-2, 3, 10))
temporal_regressor.fit(X_fc_train_s, np.array(y_fc_train))
print("🔮 Causal Temporal Forecaster trained on independent stream manifolds.")

# -------------------------------------------------------------------------------------------------
# 6. UNIFIED DUAL-COLUMN EPISTEMIC AGENT
# -------------------------------------------------------------------------------------------------
class DualColumnEpistemicAgent:
    """
    Unifies Column 1 (M20.5.7 Governance State Machine) with Column 2 (M20.5.13 Causal Forecaster).
    """
    def __init__(self, cal_profile, cal_residuals):
        self.p = cal_profile
        self.cal_raw_residuals = np.sort(cal_residuals)
        self.reset()

    def reset(self):
        self.state = "NORMAL"
        self.drift_evidence = 0.0
        self.recovery_evidence = 0.0
        self.transient_evidence = 0.0
        self.recovery_clean_streak = 0
        self.consecutive_anomalies = 0
        self.history_residuals = deque(maxlen=40)
        self.history_z_act = deque(maxlen=40)
        self.history_reports = []

    def compute_window_slope(self, values):
        if len(values) < 3: return 0.0
        x = np.arange(len(values)).reshape(-1, 1)
        return float(LinearRegression().fit(x, values).coef_[0])

    def step(self, residual, novelty, act_vector):
        self.history_residuals.append(residual)
        z = pca_temporal.transform(act_vector.reshape(1, -1))[0]
        self.history_z_act.append(z)

        res_list = list(self.history_residuals)
        z_list = list(self.history_z_act)

        # Multi-scale Trends & Persistence Ratios
        short_slope = self.compute_window_slope(res_list[-5:]) if len(res_list) >= 5 else 0.0
        med_slope = self.compute_window_slope(res_list[-15:]) if len(res_list) >= 15 else 0.0
        combined_trend = 0.6 * short_slope + 0.4 * med_slope

        recent_w8 = res_list[-8:] if len(res_list) >= 8 else res_list
        k_exceed_q90 = sum(1 for r in recent_w8 if r > self.p["q90"])
        k_clean_q75  = sum(1 for r in recent_w8 if r <= self.p["q75"])
        persistence_drift_ratio = k_exceed_q90 / len(recent_w8)
        persistence_clean_ratio = k_clean_q75 / len(recent_w8)

        # -------------------------------------------------------------
        # COLUMN 1: GOVERNANCE STATE MACHINE (LOCKED FROM M20.5.7)
        # -------------------------------------------------------------
        if residual > self.p["q90"]:
            self.consecutive_anomalies += 1
            raw_step = (min(residual, 2.0 * self.p["q99"]) - self.p["q90"]) / self.p["mad_scale"]
            if self.consecutive_anomalies < 2 and persistence_drift_ratio < 0.30:
                self.transient_evidence = float(min(raw_step, 4.0))
                self.drift_evidence = max(0.0, self.drift_evidence * 0.85 - 0.2)
            else:
                self.transient_evidence = 0.0
                raw_step *= (1.2 if combined_trend > 0 else 0.8)
                self.drift_evidence = 0.92 * self.drift_evidence + min(3.0, raw_step)
        else:
            self.consecutive_anomalies = 0
            self.transient_evidence = 0.0
            self.drift_evidence = max(0.0, 0.70 * self.drift_evidence - 0.3)

        if self.state in ["ACTIVE_DRIFT", "SEVERE_DRIFT", "RECOVERY"]:
            if residual <= self.p["q90"] and combined_trend <= 0.005:
                clean_step = (self.p["q90"] - residual) / self.p["mad_scale"]
                self.recovery_evidence = 0.90 * self.recovery_evidence + max(0.5, clean_step)
            else:
                self.recovery_evidence = max(0.0, 0.70 * self.recovery_evidence - 0.4)
        else:
            self.recovery_evidence = 0.0

        prev = self.state

        if novelty > 0.65 and residual > (2.5 * self.p["q99"]):
            self.state = "UNKNOWN"; self.recovery_clean_streak = 0; conf = 0.95
        elif prev == "NORMAL":
            self.recovery_clean_streak = 0
            if self.drift_evidence >= 2.0 and persistence_drift_ratio >= 0.35 and self.consecutive_anomalies >= 2:
                self.state = "EARLY_DRIFT"; conf = 0.70
            else: conf = 0.95
        elif prev == "EARLY_DRIFT":
            self.recovery_clean_streak = 0
            if self.drift_evidence >= 4.5 and persistence_drift_ratio >= 0.55:
                self.state = "ACTIVE_DRIFT"; conf = 0.85
            elif self.drift_evidence < 1.0 and persistence_clean_ratio >= 0.65:
                self.state = "NORMAL"; conf = 0.90
            else: conf = 0.75
        elif prev == "ACTIVE_DRIFT":
            self.recovery_clean_streak = 0
            if self.drift_evidence >= 8.0 and persistence_drift_ratio >= 0.75:
                self.state = "SEVERE_DRIFT"; conf = 0.95
            elif (self.recovery_evidence >= 1.8 or persistence_clean_ratio >= 0.50) and combined_trend < 0.005:
                self.state = "RECOVERY"; conf = 0.80
            else: conf = 0.85
        elif prev == "SEVERE_DRIFT":
            self.recovery_clean_streak = 0
            if (self.recovery_evidence >= 2.0 or persistence_clean_ratio >= 0.50) and combined_trend < 0.005:
                self.state = "RECOVERY"; conf = 0.85
            elif self.drift_evidence < 5.0 and persistence_drift_ratio < 0.50:
                self.state = "ACTIVE_DRIFT"; conf = 0.80
            else: conf = 0.95
        elif prev == "RECOVERY":
            if residual <= self.p["q75"] and combined_trend <= 0.005:
                self.recovery_clean_streak += 1
            else:
                self.recovery_clean_streak = max(0, self.recovery_clean_streak - 1)

            if self.recovery_clean_streak >= 5 and self.drift_evidence < 1.5:
                self.state = "NORMAL"; self.recovery_clean_streak = 0; conf = 0.95
            elif self.drift_evidence >= 4.5 and persistence_drift_ratio >= 0.50:
                self.state = "ACTIVE_DRIFT"; self.recovery_clean_streak = 0; conf = 0.85
            else:
                self.state = "RECOVERY"; conf = 0.80
        elif prev == "UNKNOWN":
            self.recovery_clean_streak = 0
            if residual <= self.p["q75"] and persistence_clean_ratio >= 0.80:
                self.state = "NORMAL"; conf = 0.90
            else: conf = 0.95

        # -------------------------------------------------------------
        # COLUMN 2: CAUSAL TEMPORAL FORECASTER (FROM M20.5.13)
        # -------------------------------------------------------------
        if len(res_list) >= (WINDOW_K + 1):
            pad_res = list(res_list[-WINDOW_K-1:])
            pad_act = list(np.array(z_list[-WINDOW_K-1:]).flatten())
            x_fc = np.hstack([pad_res, pad_act]).reshape(1, -1)
            raw_pred_h = float(temporal_regressor.predict(scaler_fc.transform(x_fc))[0])
            # Hazard Score: Calibrated probability of exceeding Q90 at horizon H
            hazard_score = float(1.0 / (1.0 + np.exp(-4.0 * ((raw_pred_h - self.p["q90"]) / self.p["mad_scale"]))))
        else:
            raw_pred_h = float(residual)
            hazard_score = 0.05

        # -------------------------------------------------------------
        # SYNTHESIS: EPISTEMIC SELF-STATUS REPORT
        # -------------------------------------------------------------
        if self.state == "NORMAL":
            if hazard_score >= 0.65: epistemic_status = "ANTICIPATED_RISK"
            elif novelty > 0.60:     epistemic_status = "NOVEL_BUT_VALID"
            else:                    epistemic_status = "STABLE_NOMINAL"
        else:
            epistemic_status = self.state

        # Recommended Action
        if self.state in ["ACTIVE_DRIFT", "SEVERE_DRIFT", "UNKNOWN"]:
            recommended_action = "QUARANTINE_AND_INTERVENE"
        elif epistemic_status in ["ANTICIPATED_RISK", "EARLY_DRIFT"]:
            recommended_action = "PROCEED_WITH_REDUCED_CONFIDENCE"
        elif epistemic_status == "NOVEL_BUT_VALID":
            recommended_action = "PROCEED_WITH_ELEVATED_LOGGING"
        else:
            recommended_action = "PROCEED_FULL_CONFIDENCE"

        report = {
            "operational_state": self.state,
            "epistemic_status": epistemic_status,
            "confidence": float(conf),
            "hazard_score": float(hazard_score),
            "forecast_raw_residual": float(raw_pred_h),
            "forecast_horizon_steps": HORIZON_H,
            "uncertainty_attribution": {
                "novelty_score": float(novelty),
                "transient_shock": bool(self.transient_evidence > 0),
                "drift_evidence": float(self.drift_evidence),
                "recovery_evidence": float(self.recovery_evidence)
            },
            "governance_recommendation": recommended_action
        }
        self.history_reports.append(report)
        return report

# -------------------------------------------------------------------------------------------------
# 7. CROSS-REGIME EVALUATION SUITE
# -------------------------------------------------------------------------------------------------
agent = DualColumnEpistemicAgent(cal_profile, cal_residuals)

def evaluate_regime(stream_X, stream_y):
    agent.reset()
    preds = self_model.predict(stream_X)
    resids = np.abs(stream_y - preds)
    novs = novelty_sensor.score(stream_X)
    reports = []
    for t in range(len(stream_X)):
        rep = agent.step(resids[t], novs[t], stream_X[t])
        reports.append(rep)
    return reports, resids, novs

print("\n⚙️ Evaluating Milestone 20.6 Across 4 Dynamic Regimes...")
rep_R1, res_R1, nov_R1 = evaluate_regime(R1_X, R1_y)
rep_R2, res_R2, nov_R2 = evaluate_regime(R2_X, R2_y)
rep_R3, res_R3, nov_R3 = evaluate_regime(R3_X, R3_y)
rep_R4, res_R4, nov_R4 = evaluate_regime(R4_X, R4_y)

# -------------------------------------------------------------------------------------------------
# 8. FORMAL AUDIT BENCHMARKS (CR1 TO CR4)
# -------------------------------------------------------------------------------------------------
# CR1: Gradual Ramp Horizon Anticipation (Precursor Window t in [20, 34])
r1_precursor_hazards = [r["hazard_score"] for r in rep_R1[20:35]]
r1_nominal_hazards   = [r["hazard_score"] for r in rep_R1[:20]]
r1_corr_with_future, _ = spearmanr(
    [r["forecast_raw_residual"] for r in rep_R1[20:35]],
    res_R1[20 + HORIZON_H : 35 + HORIZON_H]
)
cr1_pass = bool(
    np.mean(r1_precursor_hazards) > np.mean(r1_nominal_hazards) + 0.20 and
    r1_corr_with_future >= 0.25 and
    rep_R1[45]["operational_state"] in ["EARLY_DRIFT", "ACTIVE_DRIFT", "SEVERE_DRIFT"]
)

# CR2: Abrupt Shock Step Transition (Instant Transition without False Recovery)
r2_nominal_normal = all(r["operational_state"] == "NORMAL" for r in rep_R2[:40])
r2_shock_detection = rep_R2[43]["operational_state"] in ["ACTIVE_DRIFT", "SEVERE_DRIFT"]
cr2_pass = bool(r2_nominal_normal and r2_shock_detection)

# CR3: Oscillatory Drift Tracking & Recovery Dwell
r3_states = [r["operational_state"] for r in rep_R3]
r3_saw_drift = any(s in ["EARLY_DRIFT", "ACTIVE_DRIFT"] for s in r3_states)
r3_saw_recovery = any(s == "RECOVERY" for s in r3_states)
cr3_pass = bool(r3_saw_drift and r3_saw_recovery)

# CR4: Compound Novelty Decoupling (Novel format introduced at t=30 without premature drift)
r4_novel_valid_steps = sum(1 for t in range(30, 35) if rep_R4[t]["epistemic_status"] == "NOVEL_BUT_VALID")
r4_post_drift_detected = any(rep_R4[t]["operational_state"] in ["EARLY_DRIFT", "ACTIVE_DRIFT"] for t in range(45, 65))
cr4_pass = bool(r4_novel_valid_steps >= 2 and r4_post_drift_detected)

criteria = {
    "CR1_GRADUAL_RAMP_ANTICIPATION": cr1_pass,
    "CR2_ABRUPT_SHOCK_CONTAINMENT": cr2_pass,
    "CR3_OSCILLATORY_DWELL_TRACKING": cr3_pass,
    "CR4_COMPOUND_NOVELTY_DECOUPLING": cr4_pass
}
all_passed = all(criteria.values())

# -------------------------------------------------------------------------------------------------
# 9. REPORT & TELEMETRY
# -------------------------------------------------------------------------------------------------
print("\n" + "=" * 115)
print("🏁 MILESTONE 20.6 AUDIT RESULTS: DUAL-COLUMN ARCHITECTURE")
print("=" * 115)
for name, val in criteria.items():
    print(f"{'✅ PASS' if val else '❌ FAIL'} | {name}")
print("-" * 115)
print(f"Criteria Passed: {sum(criteria.values())}/{len(criteria)} | Overall Status: {'PASS' if all_passed else 'FAIL'}")

print("\n" + "=" * 115)
print("📊 CROSS-REGIME TELEMETRY HIGHLIGHTS")
print("=" * 115)
print(f"R1 (Gradual)   : Precursor Mean Hazard = {np.mean(r1_precursor_hazards):.4f} (Nominal = {np.mean(r1_nominal_hazards):.4f}) | Future Corr Rho = {r1_corr_with_future:.4f}")
print(f"R2 (Abrupt)    : Nominal Intact = {r2_nominal_normal} | Post-Shock Transition Step 43 = {rep_R2[43]['operational_state']}")
print(f"R3 (Oscillate) : Drift Detected = {r3_saw_drift} | Recovery Dwell Seen = {r3_saw_recovery}")
print(f"R4 (Compound)  : NOVEL_BUT_VALID Detections = {r4_novel_valid_steps}/5 steps | Shift Detected = {r4_post_drift_detected}")

print("\nSample Unified Epistemic Reports (Stream R1, steps 20..35):")
print(f"{'Step (t)':<8} | {'Operational State':<18} | {'Epistemic Status':<18} | {'Hazard':<8} | {'Recommendation'}")
print("-" * 90)
for t in range(20, 36):
    r = rep_R1[t]
    print(f"{t:<8} | {r['operational_state']:<18} | {r['epistemic_status']:<18} | {r['hazard_score']:<8.4f} | {r['governance_recommendation']}")

telemetry = {
    "milestone": "M20.6",
    "overall_pass": all_passed,
    "criteria": criteria,
    "metrics": {
        "r1_precursor_mean_hazard": float(np.mean(r1_precursor_hazards)),
        "r1_nominal_mean_hazard": float(np.mean(r1_nominal_hazards)),
        "r1_future_corr_rho": float(r1_corr_with_future),
        "r4_novel_valid_steps": int(r4_novel_valid_steps)
    }
}

with open("m20_6_dual_column_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry successfully written to 'm20_6_dual_column_results.json'")
print("=" * 115)

🔬 MILESTONE 20.6: DUAL-COLUMN EPISTEMIC ARCHITECTURE (CUDA)
Seed: 206001 | Model: Qwen/Qwen2.5-0.5B | Hook: blocks.6.hook_resid_post
🔐 Asserting partition independence and collision bounds...
🔐 Partition Orthogonality: VERIFIED
⏳ Loading Qwen/Qwen2.5-0.5B...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🔬 Extracting activations from internal representation...
🔮 Causal Temporal Forecaster trained on independent stream manifolds.

⚙️ Evaluating Milestone 20.6 Across 4 Dynamic Regimes...

🏁 MILESTONE 20.6 AUDIT RESULTS: DUAL-COLUMN ARCHITECTURE
❌ FAIL | CR1_GRADUAL_RAMP_ANTICIPATION
✅ PASS | CR2_ABRUPT_SHOCK_CONTAINMENT
✅ PASS | CR3_OSCILLATORY_DWELL_TRACKING
❌ FAIL | CR4_COMPOUND_NOVELTY_DECOUPLING
-------------------------------------------------------------------------------------------------------------------
Criteria Passed: 2/4 | Overall Status: FAIL

📊 CROSS-REGIME TELEMETRY HIGHLIGHTS
R1 (Gradual)   : Precursor Mean Hazard = 0.0905 (Nominal = 0.0549) | Future Corr Rho = -0.2393
R2 (Abrupt)    : Nominal Intact = True | Post-Shock Transition Step 43 = ACTIVE_DRIFT
R3 (Oscillate) : Drift Detected = True | Recovery Dwell Seen = True
R4 (Compound)  : NOVEL_BUT_VALID Detections = 0/5 steps | Shift Detected = False

Sample

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.6.1 — CORRECTED FAILURE DISSECTION & REPRESENTATION AUDIT
#
# Purpose:
#   Diagnose the failures observed in M20.6 without changing governance thresholds.
#
# Main safeguards:
#   1. Completely independent train / validation / calibration / test streams
#   2. Content-based collision detection independent of split labels
#   3. Padding-aware activation pooling
#   4. Frozen residual target computed only from the selected nominal self-model layer
#   5. Forecast-layer selection on independent validation streams
#   6. Test streams are not used for model/layer selection
#   7. Aligned residual + activation temporal shuffling
#   8. O3 oracle reported with and without clock information
#   9. Stream-level metrics in addition to pooled metrics
#  10. No threshold tuning to force a PASS
#
# Model:
#   Qwen/Qwen2.5-0.5B via TransformerLens
# =================================================================================================

import gc
import json
import math
import random
import hashlib
import warnings
from collections import defaultdict

import numpy as np
import torch

from scipy.stats import spearmanr
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer


# =================================================================================================
# 0. GLOBAL CONFIGURATION
# =================================================================================================

SEED = 206101

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

REQUESTED_LAYERS = [0, 3, 6, 9, 12, 15, 18, 21]

WINDOW_K = 5
HORIZON_H = 6
PCA_TEMPORAL_DIM = 12
PCA_NOVELTY_DIM = 24

N_BASE_TRAIN = 300
N_BASE_LAYER_VALIDATION = 180
N_BASE_CALIBRATION = 180

N_FORECAST_TRAIN_STREAMS = 12
N_FORECAST_VALIDATION_STREAMS = 6
N_TEST_STREAMS = 12

STREAM_LEN = 80
BATCH_SIZE = 16

EPS = 1e-8
OUTPUT_FILE = "m20_6_1_corrected_audit.json"

print("=" * 120)
print("🔬 MILESTONE 20.6.1 — CORRECTED FAILURE DISSECTION & REPRESENTATION AUDIT")
print("=" * 120)
print(f"Device       : {DEVICE.upper()}")
print(f"Model        : {MODEL_NAME}")
print(f"Window       : {WINDOW_K + 1} observations")
print(f"Horizon      : {HORIZON_H}")
print(f"Global seed  : {SEED}")
print("=" * 120)


# =================================================================================================
# 1. GENERAL HELPERS
# =================================================================================================

def safe_rho(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 3 or len(y) < 3:
        return 0.0, 1.0

    if np.std(x) < EPS or np.std(y) < EPS:
        return 0.0, 1.0

    rho, p = spearmanr(x, y)

    if np.isnan(rho):
        return 0.0, 1.0

    return float(rho), float(p)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rho, p = safe_rho(y_pred, y_true)

    return {
        "rho": rho,
        "p_value": p,
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def mean_or_zero(values):
    values = [x for x in values if np.isfinite(x)]
    return float(np.mean(values)) if values else 0.0


def flatten_streams(streams):
    return [row for stream in streams for row in stream]


def stream_lengths(streams):
    return [len(stream) for stream in streams]


def semantic_fingerprint(row):
    """
    Split-independent fingerprint.

    Unlike the previous implementation, split and stream ID are intentionally
    excluded so duplicated semantic samples across partitions can be detected.
    """
    payload = (
        f"{row['t_step']}|"
        f"{row['a']:.12f}|"
        f"{row['b']:.12f}|"
        f"{row['alpha']:.12f}|"
        f"{row['eta_a']:.12f}|"
        f"{row['eta_b']:.12f}|"
        f"{row['mechanism']:.12f}|"
        f"{row['context_type']}|"
        f"{int(row['precursor'])}|"
        f"{int(row['non_linear'])}"
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def assert_partition_disjoint(name_a, rows_a, name_b, rows_b):
    fingerprints_a = {semantic_fingerprint(row) for row in rows_a}
    fingerprints_b = {semantic_fingerprint(row) for row in rows_b}

    overlap = fingerprints_a.intersection(fingerprints_b)

    if overlap:
        raise RuntimeError(
            f"Partition collision detected: {name_a} ↔ {name_b}, "
            f"{len(overlap)} overlapping semantic samples."
        )


def to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer, np.bool_)):
        return obj.item()
    return obj


# =================================================================================================
# 2. INDEPENDENT ENVIRONMENT GENERATION
# =================================================================================================

def generate_sample(
    split,
    idx,
    rng,
    a=None,
    b=None,
    alpha=0.0,
    context_type="STANDARD",
    t_step=None,
    non_linear=False,
    precursor=False,
):
    if a is None:
        a = rng.uniform(-1.0, 1.0)
        while abs(a) < 0.10:
            a = rng.uniform(-1.0, 1.0)

    if b is None:
        b = rng.uniform(-1.0, 1.0)

    eta_a = 0.0
    eta_b = 0.0
    sigma_a = 0.0
    sigma_b = 0.0

    if non_linear:
        mechanism = float(
            np.sin(3.0 * a)
            + b ** 2
            + 1.5 * a * b
        )

    elif precursor:
        step_offset = max(0, (t_step or 20) - 20)

        sigma_a = 0.20 + 0.015 * min(15, step_offset)
        sigma_b = 0.16 + 0.015 * min(15, step_offset)

        eta_a = rng.normal(0.0, sigma_a)
        eta_b = rng.normal(0.0, sigma_b)

        mechanism = float(
            (0.8 + eta_a) * a
            - (0.55 + eta_b) * b
        )

    else:
        mechanism = float(
            (1.0 - 2.0 * alpha) * (0.8 * a)
            - 0.55 * b
        )

    row = {
        "split": split,
        "idx": int(idx),
        "t_step": None if t_step is None else int(t_step),
        "a": float(a),
        "b": float(b),
        "alpha": float(alpha),
        "eta_a": float(eta_a),
        "eta_b": float(eta_b),
        "sigma_a": float(sigma_a),
        "sigma_b": float(sigma_b),
        "sigma_instability": float(sigma_a + sigma_b),
        "context_type": context_type,
        "non_linear": bool(non_linear),
        "precursor": bool(precursor),
        "mechanism": float(mechanism),
    }

    row["fingerprint"] = semantic_fingerprint(row)
    return row


def generate_stream(
    regime,
    stream_id,
    seed,
    n_steps=STREAM_LEN,
    drift_start=35,
):
    rng = np.random.RandomState(seed)
    stream = []

    for t in range(n_steps):
        if regime == "R1_GRADUAL":
            if t < 20:
                row = generate_sample(
                    regime, t, rng, alpha=0.0, t_step=t
                )
            elif t < drift_start:
                row = generate_sample(
                    regime, t, rng, alpha=0.0, t_step=t, precursor=True
                )
            else:
                alpha = min(
                    1.0,
                    0.05 + 0.95 * ((t - drift_start) / 40.0),
                )
                row = generate_sample(
                    regime, t, rng, alpha=alpha, t_step=t
                )

        elif regime == "R2_ABRUPT":
            alpha = 0.0 if t < 40 else 0.85
            row = generate_sample(
                regime, t, rng, alpha=alpha, t_step=t
            )

        elif regime == "R3_OSCILLATORY":
            alpha = (
                0.0
                if t < 25
                else 0.45 * (1.0 + math.sin(0.25 * (t - 25)))
            )
            row = generate_sample(
                regime, t, rng, alpha=alpha, t_step=t
            )

        elif regime == "R4_COMPOUND":
            context = "STANDARD" if t < 30 else "NOVEL_FORMAT"

            if t < 20:
                row = generate_sample(
                    regime,
                    t,
                    rng,
                    alpha=0.0,
                    context_type=context,
                    t_step=t,
                )
            elif t < drift_start:
                row = generate_sample(
                    regime,
                    t,
                    rng,
                    alpha=0.0,
                    context_type=context,
                    t_step=t,
                    precursor=True,
                )
            else:
                alpha = min(
                    1.0,
                    0.10 + 0.90 * ((t - drift_start) / 40.0),
                )
                row = generate_sample(
                    regime,
                    t,
                    rng,
                    alpha=alpha,
                    context_type=context,
                    t_step=t,
                )
        else:
            raise ValueError(f"Unknown regime: {regime}")

        row["stream_id"] = stream_id
        row["regime"] = regime
        stream.append(row)

    return stream


def generate_base_rows(split, n, seed):
    rng = np.random.RandomState(seed)

    return [
        generate_sample(
            split=split,
            idx=i,
            rng=rng,
            alpha=0.0,
            context_type="STANDARD",
            t_step=None,
        )
        for i in range(n)
    ]


base_train = generate_base_rows(
    "BASE_TRAIN", N_BASE_TRAIN, SEED + 10_000
)

base_layer_validation = generate_base_rows(
    "BASE_LAYER_VALIDATION",
    N_BASE_LAYER_VALIDATION,
    SEED + 20_000,
)

base_calibration = generate_base_rows(
    "BASE_CALIBRATION",
    N_BASE_CALIBRATION,
    SEED + 30_000,
)

forecast_train_streams = [
    generate_stream(
        "R1_GRADUAL",
        f"FC_TRAIN_{i}",
        SEED + 40_000 + 101 * i,
    )
    for i in range(N_FORECAST_TRAIN_STREAMS)
]

forecast_validation_streams = [
    generate_stream(
        "R1_GRADUAL",
        f"FC_VALIDATION_{i}",
        SEED + 50_000 + 103 * i,
    )
    for i in range(N_FORECAST_VALIDATION_STREAMS)
]

test_regimes = {
    "R1_GRADUAL": [
        generate_stream(
            "R1_GRADUAL",
            f"TEST_R1_{i}",
            SEED + 60_000 + 107 * i,
        )
        for i in range(N_TEST_STREAMS)
    ],
    "R2_ABRUPT": [
        generate_stream(
            "R2_ABRUPT",
            f"TEST_R2_{i}",
            SEED + 70_000 + 109 * i,
        )
        for i in range(N_TEST_STREAMS)
    ],
    "R3_OSCILLATORY": [
        generate_stream(
            "R3_OSCILLATORY",
            f"TEST_R3_{i}",
            SEED + 80_000 + 113 * i,
        )
        for i in range(N_TEST_STREAMS)
    ],
    "R4_COMPOUND": [
        generate_stream(
            "R4_COMPOUND",
            f"TEST_R4_{i}",
            SEED + 90_000 + 127 * i,
        )
        for i in range(N_TEST_STREAMS)
    ],
}

partition_rows = {
    "base_train": base_train,
    "base_layer_validation": base_layer_validation,
    "base_calibration": base_calibration,
    "forecast_train": flatten_streams(forecast_train_streams),
    "forecast_validation": flatten_streams(forecast_validation_streams),
}

for regime_name, streams in test_regimes.items():
    partition_rows[f"test_{regime_name}"] = flatten_streams(streams)

partition_names = list(partition_rows)

for i, name_a in enumerate(partition_names):
    for name_b in partition_names[i + 1:]:
        assert_partition_disjoint(
            name_a,
            partition_rows[name_a],
            name_b,
            partition_rows[name_b],
        )

print("\n✅ Semantic partition independence verified.")


# =================================================================================================
# 3. LOAD MODEL AND LOCATE HOOKS
# =================================================================================================

print("\n⏳ Loading model...")

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device=DEVICE,
)
model.eval()

available_hooks = set(model.hook_dict.keys())

layer_hooks = {}

for layer in REQUESTED_LAYERS:
    target = f"blocks.{layer}.hook_resid_post"

    if target in available_hooks:
        layer_hooks[layer] = target

VALID_LAYERS = sorted(layer_hooks)

if not VALID_LAYERS:
    raise RuntimeError("No requested residual-post hooks were found.")

print(f"✅ Model loaded. Valid layers: {VALID_LAYERS}")


# =================================================================================================
# 4. PROMPTS AND PADDING-AWARE ACTIVATION EXTRACTION
# =================================================================================================

def make_prompt(row):
    if row["context_type"] == "NOVEL_FORMAT":
        return (
            "<<SYSTEM EXPERIMENTAL QUERY PROTOCOL>>\n"
            f"Coordinate Alpha: {row['a']:.6f}\n"
            f"Coordinate Beta: {row['b']:.6f}\n"
            "Synthesize latent transformation outcome value."
        )

    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )


def token_valid_mask(tokens):
    pad_token_id = model.tokenizer.pad_token_id

    if pad_token_id is None:
        return torch.ones_like(tokens, dtype=torch.bool)

    return tokens.ne(pad_token_id)


def pool_activations(act, valid_mask, pooling):
    """
    act        : [batch, position, d_model]
    valid_mask : [batch, position]
    """
    mask_f = valid_mask.unsqueeze(-1).to(act.dtype)

    if pooling == "mean":
        denominator = mask_f.sum(dim=1).clamp_min(1.0)
        return (act * mask_f).sum(dim=1) / denominator

    if pooling == "last":
        positions = torch.arange(
            valid_mask.shape[1],
            device=valid_mask.device,
        ).unsqueeze(0)

        last_index = (
            positions.masked_fill(~valid_mask, -1)
            .max(dim=1)
            .values
            .clamp_min(0)
        )

        batch_index = torch.arange(
            act.shape[0],
            device=act.device,
        )

        return act[batch_index, last_index]

    if pooling == "max":
        masked = act.masked_fill(
            ~valid_mask.unsqueeze(-1),
            torch.finfo(act.dtype).min,
        )
        return masked.max(dim=1).values

    raise ValueError(f"Unknown pooling mode: {pooling}")


@torch.no_grad()
def extract_multi_layer(
    rows,
    layers,
    poolings=("mean",),
    batch_size=BATCH_SIZE,
):
    output = {
        layer: {pooling: [] for pooling in poolings}
        for layer in layers
    }

    requested_hooks = {layer_hooks[layer] for layer in layers}

    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(row) for row in batch]

        tokens = model.to_tokens(
            prompts,
            prepend_bos=True,
        ).to(DEVICE)

        valid_mask = token_valid_mask(tokens)

        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name in requested_hooks,
            return_type=None,
        )

        for layer in layers:
            act = cache[layer_hooks[layer]].detach().float()

            for pooling in poolings:
                pooled = pool_activations(
                    act,
                    valid_mask,
                    pooling,
                )

                output[layer][pooling].append(
                    pooled.cpu().numpy()
                )

        del cache, tokens, valid_mask

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    for layer in layers:
        for pooling in poolings:
            output[layer][pooling] = np.concatenate(
                output[layer][pooling],
                axis=0,
            )

    return output


# =================================================================================================
# 5. EXTRACT REPRESENTATION GROUPS
# =================================================================================================

print("\n🔬 Extracting baseline representations...")

base_train_acts = extract_multi_layer(
    base_train,
    VALID_LAYERS,
    poolings=("mean",),
)

base_validation_acts = extract_multi_layer(
    base_layer_validation,
    VALID_LAYERS,
    poolings=("mean",),
)

base_calibration_acts = extract_multi_layer(
    base_calibration,
    VALID_LAYERS,
    poolings=("mean",),
)

base_train_y = np.array([row["mechanism"] for row in base_train])
base_validation_y = np.array(
    [row["mechanism"] for row in base_layer_validation]
)
base_calibration_y = np.array(
    [row["mechanism"] for row in base_calibration]
)


# =================================================================================================
# 6. NOMINAL SELF-MODEL LAYER SELECTION
# =================================================================================================

nominal_layer_results = {}
nominal_candidates = {}

print("\n" + "=" * 120)
print("🧪 TEST A — NOMINAL SELF-MODEL LAYER SELECTION")
print("=" * 120)

for layer in VALID_LAYERS:
    scaler = StandardScaler()
    regressor = RidgeCV(
        alphas=np.logspace(-3, 4, 20)
    )

    X_train = base_train_acts[layer]["mean"]
    X_validation = base_validation_acts[layer]["mean"]

    regressor.fit(
        scaler.fit_transform(X_train),
        base_train_y,
    )

    validation_prediction = regressor.predict(
        scaler.transform(X_validation)
    )

    metrics = regression_metrics(
        base_validation_y,
        validation_prediction,
    )

    nominal_layer_results[layer] = metrics
    nominal_candidates[layer] = (scaler, regressor)

    print(
        f"Layer {layer:<3} | "
        f"Rho={metrics['rho']: .4f} | "
        f"R²={metrics['r2']: .4f} | "
        f"MAE={metrics['mae']: .4f}"
    )

BEST_SELF_MODEL_LAYER = max(
    nominal_layer_results,
    key=lambda layer: nominal_layer_results[layer]["r2"],
)

self_scaler, self_regressor = nominal_candidates[
    BEST_SELF_MODEL_LAYER
]

print(
    f"\n🏆 Selected nominal self-model layer: "
    f"{BEST_SELF_MODEL_LAYER}"
)

calibration_prediction = self_regressor.predict(
    self_scaler.transform(
        base_calibration_acts[BEST_SELF_MODEL_LAYER]["mean"]
    )
)

calibration_residuals = np.abs(
    base_calibration_y - calibration_prediction
)

cal_median = float(np.median(calibration_residuals))
cal_mad = float(
    np.median(
        np.abs(calibration_residuals - cal_median)
    )
)

calibration_profile = {
    "median": cal_median,
    "mad": cal_mad,
    "mad_scale": float(1.4826 * cal_mad + EPS),
    "q75": float(np.quantile(calibration_residuals, 0.75)),
    "q90": float(np.quantile(calibration_residuals, 0.90)),
    "q95": float(np.quantile(calibration_residuals, 0.95)),
    "q99": float(np.quantile(calibration_residuals, 0.99)),
}

print("\nCalibration profile:")
for key, value in calibration_profile.items():
    print(f"  {key:<10}: {value:.6f}")


# =================================================================================================
# 7. STREAM REPRESENTATION EXTRACTION
# =================================================================================================

def extract_stream_group(streams, layers, poolings=("mean",)):
    rows = flatten_streams(streams)

    activations = extract_multi_layer(
        rows,
        layers,
        poolings=poolings,
    )

    return {
        "streams": streams,
        "rows": rows,
        "lengths": stream_lengths(streams),
        "activations": activations,
    }


print("\n🔬 Extracting independent temporal stream representations...")

forecast_train_group = extract_stream_group(
    forecast_train_streams,
    VALID_LAYERS,
)

forecast_validation_group = extract_stream_group(
    forecast_validation_streams,
    VALID_LAYERS,
)

test_groups = {}

for regime_name, streams in test_regimes.items():
    print(f"  Extracting {regime_name}...")
    test_groups[regime_name] = extract_stream_group(
        streams,
        VALID_LAYERS,
    )


# =================================================================================================
# 8. FROZEN RESIDUAL TARGETS
# =================================================================================================

def attach_frozen_residuals(group):
    """
    Residuals are always computed from BEST_SELF_MODEL_LAYER.

    They must not change during temporal-layer sweeps.
    """
    X_self = group["activations"][
        BEST_SELF_MODEL_LAYER
    ]["mean"]

    y = np.array(
        [row["mechanism"] for row in group["rows"]],
        dtype=float,
    )

    prediction = self_regressor.predict(
        self_scaler.transform(X_self)
    )

    group["mechanism"] = y
    group["self_prediction"] = prediction
    group["residuals"] = np.abs(y - prediction)


attach_frozen_residuals(forecast_train_group)
attach_frozen_residuals(forecast_validation_group)

for group in test_groups.values():
    attach_frozen_residuals(group)


# =================================================================================================
# 9. TEMPORAL DATASET UTILITIES
# =================================================================================================

class ActivationEncoder:
    def __init__(self, n_components):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.pca = None

    def fit(self, X):
        X_scaled = self.scaler.fit_transform(X)

        max_components = min(
            self.n_components,
            X_scaled.shape[0] - 1,
            X_scaled.shape[1],
        )

        self.pca = PCA(
            n_components=max_components,
            random_state=SEED,
        )

        self.pca.fit(X_scaled)
        return self

    def transform(self, X):
        return self.pca.transform(
            self.scaler.transform(X)
        )


def precursor_selector(regime, t, horizon):
    if regime in ("R1_GRADUAL", "R4_COMPOUND"):
        return 20 <= t < 35

    if regime == "R2_ABRUPT":
        # Windows whose forecast horizon approaches/crosses the abrupt shock.
        return 25 <= t < 40

    if regime == "R3_OSCILLATORY":
        return 20 <= t < (STREAM_LEN - horizon)

    return WINDOW_K <= t < (STREAM_LEN - horizon)


def split_flat_array(array, lengths):
    pieces = []
    offset = 0

    for n in lengths:
        pieces.append(array[offset:offset + n])
        offset += n

    return pieces


def build_temporal_dataset(
    group,
    layer,
    encoder,
    pooling="mean",
    regime_name="R1_GRADUAL",
    window=WINDOW_K,
    horizon=HORIZON_H,
):
    raw_activations = group["activations"][layer][pooling]
    encoded_activations = encoder.transform(raw_activations)

    residual_streams = split_flat_array(
        group["residuals"],
        group["lengths"],
    )

    encoded_streams = split_flat_array(
        encoded_activations,
        group["lengths"],
    )

    X_residual = []
    X_activation = []
    X_full = []
    y_future = []
    sample_stream_ids = []
    sample_times = []

    for stream_idx, (residuals, activations) in enumerate(
        zip(residual_streams, encoded_streams)
    ):
        n = len(residuals)

        for t in range(window, n - horizon):
            if not precursor_selector(
                regime_name,
                t,
                horizon,
            ):
                continue

            residual_history = residuals[
                t - window:t + 1
            ]

            activation_history = activations[
                t - window:t + 1
            ]

            if len(residual_history) != window + 1:
                continue

            X_residual.append(residual_history)
            X_activation.append(activation_history.flatten())
            X_full.append(
                np.hstack([
                    residual_history,
                    activation_history.flatten(),
                ])
            )

            y_future.append(residuals[t + horizon])
            sample_stream_ids.append(stream_idx)
            sample_times.append(t)

    return {
        "X_residual": np.asarray(X_residual, dtype=float),
        "X_activation": np.asarray(X_activation, dtype=float),
        "X_full": np.asarray(X_full, dtype=float),
        "y": np.asarray(y_future, dtype=float),
        "stream_ids": np.asarray(sample_stream_ids),
        "times": np.asarray(sample_times),
        "encoded_dim": int(
            encoder.pca.n_components_
        ),
    }


def fit_ridge_forecaster(X, y):
    scaler = StandardScaler()
    regressor = RidgeCV(
        alphas=np.logspace(-3, 4, 20)
    )

    regressor.fit(
        scaler.fit_transform(X),
        y,
    )

    return scaler, regressor


def predict_forecaster(model_tuple, X):
    scaler, regressor = model_tuple
    return regressor.predict(
        scaler.transform(X)
    )


def stream_macro_metrics(y, prediction, stream_ids):
    stream_rhos = []
    stream_r2s = []
    stream_maes = []

    for stream_id in np.unique(stream_ids):
        mask = stream_ids == stream_id

        if mask.sum() < 3:
            continue

        rho, _ = safe_rho(
            prediction[mask],
            y[mask],
        )

        stream_rhos.append(rho)
        stream_r2s.append(
            r2_score(y[mask], prediction[mask])
        )
        stream_maes.append(
            mean_absolute_error(y[mask], prediction[mask])
        )

    return {
        "macro_stream_rho": mean_or_zero(stream_rhos),
        "macro_stream_r2": mean_or_zero(stream_r2s),
        "macro_stream_mae": mean_or_zero(stream_maes),
        "n_streams": int(len(np.unique(stream_ids))),
    }


def evaluate_predictions(y, prediction, stream_ids):
    result = regression_metrics(y, prediction)
    result.update(
        stream_macro_metrics(
            y,
            prediction,
            stream_ids,
        )
    )
    return result


def aligned_temporal_shuffle(
    X_full,
    history_length,
    encoded_dim,
    seed,
):
    """
    Applies one shared permutation to residual and activation time slots.

    Input layout:
        [residual history,
         flattened activation history]
    """
    rng = np.random.RandomState(seed)
    X_shuffled = np.asarray(X_full).copy()

    residual_block = X_shuffled[:, :history_length]

    activation_block = X_shuffled[:, history_length:]
    activation_block = activation_block.reshape(
        len(X_shuffled),
        history_length,
        encoded_dim,
    )

    for i in range(len(X_shuffled)):
        permutation = rng.permutation(history_length)

        residual_block[i] = residual_block[i, permutation]
        activation_block[i] = activation_block[i, permutation, :]

    X_shuffled[:, :history_length] = residual_block
    X_shuffled[:, history_length:] = activation_block.reshape(
        len(X_shuffled),
        -1,
    )

    return X_shuffled


# =================================================================================================
# 10. TEMPORAL FORECAST LAYER SELECTION ON VALIDATION STREAMS
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST B — TEMPORAL FORECASTABILITY BY LAYER")
print("Selection uses forecast-validation streams, not test streams.")
print("=" * 120)

layer_temporal_results = {}
layer_temporal_artifacts = {}

for layer in VALID_LAYERS:
    encoder = ActivationEncoder(
        PCA_TEMPORAL_DIM
    ).fit(
        forecast_train_group["activations"][layer]["mean"]
    )

    train_dataset = build_temporal_dataset(
        forecast_train_group,
        layer,
        encoder,
        regime_name="R1_GRADUAL",
    )

    validation_dataset = build_temporal_dataset(
        forecast_validation_group,
        layer,
        encoder,
        regime_name="R1_GRADUAL",
    )

    forecaster = fit_ridge_forecaster(
        train_dataset["X_full"],
        train_dataset["y"],
    )

    validation_prediction = predict_forecaster(
        forecaster,
        validation_dataset["X_full"],
    )

    metrics = evaluate_predictions(
        validation_dataset["y"],
        validation_prediction,
        validation_dataset["stream_ids"],
    )

    layer_temporal_results[layer] = metrics
    layer_temporal_artifacts[layer] = {
        "encoder": encoder,
        "forecaster": forecaster,
        "train_dataset": train_dataset,
        "validation_dataset": validation_dataset,
    }

    print(
        f"Layer {layer:<3} | "
        f"Pooled rho={metrics['rho']: .4f} | "
        f"Macro rho={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )

BEST_FORECAST_LAYER = max(
    layer_temporal_results,
    key=lambda layer: (
        layer_temporal_results[layer]["macro_stream_rho"],
        layer_temporal_results[layer]["rho"],
    ),
)

print(
    f"\n🏆 Selected temporal forecast layer: "
    f"{BEST_FORECAST_LAYER}"
)

selected_encoder = layer_temporal_artifacts[
    BEST_FORECAST_LAYER
]["encoder"]

selected_train_dataset = layer_temporal_artifacts[
    BEST_FORECAST_LAYER
]["train_dataset"]

selected_forecaster = layer_temporal_artifacts[
    BEST_FORECAST_LAYER
]["forecaster"]


# =================================================================================================
# 11. REPRESENTATION / COMPRESSION LADDER
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST C — TEMPORAL REPRESENTATION LADDER")
print("=" * 120)

representation_results = {}

train_dataset = selected_train_dataset

validation_dataset = layer_temporal_artifacts[
    BEST_FORECAST_LAYER
]["validation_dataset"]

representation_variants = {
    "RESIDUAL_HISTORY": (
        train_dataset["X_residual"],
        validation_dataset["X_residual"],
    ),
    "ACTIVATION_HISTORY": (
        train_dataset["X_activation"],
        validation_dataset["X_activation"],
    ),
    "RESIDUAL_PLUS_ACTIVATION_HISTORY": (
        train_dataset["X_full"],
        validation_dataset["X_full"],
    ),
}

for name, (X_train, X_validation) in representation_variants.items():
    model_variant = fit_ridge_forecaster(
        X_train,
        train_dataset["y"],
    )

    prediction = predict_forecaster(
        model_variant,
        X_validation,
    )

    metrics = evaluate_predictions(
        validation_dataset["y"],
        prediction,
        validation_dataset["stream_ids"],
    )

    representation_results[name] = metrics

    print(
        f"{name:<38} | "
        f"Rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )


# =================================================================================================
# 12. ALIGNED TEMPORAL ORDER ABLATION
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST D — ALIGNED TEMPORAL ORDER ABLATION")
print("=" * 120)

ordered_prediction = predict_forecaster(
    selected_forecaster,
    validation_dataset["X_full"],
)

ordered_metrics = evaluate_predictions(
    validation_dataset["y"],
    ordered_prediction,
    validation_dataset["stream_ids"],
)

shuffle_repetitions = 100
shuffle_rhos = []
shuffle_r2s = []

for repetition in range(shuffle_repetitions):
    X_shuffled = aligned_temporal_shuffle(
        validation_dataset["X_full"],
        history_length=WINDOW_K + 1,
        encoded_dim=validation_dataset["encoded_dim"],
        seed=SEED + 200_000 + repetition,
    )

    shuffled_prediction = predict_forecaster(
        selected_forecaster,
        X_shuffled,
    )

    shuffled_metrics = regression_metrics(
        validation_dataset["y"],
        shuffled_prediction,
    )

    shuffle_rhos.append(shuffled_metrics["rho"])
    shuffle_r2s.append(shuffled_metrics["r2"])

shuffle_summary = {
    "ordered_rho": ordered_metrics["rho"],
    "ordered_r2": ordered_metrics["r2"],
    "shuffle_mean_rho": float(np.mean(shuffle_rhos)),
    "shuffle_p95_rho": float(np.quantile(shuffle_rhos, 0.95)),
    "shuffle_mean_r2": float(np.mean(shuffle_r2s)),
    "rho_advantage_over_shuffle_mean": float(
        ordered_metrics["rho"] - np.mean(shuffle_rhos)
    ),
    "rho_advantage_over_shuffle_p95": float(
        ordered_metrics["rho"] - np.quantile(shuffle_rhos, 0.95)
    ),
}

print(f"Ordered rho             : {shuffle_summary['ordered_rho']:.4f}")
print(f"Shuffle mean rho        : {shuffle_summary['shuffle_mean_rho']:.4f}")
print(f"Shuffle P95 rho         : {shuffle_summary['shuffle_p95_rho']:.4f}")
print(
    f"Advantage over mean     : "
    f"{shuffle_summary['rho_advantage_over_shuffle_mean']:.4f}"
)
print(
    f"Advantage over P95      : "
    f"{shuffle_summary['rho_advantage_over_shuffle_p95']:.4f}"
)


# =================================================================================================
# 13. GENERATOR ORACLE: WITH AND WITHOUT CLOCK
# =================================================================================================

def build_oracle_dataset(
    group,
    regime_name,
    include_clock,
):
    residual_streams = split_flat_array(
        group["residuals"],
        group["lengths"],
    )

    X = []
    y = []
    stream_ids = []

    for stream_idx, (stream, residuals) in enumerate(
        zip(group["streams"], residual_streams)
    ):
        n = len(stream)

        for t in range(WINDOW_K, n - HORIZON_H):
            if not precursor_selector(
                regime_name,
                t,
                HORIZON_H,
            ):
                continue

            row = stream[t]

            features = [
                row["eta_a"],
                row["eta_b"],
                row["sigma_instability"],
                row["alpha"],
            ]

            if include_clock:
                features.append(
                    row["t_step"] / max(1, n - 1)
                )

            X.append(features)
            y.append(residuals[t + HORIZON_H])
            stream_ids.append(stream_idx)

    return {
        "X": np.asarray(X, dtype=float),
        "y": np.asarray(y, dtype=float),
        "stream_ids": np.asarray(stream_ids),
    }


def fit_nonlinear_forecaster(X, y):
    regressor = GradientBoostingRegressor(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.04,
        random_state=SEED,
    )
    regressor.fit(X, y)
    return regressor


print("\n" + "=" * 120)
print("🧪 TEST E — GENERATOR ORACLE WITH/WITHOUT CLOCK")
print("=" * 120)

oracle_results = {}

for include_clock in (False, True):
    oracle_name = (
        "O3_WITH_CLOCK"
        if include_clock
        else "O3_NO_CLOCK"
    )

    oracle_train = build_oracle_dataset(
        forecast_train_group,
        "R1_GRADUAL",
        include_clock,
    )

    oracle_validation = build_oracle_dataset(
        forecast_validation_group,
        "R1_GRADUAL",
        include_clock,
    )

    oracle_model = fit_nonlinear_forecaster(
        oracle_train["X"],
        oracle_train["y"],
    )

    oracle_prediction = oracle_model.predict(
        oracle_validation["X"]
    )

    metrics = evaluate_predictions(
        oracle_validation["y"],
        oracle_prediction,
        oracle_validation["stream_ids"],
    )

    oracle_results[oracle_name] = metrics

    print(
        f"{oracle_name:<20} | "
        f"Rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )


# =================================================================================================
# 14. CROSS-REGIME GENERALIZATION ON UNTOUCHED TEST STREAMS
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST F — CROSS-REGIME GENERALIZATION")
print("=" * 120)

cross_regime_results = {}

for regime_name, group in test_groups.items():
    dataset = build_temporal_dataset(
        group,
        BEST_FORECAST_LAYER,
        selected_encoder,
        regime_name=regime_name,
    )

    prediction = predict_forecaster(
        selected_forecaster,
        dataset["X_full"],
    )

    metrics = evaluate_predictions(
        dataset["y"],
        prediction,
        dataset["stream_ids"],
    )

    persistence_prediction = dataset["X_residual"][:, -1]

    persistence_metrics = evaluate_predictions(
        dataset["y"],
        persistence_prediction,
        dataset["stream_ids"],
    )

    metrics["persistence_rho"] = persistence_metrics["rho"]
    metrics["persistence_r2"] = persistence_metrics["r2"]
    metrics["rho_gain_over_persistence"] = float(
        metrics["rho"] - persistence_metrics["rho"]
    )

    cross_regime_results[regime_name] = metrics

    print(
        f"{regime_name:<18} | "
        f"Rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"Persist={metrics['persistence_rho']: .4f} | "
        f"Δ={metrics['rho_gain_over_persistence']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )


# =================================================================================================
# 15. FORECASTER CAPACITY AUDIT
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST G — FORECASTER CAPACITY GAP")
print("=" * 120)

r1_test_dataset = build_temporal_dataset(
    test_groups["R1_GRADUAL"],
    BEST_FORECAST_LAYER,
    selected_encoder,
    regime_name="R1_GRADUAL",
)

X_train = selected_train_dataset["X_full"]
y_train = selected_train_dataset["y"]

X_test = r1_test_dataset["X_full"]
y_test = r1_test_dataset["y"]

capacity_models = {
    "RIDGE": RidgeCV(
        alphas=np.logspace(-3, 4, 20)
    ),
    "GRADIENT_BOOSTING": GradientBoostingRegressor(
        n_estimators=180,
        max_depth=3,
        learning_rate=0.04,
        random_state=SEED,
    ),
    "RANDOM_FOREST": RandomForestRegressor(
        n_estimators=250,
        max_depth=7,
        min_samples_leaf=3,
        random_state=SEED,
        n_jobs=-1,
    ),
}

capacity_results = {}

for name, regressor in capacity_models.items():
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    regressor.fit(X_train_scaled, y_train)
    prediction = regressor.predict(X_test_scaled)

    metrics = evaluate_predictions(
        y_test,
        prediction,
        r1_test_dataset["stream_ids"],
    )

    capacity_results[name] = metrics

    print(
        f"{name:<20} | "
        f"Rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f} | "
        f"MAE={metrics['mae']: .4f}"
    )


# =================================================================================================
# 16. POOLING AUDIT
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST H — PADDING-AWARE POOLING AUDIT")
print("=" * 120)

pooling_train_group = extract_stream_group(
    forecast_train_streams,
    [BEST_FORECAST_LAYER],
    poolings=("mean", "last", "max"),
)

pooling_validation_group = extract_stream_group(
    forecast_validation_streams,
    [BEST_FORECAST_LAYER],
    poolings=("mean", "last", "max"),
)

# Reuse the same frozen residual targets.
pooling_train_group["residuals"] = forecast_train_group["residuals"]
pooling_validation_group["residuals"] = forecast_validation_group["residuals"]

pooling_results = {}

for pooling in ("mean", "last", "max"):
    encoder = ActivationEncoder(
        PCA_TEMPORAL_DIM
    ).fit(
        pooling_train_group["activations"][
            BEST_FORECAST_LAYER
        ][pooling]
    )

    train_data = build_temporal_dataset(
        pooling_train_group,
        BEST_FORECAST_LAYER,
        encoder,
        pooling=pooling,
        regime_name="R1_GRADUAL",
    )

    validation_data = build_temporal_dataset(
        pooling_validation_group,
        BEST_FORECAST_LAYER,
        encoder,
        pooling=pooling,
        regime_name="R1_GRADUAL",
    )

    model_pooling = fit_ridge_forecaster(
        train_data["X_full"],
        train_data["y"],
    )

    prediction = predict_forecaster(
        model_pooling,
        validation_data["X_full"],
    )

    metrics = evaluate_predictions(
        validation_data["y"],
        prediction,
        validation_data["stream_ids"],
    )

    pooling_results[pooling] = metrics

    print(
        f"{pooling:<8} | "
        f"Rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )


# =================================================================================================
# 17. NOVELTY SENSOR AUDIT
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST I — NOVELTY SENSOR AUDIT")
print("=" * 120)

novel_rng = np.random.RandomState(SEED + 150_000)

novel_rows = [
    generate_sample(
        split="NOVELTY_TEST",
        idx=i,
        rng=novel_rng,
        alpha=0.0,
        context_type="NOVEL_FORMAT",
        t_step=None,
    )
    for i in range(N_BASE_CALIBRATION)
]

novel_acts = extract_multi_layer(
    novel_rows,
    [BEST_FORECAST_LAYER],
    poolings=("mean",),
)[BEST_FORECAST_LAYER]["mean"]

novelty_train_X = base_train_acts[
    BEST_FORECAST_LAYER
]["mean"]

novelty_standard_X = base_calibration_acts[
    BEST_FORECAST_LAYER
]["mean"]

novelty_scaler = StandardScaler()
novelty_train_scaled = novelty_scaler.fit_transform(
    novelty_train_X
)

novelty_standard_scaled = novelty_scaler.transform(
    novelty_standard_X
)

novelty_novel_scaled = novelty_scaler.transform(
    novel_acts
)

novelty_components = min(
    PCA_NOVELTY_DIM,
    novelty_train_scaled.shape[0] - 1,
    novelty_train_scaled.shape[1],
)

novelty_pca = PCA(
    n_components=novelty_components,
    random_state=SEED,
)

Z_train = novelty_pca.fit_transform(
    novelty_train_scaled
)

Z_standard = novelty_pca.transform(
    novelty_standard_scaled
)

Z_novel = novelty_pca.transform(
    novelty_novel_scaled
)

covariance_model = LedoitWolf().fit(Z_train)
novelty_center = Z_train.mean(axis=0)
novelty_precision = covariance_model.precision_


def mahalanobis_distance(Z):
    difference = Z - novelty_center

    squared_distance = np.sum(
        (difference @ novelty_precision) * difference,
        axis=1,
    )

    return np.sqrt(
        np.maximum(squared_distance, 0.0)
    )


train_novelty_scores = mahalanobis_distance(Z_train)
standard_novelty_scores = mahalanobis_distance(Z_standard)
novel_novelty_scores = mahalanobis_distance(Z_novel)

novelty_threshold = float(
    np.quantile(train_novelty_scores, 0.95)
)

labels = np.concatenate([
    np.zeros(len(standard_novelty_scores)),
    np.ones(len(novel_novelty_scores)),
])

scores = np.concatenate([
    standard_novelty_scores,
    novel_novelty_scores,
])

novelty_auc = float(
    roc_auc_score(labels, scores)
)

novelty_fpr = float(
    np.mean(
        standard_novelty_scores > novelty_threshold
    )
)

novelty_tpr = float(
    np.mean(
        novel_novelty_scores > novelty_threshold
    )
)

novelty_effect_size = float(
    (
        novel_novelty_scores.mean()
        - standard_novelty_scores.mean()
    )
    / (
        np.std(scores) + EPS
    )
)

novelty_results = {
    "auc": novelty_auc,
    "threshold_train_q95": novelty_threshold,
    "standard_false_positive_rate": novelty_fpr,
    "novel_true_positive_rate": novelty_tpr,
    "effect_size": novelty_effect_size,
    "standard_mean_distance": float(
        standard_novelty_scores.mean()
    ),
    "novel_mean_distance": float(
        novel_novelty_scores.mean()
    ),
}

print(
    f"Novelty AUC              : "
    f"{novelty_results['auc']:.4f}"
)
print(
    f"Standard FPR @ train Q95 : "
    f"{novelty_results['standard_false_positive_rate']:.4f}"
)
print(
    f"Novel TPR @ train Q95    : "
    f"{novelty_results['novel_true_positive_rate']:.4f}"
)
print(
    f"Effect size              : "
    f"{novelty_results['effect_size']:.4f}"
)


# =================================================================================================
# 18. DIAGNOSTIC INTERPRETATION
# =================================================================================================

print("\n" + "=" * 120)
print("🧠 AUTOMATED DIAGNOSTIC SUMMARY")
print("=" * 120)

diagnostic_flags = {
    "nominal_self_model_viable": bool(
        nominal_layer_results[
            BEST_SELF_MODEL_LAYER
        ]["r2"] > 0.0
    ),
    "validation_temporal_signal_above_shuffle_p95": bool(
        shuffle_summary[
            "rho_advantage_over_shuffle_p95"
        ] > 0.0
    ),
    "meaningful_order_advantage": bool(
        shuffle_summary[
            "rho_advantage_over_shuffle_mean"
        ] >= 0.05
    ),
    "r1_test_beats_persistence": bool(
        cross_regime_results[
            "R1_GRADUAL"
        ]["rho_gain_over_persistence"] > 0.0
    ),
    "cross_regime_transfer_to_abrupt": bool(
        cross_regime_results[
            "R2_ABRUPT"
        ]["rho_gain_over_persistence"] > 0.0
    ),
    "cross_regime_transfer_to_oscillatory": bool(
        cross_regime_results[
            "R3_OSCILLATORY"
        ]["rho_gain_over_persistence"] > 0.0
    ),
    "cross_regime_transfer_to_compound": bool(
        cross_regime_results[
            "R4_COMPOUND"
        ]["rho_gain_over_persistence"] > 0.0
    ),
    "novelty_sensor_discriminative": bool(
        novelty_auc >= 0.80
    ),
    "clock_is_major_oracle_source": bool(
        oracle_results["O3_WITH_CLOCK"]["rho"]
        - oracle_results["O3_NO_CLOCK"]["rho"]
        >= 0.10
    ),
}

for name, value in diagnostic_flags.items():
    print(
        f"{'✅' if value else '❌'} "
        f"{name}"
    )

if not diagnostic_flags[
    "validation_temporal_signal_above_shuffle_p95"
]:
    final_diagnosis = (
        "NO_ROBUST_TEMPORAL_ORDER_EVIDENCE: "
        "ordered history does not outperform the aligned shuffle null."
    )

elif (
    diagnostic_flags["r1_test_beats_persistence"]
    and not diagnostic_flags[
        "cross_regime_transfer_to_compound"
    ]
):
    final_diagnosis = (
        "IN_REGIME_TEMPORAL_LEARNING_WITH_TRANSFER_FAILURE: "
        "the temporal predictor uses ordered history in canonical R1, "
        "but the learned representation is not invariant across regimes."
    )

elif diagnostic_flags["clock_is_major_oracle_source"]:
    final_diagnosis = (
        "SCHEDULE_OR_CLOCK_DEPENDENCE: "
        "a substantial fraction of oracle predictability comes from "
        "episode position rather than transferable precursor dynamics."
    )

elif all([
    diagnostic_flags["r1_test_beats_persistence"],
    diagnostic_flags["cross_regime_transfer_to_abrupt"],
    diagnostic_flags["cross_regime_transfer_to_oscillatory"],
    diagnostic_flags["cross_regime_transfer_to_compound"],
]):
    final_diagnosis = (
        "ROBUST_CROSS_REGIME_TEMPORAL_SIGNAL: "
        "ordered history improves forecasting across all evaluated regimes. "
        "This is predictive evidence, not yet proof of causal mechanism learning."
    )

else:
    final_diagnosis = (
        "PARTIAL_TEMPORAL_INFORMATION_WITH_UNRESOLVED_REPRESENTATION_GAP."
    )

print(f"\nFinal diagnosis:\n{final_diagnosis}")


# =================================================================================================
# 19. SAVE TELEMETRY
# =================================================================================================

telemetry = {
    "milestone": "M20.6.1",
    "status": "DIAGNOSTIC_COMPLETE",
    "model": MODEL_NAME,
    "device": DEVICE,
    "seed": SEED,
    "configuration": {
        "requested_layers": REQUESTED_LAYERS,
        "valid_layers": VALID_LAYERS,
        "window_k": WINDOW_K,
        "history_length": WINDOW_K + 1,
        "horizon": HORIZON_H,
        "temporal_pca_dim": PCA_TEMPORAL_DIM,
        "novelty_pca_dim": PCA_NOVELTY_DIM,
    },
    "selected_layers": {
        "nominal_self_model_layer": BEST_SELF_MODEL_LAYER,
        "temporal_forecast_layer": BEST_FORECAST_LAYER,
    },
    "calibration_profile": calibration_profile,
    "nominal_layer_results": nominal_layer_results,
    "temporal_layer_validation_results": layer_temporal_results,
    "representation_ladder": representation_results,
    "temporal_shuffle_audit": shuffle_summary,
    "generator_oracles": oracle_results,
    "cross_regime_results": cross_regime_results,
    "capacity_results": capacity_results,
    "pooling_results": pooling_results,
    "novelty_results": novelty_results,
    "diagnostic_flags": diagnostic_flags,
    "final_diagnosis": final_diagnosis,
    "scientific_scope": (
        "This milestone audits predictive temporal information and "
        "cross-regime generalization. It does not by itself prove "
        "causal temporal mechanism learning."
    ),
}

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_builtin(telemetry),
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"\n💾 Telemetry written to: {OUTPUT_FILE}")
print("=" * 120)

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()

🔬 MILESTONE 20.6.1 — CORRECTED FAILURE DISSECTION & REPRESENTATION AUDIT
Device       : CUDA
Model        : Qwen/Qwen2.5-0.5B
Window       : 6 observations
Horizon      : 6
Global seed  : 206101

✅ Semantic partition independence verified.

⏳ Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Model loaded. Valid layers: [0, 3, 6, 9, 12, 15, 18, 21]

🔬 Extracting baseline representations...

🧪 TEST A — NOMINAL SELF-MODEL LAYER SELECTION
Layer 0   | Rho= 0.8653 | R²= 0.7625 | MAE= 0.2207
Layer 3   | Rho= 0.8206 | R²= 0.6655 | MAE= 0.2643
Layer 6   | Rho= 0.9308 | R²= 0.8677 | MAE= 0.1594
Layer 9   | Rho= 0.9609 | R²= 0.9266 | MAE= 0.1172
Layer 12  | Rho= 0.9737 | R²= 0.9488 | MAE= 0.1014
Layer 15  | Rho= 0.9722 | R²= 0.9411 | MAE= 0.1085
Layer 18  | Rho= 0.9316 | R²= 0.8656 | MAE= 0.1463
Layer 21  | Rho= 0.9421 | R²= 0.8891 | MAE= 0.1457

🏆 Selected nominal self-model layer: 12

Calibration profile:
  median    : 0.100974
  mad       : 0.053741
  mad_scale : 0.079677
  q75       : 0.156341
  q90       : 0.239605
  q95       : 0.262512
  q99       : 0.323124

🔬 Extracting independent temporal stream representations...
  Extracting R1_GRADUAL...
  Extracting R2_ABRUPT...
  Extracting R3_OSCILLATORY...
  Extracti

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 20.6.2 — FORECASTABILITY-GROUNDED LATENT-STATE BENCHMARK
#
# Goals
# -----
# Task A:
#   Forecast the persistent latent degradation state z[t + H].
#
# Task B:
#   Forecast whether a calibrated degradation event will occur anywhere in
#   the next H observations.
#
# Scientific safeguards
# ---------------------
# 1. Independent train / validation / calibration / test partitions
# 2. Randomized drift onset, direction, forcing strength and phase
# 3. Persistent latent dynamics rather than independent future innovations
# 4. Separate validation-only layer selection for point and hazard tasks
# 5. Separate error-only, activation-only, full and structured representations
# 6. Aligned temporal shuffle and reverse-order ablations
# 7. Stream-level macro metrics and stream bootstrap confidence intervals
# 8. Mean, persistence, autoregressive-oracle and Kalman baselines
# 9. Cross-regime evaluation on untouched test streams
# 10. Rejection gates: the least-bad layer is not automatically considered valid
#
# Model
# -----
# Qwen/Qwen2.5-0.5B through TransformerLens
# =================================================================================================

import gc
import json
import math
import random
import hashlib
import warnings
from collections import defaultdict

import numpy as np
import torch

from scipy.stats import spearmanr

from sklearn.decomposition import PCA
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
)
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer


# =================================================================================================
# 0. GLOBAL CONFIGURATION
# =================================================================================================

SEED = 206_200

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

REQUESTED_LAYERS = [0, 3, 6, 9, 12, 15, 18, 21]

# Temporal configuration
WINDOW_K = 7
HISTORY_LENGTH = WINDOW_K + 1
HORIZON_H = 6

# Representation configuration
TEMPORAL_PCA_DIM = 12
BATCH_SIZE = 16

# Dataset sizes
N_BASE_TRAIN = 320
N_BASE_VALIDATION = 180
N_BASE_CALIBRATION = 220

N_TEMPORAL_TRAIN_STREAMS = 20
N_TEMPORAL_VALIDATION_STREAMS = 8
N_TEST_STREAMS_PER_REGIME = 10

STREAM_LENGTH = 96

# Statistical configuration
BOOTSTRAP_REPETITIONS = 500
EPS = 1e-9

OUTPUT_FILE = "m20_6_2_latent_state_benchmark.json"

# Validation gates
MIN_POINT_RHO = 0.20
MIN_POINT_MACRO_RHO = 0.15
MIN_POINT_R2 = 0.00

MIN_HAZARD_AUROC = 0.65
MIN_HAZARD_AUPRC_GAIN = 0.05

# Generator observation noise
OBSERVATION_NOISE_STD = 0.025

TEST_REGIMES = [
    "AR1_TRANSFER",
    "MEAN_REVERTING",
    "RANDOM_WALK",
    "OSCILLATORY",
    "ABRUPT",
    "COMPOUND_NOVEL",
]

print("=" * 120)
print("🔬 MILESTONE 20.6.2 — FORECASTABILITY-GROUNDED LATENT-STATE BENCHMARK")
print("=" * 120)
print(f"Device                 : {DEVICE.upper()}")
print(f"Model                  : {MODEL_NAME}")
print(f"History length         : {HISTORY_LENGTH}")
print(f"Forecast horizon       : {HORIZON_H}")
print(f"Train streams          : {N_TEMPORAL_TRAIN_STREAMS}")
print(f"Validation streams     : {N_TEMPORAL_VALIDATION_STREAMS}")
print(f"Test streams/regime    : {N_TEST_STREAMS_PER_REGIME}")
print(f"Global seed            : {SEED}")
print("=" * 120)


# =================================================================================================
# 1. GENERAL HELPERS
# =================================================================================================

def safe_spearman(y_true, y_prediction):
    y_true = np.asarray(y_true, dtype=float)
    y_prediction = np.asarray(y_prediction, dtype=float)

    if len(y_true) < 3:
        return 0.0, 1.0

    if np.std(y_true) < EPS or np.std(y_prediction) < EPS:
        return 0.0, 1.0

    rho, p_value = spearmanr(y_true, y_prediction)

    if not np.isfinite(rho):
        return 0.0, 1.0

    return float(rho), float(p_value)


def safe_auroc(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    if len(np.unique(y_true)) < 2:
        return None

    return float(roc_auc_score(y_true, probability))


def safe_auprc(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    if len(np.unique(y_true)) < 2:
        return None

    return float(average_precision_score(y_true, probability))


def expected_calibration_error(y_true, probability, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.clip(
        np.asarray(probability, dtype=float),
        0.0,
        1.0,
    )

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for index in range(n_bins):
        lower = bin_edges[index]
        upper = bin_edges[index + 1]

        if index == n_bins - 1:
            mask = (probability >= lower) & (probability <= upper)
        else:
            mask = (probability >= lower) & (probability < upper)

        if not np.any(mask):
            continue

        confidence = float(np.mean(probability[mask]))
        accuracy = float(np.mean(y_true[mask]))
        weight = float(np.mean(mask))

        ece += weight * abs(confidence - accuracy)

    return float(ece)


def point_metrics(y_true, y_prediction):
    y_true = np.asarray(y_true, dtype=float)
    y_prediction = np.asarray(y_prediction, dtype=float)

    rho, p_value = safe_spearman(y_true, y_prediction)

    return {
        "rho": rho,
        "p_value": p_value,
        "r2": float(r2_score(y_true, y_prediction)),
        "mae": float(mean_absolute_error(y_true, y_prediction)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_prediction))),
    }


def hazard_metrics(y_true, probability, decision_threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.clip(
        np.asarray(probability, dtype=float),
        0.0,
        1.0,
    )

    prediction = (probability >= decision_threshold).astype(int)

    auroc = safe_auroc(y_true, probability)
    auprc = safe_auprc(y_true, probability)

    true_negative = int(np.sum((y_true == 0) & (prediction == 0)))
    false_positive = int(np.sum((y_true == 0) & (prediction == 1)))

    fpr_denominator = true_negative + false_positive

    return {
        "auroc": auroc,
        "auprc": auprc,
        "brier": float(brier_score_loss(y_true, probability)),
        "ece": expected_calibration_error(y_true, probability),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
        "precision": float(
            precision_score(y_true, prediction, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, prediction, zero_division=0)
        ),
        "false_positive_rate": (
            float(false_positive / fpr_denominator)
            if fpr_denominator > 0
            else 0.0
        ),
        "prevalence": float(np.mean(y_true)),
        "decision_threshold": float(decision_threshold),
    }


def mean_finite(values, default=0.0):
    valid = [
        float(value)
        for value in values
        if value is not None and np.isfinite(value)
    ]

    return float(np.mean(valid)) if valid else float(default)


def to_builtin(obj):
    if isinstance(obj, dict):
        return {str(key): to_builtin(value) for key, value in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [to_builtin(value) for value in obj]

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, (np.floating, np.integer, np.bool_)):
        return obj.item()

    return obj


def flatten_streams(streams):
    return [row for stream in streams for row in stream]


def split_flat_array(array, lengths):
    pieces = []
    offset = 0

    for length in lengths:
        pieces.append(array[offset:offset + length])
        offset += length

    if offset != len(array):
        raise RuntimeError(
            f"Split mismatch: consumed {offset}, received {len(array)}."
        )

    return pieces


# =================================================================================================
# 2. DATASET INDEPENDENCE
# =================================================================================================

def semantic_fingerprint(row):
    payload = (
        f"{row.get('t_step')}|"
        f"{row['a']:.12f}|"
        f"{row['b']:.12f}|"
        f"{row['nominal_mechanism']:.12f}|"
        f"{row['mechanism']:.12f}|"
        f"{row['latent_z']:.12f}|"
        f"{row['sensitivity']:.12f}|"
        f"{row['context_type']}|"
        f"{row.get('regime', 'BASE')}"
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def assert_partition_disjoint(name_a, rows_a, name_b, rows_b):
    set_a = {semantic_fingerprint(row) for row in rows_a}
    set_b = {semantic_fingerprint(row) for row in rows_b}

    overlap = set_a.intersection(set_b)

    if overlap:
        raise RuntimeError(
            f"Semantic collision between {name_a} and {name_b}: "
            f"{len(overlap)} duplicated rows."
        )


# =================================================================================================
# 3. PERSISTENT LATENT-STATE GENERATOR
# =================================================================================================

def nominal_function(a, b):
    return float(0.80 * a - 0.55 * b)


def latent_sensitivity(a, b):
    """
    The latent state changes the causal coefficients.

    mechanism =
        nominal(a, b)
        + z * sensitivity(a, b)
        + observation noise
    """
    return float(0.45 * a - 0.25 * b)


def make_base_row(split, index, rng):
    a = rng.uniform(-1.0, 1.0)
    b = rng.uniform(-1.0, 1.0)

    nominal = nominal_function(a, b)

    row = {
        "split": split,
        "idx": int(index),
        "stream_id": None,
        "regime": "BASE_NOMINAL",
        "t_step": None,
        "a": float(a),
        "b": float(b),
        "latent_z": 0.0,
        "sensitivity": latent_sensitivity(a, b),
        "nominal_mechanism": nominal,
        "mechanism": nominal,
        "observation_noise": 0.0,
        "context_type": "STANDARD",
        "drift_start": None,
        "drift_direction": 0.0,
        "forcing_strength": 0.0,
        "phi": 0.0,
    }

    row["fingerprint"] = semantic_fingerprint(row)
    return row


def generate_base_rows(split, n_rows, seed):
    rng = np.random.RandomState(seed)

    return [
        make_base_row(split, index, rng)
        for index in range(n_rows)
    ]


def generate_latent_stream(
    regime,
    stream_id,
    seed,
    n_steps=STREAM_LENGTH,
):
    rng = np.random.RandomState(seed)

    drift_start = int(rng.randint(24, 42))
    ramp_duration = int(rng.randint(18, 38))
    direction = float(rng.choice([-1.0, 1.0]))
    forcing_strength = float(rng.uniform(0.018, 0.045))
    phase = float(rng.uniform(0.0, 2.0 * np.pi))
    equilibrium_scale = float(rng.uniform(0.55, 1.10))
    shock_size = float(rng.uniform(0.75, 1.45)) * direction

    latent_z = float(rng.normal(0.0, 0.04))
    stream = []

    for t in range(n_steps):
        innovation = float(rng.normal(0.0, 0.035))

        ramp_fraction = np.clip(
            (t - drift_start) / max(1, ramp_duration),
            0.0,
            1.0,
        )

        if regime in ("AR1_TRAIN", "AR1_VALIDATION", "AR1_TRANSFER"):
            phi = float(rng.uniform(0.89, 0.95))

            forcing = (
                direction * forcing_strength * ramp_fraction
            )

            latent_z = (
                phi * latent_z
                + forcing
                + innovation
            )

        elif regime == "MEAN_REVERTING":
            phi = 0.82

            target = (
                direction
                * equilibrium_scale
                * ramp_fraction
            )

            latent_z = (
                latent_z
                + 0.18 * (target - latent_z)
                + innovation
            )

        elif regime == "RANDOM_WALK":
            phi = 1.0

            forcing = (
                direction
                * 0.45
                * forcing_strength
                * ramp_fraction
            )

            latent_z = (
                latent_z
                + forcing
                + innovation
            )

        elif regime == "OSCILLATORY":
            phi = 0.90

            oscillatory_drive = (
                0.070
                * math.sin(
                    0.22 * t + phase
                )
            )

            drift_drive = (
                direction
                * 0.35
                * forcing_strength
                * ramp_fraction
            )

            latent_z = (
                phi * latent_z
                + oscillatory_drive
                + drift_drive
                + innovation
            )

        elif regime == "ABRUPT":
            phi = 0.94

            if t == drift_start:
                latent_z += shock_size

            latent_z = (
                phi * latent_z
                + innovation
            )

        elif regime == "COMPOUND_NOVEL":
            phi = 0.91

            forcing = (
                direction
                * forcing_strength
                * ramp_fraction
            )

            oscillation = (
                0.045
                * math.sin(
                    0.18 * t + phase
                )
            )

            latent_z = (
                phi * latent_z
                + forcing
                + oscillation
                + innovation
            )

            if t == drift_start + ramp_duration // 2:
                latent_z += 0.45 * shock_size

        else:
            raise ValueError(f"Unknown regime: {regime}")

        latent_z = float(np.clip(latent_z, -2.75, 2.75))

        a = float(rng.uniform(-1.0, 1.0))
        b = float(rng.uniform(-1.0, 1.0))

        nominal = nominal_function(a, b)
        sensitivity = latent_sensitivity(a, b)

        observation_noise = float(
            rng.normal(0.0, OBSERVATION_NOISE_STD)
        )

        mechanism = float(
            nominal
            + latent_z * sensitivity
            + observation_noise
        )

        context_type = "STANDARD"

        if regime == "COMPOUND_NOVEL" and t >= drift_start:
            context_type = "NOVEL_FORMAT"

        row = {
            "split": regime,
            "idx": int(t),
            "stream_id": stream_id,
            "regime": regime,
            "t_step": int(t),
            "a": a,
            "b": b,
            "latent_z": latent_z,
            "sensitivity": sensitivity,
            "nominal_mechanism": nominal,
            "mechanism": mechanism,
            "observation_noise": observation_noise,
            "context_type": context_type,
            "drift_start": drift_start,
            "ramp_duration": ramp_duration,
            "drift_direction": direction,
            "forcing_strength": forcing_strength,
            "phi": phi,
        }

        row["fingerprint"] = semantic_fingerprint(row)
        stream.append(row)

    return stream


# =================================================================================================
# 4. CREATE INDEPENDENT PARTITIONS
# =================================================================================================

base_train = generate_base_rows(
    "BASE_TRAIN",
    N_BASE_TRAIN,
    SEED + 10_000,
)

base_validation = generate_base_rows(
    "BASE_VALIDATION",
    N_BASE_VALIDATION,
    SEED + 20_000,
)

base_calibration = generate_base_rows(
    "BASE_CALIBRATION",
    N_BASE_CALIBRATION,
    SEED + 30_000,
)

temporal_train_streams = [
    generate_latent_stream(
        "AR1_TRAIN",
        f"TRAIN_{index}",
        SEED + 40_000 + 101 * index,
    )
    for index in range(N_TEMPORAL_TRAIN_STREAMS)
]

temporal_validation_streams = [
    generate_latent_stream(
        "AR1_VALIDATION",
        f"VALIDATION_{index}",
        SEED + 50_000 + 103 * index,
    )
    for index in range(N_TEMPORAL_VALIDATION_STREAMS)
]

test_streams = {
    regime: [
        generate_latent_stream(
            regime,
            f"TEST_{regime}_{index}",
            SEED
            + 60_000
            + 10_000 * regime_index
            + 107 * index,
        )
        for index in range(N_TEST_STREAMS_PER_REGIME)
    ]
    for regime_index, regime in enumerate(TEST_REGIMES)
}

partitions = {
    "base_train": base_train,
    "base_validation": base_validation,
    "base_calibration": base_calibration,
    "temporal_train": flatten_streams(temporal_train_streams),
    "temporal_validation": flatten_streams(
        temporal_validation_streams
    ),
}

for regime, streams in test_streams.items():
    partitions[f"test_{regime}"] = flatten_streams(streams)

partition_names = list(partitions.keys())

for index, name_a in enumerate(partition_names):
    for name_b in partition_names[index + 1:]:
        assert_partition_disjoint(
            name_a,
            partitions[name_a],
            name_b,
            partitions[name_b],
        )

print("\n✅ All semantic partitions are independent.")


# =================================================================================================
# 5. LOAD MODEL
# =================================================================================================

print("\n⏳ Loading TransformerLens model...")

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device=DEVICE,
)

model.eval()

available_hooks = set(model.hook_dict.keys())

layer_hooks = {}

for layer in REQUESTED_LAYERS:
    hook_name = f"blocks.{layer}.hook_resid_post"

    if hook_name in available_hooks:
        layer_hooks[layer] = hook_name

VALID_LAYERS = sorted(layer_hooks.keys())

if not VALID_LAYERS:
    raise RuntimeError("No requested residual-post hooks were found.")

print(f"✅ Model loaded. Valid layers: {VALID_LAYERS}")


# =================================================================================================
# 6. PROMPTS AND ACTIVATION EXTRACTION
# =================================================================================================

def make_prompt(row):
    if row["context_type"] == "NOVEL_FORMAT":
        return (
            "<<EXPERIMENTAL COORDINATE PROTOCOL>>\n"
            f"Alpha-coordinate: {row['a']:.6f}\n"
            f"Beta-coordinate: {row['b']:.6f}\n"
            "Compute the latent numerical transformation output."
        )

    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.6f}\n"
        f"Input B = {row['b']:.6f}\n"
        "Infer the relationship between the quantities.\n"
        "Provide the predicted resulting value."
    )


def valid_token_mask(tokens):
    pad_token_id = model.tokenizer.pad_token_id

    if pad_token_id is None:
        return torch.ones_like(tokens, dtype=torch.bool)

    return tokens.ne(pad_token_id)


def padding_aware_mean_pool(activation, token_mask):
    mask = token_mask.unsqueeze(-1).to(activation.dtype)

    denominator = mask.sum(dim=1).clamp_min(1.0)

    return (
        (activation * mask).sum(dim=1)
        / denominator
    )


@torch.no_grad()
def extract_activations(
    rows,
    layers,
    batch_size=BATCH_SIZE,
):
    output = {
        layer: []
        for layer in layers
    }

    requested_hooks = {
        layer_hooks[layer]
        for layer in layers
    }

    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(row) for row in batch]

        tokens = model.to_tokens(
            prompts,
            prepend_bos=True,
        ).to(DEVICE)

        token_mask = valid_token_mask(tokens)

        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name in requested_hooks,
            return_type=None,
        )

        for layer in layers:
            activation = cache[
                layer_hooks[layer]
            ].detach().float()

            pooled = padding_aware_mean_pool(
                activation,
                token_mask,
            )

            output[layer].append(
                pooled.cpu().numpy()
            )

        del cache, tokens, token_mask

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return {
        layer: np.concatenate(output[layer], axis=0)
        for layer in layers
    }


# =================================================================================================
# 7. NOMINAL SELF-MODEL
# =================================================================================================

print("\n🔬 Extracting nominal representations...")

base_train_activations = extract_activations(
    base_train,
    VALID_LAYERS,
)

base_validation_activations = extract_activations(
    base_validation,
    VALID_LAYERS,
)

base_calibration_activations = extract_activations(
    base_calibration,
    VALID_LAYERS,
)

base_train_target = np.asarray(
    [row["mechanism"] for row in base_train],
    dtype=float,
)

base_validation_target = np.asarray(
    [row["mechanism"] for row in base_validation],
    dtype=float,
)

base_calibration_target = np.asarray(
    [row["mechanism"] for row in base_calibration],
    dtype=float,
)

print("\n" + "=" * 120)
print("🧪 TEST A — NOMINAL SELF-MODEL SELECTION")
print("=" * 120)

nominal_results = {}
nominal_models = {}

for layer in VALID_LAYERS:
    scaler = StandardScaler()
    regressor = RidgeCV(
        alphas=np.logspace(-4, 4, 25)
    )

    X_train = scaler.fit_transform(
        base_train_activations[layer]
    )

    regressor.fit(
        X_train,
        base_train_target,
    )

    validation_prediction = regressor.predict(
        scaler.transform(
            base_validation_activations[layer]
        )
    )

    metrics = point_metrics(
        base_validation_target,
        validation_prediction,
    )

    nominal_results[layer] = metrics
    nominal_models[layer] = {
        "scaler": scaler,
        "regressor": regressor,
    }

    print(
        f"Layer {layer:<3} | "
        f"rho={metrics['rho']: .4f} | "
        f"R²={metrics['r2']: .4f} | "
        f"MAE={metrics['mae']: .4f}"
    )

BEST_SELF_MODEL_LAYER = max(
    nominal_results,
    key=lambda layer: nominal_results[layer]["r2"],
)

self_scaler = nominal_models[
    BEST_SELF_MODEL_LAYER
]["scaler"]

self_regressor = nominal_models[
    BEST_SELF_MODEL_LAYER
]["regressor"]

print(
    f"\n🏆 Nominal self-model layer: "
    f"{BEST_SELF_MODEL_LAYER}"
)

calibration_prediction = self_regressor.predict(
    self_scaler.transform(
        base_calibration_activations[
            BEST_SELF_MODEL_LAYER
        ]
    )
)

calibration_signed_error = (
    base_calibration_target
    - calibration_prediction
)

calibration_absolute_error = np.abs(
    calibration_signed_error
)

calibration_median = float(
    np.median(calibration_absolute_error)
)

calibration_mad = float(
    np.median(
        np.abs(
            calibration_absolute_error
            - calibration_median
        )
    )
)

# A stringent instantaneous event threshold.
HAZARD_RESIDUAL_THRESHOLD = float(
    max(
        np.quantile(calibration_absolute_error, 0.95),
        calibration_median
        + 3.0 * 1.4826 * calibration_mad,
    )
)

calibration_profile = {
    "median": calibration_median,
    "mad": calibration_mad,
    "q90": float(
        np.quantile(calibration_absolute_error, 0.90)
    ),
    "q95": float(
        np.quantile(calibration_absolute_error, 0.95)
    ),
    "q99": float(
        np.quantile(calibration_absolute_error, 0.99)
    ),
    "hazard_threshold": HAZARD_RESIDUAL_THRESHOLD,
}

print("\nCalibration profile:")

for key, value in calibration_profile.items():
    print(f"  {key:<18}: {value:.6f}")


# =================================================================================================
# 8. EXTRACT TEMPORAL GROUPS
# =================================================================================================

def extract_stream_group(streams):
    rows = flatten_streams(streams)

    activations = extract_activations(
        rows,
        VALID_LAYERS,
    )

    return {
        "streams": streams,
        "rows": rows,
        "lengths": [len(stream) for stream in streams],
        "activations": activations,
    }


print("\n🔬 Extracting temporal representations...")

train_group = extract_stream_group(
    temporal_train_streams
)

validation_group = extract_stream_group(
    temporal_validation_streams
)

test_groups = {}

for regime in TEST_REGIMES:
    print(f"  Extracting {regime}...")
    test_groups[regime] = extract_stream_group(
        test_streams[regime]
    )


# =================================================================================================
# 9. ATTACH FROZEN SELF-MODEL ERRORS
# =================================================================================================

def attach_self_model_outputs(group):
    X_self = group["activations"][
        BEST_SELF_MODEL_LAYER
    ]

    true_mechanism = np.asarray(
        [row["mechanism"] for row in group["rows"]],
        dtype=float,
    )

    self_prediction = self_regressor.predict(
        self_scaler.transform(X_self)
    )

    signed_error = (
        true_mechanism
        - self_prediction
    )

    group["true_mechanism"] = true_mechanism
    group["self_prediction"] = self_prediction
    group["signed_error"] = signed_error
    group["absolute_error"] = np.abs(signed_error)
    group["latent_z"] = np.asarray(
        [row["latent_z"] for row in group["rows"]],
        dtype=float,
    )


attach_self_model_outputs(train_group)
attach_self_model_outputs(validation_group)

for group in test_groups.values():
    attach_self_model_outputs(group)


# =================================================================================================
# 10. ACTIVATION ENCODER
# =================================================================================================

class ActivationEncoder:
    def __init__(self, n_components, seed=SEED):
        self.n_components = int(n_components)
        self.seed = int(seed)
        self.scaler = StandardScaler()
        self.pca = None

    def fit(self, X):
        X_scaled = self.scaler.fit_transform(X)

        components = min(
            self.n_components,
            X_scaled.shape[0] - 1,
            X_scaled.shape[1],
        )

        if components < 1:
            raise RuntimeError("Insufficient samples for PCA.")

        self.pca = PCA(
            n_components=components,
            random_state=self.seed,
        )

        self.pca.fit(X_scaled)
        return self

    def transform(self, X):
        return self.pca.transform(
            self.scaler.transform(X)
        )

    @property
    def output_dim(self):
        return int(self.pca.n_components_)


# =================================================================================================
# 11. TEMPORAL DATASET CONSTRUCTION
# =================================================================================================

def build_temporal_dataset(
    group,
    layer,
    encoder,
):
    encoded_activation = encoder.transform(
        group["activations"][layer]
    )

    activation_streams = split_flat_array(
        encoded_activation,
        group["lengths"],
    )

    signed_error_streams = split_flat_array(
        group["signed_error"],
        group["lengths"],
    )

    absolute_error_streams = split_flat_array(
        group["absolute_error"],
        group["lengths"],
    )

    latent_streams = split_flat_array(
        group["latent_z"],
        group["lengths"],
    )

    X_error = []
    X_activation = []
    X_full = []
    X_structured = []

    point_target = []
    hazard_target = []

    stream_ids = []
    times = []
    future_event_times = []

    for stream_index, stream in enumerate(group["streams"]):
        encoded = activation_streams[stream_index]
        signed_error = signed_error_streams[stream_index]
        absolute_error = absolute_error_streams[stream_index]
        latent_z = latent_streams[stream_index]

        for t in range(
            WINDOW_K,
            len(stream) - HORIZON_H,
        ):
            start = t - WINDOW_K
            stop = t + 1

            signed_history = signed_error[start:stop]
            absolute_history = absolute_error[start:stop]
            activation_history = encoded[start:stop]

            error_history = np.column_stack([
                signed_history,
                absolute_history,
            ])

            structured_history = np.asarray([
                [
                    stream[index]["a"],
                    stream[index]["b"],
                    signed_error[index],
                    absolute_error[index],
                ]
                for index in range(start, stop)
            ], dtype=float)

            future_absolute_error = absolute_error[
                t + 1:t + HORIZON_H + 1
            ]

            future_event_mask = (
                future_absolute_error
                > HAZARD_RESIDUAL_THRESHOLD
            )

            if np.any(future_event_mask):
                first_offset = int(
                    np.argmax(future_event_mask)
                ) + 1
                event_time = t + first_offset
            else:
                event_time = -1

            X_error.append(
                error_history.flatten()
            )

            X_activation.append(
                activation_history.flatten()
            )

            X_full.append(
                np.concatenate([
                    error_history.flatten(),
                    activation_history.flatten(),
                ])
            )

            X_structured.append(
                structured_history.flatten()
            )

            point_target.append(
                latent_z[t + HORIZON_H]
            )

            hazard_target.append(
                int(np.any(future_event_mask))
            )

            stream_ids.append(stream_index)
            times.append(t)
            future_event_times.append(event_time)

    return {
        "X_error": np.asarray(X_error, dtype=float),
        "X_activation": np.asarray(X_activation, dtype=float),
        "X_full": np.asarray(X_full, dtype=float),
        "X_structured": np.asarray(X_structured, dtype=float),
        "y_point": np.asarray(point_target, dtype=float),
        "y_hazard": np.asarray(hazard_target, dtype=int),
        "stream_ids": np.asarray(stream_ids, dtype=int),
        "times": np.asarray(times, dtype=int),
        "future_event_times": np.asarray(
            future_event_times,
            dtype=int,
        ),
        "activation_dim": encoder.output_dim,
        "error_dim_per_step": 2,
        "structured_dim_per_step": 4,
    }


# =================================================================================================
# 12. MODEL HELPERS
# =================================================================================================

def fit_point_model(X, y, seed=SEED):
    model = GradientBoostingRegressor(
        n_estimators=180,
        learning_rate=0.035,
        max_depth=3,
        min_samples_leaf=5,
        loss="huber",
        random_state=seed,
    )

    model.fit(X, y)
    return model


def fit_hazard_model(X, y, seed=SEED):
    if len(np.unique(y)) < 2:
        raise RuntimeError(
            "Hazard training labels contain only one class."
        )

    model = GradientBoostingClassifier(
        n_estimators=180,
        learning_rate=0.035,
        max_depth=3,
        min_samples_leaf=5,
        subsample=0.9,
        random_state=seed,
    )

    model.fit(X, y)
    return model


def get_positive_probability(model, X):
    probabilities = model.predict_proba(X)
    classes = list(model.classes_)

    if 1 not in classes:
        return np.zeros(len(X), dtype=float)

    positive_index = classes.index(1)
    return probabilities[:, positive_index]


def select_f1_threshold(y_true, probability):
    candidates = np.linspace(0.05, 0.95, 181)

    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in candidates:
        prediction = (
            probability >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            prediction,
            zero_division=0,
        )

        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_f1


# =================================================================================================
# 13. STREAM-MACRO METRICS
# =================================================================================================

def macro_point_metrics(
    y_true,
    y_prediction,
    stream_ids,
):
    per_stream_rho = []
    per_stream_r2 = []
    per_stream_mae = []

    for stream_id in np.unique(stream_ids):
        mask = stream_ids == stream_id

        if np.sum(mask) < 3:
            continue

        rho, _ = safe_spearman(
            y_true[mask],
            y_prediction[mask],
        )

        per_stream_rho.append(rho)
        per_stream_r2.append(
            r2_score(
                y_true[mask],
                y_prediction[mask],
            )
        )
        per_stream_mae.append(
            mean_absolute_error(
                y_true[mask],
                y_prediction[mask],
            )
        )

    return {
        "macro_stream_rho": mean_finite(per_stream_rho),
        "macro_stream_r2": mean_finite(per_stream_r2),
        "macro_stream_mae": mean_finite(per_stream_mae),
        "n_streams": int(
            len(np.unique(stream_ids))
        ),
    }


def macro_hazard_metrics(
    y_true,
    probability,
    stream_ids,
):
    stream_aurocs = []
    stream_auprcs = []
    stream_briers = []

    for stream_id in np.unique(stream_ids):
        mask = stream_ids == stream_id

        stream_auc = safe_auroc(
            y_true[mask],
            probability[mask],
        )

        stream_ap = safe_auprc(
            y_true[mask],
            probability[mask],
        )

        if stream_auc is not None:
            stream_aurocs.append(stream_auc)

        if stream_ap is not None:
            stream_auprcs.append(stream_ap)

        stream_briers.append(
            brier_score_loss(
                y_true[mask],
                probability[mask],
            )
        )

    return {
        "macro_stream_auroc": mean_finite(
            stream_aurocs
        ),
        "macro_stream_auprc": mean_finite(
            stream_auprcs
        ),
        "macro_stream_brier": mean_finite(
            stream_briers
        ),
        "n_valid_auc_streams": int(
            len(stream_aurocs)
        ),
    }


def evaluate_point(
    y_true,
    prediction,
    stream_ids,
):
    output = point_metrics(
        y_true,
        prediction,
    )

    output.update(
        macro_point_metrics(
            y_true,
            prediction,
            stream_ids,
        )
    )

    return output


def evaluate_hazard(
    y_true,
    probability,
    stream_ids,
    decision_threshold,
):
    output = hazard_metrics(
        y_true,
        probability,
        decision_threshold,
    )

    output.update(
        macro_hazard_metrics(
            y_true,
            probability,
            stream_ids,
        )
    )

    return output


# =================================================================================================
# 14. STREAM BOOTSTRAP CONFIDENCE INTERVALS
# =================================================================================================

def stream_bootstrap_point_ci(
    y_true,
    prediction,
    stream_ids,
    repetitions=BOOTSTRAP_REPETITIONS,
    seed=SEED,
):
    rng = np.random.RandomState(seed)
    unique_streams = np.unique(stream_ids)

    bootstrap_rho = []
    bootstrap_r2 = []

    for _ in range(repetitions):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        sampled_true = []
        sampled_prediction = []

        for stream_id in sampled_streams:
            mask = stream_ids == stream_id
            sampled_true.append(y_true[mask])
            sampled_prediction.append(prediction[mask])

        y_boot = np.concatenate(sampled_true)
        prediction_boot = np.concatenate(
            sampled_prediction
        )

        rho, _ = safe_spearman(
            y_boot,
            prediction_boot,
        )

        bootstrap_rho.append(rho)
        bootstrap_r2.append(
            r2_score(
                y_boot,
                prediction_boot,
            )
        )

    return {
        "rho_ci95": [
            float(np.quantile(bootstrap_rho, 0.025)),
            float(np.quantile(bootstrap_rho, 0.975)),
        ],
        "r2_ci95": [
            float(np.quantile(bootstrap_r2, 0.025)),
            float(np.quantile(bootstrap_r2, 0.975)),
        ],
    }


def stream_bootstrap_hazard_ci(
    y_true,
    probability,
    stream_ids,
    repetitions=BOOTSTRAP_REPETITIONS,
    seed=SEED,
):
    rng = np.random.RandomState(seed)
    unique_streams = np.unique(stream_ids)

    bootstrap_auc = []
    bootstrap_ap = []

    for _ in range(repetitions):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        sampled_true = []
        sampled_probability = []

        for stream_id in sampled_streams:
            mask = stream_ids == stream_id
            sampled_true.append(y_true[mask])
            sampled_probability.append(
                probability[mask]
            )

        y_boot = np.concatenate(sampled_true)
        probability_boot = np.concatenate(
            sampled_probability
        )

        auc = safe_auroc(
            y_boot,
            probability_boot,
        )

        ap = safe_auprc(
            y_boot,
            probability_boot,
        )

        if auc is not None:
            bootstrap_auc.append(auc)

        if ap is not None:
            bootstrap_ap.append(ap)

    return {
        "auroc_ci95": (
            [
                float(np.quantile(bootstrap_auc, 0.025)),
                float(np.quantile(bootstrap_auc, 0.975)),
            ]
            if bootstrap_auc
            else [None, None]
        ),
        "auprc_ci95": (
            [
                float(np.quantile(bootstrap_ap, 0.025)),
                float(np.quantile(bootstrap_ap, 0.975)),
            ]
            if bootstrap_ap
            else [None, None]
        ),
    }


# =================================================================================================
# 15. LAYER SELECTION ON VALIDATION ONLY
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST B — VALIDATION-ONLY TEMPORAL LAYER SELECTION")
print("=" * 120)

layer_results = {}
layer_artifacts = {}

for layer in VALID_LAYERS:
    encoder = ActivationEncoder(
        TEMPORAL_PCA_DIM,
        seed=SEED + layer,
    ).fit(
        train_group["activations"][layer]
    )

    train_dataset = build_temporal_dataset(
        train_group,
        layer,
        encoder,
    )

    validation_dataset = build_temporal_dataset(
        validation_group,
        layer,
        encoder,
    )

    point_model = fit_point_model(
        train_dataset["X_full"],
        train_dataset["y_point"],
        seed=SEED + layer,
    )

    point_prediction = point_model.predict(
        validation_dataset["X_full"]
    )

    point_result = evaluate_point(
        validation_dataset["y_point"],
        point_prediction,
        validation_dataset["stream_ids"],
    )

    hazard_model = fit_hazard_model(
        train_dataset["X_full"],
        train_dataset["y_hazard"],
        seed=SEED + 1000 + layer,
    )

    hazard_probability = get_positive_probability(
        hazard_model,
        validation_dataset["X_full"],
    )

    hazard_result = evaluate_hazard(
        validation_dataset["y_hazard"],
        hazard_probability,
        validation_dataset["stream_ids"],
        decision_threshold=0.5,
    )

    prevalence = float(
        np.mean(
            validation_dataset["y_hazard"]
        )
    )

    auprc_gain = (
        (hazard_result["auprc"] or 0.0)
        - prevalence
    )

    hazard_result["auprc_gain_over_prevalence"] = float(
        auprc_gain
    )

    layer_results[layer] = {
        "point": point_result,
        "hazard": hazard_result,
    }

    layer_artifacts[layer] = {
        "encoder": encoder,
        "train_dataset": train_dataset,
        "validation_dataset": validation_dataset,
        "point_model": point_model,
        "hazard_model": hazard_model,
    }

    print(
        f"Layer {layer:<3} | "
        f"Point rho={point_result['rho']: .4f} | "
        f"Macro={point_result['macro_stream_rho']: .4f} | "
        f"R²={point_result['r2']: .4f} || "
        f"Hazard AUC={(hazard_result['auroc'] or 0.0): .4f} | "
        f"AP={(hazard_result['auprc'] or 0.0): .4f} | "
        f"ΔAP={auprc_gain: .4f}"
    )


point_candidate_layer = max(
    VALID_LAYERS,
    key=lambda layer: (
        layer_results[layer]["point"]["macro_stream_rho"],
        layer_results[layer]["point"]["rho"],
        layer_results[layer]["point"]["r2"],
    ),
)

hazard_candidate_layer = max(
    VALID_LAYERS,
    key=lambda layer: (
        layer_results[layer]["hazard"]["auroc"] or -1.0,
        layer_results[layer]["hazard"][
            "macro_stream_auroc"
        ],
        layer_results[layer]["hazard"]["auprc"] or -1.0,
    ),
)

point_candidate_metrics = layer_results[
    point_candidate_layer
]["point"]

hazard_candidate_metrics = layer_results[
    hazard_candidate_layer
]["hazard"]

POINT_LAYER_VALID = bool(
    point_candidate_metrics["rho"] >= MIN_POINT_RHO
    and point_candidate_metrics["macro_stream_rho"]
    >= MIN_POINT_MACRO_RHO
    and point_candidate_metrics["r2"] > MIN_POINT_R2
)

HAZARD_LAYER_VALID = bool(
    (hazard_candidate_metrics["auroc"] or 0.0)
    >= MIN_HAZARD_AUROC
    and hazard_candidate_metrics[
        "auprc_gain_over_prevalence"
    ] >= MIN_HAZARD_AUPRC_GAIN
)

BEST_POINT_LAYER = point_candidate_layer
BEST_HAZARD_LAYER = hazard_candidate_layer

print(
    f"\nPoint candidate layer  : {BEST_POINT_LAYER} "
    f"({'VALID' if POINT_LAYER_VALID else 'REJECTED BY GATE'})"
)

print(
    f"Hazard candidate layer : {BEST_HAZARD_LAYER} "
    f"({'VALID' if HAZARD_LAYER_VALID else 'REJECTED BY GATE'})"
)


# =================================================================================================
# 16. REPRESENTATION LADDER
# =================================================================================================

def evaluate_representation_ladder(
    artifacts,
    task_name,
):
    train_dataset = artifacts["train_dataset"]
    validation_dataset = artifacts["validation_dataset"]

    feature_names = [
        "ERROR_HISTORY",
        "ACTIVATION_HISTORY",
        "ERROR_PLUS_ACTIVATION",
        "STRUCTURED_SENSOR",
    ]

    feature_map = {
        "ERROR_HISTORY": "X_error",
        "ACTIVATION_HISTORY": "X_activation",
        "ERROR_PLUS_ACTIVATION": "X_full",
        "STRUCTURED_SENSOR": "X_structured",
    }

    results = {}
    models = {}

    for index, representation in enumerate(feature_names):
        feature_key = feature_map[representation]

        X_train = train_dataset[feature_key]
        X_validation = validation_dataset[feature_key]

        if task_name == "point":
            model_variant = fit_point_model(
                X_train,
                train_dataset["y_point"],
                seed=SEED + 10_000 + index,
            )

            prediction = model_variant.predict(
                X_validation
            )

            metrics = evaluate_point(
                validation_dataset["y_point"],
                prediction,
                validation_dataset["stream_ids"],
            )

        elif task_name == "hazard":
            model_variant = fit_hazard_model(
                X_train,
                train_dataset["y_hazard"],
                seed=SEED + 20_000 + index,
            )

            probability = get_positive_probability(
                model_variant,
                X_validation,
            )

            metrics = evaluate_hazard(
                validation_dataset["y_hazard"],
                probability,
                validation_dataset["stream_ids"],
                decision_threshold=0.5,
            )

            metrics["auprc_gain_over_prevalence"] = float(
                (metrics["auprc"] or 0.0)
                - metrics["prevalence"]
            )

        else:
            raise ValueError(task_name)

        results[representation] = metrics
        models[representation] = model_variant

    return results, models


print("\n" + "=" * 120)
print("🧪 TEST C — REPRESENTATION LADDERS")
print("=" * 120)

point_artifacts = layer_artifacts[
    BEST_POINT_LAYER
]

hazard_artifacts = layer_artifacts[
    BEST_HAZARD_LAYER
]

point_representation_results, point_representation_models = (
    evaluate_representation_ladder(
        point_artifacts,
        "point",
    )
)

hazard_representation_results, hazard_representation_models = (
    evaluate_representation_ladder(
        hazard_artifacts,
        "hazard",
    )
)

print("\nPoint forecasting:")

for name, metrics in point_representation_results.items():
    print(
        f"  {name:<25} | "
        f"rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )

print("\nHazard forecasting:")

for name, metrics in hazard_representation_results.items():
    print(
        f"  {name:<25} | "
        f"AUC={(metrics['auroc'] or 0.0): .4f} | "
        f"AP={(metrics['auprc'] or 0.0): .4f} | "
        f"Brier={metrics['brier']: .4f}"
    )


# =================================================================================================
# 17. ORDER ABLATIONS
# =================================================================================================

def reorder_temporal_matrix(
    X,
    block_dimensions,
    mode,
    seed,
):
    """
    block_dimensions describes concatenated temporal blocks.

    Example for X_full:
      error block      : HISTORY_LENGTH x 2
      activation block : HISTORY_LENGTH x PCA_DIM

    The same permutation is applied to every block so temporal alignment
    between error and activation remains intact.
    """
    X = np.asarray(X, dtype=float)
    output = np.empty_like(X)

    rng = np.random.RandomState(seed)

    for row_index in range(len(X)):
        if mode == "shuffle":
            permutation = rng.permutation(
                HISTORY_LENGTH
            )
        elif mode == "reverse":
            permutation = np.arange(
                HISTORY_LENGTH - 1,
                -1,
                -1,
            )
        else:
            raise ValueError(mode)

        offset = 0
        reordered_blocks = []

        for dimension in block_dimensions:
            block_size = HISTORY_LENGTH * dimension

            block = X[
                row_index,
                offset:offset + block_size
            ].reshape(
                HISTORY_LENGTH,
                dimension,
            )

            reordered_blocks.append(
                block[permutation].reshape(-1)
            )

            offset += block_size

        if offset != X.shape[1]:
            raise RuntimeError(
                f"Temporal layout mismatch: consumed {offset}, "
                f"matrix has {X.shape[1]} features."
            )

        output[row_index] = np.concatenate(
            reordered_blocks
        )

    return output


def order_ablation_point(
    model,
    dataset,
    feature_key,
    block_dimensions,
    repetitions=100,
):
    X_ordered = dataset[feature_key]
    y = dataset["y_point"]

    ordered_prediction = model.predict(X_ordered)

    ordered_metrics = evaluate_point(
        y,
        ordered_prediction,
        dataset["stream_ids"],
    )

    X_reverse = reorder_temporal_matrix(
        X_ordered,
        block_dimensions,
        mode="reverse",
        seed=SEED,
    )

    reverse_prediction = model.predict(X_reverse)
    reverse_metrics = point_metrics(
        y,
        reverse_prediction,
    )

    shuffle_rhos = []
    shuffle_r2s = []

    for repetition in range(repetitions):
        X_shuffle = reorder_temporal_matrix(
            X_ordered,
            block_dimensions,
            mode="shuffle",
            seed=SEED + 100_000 + repetition,
        )

        prediction = model.predict(X_shuffle)
        metrics = point_metrics(y, prediction)

        shuffle_rhos.append(metrics["rho"])
        shuffle_r2s.append(metrics["r2"])

    return {
        "ordered": ordered_metrics,
        "reverse": reverse_metrics,
        "shuffle_mean_rho": float(np.mean(shuffle_rhos)),
        "shuffle_p95_rho": float(
            np.quantile(shuffle_rhos, 0.95)
        ),
        "shuffle_mean_r2": float(np.mean(shuffle_r2s)),
        "rho_advantage_over_shuffle_mean": float(
            ordered_metrics["rho"]
            - np.mean(shuffle_rhos)
        ),
        "rho_advantage_over_shuffle_p95": float(
            ordered_metrics["rho"]
            - np.quantile(shuffle_rhos, 0.95)
        ),
        "rho_advantage_over_reverse": float(
            ordered_metrics["rho"]
            - reverse_metrics["rho"]
        ),
    }


def order_ablation_hazard(
    model,
    dataset,
    feature_key,
    block_dimensions,
    repetitions=100,
):
    X_ordered = dataset[feature_key]
    y = dataset["y_hazard"]

    ordered_probability = get_positive_probability(
        model,
        X_ordered,
    )

    ordered_auc = safe_auroc(
        y,
        ordered_probability,
    )

    X_reverse = reorder_temporal_matrix(
        X_ordered,
        block_dimensions,
        mode="reverse",
        seed=SEED,
    )

    reverse_probability = get_positive_probability(
        model,
        X_reverse,
    )

    reverse_auc = safe_auroc(
        y,
        reverse_probability,
    )

    shuffle_aucs = []

    for repetition in range(repetitions):
        X_shuffle = reorder_temporal_matrix(
            X_ordered,
            block_dimensions,
            mode="shuffle",
            seed=SEED + 200_000 + repetition,
        )

        probability = get_positive_probability(
            model,
            X_shuffle,
        )

        auc = safe_auroc(y, probability)

        if auc is not None:
            shuffle_aucs.append(auc)

    shuffle_mean = mean_finite(
        shuffle_aucs,
        default=0.5,
    )

    shuffle_p95 = (
        float(np.quantile(shuffle_aucs, 0.95))
        if shuffle_aucs
        else 0.5
    )

    return {
        "ordered_auroc": ordered_auc,
        "reverse_auroc": reverse_auc,
        "shuffle_mean_auroc": shuffle_mean,
        "shuffle_p95_auroc": shuffle_p95,
        "auc_advantage_over_shuffle_mean": float(
            (ordered_auc or 0.0)
            - shuffle_mean
        ),
        "auc_advantage_over_shuffle_p95": float(
            (ordered_auc or 0.0)
            - shuffle_p95
        ),
        "auc_advantage_over_reverse": float(
            (ordered_auc or 0.0)
            - (reverse_auc or 0.0)
        ),
    }


print("\n" + "=" * 120)
print("🧪 TEST D — REPRESENTATION-SPECIFIC ORDER ABLATIONS")
print("=" * 120)

point_validation_dataset = point_artifacts[
    "validation_dataset"
]

hazard_validation_dataset = hazard_artifacts[
    "validation_dataset"
]

point_order_ablations = {
    "ERROR_HISTORY": order_ablation_point(
        point_representation_models["ERROR_HISTORY"],
        point_validation_dataset,
        "X_error",
        [2],
    ),
    "ACTIVATION_HISTORY": order_ablation_point(
        point_representation_models["ACTIVATION_HISTORY"],
        point_validation_dataset,
        "X_activation",
        [point_validation_dataset["activation_dim"]],
    ),
    "ERROR_PLUS_ACTIVATION": order_ablation_point(
        point_representation_models[
            "ERROR_PLUS_ACTIVATION"
        ],
        point_validation_dataset,
        "X_full",
        [
            2,
            point_validation_dataset["activation_dim"],
        ],
    ),
    "STRUCTURED_SENSOR": order_ablation_point(
        point_representation_models["STRUCTURED_SENSOR"],
        point_validation_dataset,
        "X_structured",
        [4],
    ),
}

hazard_order_ablations = {
    "ERROR_HISTORY": order_ablation_hazard(
        hazard_representation_models["ERROR_HISTORY"],
        hazard_validation_dataset,
        "X_error",
        [2],
    ),
    "ACTIVATION_HISTORY": order_ablation_hazard(
        hazard_representation_models["ACTIVATION_HISTORY"],
        hazard_validation_dataset,
        "X_activation",
        [hazard_validation_dataset["activation_dim"]],
    ),
    "ERROR_PLUS_ACTIVATION": order_ablation_hazard(
        hazard_representation_models[
            "ERROR_PLUS_ACTIVATION"
        ],
        hazard_validation_dataset,
        "X_full",
        [
            2,
            hazard_validation_dataset["activation_dim"],
        ],
    ),
    "STRUCTURED_SENSOR": order_ablation_hazard(
        hazard_representation_models["STRUCTURED_SENSOR"],
        hazard_validation_dataset,
        "X_structured",
        [4],
    ),
}

print("\nPoint order ablations:")

for name, result in point_order_ablations.items():
    print(
        f"  {name:<25} | "
        f"Ordered={result['ordered']['rho']: .4f} | "
        f"Shuffle P95={result['shuffle_p95_rho']: .4f} | "
        f"Reverse={result['reverse']['rho']: .4f} | "
        f"Δshuffle={result['rho_advantage_over_shuffle_p95']: .4f}"
    )

print("\nHazard order ablations:")

for name, result in hazard_order_ablations.items():
    print(
        f"  {name:<25} | "
        f"Ordered={(result['ordered_auroc'] or 0.0): .4f} | "
        f"Shuffle P95={result['shuffle_p95_auroc']: .4f} | "
        f"Reverse={(result['reverse_auroc'] or 0.0): .4f} | "
        f"Δshuffle={result['auc_advantage_over_shuffle_p95']: .4f}"
    )


# =================================================================================================
# 18. SELECT DEPLOYABLE REPRESENTATIONS ON VALIDATION
# =================================================================================================

def select_point_representation(results):
    return max(
        results,
        key=lambda name: (
            results[name]["macro_stream_rho"],
            results[name]["rho"],
            results[name]["r2"],
        ),
    )


def select_hazard_representation(results):
    return max(
        results,
        key=lambda name: (
            results[name]["auroc"] or -1.0,
            results[name]["auprc"] or -1.0,
            -results[name]["brier"],
        ),
    )


BEST_POINT_REPRESENTATION = select_point_representation(
    point_representation_results
)

BEST_HAZARD_REPRESENTATION = select_hazard_representation(
    hazard_representation_results
)

feature_key_map = {
    "ERROR_HISTORY": "X_error",
    "ACTIVATION_HISTORY": "X_activation",
    "ERROR_PLUS_ACTIVATION": "X_full",
    "STRUCTURED_SENSOR": "X_structured",
}

BEST_POINT_FEATURE_KEY = feature_key_map[
    BEST_POINT_REPRESENTATION
]

BEST_HAZARD_FEATURE_KEY = feature_key_map[
    BEST_HAZARD_REPRESENTATION
]

final_point_model = point_representation_models[
    BEST_POINT_REPRESENTATION
]

final_hazard_model = hazard_representation_models[
    BEST_HAZARD_REPRESENTATION
]

validation_hazard_probability = get_positive_probability(
    final_hazard_model,
    hazard_validation_dataset[
        BEST_HAZARD_FEATURE_KEY
    ],
)

HAZARD_DECISION_THRESHOLD, validation_best_f1 = (
    select_f1_threshold(
        hazard_validation_dataset["y_hazard"],
        validation_hazard_probability,
    )
)

print("\nSelected validation representations:")
print(
    f"  Point  : {BEST_POINT_REPRESENTATION}"
)
print(
    f"  Hazard : {BEST_HAZARD_REPRESENTATION}"
)
print(
    f"  Hazard decision threshold: "
    f"{HAZARD_DECISION_THRESHOLD:.3f}"
)


# =================================================================================================
# 19. PRIVILEGED BASELINES
# =================================================================================================

def fit_latent_autoregressive_oracle(
    group,
):
    latent_streams = split_flat_array(
        group["latent_z"],
        group["lengths"],
    )

    X = []
    y = []

    for latent_z in latent_streams:
        for t in range(
            WINDOW_K,
            len(latent_z) - HORIZON_H,
        ):
            X.append(
                latent_z[
                    t - WINDOW_K:t + 1
                ]
            )

            y.append(
                latent_z[t + HORIZON_H]
            )

    scaler = StandardScaler()
    regressor = RidgeCV(
        alphas=np.logspace(-5, 3, 20)
    )

    regressor.fit(
        scaler.fit_transform(np.asarray(X)),
        np.asarray(y),
    )

    return scaler, regressor


latent_oracle_scaler, latent_oracle_model = (
    fit_latent_autoregressive_oracle(
        train_group
    )
)


def autoregressive_oracle_predictions(
    group,
    dataset,
):
    latent_streams = split_flat_array(
        group["latent_z"],
        group["lengths"],
    )

    X = []

    for stream_id, t in zip(
        dataset["stream_ids"],
        dataset["times"],
    ):
        latent_z = latent_streams[stream_id]

        X.append(
            latent_z[
                t - WINDOW_K:t + 1
            ]
        )

    X = np.asarray(X)

    return latent_oracle_model.predict(
        latent_oracle_scaler.transform(X)
    )


def latent_persistence_predictions(
    group,
    dataset,
):
    latent_streams = split_flat_array(
        group["latent_z"],
        group["lengths"],
    )

    predictions = []

    for stream_id, t in zip(
        dataset["stream_ids"],
        dataset["times"],
    ):
        predictions.append(
            latent_streams[stream_id][t]
        )

    return np.asarray(predictions, dtype=float)


def kalman_filter_predictions(
    group,
    dataset,
    phi=0.93,
    process_variance=0.035 ** 2,
    measurement_variance=OBSERVATION_NOISE_STD ** 2,
):
    """
    Privileged mechanism-aware Kalman baseline.

    Observation:
        mechanism - nominal = sensitivity * z + observation_noise

    It knows the synthetic nominal function and sensitivity. Therefore this
    is an upper-bound diagnostic, not a deployable self-model baseline.
    """
    prediction_map = {}

    for stream_index, stream in enumerate(group["streams"]):
        estimate = 0.0
        covariance = 1.0

        for t, row in enumerate(stream):
            # Predict one step
            estimate_prior = phi * estimate
            covariance_prior = (
                phi ** 2 * covariance
                + process_variance
            )

            observation = (
                row["mechanism"]
                - row["nominal_mechanism"]
            )

            observation_matrix = row["sensitivity"]

            innovation_variance = (
                observation_matrix ** 2
                * covariance_prior
                + measurement_variance
            )

            kalman_gain = (
                covariance_prior
                * observation_matrix
                / max(innovation_variance, EPS)
            )

            innovation = (
                observation
                - observation_matrix
                * estimate_prior
            )

            estimate = (
                estimate_prior
                + kalman_gain * innovation
            )

            covariance = (
                1.0
                - kalman_gain * observation_matrix
            ) * covariance_prior

            # H-step forecast under the nominal AR(1) transition.
            prediction_map[
                (stream_index, t)
            ] = float(
                (phi ** HORIZON_H) * estimate
            )

    predictions = []

    for stream_id, t in zip(
        dataset["stream_ids"],
        dataset["times"],
    ):
        predictions.append(
            prediction_map[
                (int(stream_id), int(t))
            ]
        )

    return np.asarray(predictions, dtype=float)


# =================================================================================================
# 20. HAZARD ORACLE BASELINES
# =================================================================================================

def latent_history_matrix(group, dataset):
    """
    Builds a privileged feature matrix containing the true latent-state
    history. This is an oracle diagnostic and must not be interpreted as
    a deployable model.
    """
    latent_streams = split_flat_array(
        group["latent_z"],
        group["lengths"],
    )

    features = []

    for stream_id, t in zip(
        dataset["stream_ids"],
        dataset["times"],
    ):
        latent_z = latent_streams[int(stream_id)]

        features.append(
            latent_z[
                int(t) - WINDOW_K:int(t) + 1
            ]
        )

    return np.asarray(features, dtype=float)


def fit_latent_hazard_oracle(group, dataset):
    X = latent_history_matrix(
        group,
        dataset,
    )

    y = dataset["y_hazard"]

    model_variant = fit_hazard_model(
        X,
        y,
        seed=SEED + 300_000,
    )

    return model_variant


latent_hazard_oracle = fit_latent_hazard_oracle(
    train_group,
    hazard_artifacts["train_dataset"],
)


def latent_hazard_oracle_probability(
    group,
    dataset,
):
    X = latent_history_matrix(
        group,
        dataset,
    )

    return get_positive_probability(
        latent_hazard_oracle,
        X,
    )


def current_error_hazard_probability(
    group,
    dataset,
):
    """
    A persistence-style hazard baseline.

    It maps the current instantaneous self-model error into a soft
    probability relative to the calibrated hazard threshold.
    """
    absolute_error_streams = split_flat_array(
        group["absolute_error"],
        group["lengths"],
    )

    current_errors = []

    for stream_id, t in zip(
        dataset["stream_ids"],
        dataset["times"],
    ):
        current_errors.append(
            absolute_error_streams[
                int(stream_id)
            ][int(t)]
        )

    current_errors = np.asarray(
        current_errors,
        dtype=float,
    )

    scale = max(
        calibration_profile["mad"] * 1.4826,
        0.02,
    )

    logits = (
        current_errors
        - HAZARD_RESIDUAL_THRESHOLD
    ) / scale

    logits = np.clip(logits, -30.0, 30.0)

    return 1.0 / (1.0 + np.exp(-logits))


# =================================================================================================
# 21. VALIDATION BASELINE AUDIT
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST E — VALIDATION BASELINES AND ORACLE CEILING")
print("=" * 120)

# -------------------------------------------------------------------------------------------------
# Point baselines
# -------------------------------------------------------------------------------------------------

point_train_dataset = point_artifacts[
    "train_dataset"
]

point_validation_dataset = point_artifacts[
    "validation_dataset"
]

point_train_mean = float(
    np.mean(
        point_train_dataset["y_point"]
    )
)

validation_point_mean_prediction = np.full(
    len(point_validation_dataset["y_point"]),
    point_train_mean,
    dtype=float,
)

validation_point_persistence_prediction = (
    latent_persistence_predictions(
        validation_group,
        point_validation_dataset,
    )
)

validation_point_oracle_prediction = (
    autoregressive_oracle_predictions(
        validation_group,
        point_validation_dataset,
    )
)

validation_point_kalman_prediction = (
    kalman_filter_predictions(
        validation_group,
        point_validation_dataset,
    )
)

validation_point_model_prediction = (
    final_point_model.predict(
        point_validation_dataset[
            BEST_POINT_FEATURE_KEY
        ]
    )
)

validation_point_baselines = {
    "MEAN": evaluate_point(
        point_validation_dataset["y_point"],
        validation_point_mean_prediction,
        point_validation_dataset["stream_ids"],
    ),
    "LATENT_PERSISTENCE_PRIVILEGED": evaluate_point(
        point_validation_dataset["y_point"],
        validation_point_persistence_prediction,
        point_validation_dataset["stream_ids"],
    ),
    "LATENT_AR_ORACLE": evaluate_point(
        point_validation_dataset["y_point"],
        validation_point_oracle_prediction,
        point_validation_dataset["stream_ids"],
    ),
    "MECHANISM_KALMAN_ORACLE": evaluate_point(
        point_validation_dataset["y_point"],
        validation_point_kalman_prediction,
        point_validation_dataset["stream_ids"],
    ),
    "SELECTED_FORECASTER": evaluate_point(
        point_validation_dataset["y_point"],
        validation_point_model_prediction,
        point_validation_dataset["stream_ids"],
    ),
}

for name, metrics in validation_point_baselines.items():
    print(
        f"{name:<30} | "
        f"rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f} | "
        f"MAE={metrics['mae']: .4f}"
    )

# -------------------------------------------------------------------------------------------------
# Hazard baselines
# -------------------------------------------------------------------------------------------------

hazard_train_dataset = hazard_artifacts[
    "train_dataset"
]

hazard_validation_dataset = hazard_artifacts[
    "validation_dataset"
]

hazard_train_prevalence = float(
    np.mean(
        hazard_train_dataset["y_hazard"]
    )
)

validation_hazard_constant_probability = np.full(
    len(hazard_validation_dataset["y_hazard"]),
    hazard_train_prevalence,
    dtype=float,
)

validation_hazard_persistence_probability = (
    current_error_hazard_probability(
        validation_group,
        hazard_validation_dataset,
    )
)

validation_hazard_oracle_probability = (
    latent_hazard_oracle_probability(
        validation_group,
        hazard_validation_dataset,
    )
)

validation_hazard_model_probability = (
    get_positive_probability(
        final_hazard_model,
        hazard_validation_dataset[
            BEST_HAZARD_FEATURE_KEY
        ],
    )
)

validation_hazard_baselines = {
    "CONSTANT_PREVALENCE": evaluate_hazard(
        hazard_validation_dataset["y_hazard"],
        validation_hazard_constant_probability,
        hazard_validation_dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    ),
    "CURRENT_ERROR_PERSISTENCE": evaluate_hazard(
        hazard_validation_dataset["y_hazard"],
        validation_hazard_persistence_probability,
        hazard_validation_dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    ),
    "LATENT_HISTORY_ORACLE": evaluate_hazard(
        hazard_validation_dataset["y_hazard"],
        validation_hazard_oracle_probability,
        hazard_validation_dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    ),
    "SELECTED_HAZARD_FORECASTER": evaluate_hazard(
        hazard_validation_dataset["y_hazard"],
        validation_hazard_model_probability,
        hazard_validation_dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    ),
}

print("\nHazard baselines:")

for name, metrics in validation_hazard_baselines.items():
    print(
        f"{name:<30} | "
        f"AUC={(metrics['auroc'] or 0.0): .4f} | "
        f"AP={(metrics['auprc'] or 0.0): .4f} | "
        f"Brier={metrics['brier']: .4f} | "
        f"ECE={metrics['ece']: .4f}"
    )


# =================================================================================================
# 22. LEAD-TIME AND FALSE-ALARM AUDIT
# =================================================================================================

def lead_time_audit(
    group,
    dataset,
    probability,
    decision_threshold,
):
    """
    Measures warning behavior against instantaneous threshold crossings.

    For every real instantaneous event at time e, a warning is considered
    useful if at least one positive forecast was emitted during:

        [e - HORIZON_H, e - 1]

    The reported lead time is measured from the earliest alert inside that
    interval. Predictions that are not followed by an event in the next H
    steps are counted as false-alert samples.
    """
    probability = np.asarray(
        probability,
        dtype=float,
    )

    predicted_alert = (
        probability >= decision_threshold
    )

    absolute_error_streams = split_flat_array(
        group["absolute_error"],
        group["lengths"],
    )

    detected_event_leads = []
    total_events = 0
    detected_events = 0

    total_alert_samples = 0
    false_alert_samples = 0

    for stream_id in np.unique(
        dataset["stream_ids"]
    ):
        sample_mask = (
            dataset["stream_ids"] == stream_id
        )

        stream_times = dataset["times"][sample_mask]
        stream_alerts = predicted_alert[sample_mask]

        time_to_alert = {
            int(t): bool(alert)
            for t, alert in zip(
                stream_times,
                stream_alerts,
            )
        }

        instantaneous_event = (
            absolute_error_streams[int(stream_id)]
            > HAZARD_RESIDUAL_THRESHOLD
        )

        event_times = np.flatnonzero(
            instantaneous_event
        )

        # Count onsets, not every consecutive threshold-exceeding step.
        event_onsets = [
            int(event_time)
            for event_time in event_times
            if (
                event_time == 0
                or not instantaneous_event[
                    event_time - 1
                ]
            )
        ]

        for event_time in event_onsets:
            candidate_alert_times = [
                t
                for t in range(
                    max(WINDOW_K, event_time - HORIZON_H),
                    event_time,
                )
                if time_to_alert.get(t, False)
            ]

            # Only score events for which a complete/valid forecast
            # opportunity existed.
            eligible_times = [
                t
                for t in range(
                    max(WINDOW_K, event_time - HORIZON_H),
                    event_time,
                )
                if t in time_to_alert
            ]

            if not eligible_times:
                continue

            total_events += 1

            if candidate_alert_times:
                detected_events += 1

                earliest_alert = min(
                    candidate_alert_times
                )

                detected_event_leads.append(
                    int(
                        event_time
                        - earliest_alert
                    )
                )

        for t, alert in zip(
            stream_times,
            stream_alerts,
        ):
            if not alert:
                continue

            total_alert_samples += 1

            future_start = int(t) + 1
            future_stop = min(
                len(instantaneous_event),
                int(t) + HORIZON_H + 1,
            )

            followed_by_event = bool(
                np.any(
                    instantaneous_event[
                        future_start:future_stop
                    ]
                )
            )

            if not followed_by_event:
                false_alert_samples += 1

    event_recall = (
        detected_events / total_events
        if total_events > 0
        else None
    )

    false_alert_fraction = (
        false_alert_samples / total_alert_samples
        if total_alert_samples > 0
        else 0.0
    )

    return {
        "total_event_onsets": int(total_events),
        "detected_event_onsets": int(
            detected_events
        ),
        "event_recall": (
            float(event_recall)
            if event_recall is not None
            else None
        ),
        "mean_detected_lead_time": (
            float(np.mean(detected_event_leads))
            if detected_event_leads
            else None
        ),
        "median_detected_lead_time": (
            float(np.median(detected_event_leads))
            if detected_event_leads
            else None
        ),
        "maximum_detected_lead_time": (
            int(np.max(detected_event_leads))
            if detected_event_leads
            else None
        ),
        "total_alert_samples": int(
            total_alert_samples
        ),
        "false_alert_samples": int(
            false_alert_samples
        ),
        "false_alert_fraction": float(
            false_alert_fraction
        ),
    }


validation_lead_time = lead_time_audit(
    validation_group,
    hazard_validation_dataset,
    validation_hazard_model_probability,
    HAZARD_DECISION_THRESHOLD,
)

print("\nValidation warning behavior:")

for key, value in validation_lead_time.items():
    print(f"  {key:<30}: {value}")


# =================================================================================================
# 23. BUILD UNTOUCHED CROSS-REGIME TEST DATASETS
# =================================================================================================

point_encoder = point_artifacts["encoder"]
hazard_encoder = hazard_artifacts["encoder"]

point_test_datasets = {}
hazard_test_datasets = {}

for regime in TEST_REGIMES:
    point_test_datasets[regime] = (
        build_temporal_dataset(
            test_groups[regime],
            BEST_POINT_LAYER,
            point_encoder,
        )
    )

    hazard_test_datasets[regime] = (
        build_temporal_dataset(
            test_groups[regime],
            BEST_HAZARD_LAYER,
            hazard_encoder,
        )
    )


# =================================================================================================
# 24. CROSS-REGIME POINT FORECASTING
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST F — CROSS-REGIME POINT FORECASTING")
print("=" * 120)

cross_regime_point_results = {}

for regime_index, regime in enumerate(TEST_REGIMES):
    group = test_groups[regime]
    dataset = point_test_datasets[regime]

    selected_prediction = final_point_model.predict(
        dataset[BEST_POINT_FEATURE_KEY]
    )

    mean_prediction = np.full(
        len(dataset["y_point"]),
        point_train_mean,
        dtype=float,
    )

    persistence_prediction = (
        latent_persistence_predictions(
            group,
            dataset,
        )
    )

    oracle_prediction = (
        autoregressive_oracle_predictions(
            group,
            dataset,
        )
    )

    kalman_prediction = (
        kalman_filter_predictions(
            group,
            dataset,
        )
    )

    selected_metrics = evaluate_point(
        dataset["y_point"],
        selected_prediction,
        dataset["stream_ids"],
    )

    mean_metrics = evaluate_point(
        dataset["y_point"],
        mean_prediction,
        dataset["stream_ids"],
    )

    persistence_metrics = evaluate_point(
        dataset["y_point"],
        persistence_prediction,
        dataset["stream_ids"],
    )

    oracle_metrics = evaluate_point(
        dataset["y_point"],
        oracle_prediction,
        dataset["stream_ids"],
    )

    kalman_metrics = evaluate_point(
        dataset["y_point"],
        kalman_prediction,
        dataset["stream_ids"],
    )

    bootstrap_ci = stream_bootstrap_point_ci(
        dataset["y_point"],
        selected_prediction,
        dataset["stream_ids"],
        seed=SEED + 400_000 + regime_index,
    )

    selected_metrics.update(bootstrap_ci)

    selected_metrics[
        "rho_gain_over_persistence"
    ] = float(
        selected_metrics["rho"]
        - persistence_metrics["rho"]
    )

    selected_metrics[
        "r2_gain_over_mean"
    ] = float(
        selected_metrics["r2"]
        - mean_metrics["r2"]
    )

    cross_regime_point_results[regime] = {
        "selected_model": selected_metrics,
        "mean": mean_metrics,
        "latent_persistence_privileged": persistence_metrics,
        "latent_ar_oracle": oracle_metrics,
        "mechanism_kalman_oracle": kalman_metrics,
    }

    print(
        f"{regime:<18} | "
        f"rho={selected_metrics['rho']: .4f} | "
        f"Macro={selected_metrics['macro_stream_rho']: .4f} | "
        f"R²={selected_metrics['r2']: .4f} | "
        f"Persist={persistence_metrics['rho']: .4f} | "
        f"Oracle={oracle_metrics['rho']: .4f}"
    )


# =================================================================================================
# 25. CROSS-REGIME HAZARD FORECASTING
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST G — CROSS-REGIME HAZARD FORECASTING")
print("=" * 120)

cross_regime_hazard_results = {}

for regime_index, regime in enumerate(TEST_REGIMES):
    group = test_groups[regime]
    dataset = hazard_test_datasets[regime]

    selected_probability = get_positive_probability(
        final_hazard_model,
        dataset[BEST_HAZARD_FEATURE_KEY],
    )

    constant_probability = np.full(
        len(dataset["y_hazard"]),
        hazard_train_prevalence,
        dtype=float,
    )

    persistence_probability = (
        current_error_hazard_probability(
            group,
            dataset,
        )
    )

    oracle_probability = (
        latent_hazard_oracle_probability(
            group,
            dataset,
        )
    )

    selected_metrics = evaluate_hazard(
        dataset["y_hazard"],
        selected_probability,
        dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    )

    constant_metrics = evaluate_hazard(
        dataset["y_hazard"],
        constant_probability,
        dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    )

    persistence_metrics = evaluate_hazard(
        dataset["y_hazard"],
        persistence_probability,
        dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    )

    oracle_metrics = evaluate_hazard(
        dataset["y_hazard"],
        oracle_probability,
        dataset["stream_ids"],
        HAZARD_DECISION_THRESHOLD,
    )

    bootstrap_ci = stream_bootstrap_hazard_ci(
        dataset["y_hazard"],
        selected_probability,
        dataset["stream_ids"],
        seed=SEED + 500_000 + regime_index,
    )

    selected_metrics.update(bootstrap_ci)

    selected_metrics[
        "auprc_gain_over_prevalence"
    ] = float(
        (selected_metrics["auprc"] or 0.0)
        - selected_metrics["prevalence"]
    )

    selected_metrics[
        "auroc_gain_over_persistence"
    ] = float(
        (selected_metrics["auroc"] or 0.0)
        - (persistence_metrics["auroc"] or 0.0)
    )

    lead_time_metrics = lead_time_audit(
        group,
        dataset,
        selected_probability,
        HAZARD_DECISION_THRESHOLD,
    )

    cross_regime_hazard_results[regime] = {
        "selected_model": selected_metrics,
        "constant_prevalence": constant_metrics,
        "current_error_persistence": persistence_metrics,
        "latent_history_oracle": oracle_metrics,
        "lead_time": lead_time_metrics,
    }

    print(
        f"{regime:<18} | "
        f"AUC={(selected_metrics['auroc'] or 0.0): .4f} | "
        f"AP={(selected_metrics['auprc'] or 0.0): .4f} | "
        f"Prev={selected_metrics['prevalence']: .4f} | "
        f"Brier={selected_metrics['brier']: .4f} | "
        f"Oracle={(oracle_metrics['auroc'] or 0.0): .4f}"
    )


# =================================================================================================
# 26. NOVELTY SENSOR AUDIT
# =================================================================================================

print("\n" + "=" * 120)
print("🧪 TEST H — NOVELTY SENSOR AUDIT")
print("=" * 120)

standard_reference_activation = (
    base_calibration_activations[
        BEST_SELF_MODEL_LAYER
    ]
)

novel_group = test_groups[
    "COMPOUND_NOVEL"
]

novel_activation = novel_group[
    "activations"
][BEST_SELF_MODEL_LAYER]

novel_labels = np.asarray([
    int(
        row["context_type"]
        == "NOVEL_FORMAT"
    )
    for row in novel_group["rows"]
], dtype=int)

novelty_scaler = StandardScaler()
reference_scaled = novelty_scaler.fit_transform(
    standard_reference_activation
)

novel_scaled = novelty_scaler.transform(
    novel_activation
)

# Diagonal Mahalanobis distance after standardization.
reference_center = np.mean(
    reference_scaled,
    axis=0,
)

reference_scores = np.sqrt(
    np.mean(
        (
            reference_scaled
            - reference_center
        ) ** 2,
        axis=1,
    )
)

novelty_scores = np.sqrt(
    np.mean(
        (
            novel_scaled
            - reference_center
        ) ** 2,
        axis=1,
    )
)

NOVELTY_THRESHOLD = float(
    np.quantile(
        reference_scores,
        0.95,
    )
)

novelty_auc = safe_auroc(
    novel_labels,
    novelty_scores,
)

standard_fpr = float(
    np.mean(
        reference_scores
        > NOVELTY_THRESHOLD
    )
)

if np.any(novel_labels == 1):
    novel_tpr = float(
        np.mean(
            novelty_scores[
                novel_labels == 1
            ] > NOVELTY_THRESHOLD
        )
    )
else:
    novel_tpr = None

novelty_audit = {
    "auroc": novelty_auc,
    "threshold": NOVELTY_THRESHOLD,
    "standard_fpr": standard_fpr,
    "novel_tpr": novel_tpr,
    "standard_score_mean": float(
        np.mean(reference_scores)
    ),
    "novel_score_mean": float(
        np.mean(
            novelty_scores[
                novel_labels == 1
            ]
        )
    ),
}

print(
    f"Novelty AUROC            : "
    f"{(novelty_auc or 0.0):.4f}"
)
print(
    f"Standard FPR @ Q95       : "
    f"{standard_fpr:.4f}"
)
print(
    f"Novel TPR @ threshold    : "
    f"{(novel_tpr or 0.0):.4f}"
)


# =================================================================================================
# 27. FORECASTABILITY AND TRANSFER GATES
# =================================================================================================

def point_transfer_pass(result):
    selected = result["selected_model"]
    mean_baseline = result["mean"]

    rho_lower_bound = selected[
        "rho_ci95"
    ][0]

    return bool(
        selected["rho"] >= 0.20
        and selected["macro_stream_rho"] >= 0.15
        and selected["r2"] > 0.0
        and selected["r2"] > mean_baseline["r2"]
        and rho_lower_bound > 0.0
    )


def hazard_transfer_pass(result):
    selected = result["selected_model"]

    auc_lower_bound = selected[
        "auroc_ci95"
    ][0]

    if auc_lower_bound is None:
        return False

    return bool(
        (selected["auroc"] or 0.0) >= 0.65
        and selected[
            "auprc_gain_over_prevalence"
        ] >= 0.05
        and auc_lower_bound > 0.50
    )


point_transfer_flags = {
    regime: point_transfer_pass(
        cross_regime_point_results[regime]
    )
    for regime in TEST_REGIMES
}

hazard_transfer_flags = {
    regime: hazard_transfer_pass(
        cross_regime_hazard_results[regime]
    )
    for regime in TEST_REGIMES
}

point_pass_count = int(
    sum(point_transfer_flags.values())
)

hazard_pass_count = int(
    sum(hazard_transfer_flags.values())
)


# =================================================================================================
# 28. ORDER-EVIDENCE GATES
# =================================================================================================

selected_point_order_result = (
    point_order_ablations[
        BEST_POINT_REPRESENTATION
    ]
)

selected_hazard_order_result = (
    hazard_order_ablations[
        BEST_HAZARD_REPRESENTATION
    ]
)

POINT_ORDER_VALID = bool(
    selected_point_order_result[
        "rho_advantage_over_shuffle_p95"
    ] > 0.0
    and selected_point_order_result[
        "rho_advantage_over_reverse"
    ] > 0.02
)

HAZARD_ORDER_VALID = bool(
    selected_hazard_order_result[
        "auc_advantage_over_shuffle_p95"
    ] > 0.0
    and selected_hazard_order_result[
        "auc_advantage_over_reverse"
    ] > 0.01
)


# =================================================================================================
# 29. BENCHMARK FORECASTABILITY GATES
# =================================================================================================

validation_oracle_point = (
    validation_point_baselines[
        "LATENT_AR_ORACLE"
    ]
)

validation_kalman_point = (
    validation_point_baselines[
        "MECHANISM_KALMAN_ORACLE"
    ]
)

validation_oracle_hazard = (
    validation_hazard_baselines[
        "LATENT_HISTORY_ORACLE"
    ]
)

POINT_TASK_FORECASTABLE = bool(
    validation_oracle_point["rho"] >= 0.50
    and validation_oracle_point["r2"] > 0.10
)

MECHANISM_OBSERVATION_INFORMATIVE = bool(
    validation_kalman_point["rho"] >= 0.30
)

HAZARD_TASK_FORECASTABLE = bool(
    (validation_oracle_hazard["auroc"] or 0.0)
    >= 0.70
)


# =================================================================================================
# 30. FINAL AUTOMATED DIAGNOSTIC
# =================================================================================================

nominal_metrics = nominal_results[
    BEST_SELF_MODEL_LAYER
]

NOMINAL_SELF_MODEL_VALID = bool(
    nominal_metrics["rho"] >= 0.80
    and nominal_metrics["r2"] >= 0.60
)

SELECTED_POINT_VALIDATION_VALID = bool(
    point_representation_results[
        BEST_POINT_REPRESENTATION
    ]["rho"] >= MIN_POINT_RHO
    and point_representation_results[
        BEST_POINT_REPRESENTATION
    ]["macro_stream_rho"] >= MIN_POINT_MACRO_RHO
    and point_representation_results[
        BEST_POINT_REPRESENTATION
    ]["r2"] > MIN_POINT_R2
)

selected_hazard_validation = (
    hazard_representation_results[
        BEST_HAZARD_REPRESENTATION
    ]
)

SELECTED_HAZARD_VALIDATION_VALID = bool(
    (selected_hazard_validation["auroc"] or 0.0)
    >= MIN_HAZARD_AUROC
    and selected_hazard_validation[
        "auprc_gain_over_prevalence"
    ] >= MIN_HAZARD_AUPRC_GAIN
)

NOVELTY_SENSOR_VALID = bool(
    (novelty_auc or 0.0) >= 0.85
    and (novel_tpr or 0.0) >= 0.80
    and standard_fpr <= 0.10
)

diagnostic_flags = {
    "nominal_self_model_valid": (
        NOMINAL_SELF_MODEL_VALID
    ),
    "point_task_oracle_forecastable": (
        POINT_TASK_FORECASTABLE
    ),
    "mechanism_observation_informative": (
        MECHANISM_OBSERVATION_INFORMATIVE
    ),
    "hazard_task_oracle_forecastable": (
        HAZARD_TASK_FORECASTABLE
    ),
    "activation_point_layer_passes_gate": (
        POINT_LAYER_VALID
    ),
    "activation_hazard_layer_passes_gate": (
        HAZARD_LAYER_VALID
    ),
    "selected_point_representation_valid": (
        SELECTED_POINT_VALIDATION_VALID
    ),
    "selected_hazard_representation_valid": (
        SELECTED_HAZARD_VALIDATION_VALID
    ),
    "selected_point_representation_uses_order": (
        POINT_ORDER_VALID
    ),
    "selected_hazard_representation_uses_order": (
        HAZARD_ORDER_VALID
    ),
    "point_transfer_majority": bool(
        point_pass_count
        >= math.ceil(len(TEST_REGIMES) / 2)
    ),
    "hazard_transfer_majority": bool(
        hazard_pass_count
        >= math.ceil(len(TEST_REGIMES) / 2)
    ),
    "compound_point_transfer": (
        point_transfer_flags[
            "COMPOUND_NOVEL"
        ]
    ),
    "compound_hazard_transfer": (
        hazard_transfer_flags[
            "COMPOUND_NOVEL"
        ]
    ),
    "novelty_sensor_valid": (
        NOVELTY_SENSOR_VALID
    ),
}

core_scientific_success = bool(
    NOMINAL_SELF_MODEL_VALID
    and POINT_TASK_FORECASTABLE
    and HAZARD_TASK_FORECASTABLE
)

temporal_self_model_success = bool(
    SELECTED_POINT_VALIDATION_VALID
    and SELECTED_HAZARD_VALIDATION_VALID
    and (
        POINT_ORDER_VALID
        or HAZARD_ORDER_VALID
    )
)

generalization_success = bool(
    diagnostic_flags["point_transfer_majority"]
    and diagnostic_flags[
        "hazard_transfer_majority"
    ]
)

if not POINT_TASK_FORECASTABLE:
    final_diagnosis = (
        "POINT_BENCHMARK_NOT_FORECASTABLE: the privileged latent-state "
        "oracle did not establish an adequate forecasting ceiling. "
        "No negative conclusion about representation learning should be "
        "drawn until generator parameters are revised."
    )

elif not HAZARD_TASK_FORECASTABLE:
    final_diagnosis = (
        "HAZARD_TARGET_NOT_FORECASTABLE: the privileged latent-history "
        "oracle did not establish adequate event predictability. "
        "The event threshold or latent dynamics require recalibration."
    )

elif (
    not SELECTED_POINT_VALIDATION_VALID
    and not SELECTED_HAZARD_VALIDATION_VALID
):
    final_diagnosis = (
        "FORECASTABLE_PROCESS_BUT_NO_DEPLOYABLE_TEMPORAL_SIGNAL: privileged "
        "oracles predict the future, but residual and activation histories "
        "do not provide robust validation performance."
    )

elif temporal_self_model_success and generalization_success:
    final_diagnosis = (
        "TEMPORAL_SELF_MODEL_VALIDATED: the persistent latent process is "
        "forecastable, the selected representation passes validation and "
        "order ablations, and a majority of untouched regimes transfer."
    )

elif temporal_self_model_success:
    final_diagnosis = (
        "IN_DISTRIBUTION_TEMPORAL_SELF_MODEL_ONLY: validation and temporal "
        "order evidence pass, but robust cross-regime generalization is not "
        "yet established."
    )

elif (
    SELECTED_POINT_VALIDATION_VALID
    or SELECTED_HAZARD_VALIDATION_VALID
):
    final_diagnosis = (
        "PREDICTIVE_WITHOUT_ROBUST_ORDER_EVIDENCE: at least one temporal "
        "task is predictive, but aligned shuffle/reversal tests do not "
        "confirm that ordered history is causally necessary."
    )

else:
    final_diagnosis = (
        "NO_VALIDATED_TEMPORAL_SELF_MODEL: the corrected benchmark is "
        "forecastable at the oracle level, but the deployable feature "
        "representations do not pass the required validation gates."
    )


# =================================================================================================
# 31. PRINT SUMMARY
# =================================================================================================

print("\n" + "=" * 120)
print("🧠 AUTOMATED DIAGNOSTIC SUMMARY")
print("=" * 120)

for flag_name, flag_value in diagnostic_flags.items():
    icon = "✅" if flag_value else "❌"

    print(
        f"{icon} {flag_name}"
    )

print("\nCross-regime point gates:")

for regime, passed in point_transfer_flags.items():
    print(
        f"  {'✅' if passed else '❌'} "
        f"{regime}"
    )

print("\nCross-regime hazard gates:")

for regime, passed in hazard_transfer_flags.items():
    print(
        f"  {'✅' if passed else '❌'} "
        f"{regime}"
    )

print("\nSelected architecture:")
print(
    f"  Nominal layer          : "
    f"{BEST_SELF_MODEL_LAYER}"
)
print(
    f"  Point layer            : "
    f"{BEST_POINT_LAYER}"
)
print(
    f"  Point representation   : "
    f"{BEST_POINT_REPRESENTATION}"
)
print(
    f"  Hazard layer           : "
    f"{BEST_HAZARD_LAYER}"
)
print(
    f"  Hazard representation  : "
    f"{BEST_HAZARD_REPRESENTATION}"
)
print(
    f"  Hazard threshold       : "
    f"{HAZARD_RESIDUAL_THRESHOLD:.6f}"
)
print(
    f"  Decision threshold     : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)

print("\nFinal diagnosis:")
print(final_diagnosis)

print("\nOverall scientific statuses:")
print(
    f"  Benchmark construction : "
    f"{'PASS' if core_scientific_success else 'FAIL'}"
)
print(
    f"  Temporal self-model    : "
    f"{'PASS' if temporal_self_model_success else 'FAIL'}"
)
print(
    f"  Cross-regime transfer  : "
    f"{'PASS' if generalization_success else 'FAIL'}"
)


# =================================================================================================
# 32. TELEMETRY
# =================================================================================================

telemetry = {
    "milestone": "M20.6.2",
    "title": (
        "Forecastability-Grounded Latent-State Benchmark"
    ),
    "configuration": {
        "seed": SEED,
        "device": DEVICE,
        "model_name": MODEL_NAME,
        "valid_layers": VALID_LAYERS,
        "window_k": WINDOW_K,
        "history_length": HISTORY_LENGTH,
        "horizon_h": HORIZON_H,
        "temporal_pca_dim": TEMPORAL_PCA_DIM,
        "stream_length": STREAM_LENGTH,
        "n_base_train": N_BASE_TRAIN,
        "n_base_validation": N_BASE_VALIDATION,
        "n_base_calibration": N_BASE_CALIBRATION,
        "n_temporal_train_streams": (
            N_TEMPORAL_TRAIN_STREAMS
        ),
        "n_temporal_validation_streams": (
            N_TEMPORAL_VALIDATION_STREAMS
        ),
        "n_test_streams_per_regime": (
            N_TEST_STREAMS_PER_REGIME
        ),
        "test_regimes": TEST_REGIMES,
        "hazard_residual_threshold": (
            HAZARD_RESIDUAL_THRESHOLD
        ),
        "hazard_decision_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
    },
    "nominal_self_model": {
        "selected_layer": BEST_SELF_MODEL_LAYER,
        "layer_results": nominal_results,
        "calibration_profile": calibration_profile,
    },
    "temporal_layer_selection": {
        "layer_results": layer_results,
        "point_candidate_layer": (
            BEST_POINT_LAYER
        ),
        "point_candidate_valid": (
            POINT_LAYER_VALID
        ),
        "hazard_candidate_layer": (
            BEST_HAZARD_LAYER
        ),
        "hazard_candidate_valid": (
            HAZARD_LAYER_VALID
        ),
    },
    "representation_selection": {
        "point_results": (
            point_representation_results
        ),
        "hazard_results": (
            hazard_representation_results
        ),
        "selected_point_representation": (
            BEST_POINT_REPRESENTATION
        ),
        "selected_hazard_representation": (
            BEST_HAZARD_REPRESENTATION
        ),
    },
    "order_ablations": {
        "point": point_order_ablations,
        "hazard": hazard_order_ablations,
    },
    "validation_baselines": {
        "point": validation_point_baselines,
        "hazard": validation_hazard_baselines,
        "lead_time": validation_lead_time,
    },
    "cross_regime": {
        "point": cross_regime_point_results,
        "hazard": cross_regime_hazard_results,
        "point_transfer_flags": (
            point_transfer_flags
        ),
        "hazard_transfer_flags": (
            hazard_transfer_flags
        ),
    },
    "novelty_audit": novelty_audit,
    "diagnostic_flags": diagnostic_flags,
    "statuses": {
        "core_scientific_success": (
            core_scientific_success
        ),
        "temporal_self_model_success": (
            temporal_self_model_success
        ),
        "generalization_success": (
            generalization_success
        ),
    },
    "final_diagnosis": final_diagnosis,
}

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_builtin(telemetry),
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"\n💾 Telemetry written to: "
    f"{OUTPUT_FILE}"
)


# =================================================================================================
# 33. MEMORY CLEANUP
# =================================================================================================

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("=" * 120)
print("✅ M20.6.2 execution completed.")
print("=" * 120)

🔬 MILESTONE 20.6.2 — FORECASTABILITY-GROUNDED LATENT-STATE BENCHMARK
Device                 : CUDA
Model                  : Qwen/Qwen2.5-0.5B
History length         : 8
Forecast horizon       : 6
Train streams          : 20
Validation streams     : 8
Test streams/regime    : 10
Global seed            : 206200

✅ All semantic partitions are independent.

⏳ Loading TransformerLens model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Model loaded. Valid layers: [0, 3, 6, 9, 12, 15, 18, 21]

🔬 Extracting nominal representations...

🧪 TEST A — NOMINAL SELF-MODEL SELECTION
Layer 0   | rho= 0.8487 | R²= 0.7415 | MAE= 0.2250
Layer 3   | rho= 0.7994 | R²= 0.6193 | MAE= 0.2837
Layer 6   | rho= 0.9093 | R²= 0.8307 | MAE= 0.1813
Layer 9   | rho= 0.9598 | R²= 0.9280 | MAE= 0.1204
Layer 12  | rho= 0.9687 | R²= 0.9365 | MAE= 0.1167
Layer 15  | rho= 0.9687 | R²= 0.9285 | MAE= 0.1209
Layer 18  | rho= 0.9568 | R²= 0.9095 | MAE= 0.1389
Layer 21  | rho= 0.9340 | R²= 0.8627 | MAE= 0.1599

🏆 Nominal self-model layer: 12

Calibration profile:
  median            : 0.094396
  mad               : 0.059046
  q90               : 0.236124
  q95               : 0.267156
  q99               : 0.360170
  hazard_threshold  : 0.357021

🔬 Extracting temporal representations...
  Extracting AR1_TRANSFER...
  Extracting MEAN_REVERTING...
  Extracting RANDOM_WALK...
  Extracting OSC

In [ ]:
# =================================================================================================
# MILESTONE 20.6.3
# OBSERVABLE FEEDBACK AND EVENT-ONSET FORECASTING
# =================================================================================================
#
# Required packages:
#
# pip install torch transformer-lens transformers scikit-learn scipy numpy
#
# Scientific goals:
#
# 1. Make latent degradation causally observable through explicit environmental feedback.
# 2. Extract activations after the model has observed prediction, outcome and residual.
# 3. Forecast future latent state rather than future random realized residual.
# 4. Predict first-passage/event-onset hazard while the system is currently safe.
# 5. Separate novelty from degradation through factorial assignment.
# 6. Compare genuine sequence models against permutation-invariant baselines.
# 7. Retrain shuffled and reversed controls rather than only corrupting evaluation data.
# 8. Select operational warning thresholds under a false-positive-rate constraint.
# 9. Audit cross-regime transfer, order dependence, novelty and calibration.
#
# =================================================================================================

import os
import gc
import json
import math
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn

from scipy.stats import spearmanr

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    brier_score_loss,
)
from sklearn.preprocessing import StandardScaler

from transformer_lens import HookedTransformer


warnings.filterwarnings("ignore")


# =================================================================================================
# 1. CONFIGURATION
# =================================================================================================

@dataclass
class Config:
    milestone: str = "M20.6.3"

    title: str = (
        "Observable Feedback and Event-Onset Forecasting"
    )

    seed: int = 206300

    model_name: str = "Qwen/Qwen2.5-0.5B"

    output_file: str = (
        "m20_6_3_observable_feedback_event_onset.json"
    )

    # Transformer layers inspected during observability selection.
    candidate_layers: Tuple[int, ...] = (
        0, 3, 6, 9, 12, 15, 18, 21
    )

    # Temporal configuration.
    history_length: int = 8
    horizon: int = 6
    stream_length: int = 96

    # Dataset sizes.
    n_train_streams: int = 24
    n_calibration_streams: int = 10
    n_validation_streams: int = 10
    n_test_streams_per_regime: int = 10

    # Latent dynamics.
    initial_state_std: float = 0.25
    process_noise_std: float = 0.14
    observation_noise_std: float = 0.035

    # Event onset:
    # positive only if currently safe and |z| crosses this level
    # inside the next horizon.
    latent_hazard_threshold: float = 0.72

    # Activation compression.
    activation_pca_dim: int = 12

    # Ridge/logistic regularization.
    ridge_alpha: float = 3.0
    logistic_c: float = 0.5

    # GRU.
    gru_hidden_dim: int = 48
    gru_layers: int = 1
    gru_dropout: float = 0.0
    gru_epochs: int = 45
    gru_batch_size: int = 128
    gru_learning_rate: float = 1e-3
    gru_weight_decay: float = 1e-4
    gru_patience: int = 8

    # Activation extraction.
    transformer_batch_size: int = 24

    # Warning threshold policy.
    maximum_calibration_fpr: float = 0.10

    # Order ablation.
    n_shuffle_controls: int = 5

    # Bootstrap.
    bootstrap_iterations: int = 300

    # Validation gates.
    minimum_point_rho: float = 0.25
    minimum_point_macro_rho: float = 0.20
    minimum_point_r2: float = 0.03

    minimum_hazard_auc: float = 0.65
    minimum_hazard_ap_gain: float = 0.05

    minimum_order_rho_advantage: float = 0.03
    minimum_order_auc_advantage: float = 0.02

    minimum_novelty_auc: float = 0.85
    minimum_novelty_tpr: float = 0.80
    maximum_novelty_fpr: float = 0.10


CFG = Config()

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

TEST_REGIMES = (
    "AR1_TRANSFER",
    "MEAN_REVERTING",
    "RANDOM_WALK",
    "OSCILLATORY",
    "ABRUPT",
    "COMPOUND_FACTORIAL",
)


# =================================================================================================
# 2. REPRODUCIBILITY
# =================================================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass


seed_everything(CFG.seed)


# =================================================================================================
# 3. GENERAL UTILITIES
# =================================================================================================

def to_builtin(value):
    if isinstance(value, dict):
        return {
            str(k): to_builtin(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(v)
            for v in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        value = float(value)

        if not np.isfinite(value):
            return None

        return value

    if isinstance(value, float):
        if not np.isfinite(value):
            return None

        return value

    if isinstance(value, torch.Tensor):
        return to_builtin(
            value.detach().cpu().numpy()
        )

    return value


def safe_spearman(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (
        len(y_true) < 3
        or np.std(y_true) < 1e-12
        or np.std(y_pred) < 1e-12
    ):
        return 0.0

    rho = spearmanr(
        y_true,
        y_pred,
    ).statistic

    if rho is None or not np.isfinite(rho):
        return 0.0

    return float(rho)


def safe_auroc(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> Optional[float]:
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(
        probability,
        dtype=float,
    )

    if len(np.unique(y_true)) < 2:
        return None

    return float(
        roc_auc_score(
            y_true,
            probability,
        )
    )


def safe_auprc(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> Optional[float]:
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(
        probability,
        dtype=float,
    )

    if len(np.unique(y_true)) < 2:
        return None

    return float(
        average_precision_score(
            y_true,
            probability,
        )
    )


def expected_calibration_error(
    y_true: np.ndarray,
    probability: np.ndarray,
    n_bins: int = 10,
) -> float:
    y_true = np.asarray(y_true, dtype=float)

    probability = np.clip(
        np.asarray(probability, dtype=float),
        0.0,
        1.0,
    )

    bins = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for index in range(n_bins):
        if index == n_bins - 1:
            mask = (
                (probability >= bins[index])
                & (probability <= bins[index + 1])
            )
        else:
            mask = (
                (probability >= bins[index])
                & (probability < bins[index + 1])
            )

        if not np.any(mask):
            continue

        confidence = float(
            np.mean(probability[mask])
        )

        accuracy = float(
            np.mean(y_true[mask])
        )

        ece += (
            np.mean(mask)
            * abs(confidence - accuracy)
        )

    return float(ece)


def macro_stream_rho(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    stream_ids: np.ndarray,
) -> float:
    values = []

    for stream_id in np.unique(stream_ids):
        mask = stream_ids == stream_id

        if np.sum(mask) < 3:
            continue

        values.append(
            safe_spearman(
                y_true[mask],
                y_pred[mask],
            )
        )

    if not values:
        return 0.0

    return float(np.mean(values))


def evaluate_point(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    stream_ids: np.ndarray,
) -> Dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return {
        "n": int(len(y_true)),
        "rho": safe_spearman(
            y_true,
            y_pred,
        ),
        "macro_stream_rho": macro_stream_rho(
            y_true,
            y_pred,
            stream_ids,
        ),
        "r2": float(
            r2_score(
                y_true,
                y_pred,
            )
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred,
                )
            )
        ),
    }


def evaluate_hazard(
    y_true: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> Dict:
    y_true = np.asarray(y_true, dtype=int)

    probability = np.clip(
        np.asarray(probability, dtype=float),
        0.0,
        1.0,
    )

    prediction = (
        probability >= threshold
    ).astype(int)

    tp = int(
        np.sum(
            (prediction == 1)
            & (y_true == 1)
        )
    )

    fp = int(
        np.sum(
            (prediction == 1)
            & (y_true == 0)
        )
    )

    tn = int(
        np.sum(
            (prediction == 0)
            & (y_true == 0)
        )
    )

    fn = int(
        np.sum(
            (prediction == 0)
            & (y_true == 1)
        )
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0.0
    )

    return {
        "n": int(len(y_true)),
        "positives": int(np.sum(y_true)),
        "prevalence": float(np.mean(y_true)),
        "auroc": safe_auroc(
            y_true,
            probability,
        ),
        "auprc": safe_auprc(
            y_true,
            probability,
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probability,
            )
        ),
        "ece": expected_calibration_error(
            y_true,
            probability,
        ),
        "threshold": float(threshold),
        "recall": float(recall),
        "precision": float(precision),
        "fpr": float(fpr),
        "specificity": float(specificity),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


def sigmoid_numpy(logits: np.ndarray) -> np.ndarray:
    logits = np.clip(
        np.asarray(logits, dtype=float),
        -30.0,
        30.0,
    )

    return 1.0 / (
        1.0 + np.exp(-logits)
    )


# =================================================================================================
# 4. DYNAMICAL PROCESS
# =================================================================================================

def mechanism_baseline(
    a: float,
    b: float,
) -> float:
    return (
        0.55 * np.sin(1.4 * a)
        + 0.30 * np.cos(1.8 * b)
        + 0.18 * a * b
        + 0.08 * a ** 2
    )


def mechanism_sensitivity(
    a: float,
    b: float,
) -> float:
    # Positive and bounded sensitivity.
    return float(
        0.60
        + 0.16 * np.tanh(a)
        + 0.12 * np.sin(b)
        + 0.08 * np.cos(a - b)
    )


def evolve_latent(
    regime: str,
    previous: float,
    previous_previous: float,
    t: int,
    rng: np.random.Generator,
) -> float:
    noise = rng.normal(
        0.0,
        CFG.process_noise_std,
    )

    if regime in (
        "TRAIN_AR1",
        "CALIBRATION_AR1",
        "VALIDATION_AR1",
        "AR1_TRANSFER",
    ):
        value = 0.90 * previous + noise

    elif regime == "MEAN_REVERTING":
        value = (
            0.68 * previous
            + 0.05 * np.sin(t / 7.0)
            + noise
        )

    elif regime == "RANDOM_WALK":
        value = previous + 0.75 * noise

    elif regime == "OSCILLATORY":
        value = (
            1.42 * previous
            - 0.72 * previous_previous
            + 0.80 * noise
        )

    elif regime == "ABRUPT":
        jump = 0.0

        if rng.random() < 0.055:
            jump = rng.choice(
                [-1.0, 1.0]
            ) * rng.uniform(
                0.55,
                1.10,
            )

        value = (
            0.84 * previous
            + noise
            + jump
        )

    elif regime == "COMPOUND_FACTORIAL":
        jump = 0.0

        if rng.random() < 0.045:
            jump = rng.choice(
                [-1.0, 1.0]
            ) * rng.uniform(
                0.45,
                0.95,
            )

        periodic = (
            0.10 * np.sin(t / 3.3)
        )

        value = (
            0.88 * previous
            + periodic
            + noise
            + jump
        )

    else:
        raise ValueError(
            f"Unknown regime: {regime}"
        )

    return float(
        np.clip(
            value,
            -1.65,
            1.65,
        )
    )


# =================================================================================================
# 5. OBSERVABLE FEEDBACK PROMPTS
# =================================================================================================

def standard_feedback_prompt(row: Dict) -> str:
    return (
        "You are monitoring a numerical prediction system.\n"
        f"Input A: {row['a']:+.5f}\n"
        f"Input B: {row['b']:+.5f}\n"
        f"System prediction: {row['baseline_prediction']:+.5f}\n"
        f"Observed environmental outcome: {row['observed_outcome']:+.5f}\n"
        f"Signed prediction error: {row['signed_error']:+.5f}\n"
        f"Absolute prediction error: {row['absolute_error']:.5f}\n"
        "Update the internal diagnostic state using this feedback.\n"
        "Diagnostic state:"
    )


def novel_feedback_prompt(row: Dict) -> str:
    # Same semantics, unseen structure and vocabulary.
    return (
        "<telemetry_packet>\n"
        f"beta_channel={row['b']:+.5f}; "
        f"alpha_channel={row['a']:+.5f}\n"
        f"forecast::{row['baseline_prediction']:+.5f}\n"
        f"measurement::{row['observed_outcome']:+.5f}\n"
        f"delta_signed::{row['signed_error']:+.5f}\n"
        f"delta_magnitude::{row['absolute_error']:.5f}\n"
        "</telemetry_packet>\n"
        "Infer the updated health-state representation.\n"
        "state_vector:"
    )


# =================================================================================================
# 6. STREAM GENERATION
# =================================================================================================

def generate_stream(
    stream_id: int,
    regime: str,
    seed: int,
    allow_novel_format: bool,
) -> Dict:
    rng = np.random.default_rng(seed)

    length = CFG.stream_length

    latent = np.zeros(
        length,
        dtype=np.float32,
    )

    latent[0] = rng.normal(
        0.0,
        CFG.initial_state_std,
    )

    latent[1] = (
        0.90 * latent[0]
        + rng.normal(
            0.0,
            CFG.process_noise_std,
        )
    )

    for t in range(2, length):
        latent[t] = evolve_latent(
            regime=regime,
            previous=float(latent[t - 1]),
            previous_previous=float(
                latent[t - 2]
            ),
            t=t,
            rng=rng,
        )

    rows = []

    for t in range(length):
        a = float(
            rng.uniform(-1.6, 1.6)
        )

        b = float(
            rng.uniform(-1.6, 1.6)
        )

        baseline = float(
            mechanism_baseline(a, b)
        )

        sensitivity = float(
            mechanism_sensitivity(a, b)
        )

        observed_outcome = float(
            baseline
            + sensitivity * latent[t]
            + rng.normal(
                0.0,
                CFG.observation_noise_std,
            )
        )

        signed_error = float(
            observed_outcome - baseline
        )

        absolute_error = float(
            abs(signed_error)
        )

        # Factorial novelty assignment:
        # format is sampled independently of z, error and hazard.
        novel_format = bool(
            allow_novel_format
            and rng.random() < 0.50
        )

        row = {
            "stream_id": int(stream_id),
            "time": int(t),
            "regime": regime,
            "a": a,
            "b": b,
            "baseline_prediction": baseline,
            "observed_outcome": observed_outcome,
            "signed_error": signed_error,
            "absolute_error": absolute_error,
            "sensitivity": sensitivity,
            "latent_z": float(latent[t]),
            "novel_format": int(novel_format),
        }

        if novel_format:
            prompt = novel_feedback_prompt(row)
        else:
            prompt = standard_feedback_prompt(row)

        row["prompt"] = prompt

        rows.append(row)

    return {
        "stream_id": int(stream_id),
        "regime": regime,
        "rows": rows,
        "latent_z": latent,
    }


def generate_group(
    group_name: str,
    regime: str,
    n_streams: int,
    seed_offset: int,
    allow_novel_format: bool = False,
) -> Dict:
    streams = []

    for stream_id in range(n_streams):
        streams.append(
            generate_stream(
                stream_id=stream_id,
                regime=regime,
                seed=(
                    CFG.seed
                    + seed_offset
                    + 1009 * stream_id
                ),
                allow_novel_format=(
                    allow_novel_format
                ),
            )
        )

    return {
        "name": group_name,
        "regime": regime,
        "streams": streams,
    }


print("=" * 120)
print(
    "🔬 MILESTONE 20.6.3 — "
    "OBSERVABLE FEEDBACK AND EVENT-ONSET FORECASTING"
)
print("=" * 120)
print(f"Device                 : {DEVICE.upper()}")
print(f"Model                  : {CFG.model_name}")
print(f"History length         : {CFG.history_length}")
print(f"Forecast horizon       : {CFG.horizon}")
print(f"Latent event threshold : {CFG.latent_hazard_threshold}")
print(f"Train streams          : {CFG.n_train_streams}")
print(f"Calibration streams    : {CFG.n_calibration_streams}")
print(f"Validation streams     : {CFG.n_validation_streams}")
print(
    f"Test streams/regime    : "
    f"{CFG.n_test_streams_per_regime}"
)
print(f"Global seed            : {CFG.seed}")
print("=" * 120)


train_group = generate_group(
    group_name="TRAIN",
    regime="TRAIN_AR1",
    n_streams=CFG.n_train_streams,
    seed_offset=10_000,
)

calibration_group = generate_group(
    group_name="CALIBRATION",
    regime="CALIBRATION_AR1",
    n_streams=CFG.n_calibration_streams,
    seed_offset=30_000,
)

validation_group = generate_group(
    group_name="VALIDATION",
    regime="VALIDATION_AR1",
    n_streams=CFG.n_validation_streams,
    seed_offset=50_000,
)

test_groups = {}

for regime_index, regime in enumerate(TEST_REGIMES):
    test_groups[regime] = generate_group(
        group_name=f"TEST_{regime}",
        regime=regime,
        n_streams=(
            CFG.n_test_streams_per_regime
        ),
        seed_offset=(
            100_000
            + 20_000 * regime_index
        ),
        allow_novel_format=(
            regime == "COMPOUND_FACTORIAL"
        ),
    )

print("✅ Independent train/calibration/validation/test groups generated.")


# =================================================================================================
# 7. LOAD TRANSFORMERLENS MODEL
# =================================================================================================

print("\n⏳ Loading TransformerLens model...")

model = HookedTransformer.from_pretrained(
    CFG.model_name,
    device=DEVICE,
    dtype=(
        torch.float16
        if DEVICE == "cuda"
        else torch.float32
    ),
)

model.eval()

if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = (
        model.tokenizer.eos_token
    )

model.tokenizer.padding_side = "left"

valid_layers = tuple(
    layer
    for layer in CFG.candidate_layers
    if layer < model.cfg.n_layers
)

if not valid_layers:
    raise RuntimeError(
        "No configured candidate layer exists "
        "in the loaded model."
    )

print(
    f"✅ Model loaded. Valid layers: "
    f"{list(valid_layers)}"
)


# =================================================================================================
# 8. ACTIVATION EXTRACTION
# =================================================================================================

def flatten_group_rows(group: Dict) -> List[Dict]:
    rows = []

    for stream in group["streams"]:
        rows.extend(stream["rows"])

    return rows


@torch.no_grad()
def extract_group_activations(
    group: Dict,
    layers: Tuple[int, ...],
) -> None:
    rows = flatten_group_rows(group)

    prompts = [
        row["prompt"]
        for row in rows
    ]

    activation_chunks = {
        layer: []
        for layer in layers
    }

    hook_names = {
        layer: f"blocks.{layer}.hook_resid_post"
        for layer in layers
    }

    requested_names = set(
        hook_names.values()
    )

    for start in range(
        0,
        len(prompts),
        CFG.transformer_batch_size,
    ):
        batch_prompts = prompts[
            start:
            start + CFG.transformer_batch_size
        ]

        tokenized = model.tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        )

        tokens = tokenized[
            "input_ids"
        ].to(DEVICE)

        attention_mask = tokenized[
            "attention_mask"
        ].to(DEVICE)

        _, cache = model.run_with_cache(
            tokens,
            attention_mask=attention_mask,
            names_filter=lambda name: (
                name in requested_names
            ),
            return_type="logits",
        )

        # Left padding guarantees the final token is valid
        # for every prompt.
        for layer in layers:
            tensor = cache[
                hook_names[layer]
            ][:, -1, :]

            activation_chunks[layer].append(
                tensor.float().cpu().numpy()
            )

        del cache
        del tokens
        del attention_mask

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    group["activations"] = {
        layer: np.concatenate(
            activation_chunks[layer],
            axis=0,
        ).astype(np.float32)
        for layer in layers
    }

    row_index = 0

    for stream in group["streams"]:
        stream_length = len(stream["rows"])

        stream["activations"] = {
            layer: group["activations"][layer][
                row_index:
                row_index + stream_length
            ]
            for layer in layers
        }

        row_index += stream_length


print("\n🔬 Extracting observable-feedback activations...")

extract_group_activations(
    train_group,
    valid_layers,
)

extract_group_activations(
    calibration_group,
    valid_layers,
)

extract_group_activations(
    validation_group,
    valid_layers,
)

for regime in TEST_REGIMES:
    print(f"  Extracting {regime}...")

    extract_group_activations(
        test_groups[regime],
        valid_layers,
    )


# =================================================================================================
# 9. FEEDBACK OBSERVABILITY LAYER SELECTION
# =================================================================================================

def flattened_latent(group: Dict) -> np.ndarray:
    return np.concatenate([
        stream["latent_z"]
        for stream in group["streams"]
    ]).astype(np.float32)


train_latent_flat = flattened_latent(
    train_group
)

calibration_latent_flat = flattened_latent(
    calibration_group
)

layer_observability_results = {}

print("\n" + "=" * 120)
print("🧪 TEST A — POST-FEEDBACK LATENT OBSERVABILITY")
print("=" * 120)

for layer in valid_layers:
    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_group["activations"][layer]
    )

    X_calibration = scaler.transform(
        calibration_group["activations"][layer]
    )

    layer_model = Ridge(
        alpha=CFG.ridge_alpha
    )

    layer_model.fit(
        X_train,
        train_latent_flat,
    )

    prediction = layer_model.predict(
        X_calibration
    )

    metrics = evaluate_point(
        calibration_latent_flat,
        prediction,
        np.repeat(
            np.arange(
                CFG.n_calibration_streams
            ),
            CFG.stream_length,
        ),
    )

    layer_observability_results[layer] = metrics

    print(
        f"Layer {layer:<3} | "
        f"rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f} | "
        f"MAE={metrics['mae']: .4f}"
    )


BEST_FEEDBACK_LAYER = max(
    valid_layers,
    key=lambda layer: (
        layer_observability_results[layer][
            "macro_stream_rho"
        ],
        layer_observability_results[layer]["rho"],
        layer_observability_results[layer]["r2"],
    ),
)

print(
    f"\n🏆 Selected post-feedback layer: "
    f"{BEST_FEEDBACK_LAYER}"
)


# =================================================================================================
# 10. FIT ACTIVATION PREPROCESSOR ON TRAIN ONLY
# =================================================================================================

activation_scaler = StandardScaler()

train_activation_scaled = (
    activation_scaler.fit_transform(
        train_group["activations"][
            BEST_FEEDBACK_LAYER
        ]
    )
)

pca_components = min(
    CFG.activation_pca_dim,
    train_activation_scaled.shape[0] - 1,
    train_activation_scaled.shape[1],
)

activation_pca = PCA(
    n_components=pca_components,
    random_state=CFG.seed,
)

activation_pca.fit(
    train_activation_scaled
)


def attach_compressed_activations(
    group: Dict,
) -> None:
    for stream in group["streams"]:
        activation = stream[
            "activations"
        ][BEST_FEEDBACK_LAYER]

        scaled = activation_scaler.transform(
            activation
        )

        compressed = activation_pca.transform(
            scaled
        ).astype(np.float32)

        stream["activation_pca"] = compressed


attach_compressed_activations(train_group)
attach_compressed_activations(calibration_group)
attach_compressed_activations(validation_group)

for regime in TEST_REGIMES:
    attach_compressed_activations(
        test_groups[regime]
    )


# =================================================================================================
# 11. TEMPORAL DATASET CONSTRUCTION
# =================================================================================================

def row_sensor_vector(
    row: Dict,
    activation_vector: np.ndarray,
) -> np.ndarray:
    sensitivity = max(
        abs(row["sensitivity"]),
        1e-3,
    )

    direct_latent_estimate = (
        row["signed_error"]
        / sensitivity
    )

    return np.concatenate([
        np.asarray([
            row["signed_error"],
            row["absolute_error"],
            row["sensitivity"],
            direct_latent_estimate,
            row["a"],
            row["b"],
            row["baseline_prediction"],
            row["observed_outcome"],
        ], dtype=np.float32),
        activation_vector.astype(np.float32),
    ])


def build_temporal_dataset(
    group: Dict,
) -> Dict:
    sequences = []
    point_targets = []
    hazard_targets = []
    hazard_eligible = []
    stream_ids = []
    times = []
    current_latent = []
    future_latent = []
    format_labels = []

    k = CFG.history_length
    h = CFG.horizon
    threshold = CFG.latent_hazard_threshold

    for stream in group["streams"]:
        rows = stream["rows"]
        latent = stream["latent_z"]
        activation = stream["activation_pca"]

        row_vectors = np.stack([
            row_sensor_vector(
                rows[t],
                activation[t],
            )
            for t in range(len(rows))
        ]).astype(np.float32)

        for t in range(
            k - 1,
            len(rows) - h,
        ):
            sequence = row_vectors[
                t - k + 1:
                t + 1
            ]

            point_target = float(
                latent[t + h]
            )

            currently_safe = bool(
                abs(latent[t]) <= threshold
            )

            future_path = latent[
                t + 1:
                t + h + 1
            ]

            future_crossing = bool(
                np.any(
                    np.abs(future_path)
                    > threshold
                )
            )

            hazard_target = int(
                currently_safe
                and future_crossing
            )

            sequences.append(sequence)
            point_targets.append(point_target)
            hazard_targets.append(hazard_target)
            hazard_eligible.append(
                int(currently_safe)
            )
            stream_ids.append(
                stream["stream_id"]
            )
            times.append(t)
            current_latent.append(
                float(latent[t])
            )
            future_latent.append(
                future_path.astype(
                    np.float32
                )
            )
            format_labels.append(
                int(rows[t]["novel_format"])
            )

    return {
        "X": np.asarray(
            sequences,
            dtype=np.float32,
        ),
        "y_point": np.asarray(
            point_targets,
            dtype=np.float32,
        ),
        "y_hazard": np.asarray(
            hazard_targets,
            dtype=np.int64,
        ),
        "hazard_eligible": np.asarray(
            hazard_eligible,
            dtype=bool,
        ),
        "stream_ids": np.asarray(
            stream_ids,
            dtype=np.int64,
        ),
        "times": np.asarray(
            times,
            dtype=np.int64,
        ),
        "current_latent": np.asarray(
            current_latent,
            dtype=np.float32,
        ),
        "future_latent": np.asarray(
            future_latent,
            dtype=np.float32,
        ),
        "novel_format": np.asarray(
            format_labels,
            dtype=np.int64,
        ),
    }


train_dataset = build_temporal_dataset(
    train_group
)

calibration_dataset = build_temporal_dataset(
    calibration_group
)

validation_dataset = build_temporal_dataset(
    validation_group
)

test_datasets = {
    regime: build_temporal_dataset(
        test_groups[regime]
    )
    for regime in TEST_REGIMES
}

INPUT_DIM = int(
    train_dataset["X"].shape[-1]
)

print("\nTemporal dataset shapes:")
print(f"  Train       : {train_dataset['X'].shape}")
print(f"  Calibration : {calibration_dataset['X'].shape}")
print(f"  Validation  : {validation_dataset['X'].shape}")
print(f"  Input dim   : {INPUT_DIM}")


# =================================================================================================
# 12. TEMPORAL FEATURE SCALING
# =================================================================================================

temporal_scaler = StandardScaler()

temporal_scaler.fit(
    train_dataset["X"].reshape(
        -1,
        INPUT_DIM,
    )
)


def scale_sequence_dataset(
    dataset: Dict,
) -> np.ndarray:
    original_shape = dataset["X"].shape

    scaled = temporal_scaler.transform(
        dataset["X"].reshape(
            -1,
            INPUT_DIM,
        )
    )

    return scaled.reshape(
        original_shape
    ).astype(np.float32)


X_train = scale_sequence_dataset(
    train_dataset
)

X_calibration = scale_sequence_dataset(
    calibration_dataset
)

X_validation = scale_sequence_dataset(
    validation_dataset
)

X_tests = {
    regime: scale_sequence_dataset(
        test_datasets[regime]
    )
    for regime in TEST_REGIMES
}


# =================================================================================================
# 13. PERMUTATION-INVARIANT AND STRUCTURED FEATURES
# =================================================================================================

def latest_features(
    X: np.ndarray,
) -> np.ndarray:
    return X[:, -1, :]


def bag_features(
    X: np.ndarray,
) -> np.ndarray:
    return np.concatenate([
        np.mean(X, axis=1),
        np.std(X, axis=1),
        np.min(X, axis=1),
        np.max(X, axis=1),
    ], axis=1)


def slope_features(
    X: np.ndarray,
) -> np.ndarray:
    k = X.shape[1]

    time = np.arange(
        k,
        dtype=np.float32,
    )

    centered_time = (
        time - np.mean(time)
    )

    denominator = float(
        np.sum(centered_time ** 2)
    )

    centered_X = (
        X - np.mean(
            X,
            axis=1,
            keepdims=True,
        )
    )

    slopes = np.sum(
        centered_X
        * centered_time[
            None, :, None
        ],
        axis=1,
    ) / max(denominator, 1e-8)

    return np.concatenate([
        X[:, -1, :],
        np.mean(X, axis=1),
        slopes,
    ], axis=1)


# =================================================================================================
# 14. GRU MODELS
# =================================================================================================

class PointGRU(nn.Module):
    def __init__(
        self,
        input_dim: int,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=CFG.gru_hidden_dim,
            num_layers=CFG.gru_layers,
            batch_first=True,
            dropout=(
                CFG.gru_dropout
                if CFG.gru_layers > 1
                else 0.0
            ),
        )

        self.head = nn.Sequential(
            nn.LayerNorm(
                CFG.gru_hidden_dim
            ),
            nn.Linear(
                CFG.gru_hidden_dim,
                CFG.gru_hidden_dim // 2,
            ),
            nn.GELU(),
            nn.Linear(
                CFG.gru_hidden_dim // 2,
                1,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        output, _ = self.gru(x)

        return self.head(
            output[:, -1, :]
        ).squeeze(-1)


class HazardGRU(nn.Module):
    def __init__(
        self,
        input_dim: int,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=CFG.gru_hidden_dim,
            num_layers=CFG.gru_layers,
            batch_first=True,
            dropout=(
                CFG.gru_dropout
                if CFG.gru_layers > 1
                else 0.0
            ),
        )

        self.head = nn.Sequential(
            nn.LayerNorm(
                CFG.gru_hidden_dim
            ),
            nn.Linear(
                CFG.gru_hidden_dim,
                CFG.gru_hidden_dim // 2,
            ),
            nn.GELU(),
            nn.Linear(
                CFG.gru_hidden_dim // 2,
                1,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        output, _ = self.gru(x)

        return self.head(
            output[:, -1, :]
        ).squeeze(-1)


def mini_batches(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    rng: np.random.Generator,
):
    indices = rng.permutation(
        len(X)
    )

    for start in range(
        0,
        len(indices),
        batch_size,
    ):
        batch_indices = indices[
            start:
            start + batch_size
        ]

        yield (
            torch.tensor(
                X[batch_indices],
                dtype=torch.float32,
                device=DEVICE,
            ),
            torch.tensor(
                y[batch_indices],
                dtype=torch.float32,
                device=DEVICE,
            ),
        )


@torch.no_grad()
def predict_point_gru(
    network: nn.Module,
    X: np.ndarray,
) -> np.ndarray:
    network.eval()

    outputs = []

    for start in range(
        0,
        len(X),
        CFG.gru_batch_size,
    ):
        batch = torch.tensor(
            X[
                start:
                start + CFG.gru_batch_size
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        outputs.append(
            network(batch)
            .float()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        outputs
    )


@torch.no_grad()
def predict_hazard_gru(
    network: nn.Module,
    X: np.ndarray,
) -> np.ndarray:
    network.eval()

    logits = []

    for start in range(
        0,
        len(X),
        CFG.gru_batch_size,
    ):
        batch = torch.tensor(
            X[
                start:
                start + CFG.gru_batch_size
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        logits.append(
            network(batch)
            .float()
            .cpu()
            .numpy()
        )

    return sigmoid_numpy(
        np.concatenate(logits)
    )


def train_point_gru(
    X_train_local: np.ndarray,
    y_train_local: np.ndarray,
    X_monitor: np.ndarray,
    y_monitor: np.ndarray,
    seed: int,
) -> PointGRU:
    seed_everything(seed)

    network = PointGRU(
        X_train_local.shape[-1]
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        network.parameters(),
        lr=CFG.gru_learning_rate,
        weight_decay=CFG.gru_weight_decay,
    )

    loss_function = nn.MSELoss()

    best_state = None
    best_monitor_loss = float("inf")
    patience_counter = 0

    rng = np.random.default_rng(seed)

    for _ in range(CFG.gru_epochs):
        network.train()

        for X_batch, y_batch in mini_batches(
            X_train_local,
            y_train_local,
            CFG.gru_batch_size,
            rng,
        ):
            optimizer.zero_grad(
                set_to_none=True
            )

            prediction = network(X_batch)

            loss = loss_function(
                prediction,
                y_batch,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                network.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

        monitor_prediction = (
            predict_point_gru(
                network,
                X_monitor,
            )
        )

        monitor_loss = float(
            mean_squared_error(
                y_monitor,
                monitor_prediction,
            )
        )

        if monitor_loss < (
            best_monitor_loss - 1e-5
        ):
            best_monitor_loss = monitor_loss

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in network.state_dict().items()
            }

            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CFG.gru_patience:
            break

    if best_state is not None:
        network.load_state_dict(
            best_state
        )

    return network


def train_hazard_gru(
    X_train_local: np.ndarray,
    y_train_local: np.ndarray,
    X_monitor: np.ndarray,
    y_monitor: np.ndarray,
    seed: int,
) -> HazardGRU:
    seed_everything(seed)

    network = HazardGRU(
        X_train_local.shape[-1]
    ).to(DEVICE)

    positive_count = max(
        int(np.sum(y_train_local == 1)),
        1,
    )

    negative_count = max(
        int(np.sum(y_train_local == 0)),
        1,
    )

    positive_weight = (
        negative_count
        / positive_count
    )

    loss_function = (
        nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                positive_weight,
                dtype=torch.float32,
                device=DEVICE,
            )
        )
    )

    optimizer = torch.optim.AdamW(
        network.parameters(),
        lr=CFG.gru_learning_rate,
        weight_decay=CFG.gru_weight_decay,
    )

    best_state = None
    best_score = -float("inf")
    patience_counter = 0

    rng = np.random.default_rng(seed)

    for _ in range(CFG.gru_epochs):
        network.train()

        for X_batch, y_batch in mini_batches(
            X_train_local,
            y_train_local,
            CFG.gru_batch_size,
            rng,
        ):
            optimizer.zero_grad(
                set_to_none=True
            )

            logits = network(X_batch)

            loss = loss_function(
                logits,
                y_batch,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                network.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

        monitor_probability = (
            predict_hazard_gru(
                network,
                X_monitor,
            )
        )

        monitor_auc = safe_auroc(
            y_monitor,
            monitor_probability,
        )

        monitor_ap = safe_auprc(
            y_monitor,
            monitor_probability,
        )

        score = (
            (monitor_auc or 0.0)
            + 0.25 * (monitor_ap or 0.0)
        )

        if score > best_score + 1e-5:
            best_score = score

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in network.state_dict().items()
            }

            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CFG.gru_patience:
            break

    if best_state is not None:
        network.load_state_dict(
            best_state
        )

    return network


# =================================================================================================
# 15. HAZARD-ELIGIBLE SUBSETS
# =================================================================================================

def hazard_subset(
    X: np.ndarray,
    dataset: Dict,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    mask = dataset["hazard_eligible"]

    return (
        X[mask],
        dataset["y_hazard"][mask],
        dataset["stream_ids"][mask],
    )


X_train_hazard, y_train_hazard, _ = (
    hazard_subset(
        X_train,
        train_dataset,
    )
)

X_calibration_hazard, y_calibration_hazard, _ = (
    hazard_subset(
        X_calibration,
        calibration_dataset,
    )
)

X_validation_hazard, y_validation_hazard, validation_hazard_stream_ids = (
    hazard_subset(
        X_validation,
        validation_dataset,
    )
)

print("\nHazard onset prevalence:")
print(
    f"  Train       : "
    f"{np.mean(y_train_hazard):.4f} "
    f"({np.sum(y_train_hazard)}/{len(y_train_hazard)})"
)
print(
    f"  Calibration : "
    f"{np.mean(y_calibration_hazard):.4f} "
    f"({np.sum(y_calibration_hazard)}/{len(y_calibration_hazard)})"
)
print(
    f"  Validation  : "
    f"{np.mean(y_validation_hazard):.4f} "
    f"({np.sum(y_validation_hazard)}/{len(y_validation_hazard)})"
)


# =================================================================================================
# 16. BASELINE MODELS
# =================================================================================================

point_models = {}

for name, feature_function in {
    "LATEST": latest_features,
    "BAG": bag_features,
    "SLOPE": slope_features,
}.items():
    regression = Ridge(
        alpha=CFG.ridge_alpha
    )

    regression.fit(
        feature_function(X_train),
        train_dataset["y_point"],
    )

    point_models[name] = {
        "model": regression,
        "feature_function": feature_function,
    }


hazard_models = {}

for name, feature_function in {
    "LATEST": latest_features,
    "BAG": bag_features,
    "SLOPE": slope_features,
}.items():
    classifier = LogisticRegression(
        C=CFG.logistic_c,
        class_weight="balanced",
        max_iter=2000,
        random_state=CFG.seed,
    )

    classifier.fit(
        feature_function(
            X_train_hazard
        ),
        y_train_hazard,
    )

    hazard_models[name] = {
        "model": classifier,
        "feature_function": feature_function,
    }


# =================================================================================================
# 17. TRAIN ORDERED GRU MODELS
# =================================================================================================

print("\n⏳ Training ordered point GRU...")

ordered_point_gru = train_point_gru(
    X_train,
    train_dataset["y_point"],
    X_calibration,
    calibration_dataset["y_point"],
    seed=CFG.seed + 1,
)

print("⏳ Training ordered hazard GRU...")

ordered_hazard_gru = train_hazard_gru(
    X_train_hazard,
    y_train_hazard,
    X_calibration_hazard,
    y_calibration_hazard,
    seed=CFG.seed + 2,
)


# =================================================================================================
# 18. CALIBRATION-ONLY MODEL SELECTION
# =================================================================================================

point_calibration_results = {}

for name, artifact in point_models.items():
    prediction = artifact[
        "model"
    ].predict(
        artifact[
            "feature_function"
        ](X_calibration)
    )

    point_calibration_results[name] = (
        evaluate_point(
            calibration_dataset["y_point"],
            prediction,
            calibration_dataset["stream_ids"],
        )
    )

gru_point_calibration_prediction = (
    predict_point_gru(
        ordered_point_gru,
        X_calibration,
    )
)

point_calibration_results["GRU_ORDERED"] = (
    evaluate_point(
        calibration_dataset["y_point"],
        gru_point_calibration_prediction,
        calibration_dataset["stream_ids"],
    )
)


hazard_calibration_probabilities = {}
hazard_calibration_results = {}

for name, artifact in hazard_models.items():
    probability = artifact[
        "model"
    ].predict_proba(
        artifact[
            "feature_function"
        ](X_calibration_hazard)
    )[:, 1]

    hazard_calibration_probabilities[
        name
    ] = probability

    hazard_calibration_results[name] = (
        evaluate_hazard(
            y_calibration_hazard,
            probability,
            threshold=0.5,
        )
    )

gru_hazard_calibration_probability = (
    predict_hazard_gru(
        ordered_hazard_gru,
        X_calibration_hazard,
    )
)

hazard_calibration_probabilities[
    "GRU_ORDERED"
] = gru_hazard_calibration_probability

hazard_calibration_results[
    "GRU_ORDERED"
] = evaluate_hazard(
    y_calibration_hazard,
    gru_hazard_calibration_probability,
    threshold=0.5,
)


BEST_POINT_MODEL_NAME = max(
    point_calibration_results,
    key=lambda name: (
        point_calibration_results[name][
            "macro_stream_rho"
        ],
        point_calibration_results[name]["rho"],
        point_calibration_results[name]["r2"],
    ),
)

BEST_HAZARD_MODEL_NAME = max(
    hazard_calibration_results,
    key=lambda name: (
        hazard_calibration_results[name][
            "auroc"
        ] or 0.0,
        hazard_calibration_results[name][
            "auprc"
        ] or 0.0,
    ),
)


print("\n" + "=" * 120)
print("🧪 TEST B — CALIBRATION-ONLY MODEL SELECTION")
print("=" * 120)

print("\nPoint models:")

for name, metrics in point_calibration_results.items():
    print(
        f"  {name:<14} | "
        f"rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )

print("\nHazard models:")

for name, metrics in hazard_calibration_results.items():
    print(
        f"  {name:<14} | "
        f"AUC={(metrics['auroc'] or 0.0): .4f} | "
        f"AP={(metrics['auprc'] or 0.0): .4f} | "
        f"Prev={metrics['prevalence']: .4f}"
    )

print(
    f"\n🏆 Selected point model : "
    f"{BEST_POINT_MODEL_NAME}"
)
print(
    f"🏆 Selected hazard model: "
    f"{BEST_HAZARD_MODEL_NAME}"
)


# =================================================================================================
# 19. OPERATIONAL THRESHOLD UNDER FPR CONSTRAINT
# =================================================================================================

def select_threshold_under_fpr(
    y_true: np.ndarray,
    probability: np.ndarray,
    maximum_fpr: float,
) -> Dict:
    candidates = np.unique(
        np.concatenate([
            np.linspace(
                0.0,
                1.0,
                501,
            ),
            probability,
        ])
    )

    feasible = []

    for threshold in candidates:
        metrics = evaluate_hazard(
            y_true,
            probability,
            float(threshold),
        )

        if metrics["fpr"] <= maximum_fpr:
            feasible.append(metrics)

    if not feasible:
        return evaluate_hazard(
            y_true,
            probability,
            threshold=1.0,
        )

    return max(
        feasible,
        key=lambda metrics: (
            metrics["recall"],
            metrics["precision"],
            -metrics["brier"],
            metrics["threshold"],
        ),
    )


selected_hazard_calibration_probability = (
    hazard_calibration_probabilities[
        BEST_HAZARD_MODEL_NAME
    ]
)

threshold_selection = select_threshold_under_fpr(
    y_calibration_hazard,
    selected_hazard_calibration_probability,
    CFG.maximum_calibration_fpr,
)

HAZARD_DECISION_THRESHOLD = float(
    threshold_selection["threshold"]
)

print(
    f"\nOperational hazard threshold: "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)
print(
    f"Calibration FPR             : "
    f"{threshold_selection['fpr']:.4f}"
)
print(
    f"Calibration recall          : "
    f"{threshold_selection['recall']:.4f}"
)


# =================================================================================================
# 20. PREDICTION ROUTERS
# =================================================================================================

def point_predict(
    model_name: str,
    X: np.ndarray,
) -> np.ndarray:
    if model_name == "GRU_ORDERED":
        return predict_point_gru(
            ordered_point_gru,
            X,
        )

    artifact = point_models[
        model_name
    ]

    return artifact["model"].predict(
        artifact[
            "feature_function"
        ](X)
    )


def hazard_predict(
    model_name: str,
    X: np.ndarray,
) -> np.ndarray:
    if model_name == "GRU_ORDERED":
        return predict_hazard_gru(
            ordered_hazard_gru,
            X,
        )

    artifact = hazard_models[
        model_name
    ]

    return artifact[
        "model"
    ].predict_proba(
        artifact[
            "feature_function"
        ](X)
    )[:, 1]


# =================================================================================================
# 21. PRIVILEGED LATENT ORACLES
# =================================================================================================

def latent_history_features(
    dataset: Dict,
) -> np.ndarray:
    # Current latent state plus the known future-process-independent
    # information available up to time t.
    #
    # Because each temporal sample stores current z, we reconstruct
    # useful privileged features from current z and sensor history.
    current = dataset[
        "current_latent"
    ][:, None]

    return np.concatenate([
        current,
        current ** 2,
        np.abs(current),
    ], axis=1)


point_oracle = Ridge(
    alpha=0.1
)

point_oracle.fit(
    latent_history_features(
        train_dataset
    ),
    train_dataset["y_point"],
)

hazard_oracle = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=CFG.seed,
)

train_hazard_mask = train_dataset[
    "hazard_eligible"
]

hazard_oracle.fit(
    latent_history_features(
        train_dataset
    )[train_hazard_mask],
    train_dataset[
        "y_hazard"
    ][train_hazard_mask],
)


# =================================================================================================
# 22. VALIDATION EVALUATION
# =================================================================================================

validation_point_prediction = point_predict(
    BEST_POINT_MODEL_NAME,
    X_validation,
)

validation_point_metrics = evaluate_point(
    validation_dataset["y_point"],
    validation_point_prediction,
    validation_dataset["stream_ids"],
)

validation_hazard_probability = hazard_predict(
    BEST_HAZARD_MODEL_NAME,
    X_validation_hazard,
)

validation_hazard_metrics = evaluate_hazard(
    y_validation_hazard,
    validation_hazard_probability,
    HAZARD_DECISION_THRESHOLD,
)

validation_point_oracle_prediction = (
    point_oracle.predict(
        latent_history_features(
            validation_dataset
        )
    )
)

validation_point_oracle_metrics = (
    evaluate_point(
        validation_dataset["y_point"],
        validation_point_oracle_prediction,
        validation_dataset["stream_ids"],
    )
)

validation_hazard_mask = validation_dataset[
    "hazard_eligible"
]

validation_hazard_oracle_probability = (
    hazard_oracle.predict_proba(
        latent_history_features(
            validation_dataset
        )[validation_hazard_mask]
    )[:, 1]
)

validation_hazard_oracle_metrics = (
    evaluate_hazard(
        y_validation_hazard,
        validation_hazard_oracle_probability,
        HAZARD_DECISION_THRESHOLD,
    )
)


print("\n" + "=" * 120)
print("🧪 TEST C — UNTOUCHED VALIDATION")
print("=" * 120)

print("\nSelected point forecaster:")
print(
    f"  Model       : {BEST_POINT_MODEL_NAME}\n"
    f"  rho         : {validation_point_metrics['rho']:.4f}\n"
    f"  Macro rho   : {validation_point_metrics['macro_stream_rho']:.4f}\n"
    f"  R²          : {validation_point_metrics['r2']:.4f}\n"
    f"  MAE         : {validation_point_metrics['mae']:.4f}"
)

print("\nPrivileged point oracle:")
print(
    f"  rho         : {validation_point_oracle_metrics['rho']:.4f}\n"
    f"  Macro rho   : {validation_point_oracle_metrics['macro_stream_rho']:.4f}\n"
    f"  R²          : {validation_point_oracle_metrics['r2']:.4f}"
)

print("\nSelected hazard forecaster:")
print(
    f"  Model       : {BEST_HAZARD_MODEL_NAME}\n"
    f"  AUC         : {(validation_hazard_metrics['auroc'] or 0.0):.4f}\n"
    f"  AP          : {(validation_hazard_metrics['auprc'] or 0.0):.4f}\n"
    f"  Prevalence  : {validation_hazard_metrics['prevalence']:.4f}\n"
    f"  Recall      : {validation_hazard_metrics['recall']:.4f}\n"
    f"  FPR         : {validation_hazard_metrics['fpr']:.4f}\n"
    f"  Brier       : {validation_hazard_metrics['brier']:.4f}"
)

print("\nPrivileged hazard oracle:")
print(
    f"  AUC         : {(validation_hazard_oracle_metrics['auroc'] or 0.0):.4f}\n"
    f"  AP          : {(validation_hazard_oracle_metrics['auprc'] or 0.0):.4f}"
)


# =================================================================================================
# 23. ORDER ABLATIONS WITH INDEPENDENT RETRAINING
# =================================================================================================

def reverse_sequences(
    X: np.ndarray,
) -> np.ndarray:
    return X[:, ::-1, :].copy()


def shuffle_sequences(
    X: np.ndarray,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)

    output = np.empty_like(X)

    for index in range(len(X)):
        permutation = rng.permutation(
            X.shape[1]
        )

        output[index] = X[
            index,
            permutation,
            :,
        ]

    return output


order_ablation = {
    "point": {},
    "hazard": {},
}

print("\n" + "=" * 120)
print("🧪 TEST D — RETRAINED ORDER ABLATIONS")
print("=" * 120)

if BEST_POINT_MODEL_NAME == "GRU_ORDERED":
    reversed_point_gru = train_point_gru(
        reverse_sequences(X_train),
        train_dataset["y_point"],
        reverse_sequences(X_calibration),
        calibration_dataset["y_point"],
        seed=CFG.seed + 100,
    )

    reversed_point_prediction = (
        predict_point_gru(
            reversed_point_gru,
            reverse_sequences(X_validation),
        )
    )

    reversed_point_metrics = evaluate_point(
        validation_dataset["y_point"],
        reversed_point_prediction,
        validation_dataset["stream_ids"],
    )

    shuffled_point_metrics = []

    for control_index in range(
        CFG.n_shuffle_controls
    ):
        control_seed = (
            CFG.seed
            + 1000
            + control_index
        )

        shuffled_train = shuffle_sequences(
            X_train,
            control_seed,
        )

        shuffled_calibration = shuffle_sequences(
            X_calibration,
            control_seed + 10_000,
        )

        shuffled_validation = shuffle_sequences(
            X_validation,
            control_seed + 20_000,
        )

        shuffled_model = train_point_gru(
            shuffled_train,
            train_dataset["y_point"],
            shuffled_calibration,
            calibration_dataset["y_point"],
            seed=control_seed,
        )

        shuffled_prediction = (
            predict_point_gru(
                shuffled_model,
                shuffled_validation,
            )
        )

        shuffled_point_metrics.append(
            evaluate_point(
                validation_dataset["y_point"],
                shuffled_prediction,
                validation_dataset["stream_ids"],
            )
        )

        del shuffled_model

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    shuffle_rhos = np.asarray([
        result["rho"]
        for result in shuffled_point_metrics
    ])

    point_shuffle_p95 = float(
        np.quantile(
            shuffle_rhos,
            0.95,
        )
    )

    point_order_advantage = float(
        validation_point_metrics["rho"]
        - point_shuffle_p95
    )

    point_reverse_advantage = float(
        validation_point_metrics["rho"]
        - reversed_point_metrics["rho"]
    )

    order_ablation["point"] = {
        "applicable": True,
        "ordered": validation_point_metrics,
        "reversed": reversed_point_metrics,
        "shuffled_controls": (
            shuffled_point_metrics
        ),
        "shuffle_rho_p95": point_shuffle_p95,
        "advantage_over_shuffle_p95": (
            point_order_advantage
        ),
        "advantage_over_reverse": (
            point_reverse_advantage
        ),
    }

    print(
        f"Point | "
        f"Ordered={validation_point_metrics['rho']:.4f} | "
        f"Shuffle P95={point_shuffle_p95:.4f} | "
        f"Reverse={reversed_point_metrics['rho']:.4f} | "
        f"Δshuffle={point_order_advantage:.4f}"
    )

else:
    order_ablation["point"] = {
        "applicable": False,
        "reason": (
            "Selected point model is permutation-invariant "
            "or non-recurrent."
        ),
    }

    print(
        "Point | Order test not applicable because "
        f"{BEST_POINT_MODEL_NAME} was selected."
    )


if BEST_HAZARD_MODEL_NAME == "GRU_ORDERED":
    reversed_train_hazard = reverse_sequences(
        X_train_hazard
    )

    reversed_calibration_hazard = (
        reverse_sequences(
            X_calibration_hazard
        )
    )

    reversed_validation_hazard = (
        reverse_sequences(
            X_validation_hazard
        )
    )

    reversed_hazard_gru = train_hazard_gru(
        reversed_train_hazard,
        y_train_hazard,
        reversed_calibration_hazard,
        y_calibration_hazard,
        seed=CFG.seed + 200,
    )

    reversed_hazard_probability = (
        predict_hazard_gru(
            reversed_hazard_gru,
            reversed_validation_hazard,
        )
    )

    reversed_hazard_metrics = (
        evaluate_hazard(
            y_validation_hazard,
            reversed_hazard_probability,
            HAZARD_DECISION_THRESHOLD,
        )
    )

    shuffled_hazard_metrics = []

    for control_index in range(
        CFG.n_shuffle_controls
    ):
        control_seed = (
            CFG.seed
            + 2000
            + control_index
        )

        shuffled_train = shuffle_sequences(
            X_train_hazard,
            control_seed,
        )

        shuffled_calibration = shuffle_sequences(
            X_calibration_hazard,
            control_seed + 10_000,
        )

        shuffled_validation = shuffle_sequences(
            X_validation_hazard,
            control_seed + 20_000,
        )

        shuffled_model = train_hazard_gru(
            shuffled_train,
            y_train_hazard,
            shuffled_calibration,
            y_calibration_hazard,
            seed=control_seed,
        )

        shuffled_probability = (
            predict_hazard_gru(
                shuffled_model,
                shuffled_validation,
            )
        )

        shuffled_hazard_metrics.append(
            evaluate_hazard(
                y_validation_hazard,
                shuffled_probability,
                HAZARD_DECISION_THRESHOLD,
            )
        )

        del shuffled_model

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    shuffle_aucs = np.asarray([
        result["auroc"] or 0.0
        for result in shuffled_hazard_metrics
    ])

    hazard_shuffle_p95 = float(
        np.quantile(
            shuffle_aucs,
            0.95,
        )
    )

    hazard_order_advantage = float(
        (validation_hazard_metrics["auroc"] or 0.0)
        - hazard_shuffle_p95
    )

    hazard_reverse_advantage = float(
        (validation_hazard_metrics["auroc"] or 0.0)
        - (
            reversed_hazard_metrics[
                "auroc"
            ] or 0.0
        )
    )

    order_ablation["hazard"] = {
        "applicable": True,
        "ordered": validation_hazard_metrics,
        "reversed": reversed_hazard_metrics,
        "shuffled_controls": (
            shuffled_hazard_metrics
        ),
        "shuffle_auc_p95": hazard_shuffle_p95,
        "advantage_over_shuffle_p95": (
            hazard_order_advantage
        ),
        "advantage_over_reverse": (
            hazard_reverse_advantage
        ),
    }

    print(
        f"Hazard | "
        f"Ordered={(validation_hazard_metrics['auroc'] or 0.0):.4f} | "
        f"Shuffle P95={hazard_shuffle_p95:.4f} | "
        f"Reverse={(reversed_hazard_metrics['auroc'] or 0.0):.4f} | "
        f"Δshuffle={hazard_order_advantage:.4f}"
    )

else:
    order_ablation["hazard"] = {
        "applicable": False,
        "reason": (
            "Selected hazard model is permutation-invariant "
            "or non-recurrent."
        ),
    }

    print(
        "Hazard | Order test not applicable because "
        f"{BEST_HAZARD_MODEL_NAME} was selected."
    )


# =================================================================================================
# 24. LEAD-TIME AUDIT FOR EVENT ONSETS
# =================================================================================================

def event_onset_lead_time_audit(
    dataset: Dict,
    probability_eligible: np.ndarray,
    threshold: float,
) -> Dict:
    eligible_indices = np.flatnonzero(
        dataset["hazard_eligible"]
    )

    alert = (
        probability_eligible >= threshold
    )

    alert_by_sample = {
        int(index): bool(value)
        for index, value in zip(
            eligible_indices,
            alert,
        )
    }

    detected_leads = []
    total_events = 0
    detected_events = 0
    total_alerts = int(np.sum(alert))
    false_alerts = 0

    for local_index, sample_index in enumerate(
        eligible_indices
    ):
        target = int(
            dataset["y_hazard"][sample_index]
        )

        predicted_alert = bool(
            alert_by_sample[sample_index]
        )

        if target == 1:
            total_events += 1

            if predicted_alert:
                detected_events += 1

                future_path = dataset[
                    "future_latent"
                ][sample_index]

                crossing_positions = np.flatnonzero(
                    np.abs(future_path)
                    > CFG.latent_hazard_threshold
                )

                if len(crossing_positions) > 0:
                    # Position 0 means that crossing occurs at t+1.
                    lead_time = int(
                        crossing_positions[0] + 1
                    )

                    detected_leads.append(
                        lead_time
                    )

        elif predicted_alert:
            false_alerts += 1

    event_recall = (
        detected_events / total_events
        if total_events > 0
        else 0.0
    )

    false_alert_fraction = (
        false_alerts / total_alerts
        if total_alerts > 0
        else 0.0
    )

    return {
        "total_positive_windows": int(total_events),
        "detected_positive_windows": int(detected_events),
        "window_recall": float(event_recall),
        "total_alert_samples": int(total_alerts),
        "false_alert_samples": int(false_alerts),
        "false_alert_fraction": float(
            false_alert_fraction
        ),
        "mean_detected_lead_time": (
            float(np.mean(detected_leads))
            if detected_leads
            else None
        ),
        "median_detected_lead_time": (
            float(np.median(detected_leads))
            if detected_leads
            else None
        ),
        "minimum_detected_lead_time": (
            int(np.min(detected_leads))
            if detected_leads
            else None
        ),
        "maximum_detected_lead_time": (
            int(np.max(detected_leads))
            if detected_leads
            else None
        ),
    }


def unique_event_onset_audit(
    dataset: Dict,
    probability_eligible: np.ndarray,
    threshold: float,
) -> Dict:
    """
    Event-centric audit.

    Unlike the window-level audit, each physical threshold
    crossing is counted only once. An onset at time tau is
    considered detected if at least one alert was emitted in
    [tau - horizon, tau - 1].
    """

    eligible_mask = dataset[
        "hazard_eligible"
    ]

    eligible_indices = np.flatnonzero(
        eligible_mask
    )

    alert_flags = (
        probability_eligible >= threshold
    )

    probability_by_stream_time = {}

    for sample_index, probability, alert in zip(
        eligible_indices,
        probability_eligible,
        alert_flags,
    ):
        key = (
            int(dataset["stream_ids"][sample_index]),
            int(dataset["times"][sample_index]),
        )

        probability_by_stream_time[key] = {
            "probability": float(probability),
            "alert": bool(alert),
            "target": int(
                dataset["y_hazard"][sample_index]
            ),
        }

    unique_stream_ids = np.unique(
        dataset["stream_ids"]
    )

    total_onsets = 0
    detected_onsets = 0
    onset_lead_times = []
    onset_records = []

    for stream_id in unique_stream_ids:
        stream_mask = (
            dataset["stream_ids"] == stream_id
        )

        stream_indices = np.flatnonzero(
            stream_mask
        )

        if len(stream_indices) == 0:
            continue

        ordering = np.argsort(
            dataset["times"][stream_indices]
        )

        stream_indices = stream_indices[
            ordering
        ]

        # Recover threshold crossings from stored current latent
        # states and future paths. Candidate onset times may extend
        # beyond the final current-time sample.
        latent_by_time = {}

        for sample_index in stream_indices:
            current_time = int(
                dataset["times"][sample_index]
            )

            latent_by_time[current_time] = float(
                dataset["current_latent"][sample_index]
            )

            future_path = dataset[
                "future_latent"
            ][sample_index]

            for offset, value in enumerate(
                future_path,
                start=1,
            ):
                latent_by_time[
                    current_time + offset
                ] = float(value)

        available_times = sorted(
            latent_by_time.keys()
        )

        if len(available_times) < 2:
            continue

        onset_times = []

        for previous_time, current_time in zip(
            available_times[:-1],
            available_times[1:],
        ):
            # Require adjacent timestamps.
            if current_time != previous_time + 1:
                continue

            previous_safe = (
                abs(latent_by_time[previous_time])
                <= CFG.latent_hazard_threshold
            )

            current_degraded = (
                abs(latent_by_time[current_time])
                > CFG.latent_hazard_threshold
            )

            if previous_safe and current_degraded:
                onset_times.append(
                    current_time
                )

        for onset_time in onset_times:
            total_onsets += 1

            candidate_alerts = []

            for warning_time in range(
                onset_time - CFG.horizon,
                onset_time,
            ):
                key = (
                    int(stream_id),
                    int(warning_time),
                )

                if key not in probability_by_stream_time:
                    continue

                record = probability_by_stream_time[key]

                if record["alert"]:
                    lead_time = int(
                        onset_time - warning_time
                    )

                    candidate_alerts.append({
                        "warning_time": int(
                            warning_time
                        ),
                        "lead_time": lead_time,
                        "probability": float(
                            record["probability"]
                        ),
                    })

            if candidate_alerts:
                detected_onsets += 1

                # Earliest alert gives the largest lead time.
                selected_alert = max(
                    candidate_alerts,
                    key=lambda item: (
                        item["lead_time"],
                        item["probability"],
                    ),
                )

                onset_lead_times.append(
                    selected_alert["lead_time"]
                )

                onset_records.append({
                    "stream_id": int(stream_id),
                    "onset_time": int(onset_time),
                    "detected": True,
                    "lead_time": int(
                        selected_alert["lead_time"]
                    ),
                    "warning_probability": float(
                        selected_alert["probability"]
                    ),
                })

            else:
                onset_records.append({
                    "stream_id": int(stream_id),
                    "onset_time": int(onset_time),
                    "detected": False,
                    "lead_time": None,
                    "warning_probability": None,
                })

    recall = (
        detected_onsets / total_onsets
        if total_onsets > 0
        else 0.0
    )

    false_alert_count = 0
    total_alert_count = 0

    for sample_index, alert in zip(
        eligible_indices,
        alert_flags,
    ):
        if not alert:
            continue

        total_alert_count += 1

        if dataset["y_hazard"][sample_index] == 0:
            false_alert_count += 1

    false_alert_fraction = (
        false_alert_count / total_alert_count
        if total_alert_count > 0
        else 0.0
    )

    return {
        "total_unique_event_onsets": int(
            total_onsets
        ),
        "detected_unique_event_onsets": int(
            detected_onsets
        ),
        "event_recall": float(recall),
        "mean_detected_lead_time": (
            float(np.mean(onset_lead_times))
            if onset_lead_times
            else None
        ),
        "median_detected_lead_time": (
            float(np.median(onset_lead_times))
            if onset_lead_times
            else None
        ),
        "minimum_detected_lead_time": (
            int(np.min(onset_lead_times))
            if onset_lead_times
            else None
        ),
        "maximum_detected_lead_time": (
            int(np.max(onset_lead_times))
            if onset_lead_times
            else None
        ),
        "total_alert_samples": int(
            total_alert_count
        ),
        "false_alert_samples": int(
            false_alert_count
        ),
        "false_alert_fraction": float(
            false_alert_fraction
        ),
        "onset_records": onset_records,
    }


validation_window_warning_audit = (
    event_onset_lead_time_audit(
        validation_dataset,
        validation_hazard_probability,
        HAZARD_DECISION_THRESHOLD,
    )
)

validation_unique_onset_audit = (
    unique_event_onset_audit(
        validation_dataset,
        validation_hazard_probability,
        HAZARD_DECISION_THRESHOLD,
    )
)

print("\n" + "=" * 120)
print("🧪 TEST E — EVENT-ONSET WARNING AUDIT")
print("=" * 120)

print("\nWindow-level behavior:")

for key, value in (
    validation_window_warning_audit.items()
):
    print(f"  {key:<34}: {value}")

print("\nUnique physical event onsets:")

for key, value in (
    validation_unique_onset_audit.items()
):
    if key == "onset_records":
        continue

    print(f"  {key:<34}: {value}")


# =================================================================================================
# 25. CROSS-REGIME POINT AND HAZARD TRANSFER
# =================================================================================================

cross_regime_results = {}

print("\n" + "=" * 120)
print("🧪 TEST F — CROSS-REGIME TRANSFER")
print("=" * 120)

for regime in TEST_REGIMES:
    dataset = test_datasets[regime]
    X_regime = X_tests[regime]

    point_prediction = point_predict(
        BEST_POINT_MODEL_NAME,
        X_regime,
    )

    point_metrics = evaluate_point(
        dataset["y_point"],
        point_prediction,
        dataset["stream_ids"],
    )

    point_oracle_prediction = (
        point_oracle.predict(
            latent_history_features(
                dataset
            )
        )
    )

    point_oracle_metrics = evaluate_point(
        dataset["y_point"],
        point_oracle_prediction,
        dataset["stream_ids"],
    )

    # Privileged persistence baseline:
    # future z is predicted as current z.
    persistence_prediction = dataset[
        "current_latent"
    ]

    persistence_metrics = evaluate_point(
        dataset["y_point"],
        persistence_prediction,
        dataset["stream_ids"],
    )

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    X_regime_hazard = X_regime[
        hazard_mask
    ]

    y_regime_hazard = dataset[
        "y_hazard"
    ][hazard_mask]

    hazard_probability = hazard_predict(
        BEST_HAZARD_MODEL_NAME,
        X_regime_hazard,
    )

    hazard_metrics = evaluate_hazard(
        y_regime_hazard,
        hazard_probability,
        HAZARD_DECISION_THRESHOLD,
    )

    hazard_oracle_probability = (
        hazard_oracle.predict_proba(
            latent_history_features(
                dataset
            )[hazard_mask]
        )[:, 1]
    )

    hazard_oracle_metrics = evaluate_hazard(
        y_regime_hazard,
        hazard_oracle_probability,
        HAZARD_DECISION_THRESHOLD,
    )

    constant_probability = np.full(
        len(y_regime_hazard),
        fill_value=float(
            np.mean(y_train_hazard)
        ),
        dtype=np.float64,
    )

    constant_metrics = evaluate_hazard(
        y_regime_hazard,
        constant_probability,
        HAZARD_DECISION_THRESHOLD,
    )

    warning_audit = (
        unique_event_onset_audit(
            dataset,
            hazard_probability,
            HAZARD_DECISION_THRESHOLD,
        )
    )

    prevalence = float(
        np.mean(y_regime_hazard)
    )

    hazard_ap = (
        hazard_metrics["auprc"]
        if hazard_metrics["auprc"] is not None
        else 0.0
    )

    hazard_ap_gain = float(
        hazard_ap - prevalence
    )

    point_pass = bool(
        point_metrics["rho"]
        >= CFG.minimum_point_rho
        and point_metrics[
            "macro_stream_rho"
        ] >= CFG.minimum_point_macro_rho
        and point_metrics["r2"]
        >= CFG.minimum_point_r2
    )

    hazard_pass = bool(
        (
            hazard_metrics["auroc"]
            if hazard_metrics["auroc"] is not None
            else 0.0
        ) >= CFG.minimum_hazard_auc
        and hazard_ap_gain
        >= CFG.minimum_hazard_ap_gain
    )

    cross_regime_results[regime] = {
        "point": point_metrics,
        "point_persistence": (
            persistence_metrics
        ),
        "point_privileged_oracle": (
            point_oracle_metrics
        ),
        "hazard": hazard_metrics,
        "hazard_constant_baseline": (
            constant_metrics
        ),
        "hazard_privileged_oracle": (
            hazard_oracle_metrics
        ),
        "hazard_ap_gain_over_prevalence": (
            hazard_ap_gain
        ),
        "warning_audit": warning_audit,
        "point_gate_pass": point_pass,
        "hazard_gate_pass": hazard_pass,
    }

    print(f"\n{regime}")

    print(
        f"  Point  | "
        f"rho={point_metrics['rho']: .4f} | "
        f"Macro={point_metrics['macro_stream_rho']: .4f} | "
        f"R²={point_metrics['r2']: .4f} | "
        f"Persist={persistence_metrics['rho']: .4f} | "
        f"Oracle={point_oracle_metrics['rho']: .4f} | "
        f"Gate={'PASS' if point_pass else 'FAIL'}"
    )

    print(
        f"  Hazard | "
        f"AUC={(hazard_metrics['auroc'] or 0.0): .4f} | "
        f"AP={(hazard_metrics['auprc'] or 0.0): .4f} | "
        f"Prev={prevalence: .4f} | "
        f"ΔAP={hazard_ap_gain: .4f} | "
        f"FPR={hazard_metrics['fpr']: .4f} | "
        f"Gate={'PASS' if hazard_pass else 'FAIL'}"
    )


# =================================================================================================
# 26. NOVELTY SENSOR FITTED ONLY ON STANDARD TRAIN/CALIBRATION DATA
# =================================================================================================

def flattened_compressed_activations(
    group: Dict,
) -> np.ndarray:
    return np.concatenate([
        stream["activation_pca"]
        for stream in group["streams"]
    ], axis=0).astype(np.float32)


novelty_reference_scaler = StandardScaler()

novelty_reference_scaler.fit(
    flattened_compressed_activations(
        train_group
    )
)


def activation_novelty_score(
    compressed_activation: np.ndarray,
) -> np.ndarray:
    standardized = (
        novelty_reference_scaler.transform(
            compressed_activation
        )
    )

    # Root mean squared standardized displacement.
    return np.sqrt(
        np.mean(
            standardized ** 2,
            axis=1,
        )
    )


calibration_novelty_score = (
    activation_novelty_score(
        flattened_compressed_activations(
            calibration_group
        )
    )
)

NOVELTY_THRESHOLD = float(
    np.quantile(
        calibration_novelty_score,
        1.0 - CFG.maximum_novelty_fpr,
    )
)

compound_group = test_groups[
    "COMPOUND_FACTORIAL"
]

compound_activation = (
    flattened_compressed_activations(
        compound_group
    )
)

compound_novelty_scores = (
    activation_novelty_score(
        compound_activation
    )
)

compound_rows = flatten_group_rows(
    compound_group
)

compound_novelty_labels = np.asarray([
    row["novel_format"]
    for row in compound_rows
], dtype=np.int64)

novelty_auc = safe_auroc(
    compound_novelty_labels,
    compound_novelty_scores,
)

novelty_predictions = (
    compound_novelty_scores
    >= NOVELTY_THRESHOLD
).astype(np.int64)

novel_mask = (
    compound_novelty_labels == 1
)

standard_mask = (
    compound_novelty_labels == 0
)

novelty_tpr = (
    float(
        np.mean(
            novelty_predictions[novel_mask]
            == 1
        )
    )
    if np.any(novel_mask)
    else 0.0
)

novelty_fpr = (
    float(
        np.mean(
            novelty_predictions[standard_mask]
            == 1
        )
    )
    if np.any(standard_mask)
    else 0.0
)

novelty_sensor_metrics = {
    "threshold": NOVELTY_THRESHOLD,
    "auroc": novelty_auc,
    "novel_tpr": novelty_tpr,
    "standard_fpr": novelty_fpr,
    "n_standard": int(
        np.sum(standard_mask)
    ),
    "n_novel": int(
        np.sum(novel_mask)
    ),
}


# =================================================================================================
# 27. FACTORIAL NOVELTY–DEGRADATION INDEPENDENCE AUDIT
# =================================================================================================

compound_latent_flat = flattened_latent(
    compound_group
)

compound_degraded_labels = (
    np.abs(compound_latent_flat)
    > CFG.latent_hazard_threshold
).astype(np.int64)


def binary_phi_coefficient(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    first = np.asarray(
        first,
        dtype=np.float64,
    )

    second = np.asarray(
        second,
        dtype=np.float64,
    )

    if (
        np.std(first) < 1e-12
        or np.std(second) < 1e-12
    ):
        return 0.0

    return float(
        np.corrcoef(
            first,
            second,
        )[0, 1]
    )


format_degradation_phi = (
    binary_phi_coefficient(
        compound_novelty_labels,
        compound_degraded_labels,
    )
)

degradation_rate_standard = (
    float(
        np.mean(
            compound_degraded_labels[
                standard_mask
            ]
        )
    )
    if np.any(standard_mask)
    else 0.0
)

degradation_rate_novel = (
    float(
        np.mean(
            compound_degraded_labels[
                novel_mask
            ]
        )
    )
    if np.any(novel_mask)
    else 0.0
)

factorial_counts = {}

for format_value, format_name in (
    (0, "STANDARD"),
    (1, "NOVEL"),
):
    for degradation_value, degradation_name in (
        (0, "SAFE"),
        (1, "DEGRADED"),
    ):
        count = int(
            np.sum(
                (
                    compound_novelty_labels
                    == format_value
                )
                & (
                    compound_degraded_labels
                    == degradation_value
                )
            )
        )

        factorial_counts[
            f"{format_name}_{degradation_name}"
        ] = count


factorial_independence_metrics = {
    "phi_format_vs_degradation": (
        format_degradation_phi
    ),
    "absolute_phi": float(
        abs(format_degradation_phi)
    ),
    "degradation_rate_standard": (
        degradation_rate_standard
    ),
    "degradation_rate_novel": (
        degradation_rate_novel
    ),
    "absolute_rate_difference": float(
        abs(
            degradation_rate_standard
            - degradation_rate_novel
        )
    ),
    "cell_counts": factorial_counts,
}


print("\n" + "=" * 120)
print("🧪 TEST G — NOVELTY AND FACTORIAL INDEPENDENCE AUDIT")
print("=" * 120)

print(
    f"Novelty AUROC                : "
    f"{(novelty_auc or 0.0):.4f}"
)
print(
    f"Standard FPR                 : "
    f"{novelty_fpr:.4f}"
)
print(
    f"Novel TPR                    : "
    f"{novelty_tpr:.4f}"
)
print(
    f"Format/degradation phi       : "
    f"{format_degradation_phi:.4f}"
)
print(
    f"Standard degradation rate    : "
    f"{degradation_rate_standard:.4f}"
)
print(
    f"Novel degradation rate       : "
    f"{degradation_rate_novel:.4f}"
)


# =================================================================================================
# 28. FORMAT-CONDITIONAL PERFORMANCE IN COMPOUND REGIME
# =================================================================================================

compound_dataset = test_datasets[
    "COMPOUND_FACTORIAL"
]

compound_X = X_tests[
    "COMPOUND_FACTORIAL"
]

compound_point_prediction = point_predict(
    BEST_POINT_MODEL_NAME,
    compound_X,
)

compound_format_point_results = {}

for format_value, format_name in (
    (0, "STANDARD"),
    (1, "NOVEL"),
):
    mask = (
        compound_dataset["novel_format"]
        == format_value
    )

    if np.sum(mask) < 3:
        compound_format_point_results[
            format_name
        ] = None

        continue

    compound_format_point_results[
        format_name
    ] = evaluate_point(
        compound_dataset["y_point"][mask],
        compound_point_prediction[mask],
        compound_dataset["stream_ids"][mask],
    )


compound_hazard_mask = compound_dataset[
    "hazard_eligible"
]

compound_hazard_X = compound_X[
    compound_hazard_mask
]

compound_hazard_probability = hazard_predict(
    BEST_HAZARD_MODEL_NAME,
    compound_hazard_X,
)

compound_hazard_labels = compound_dataset[
    "y_hazard"
][compound_hazard_mask]

compound_hazard_formats = compound_dataset[
    "novel_format"
][compound_hazard_mask]

compound_format_hazard_results = {}

for format_value, format_name in (
    (0, "STANDARD"),
    (1, "NOVEL"),
):
    mask = (
        compound_hazard_formats
        == format_value
    )

    if (
        np.sum(mask) < 3
        or len(
            np.unique(
                compound_hazard_labels[mask]
            )
        ) < 2
    ):
        compound_format_hazard_results[
            format_name
        ] = None

        continue

    compound_format_hazard_results[
        format_name
    ] = evaluate_hazard(
        compound_hazard_labels[mask],
        compound_hazard_probability[mask],
        HAZARD_DECISION_THRESHOLD,
    )


print("\n" + "=" * 120)
print("🧪 TEST H — FORMAT-CONDITIONAL COMPOUND PERFORMANCE")
print("=" * 120)

for format_name, metrics in (
    compound_format_point_results.items()
):
    if metrics is None:
        print(
            f"Point  {format_name:<8}: insufficient samples"
        )

        continue

    print(
        f"Point  {format_name:<8} | "
        f"rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )

for format_name, metrics in (
    compound_format_hazard_results.items()
):
    if metrics is None:
        print(
            f"Hazard {format_name:<8}: "
            "insufficient class variation"
        )

        continue

    print(
        f"Hazard {format_name:<8} | "
        f"AUC={(metrics['auroc'] or 0.0): .4f} | "
        f"AP={(metrics['auprc'] or 0.0): .4f} | "
        f"Brier={metrics['brier']: .4f}"
    )


# =================================================================================================
# 29. BOOTSTRAP CONFIDENCE INTERVALS
# =================================================================================================

def percentile_interval(
    values: List[float],
    confidence: float = 0.95,
) -> Dict:
    finite_values = np.asarray([
        value
        for value in values
        if value is not None
        and np.isfinite(value)
    ], dtype=np.float64)

    if len(finite_values) == 0:
        return {
            "lower": None,
            "median": None,
            "upper": None,
            "n_valid": 0,
        }

    alpha = (
        1.0 - confidence
    ) / 2.0

    return {
        "lower": float(
            np.quantile(
                finite_values,
                alpha,
            )
        ),
        "median": float(
            np.quantile(
                finite_values,
                0.5,
            )
        ),
        "upper": float(
            np.quantile(
                finite_values,
                1.0 - alpha,
            )
        ),
        "n_valid": int(
            len(finite_values)
        ),
    }


def stream_bootstrap_point(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    stream_ids: np.ndarray,
    iterations: int,
    seed: int,
) -> Dict:
    rng = np.random.default_rng(seed)

    unique_streams = np.unique(
        stream_ids
    )

    rho_values = []
    macro_values = []
    r2_values = []

    for _ in range(iterations):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        sampled_true = []
        sampled_pred = []
        sampled_ids = []

        for bootstrap_id, stream_id in enumerate(
            sampled_streams
        ):
            mask = (
                stream_ids == stream_id
            )

            sampled_true.append(
                y_true[mask]
            )

            sampled_pred.append(
                y_pred[mask]
            )

            sampled_ids.append(
                np.full(
                    np.sum(mask),
                    bootstrap_id,
                    dtype=np.int64,
                )
            )

        bootstrap_true = np.concatenate(
            sampled_true
        )

        bootstrap_pred = np.concatenate(
            sampled_pred
        )

        bootstrap_ids = np.concatenate(
            sampled_ids
        )

        rho_values.append(
            safe_spearman(
                bootstrap_true,
                bootstrap_pred,
            )
        )

        macro_values.append(
            macro_stream_rho(
                bootstrap_true,
                bootstrap_pred,
                bootstrap_ids,
            )
        )

        try:
            r2_values.append(
                float(
                    r2_score(
                        bootstrap_true,
                        bootstrap_pred,
                    )
                )
            )
        except Exception:
            pass

    return {
        "rho": percentile_interval(
            rho_values
        ),
        "macro_stream_rho": (
            percentile_interval(
                macro_values
            )
        ),
        "r2": percentile_interval(
            r2_values
        ),
    }


def stream_bootstrap_hazard(
    y_true: np.ndarray,
    probability: np.ndarray,
    stream_ids: np.ndarray,
    iterations: int,
    seed: int,
) -> Dict:
    rng = np.random.default_rng(seed)

    unique_streams = np.unique(
        stream_ids
    )

    auc_values = []
    ap_values = []
    brier_values = []

    for _ in range(iterations):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        sampled_true = []
        sampled_probability = []

        for stream_id in sampled_streams:
            mask = (
                stream_ids == stream_id
            )

            sampled_true.append(
                y_true[mask]
            )

            sampled_probability.append(
                probability[mask]
            )

        bootstrap_true = np.concatenate(
            sampled_true
        )

        bootstrap_probability = np.concatenate(
            sampled_probability
        )

        auc = safe_auroc(
            bootstrap_true,
            bootstrap_probability,
        )

        ap = safe_auprc(
            bootstrap_true,
            bootstrap_probability,
        )

        if auc is not None:
            auc_values.append(auc)

        if ap is not None:
            ap_values.append(ap)

        brier_values.append(
            float(
                brier_score_loss(
                    bootstrap_true,
                    bootstrap_probability,
                )
            )
        )

    return {
        "auroc": percentile_interval(
            auc_values
        ),
        "auprc": percentile_interval(
            ap_values
        ),
        "brier": percentile_interval(
            brier_values
        ),
    }


validation_point_bootstrap = (
    stream_bootstrap_point(
        validation_dataset["y_point"],
        validation_point_prediction,
        validation_dataset["stream_ids"],
        CFG.bootstrap_iterations,
        CFG.seed + 50_000,
    )
)

validation_hazard_bootstrap = (
    stream_bootstrap_hazard(
        y_validation_hazard,
        validation_hazard_probability,
        validation_hazard_stream_ids,
        CFG.bootstrap_iterations,
        CFG.seed + 60_000,
    )
)

print("\n" + "=" * 120)
print("🧪 TEST I — STREAM-LEVEL BOOTSTRAP INTERVALS")
print("=" * 120)

print(
    "Point rho 95% CI       : "
    f"[{validation_point_bootstrap['rho']['lower']}, "
    f"{validation_point_bootstrap['rho']['upper']}]"
)

print(
    "Point macro rho 95% CI : "
    f"[{validation_point_bootstrap['macro_stream_rho']['lower']}, "
    f"{validation_point_bootstrap['macro_stream_rho']['upper']}]"
)

print(
    "Hazard AUC 95% CI      : "
    f"[{validation_hazard_bootstrap['auroc']['lower']}, "
    f"{validation_hazard_bootstrap['auroc']['upper']}]"
)

print(
    "Hazard AP 95% CI       : "
    f"[{validation_hazard_bootstrap['auprc']['lower']}, "
    f"{validation_hazard_bootstrap['auprc']['upper']}]"
)


# =================================================================================================
# 30. SCIENTIFIC GATES
# =================================================================================================

validation_hazard_prevalence = float(
    np.mean(y_validation_hazard)
)

validation_hazard_ap = (
    validation_hazard_metrics["auprc"]
    if validation_hazard_metrics["auprc"] is not None
    else 0.0
)

validation_hazard_ap_gain = float(
    validation_hazard_ap
    - validation_hazard_prevalence
)

point_oracle_forecastable = bool(
    validation_point_oracle_metrics["rho"]
    >= 0.60
    and validation_point_oracle_metrics["r2"]
    >= 0.20
)

hazard_oracle_forecastable = bool(
    (
        validation_hazard_oracle_metrics[
            "auroc"
        ] or 0.0
    ) >= 0.70
    and (
        (
            validation_hazard_oracle_metrics[
                "auprc"
            ] or 0.0
        )
        - validation_hazard_prevalence
    ) >= 0.05
)

feedback_observability_valid = bool(
    layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]["rho"] >= 0.50
    and layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]["r2"] >= 0.20
)

selected_point_valid = bool(
    validation_point_metrics["rho"]
    >= CFG.minimum_point_rho
    and validation_point_metrics[
        "macro_stream_rho"
    ] >= CFG.minimum_point_macro_rho
    and validation_point_metrics["r2"]
    >= CFG.minimum_point_r2
)

selected_hazard_valid = bool(
    (
        validation_hazard_metrics["auroc"]
        or 0.0
    ) >= CFG.minimum_hazard_auc
    and validation_hazard_ap_gain
    >= CFG.minimum_hazard_ap_gain
)

point_order_valid = bool(
    order_ablation["point"].get(
        "applicable",
        False,
    )
    and order_ablation["point"].get(
        "advantage_over_shuffle_p95",
        -np.inf,
    ) >= CFG.minimum_order_rho_advantage
    and order_ablation["point"].get(
        "advantage_over_reverse",
        -np.inf,
    ) > 0.0
)

hazard_order_valid = bool(
    order_ablation["hazard"].get(
        "applicable",
        False,
    )
    and order_ablation["hazard"].get(
        "advantage_over_shuffle_p95",
        -np.inf,
    ) >= CFG.minimum_order_auc_advantage
    and order_ablation["hazard"].get(
        "advantage_over_reverse",
        -np.inf,
    ) > 0.0
)

novelty_sensor_valid = bool(
    (novelty_auc or 0.0)
    >= CFG.minimum_novelty_auc
    and novelty_tpr
    >= CFG.minimum_novelty_tpr
    and novelty_fpr
    <= CFG.maximum_novelty_fpr
)

factorial_independence_valid = bool(
    abs(format_degradation_phi)
    <= 0.10
    and factorial_independence_metrics[
        "absolute_rate_difference"
    ] <= 0.10
    and all(
        count > 0
        for count in factorial_counts.values()
    )
)

operational_warning_valid = bool(
    validation_hazard_metrics["fpr"]
    <= CFG.maximum_calibration_fpr + 0.05
    and validation_unique_onset_audit[
        "event_recall"
    ] >= 0.60
)

point_pass_count = int(
    sum(
        result["point_gate_pass"]
        for result
        in cross_regime_results.values()
    )
)

hazard_pass_count = int(
    sum(
        result["hazard_gate_pass"]
        for result
        in cross_regime_results.values()
    )
)

# Strict majority: 4/6 rather than 3/6.
strict_majority_required = (
    len(TEST_REGIMES) // 2 + 1
)

point_transfer_majority = bool(
    point_pass_count
    >= strict_majority_required
)

hazard_transfer_majority = bool(
    hazard_pass_count
    >= strict_majority_required
)

compound_point_transfer = bool(
    cross_regime_results[
        "COMPOUND_FACTORIAL"
    ]["point_gate_pass"]
)

compound_hazard_transfer = bool(
    cross_regime_results[
        "COMPOUND_FACTORIAL"
    ]["hazard_gate_pass"]
)


scientific_gates = {
    "feedback_observability_valid": (
        feedback_observability_valid
    ),
    "point_task_oracle_forecastable": (
        point_oracle_forecastable
    ),
    "hazard_task_oracle_forecastable": (
        hazard_oracle_forecastable
    ),
    "selected_point_forecaster_valid": (
        selected_point_valid
    ),
    "selected_hazard_forecaster_valid": (
        selected_hazard_valid
    ),
    "selected_point_forecaster_uses_order": (
        point_order_valid
    ),
    "selected_hazard_forecaster_uses_order": (
        hazard_order_valid
    ),
    "operational_warning_policy_valid": (
        operational_warning_valid
    ),
    "novelty_sensor_valid": (
        novelty_sensor_valid
    ),
    "factorial_novelty_independence_valid": (
        factorial_independence_valid
    ),
    "point_transfer_strict_majority": (
        point_transfer_majority
    ),
    "hazard_transfer_strict_majority": (
        hazard_transfer_majority
    ),
    "compound_point_transfer": (
        compound_point_transfer
    ),
    "compound_hazard_transfer": (
        compound_hazard_transfer
    ),
}


# =================================================================================================
# 31. FINAL SCIENTIFIC DIAGNOSIS
# =================================================================================================

benchmark_construction_pass = bool(
    point_oracle_forecastable
    and hazard_oracle_forecastable
    and factorial_independence_valid
)

observable_self_model_pass = bool(
    feedback_observability_valid
    and selected_point_valid
)

temporal_order_learning_pass = bool(
    point_order_valid
    and hazard_order_valid
)

hazard_system_pass = bool(
    selected_hazard_valid
    and operational_warning_valid
)

cross_regime_transfer_pass = bool(
    point_transfer_majority
    and hazard_transfer_majority
)

full_milestone_pass = bool(
    benchmark_construction_pass
    and observable_self_model_pass
    and temporal_order_learning_pass
    and hazard_system_pass
    and cross_regime_transfer_pass
    and novelty_sensor_valid
)


def construct_final_diagnosis() -> str:
    failures = []

    if not point_oracle_forecastable:
        failures.append(
            "POINT_TARGET_NOT_FORECASTABLE"
        )

    if not hazard_oracle_forecastable:
        failures.append(
            "EVENT_ONSET_TARGET_NOT_FORECASTABLE"
        )

    if not feedback_observability_valid:
        failures.append(
            "POST_FEEDBACK_LATENT_NOT_OBSERVABLE"
        )

    if not selected_point_valid:
        failures.append(
            "POINT_FORECASTER_NOT_VALID"
        )

    if not selected_hazard_valid:
        failures.append(
            "HAZARD_FORECASTER_NOT_VALID"
        )

    if not point_order_valid:
        failures.append(
            "POINT_TEMPORAL_ORDER_NOT_ESTABLISHED"
        )

    if not hazard_order_valid:
        failures.append(
            "HAZARD_TEMPORAL_ORDER_NOT_ESTABLISHED"
        )

    if not operational_warning_valid:
        failures.append(
            "WARNING_POLICY_NOT_OPERATIONALLY_VALID"
        )

    if not factorial_independence_valid:
        failures.append(
            "NOVELTY_DEGRADATION_CONFOUNDING_DETECTED"
        )

    if not novelty_sensor_valid:
        failures.append(
            "NOVELTY_SENSOR_NOT_VALID"
        )

    if not point_transfer_majority:
        failures.append(
            "POINT_CROSS_REGIME_TRANSFER_FAILED"
        )

    if not hazard_transfer_majority:
        failures.append(
            "HAZARD_CROSS_REGIME_TRANSFER_FAILED"
        )

    if not failures:
        return (
            "PASS: observable feedback supports latent-state "
            "estimation, event-onset forecasting, temporal-order "
            "learning and strict-majority cross-regime transfer."
        )

    return " | ".join(failures)


final_diagnosis = (
    construct_final_diagnosis()
)

overall_statuses = {
    "benchmark_construction": (
        "PASS"
        if benchmark_construction_pass
        else "FAIL"
    ),
    "observable_feedback_self_model": (
        "PASS"
        if observable_self_model_pass
        else "FAIL"
    ),
    "temporal_order_learning": (
        "PASS"
        if temporal_order_learning_pass
        else "FAIL"
    ),
    "hazard_event_onset_system": (
        "PASS"
        if hazard_system_pass
        else "FAIL"
    ),
    "cross_regime_transfer": (
        "PASS"
        if cross_regime_transfer_pass
        else "FAIL"
    ),
    "novelty_detection": (
        "PASS"
        if novelty_sensor_valid
        else "FAIL"
    ),
    "overall": (
        "PASS"
        if full_milestone_pass
        else "FAIL"
    ),
}


# =================================================================================================
# 32. AUTOMATED SUMMARY
# =================================================================================================

print("\n" + "=" * 120)
print("🧠 AUTOMATED DIAGNOSTIC SUMMARY")
print("=" * 120)

for gate_name, gate_value in (
    scientific_gates.items()
):
    symbol = "✅" if gate_value else "❌"

    print(
        f"{symbol} {gate_name}"
    )

print("\nCross-regime point gates:")

for regime in TEST_REGIMES:
    passed = cross_regime_results[
        regime
    ]["point_gate_pass"]

    print(
        f"  {'✅' if passed else '❌'} "
        f"{regime}"
    )

print(
    f"\nPoint transfer count: "
    f"{point_pass_count}/{len(TEST_REGIMES)} "
    f"(strict majority requires "
    f"{strict_majority_required})"
)

print("\nCross-regime hazard gates:")

for regime in TEST_REGIMES:
    passed = cross_regime_results[
        regime
    ]["hazard_gate_pass"]

    print(
        f"  {'✅' if passed else '❌'} "
        f"{regime}"
    )

print(
    f"\nHazard transfer count: "
    f"{hazard_pass_count}/{len(TEST_REGIMES)} "
    f"(strict majority requires "
    f"{strict_majority_required})"
)

print("\nSelected architecture:")
print(
    f"  Feedback layer       : "
    f"{BEST_FEEDBACK_LAYER}"
)
print(
    f"  Activation PCA dim   : "
    f"{pca_components}"
)
print(
    f"  Point model          : "
    f"{BEST_POINT_MODEL_NAME}"
)
print(
    f"  Hazard model         : "
    f"{BEST_HAZARD_MODEL_NAME}"
)
print(
    f"  Latent threshold     : "
    f"{CFG.latent_hazard_threshold:.4f}"
)
print(
    f"  Decision threshold   : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)
print(
    f"  Novelty threshold    : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

print("\nOverall scientific statuses:")

for status_name, status_value in (
    overall_statuses.items()
):
    print(
        f"  {status_name:<34}: "
        f"{status_value}"
    )

print("\nFinal diagnosis:")
print(final_diagnosis)


# =================================================================================================
# 33. TELEMETRY
# =================================================================================================

telemetry = {
    "milestone": CFG.milestone,
    "title": CFG.title,
    "device": DEVICE,
    "model_name": CFG.model_name,
    "configuration": asdict(CFG),
    "valid_layers": list(valid_layers),

    "data_summary": {
        "train_sequence_shape": list(
            train_dataset["X"].shape
        ),
        "calibration_sequence_shape": list(
            calibration_dataset["X"].shape
        ),
        "validation_sequence_shape": list(
            validation_dataset["X"].shape
        ),
        "input_dimension": INPUT_DIM,
        "activation_pca_dimension": (
            pca_components
        ),
        "hazard_prevalence": {
            "train": float(
                np.mean(y_train_hazard)
            ),
            "calibration": float(
                np.mean(
                    y_calibration_hazard
                )
            ),
            "validation": float(
                np.mean(
                    y_validation_hazard
                )
            ),
        },
    },

    "post_feedback_observability": {
        "selected_layer": (
            BEST_FEEDBACK_LAYER
        ),
        "layer_results": {
            str(layer): metrics
            for layer, metrics in (
                layer_observability_results.items()
            )
        },
    },

    "calibration_model_selection": {
        "selected_point_model": (
            BEST_POINT_MODEL_NAME
        ),
        "selected_hazard_model": (
            BEST_HAZARD_MODEL_NAME
        ),
        "point_results": (
            point_calibration_results
        ),
        "hazard_results": (
            hazard_calibration_results
        ),
        "hazard_threshold_selection": (
            threshold_selection
        ),
    },

    "validation": {
        "point_selected": (
            validation_point_metrics
        ),
        "point_privileged_oracle": (
            validation_point_oracle_metrics
        ),
        "hazard_selected": (
            validation_hazard_metrics
        ),
        "hazard_privileged_oracle": (
            validation_hazard_oracle_metrics
        ),
        "hazard_ap_gain_over_prevalence": (
            validation_hazard_ap_gain
        ),
        "window_warning_audit": (
            validation_window_warning_audit
        ),
        "unique_onset_warning_audit": (
            validation_unique_onset_audit
        ),
        "bootstrap": {
            "point": (
                validation_point_bootstrap
            ),
            "hazard": (
                validation_hazard_bootstrap
            ),
        },
    },

    "order_ablation": order_ablation,

    "cross_regime_results": (
        cross_regime_results
    ),

    "novelty": {
        "sensor_metrics": (
            novelty_sensor_metrics
        ),
        "factorial_independence": (
            factorial_independence_metrics
        ),
        "compound_format_point_results": (
            compound_format_point_results
        ),
        "compound_format_hazard_results": (
            compound_format_hazard_results
        ),
    },

    "scientific_gates": scientific_gates,

    "transfer_summary": {
        "point_pass_count": (
            point_pass_count
        ),
        "hazard_pass_count": (
            hazard_pass_count
        ),
        "total_regimes": len(
            TEST_REGIMES
        ),
        "strict_majority_required": (
            strict_majority_required
        ),
    },

    "overall_statuses": overall_statuses,
    "final_diagnosis": final_diagnosis,
}


with open(
    CFG.output_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_builtin(telemetry),
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    f"\n💾 Telemetry written to: "
    f"{CFG.output_file}"
)


# =================================================================================================
# 34. OPTIONAL MEMORY CLEANUP
# =================================================================================================

try:
    del train_activation_scaled
except Exception:
    pass

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("=" * 120)
print("✅ M20.6.3 execution completed.")
print("=" * 120)


🔬 MILESTONE 20.6.3 — OBSERVABLE FEEDBACK AND EVENT-ONSET FORECASTING
Device                 : CUDA
Model                  : Qwen/Qwen2.5-0.5B
History length         : 8
Forecast horizon       : 6
Latent event threshold : 0.72
Train streams          : 24
Calibration streams    : 10
Validation streams     : 10
Test streams/regime    : 10
Global seed            : 206300
✅ Independent train/calibration/validation/test groups generated.

⏳ Loading TransformerLens model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Model loaded. Valid layers: [0, 3, 6, 9, 12, 15, 18, 21]

🔬 Extracting observable-feedback activations...
  Extracting AR1_TRANSFER...
  Extracting MEAN_REVERTING...
  Extracting RANDOM_WALK...
  Extracting OSCILLATORY...
  Extracting ABRUPT...
  Extracting COMPOUND_FACTORIAL...

🧪 TEST A — POST-FEEDBACK LATENT OBSERVABILITY
Layer 0   | rho= 0.7692 | Macro= 0.7279 | R²= 0.5664 | MAE= 0.1657
Layer 3   | rho= 0.8114 | Macro= 0.7727 | R²= 0.6077 | MAE= 0.1528
Layer 6   | rho= 0.8837 | Macro= 0.8541 | R²= 0.7201 | MAE= 0.1279
Layer 9   | rho= 0.8954 | Macro= 0.8694 | R²= 0.7436 | MAE= 0.1244
Layer 12  | rho= 0.8817 | Macro= 0.8509 | R²= 0.7306 | MAE= 0.1290
Layer 15  | rho= 0.8953 | Macro= 0.8593 | R²= 0.7454 | MAE= 0.1233
Layer 18  | rho= 0.9083 | Macro= 0.8797 | R²= 0.7755 | MAE= 0.1146
Layer 21  | rho= 0.8962 | Macro= 0.8664 | R²= 0.7546 | MAE= 0.1208

🏆 Selected post-feedback layer: 18

Temporal dataset shapes:
  Train 

In [ ]:
# =================================================================================================
# MILESTONE 20.6.3.1
# BENCHMARK REPAIR AND DISENTANGLED TEMPORAL VALIDATION
# =================================================================================================

import os
import gc
import json
import math
import random
import warnings

from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch

from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm
from transformer_lens import HookedTransformer


warnings.filterwarnings("ignore")


# =================================================================================================
# 1. CONFIGURATION
# =================================================================================================

@dataclass
class Configuration:
    milestone: str = "M20.6.3.1"

    title: str = (
        "Benchmark Repair and Disentangled Temporal Validation"
    )

    model_name: str = "Qwen/Qwen2.5-0.5B"

    seed: int = 206311

    history_length: int = 8
    forecast_horizon: int = 6

    stream_length: int = 104

    train_streams: int = 36
    calibration_selection_streams: int = 12
    calibration_probability_streams: int = 12
    calibration_policy_streams: int = 12
    validation_streams: int = 12
    test_streams_per_regime: int = 12

    latent_event_threshold: float = 0.72

    activation_pca_dimension: int = 12

    extraction_batch_size: int = 12

    ridge_alpha: float = 8.0
    logistic_c: float = 0.35

    maximum_policy_fpr: float = 0.10

    order_shuffle_repetitions: int = 20
    bootstrap_iterations: int = 300

    minimum_observability_r2: float = 0.20
    minimum_observability_rho: float = 0.50

    minimum_point_rho: float = 0.30
    minimum_point_macro_rho: float = 0.20
    minimum_point_r2: float = 0.05

    minimum_hazard_auc: float = 0.68
    minimum_hazard_ap_gain: float = 0.04

    minimum_point_order_advantage: float = 0.04
    minimum_hazard_order_advantage: float = 0.03

    minimum_event_recall: float = 0.60

    maximum_false_alert_episodes_per_100: float = 8.0

    maximum_novelty_fpr: float = 0.10
    minimum_novelty_auc: float = 0.70
    minimum_novelty_tpr: float = 0.60

    output_file: str = (
        "m20_6_3_1_benchmark_repair_disentangled.json"
    )


CFG = Configuration()

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

TORCH_DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)


# =================================================================================================
# 2. REPRODUCIBILITY
# =================================================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass


seed_everything(CFG.seed)


# =================================================================================================
# 3. GENERAL HELPERS
# =================================================================================================

def to_builtin(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()

    if isinstance(value, float):
        if not np.isfinite(value):
            return None

    return value


def safe_spearman(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64,
    )

    if len(y_true) < 3:
        return 0.0

    if (
        np.std(y_true) < 1e-12
        or np.std(y_pred) < 1e-12
    ):
        return 0.0

    result = spearmanr(
        y_true,
        y_pred,
    ).correlation

    if result is None or not np.isfinite(result):
        return 0.0

    return float(result)


def safe_auroc(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> Optional[float]:
    y_true = np.asarray(y_true)

    if len(np.unique(y_true)) < 2:
        return None

    try:
        return float(
            roc_auc_score(
                y_true,
                probability,
            )
        )
    except Exception:
        return None


def safe_auprc(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> Optional[float]:
    y_true = np.asarray(y_true)

    if len(np.unique(y_true)) < 2:
        return None

    try:
        return float(
            average_precision_score(
                y_true,
                probability,
            )
        )
    except Exception:
        return None


def macro_stream_rho(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    stream_ids: np.ndarray,
) -> float:
    values = []

    for stream_id in np.unique(stream_ids):
        mask = (
            stream_ids == stream_id
        )

        if np.sum(mask) < 3:
            continue

        value = safe_spearman(
            y_true[mask],
            y_pred[mask],
        )

        values.append(value)

    if not values:
        return 0.0

    return float(np.mean(values))


def evaluate_point(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    stream_ids: np.ndarray,
) -> Dict:
    return {
        "rho": safe_spearman(
            y_true,
            y_pred,
        ),
        "macro_stream_rho": macro_stream_rho(
            y_true,
            y_pred,
            stream_ids,
        ),
        "r2": float(
            r2_score(
                y_true,
                y_pred,
            )
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
    }


def evaluate_hazard(
    y_true: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> Dict:
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    probability = np.asarray(
        probability,
        dtype=np.float64,
    )

    prediction = (
        probability >= threshold
    ).astype(np.int64)

    true_positive = int(
        np.sum(
            (prediction == 1)
            & (y_true == 1)
        )
    )

    false_positive = int(
        np.sum(
            (prediction == 1)
            & (y_true == 0)
        )
    )

    true_negative = int(
        np.sum(
            (prediction == 0)
            & (y_true == 0)
        )
    )

    false_negative = int(
        np.sum(
            (prediction == 0)
            & (y_true == 1)
        )
    )

    recall = (
        true_positive
        / max(
            true_positive + false_negative,
            1,
        )
    )

    precision = (
        true_positive
        / max(
            true_positive + false_positive,
            1,
        )
    )

    fpr = (
        false_positive
        / max(
            false_positive + true_negative,
            1,
        )
    )

    return {
        "auroc": safe_auroc(
            y_true,
            probability,
        ),
        "auprc": safe_auprc(
            y_true,
            probability,
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probability,
            )
        ),
        "prevalence": float(
            np.mean(y_true)
        ),
        "recall": float(recall),
        "precision": float(precision),
        "fpr": float(fpr),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "true_negative": true_negative,
        "false_negative": false_negative,
    }


def print_header(title: str) -> None:
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


# =================================================================================================
# 4. DYNAMIC MECHANISMS
# =================================================================================================

TRAIN_FAMILIES = (
    "AR2",
    "DELAYED",
    "OSCILLATORY",
    "MEAN_REVERTING",
)

TEST_REGIMES = (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
)


def dynamic_step(
    family: str,
    history: np.ndarray,
    rng: np.random.Generator,
    parameter_mode: str,
) -> float:
    z_t = float(history[-1])

    z_tm1 = float(
        history[-2]
        if len(history) >= 2
        else 0.0
    )

    z_tm3 = float(
        history[-4]
        if len(history) >= 4
        else 0.0
    )

    if family == "AR2":
        if parameter_mode == "train":
            a1 = rng.uniform(1.08, 1.22)
            a2 = rng.uniform(-0.42, -0.28)
            noise_scale = 0.105
        else:
            a1 = rng.uniform(1.24, 1.34)
            a2 = rng.uniform(-0.52, -0.42)
            noise_scale = 0.115

        value = (
            a1 * z_t
            + a2 * z_tm1
            + rng.normal(0.0, noise_scale)
        )

    elif family == "DELAYED":
        if parameter_mode == "train":
            a = rng.uniform(0.43, 0.56)
            b = rng.uniform(0.28, 0.38)
            noise_scale = 0.105
        else:
            a = rng.uniform(0.34, 0.43)
            b = rng.uniform(0.40, 0.50)
            noise_scale = 0.115

        value = (
            a * z_t
            + b * z_tm3
            + rng.normal(0.0, noise_scale)
        )

    elif family == "OSCILLATORY":
        if parameter_mode == "train":
            a1 = rng.uniform(1.28, 1.42)
            a2 = rng.uniform(-0.76, -0.62)
            noise_scale = 0.085
        else:
            a1 = rng.uniform(1.43, 1.54)
            a2 = rng.uniform(-0.86, -0.76)
            noise_scale = 0.095

        value = (
            a1 * z_t
            + a2 * z_tm1
            + rng.normal(0.0, noise_scale)
        )

    elif family == "MEAN_REVERTING":
        if parameter_mode == "train":
            coefficient = rng.uniform(0.58, 0.72)
            noise_scale = 0.16
        else:
            coefficient = rng.uniform(0.43, 0.56)
            noise_scale = 0.18

        value = (
            coefficient * z_t
            + rng.normal(0.0, noise_scale)
        )

    elif family == "ABRUPT_UNKNOWN":
        value = (
            0.68 * z_t
            + rng.normal(0.0, 0.11)
        )

        if rng.random() < 0.075:
            value += rng.choice(
                [-1.0, 1.0]
            ) * rng.uniform(0.55, 1.05)

    elif family == "MECHANISM_NOVEL":
        # Fixed prompt format; only the causal law is novel.
        value = (
            0.42 * z_t
            - 0.48 * z_tm1
            + 0.52 * np.tanh(
                3.0 * z_tm3
            )
            + 0.16 * np.sin(
                4.5 * z_t
            )
            + rng.normal(0.0, 0.105)
        )

    else:
        raise ValueError(
            f"Unknown dynamic family: {family}"
        )

    # Sparse impulses occur in all known families. This prevents
    # novelty from being equivalent to the mere presence of shocks.
    if (
        family not in (
            "ABRUPT_UNKNOWN",
            "MECHANISM_NOVEL",
        )
        and rng.random() < 0.025
    ):
        value += rng.choice(
            [-1.0, 1.0]
        ) * rng.uniform(0.28, 0.58)

    return float(
        np.clip(
            value,
            -1.50,
            1.50,
        )
    )


def generate_latent_stream(
    family: str,
    length: int,
    seed: int,
    parameter_mode: str,
) -> np.ndarray:
    rng = np.random.default_rng(seed)

    latent = np.zeros(
        length,
        dtype=np.float32,
    )

    latent[:4] = rng.normal(
        0.0,
        0.18,
        size=4,
    )

    for time_index in range(
        4,
        length,
    ):
        latent[time_index] = dynamic_step(
            family=family,
            history=latent[:time_index],
            rng=rng,
            parameter_mode=parameter_mode,
        )

    return latent


# =================================================================================================
# 5. OBSERVABLE FEEDBACK GENERATION
# =================================================================================================

RAW_FEATURE_NAMES = (
    "sensor_a",
    "sensor_b",
    "prediction",
    "outcome",
    "signed_error",
    "absolute_error",
)


def generate_observable_rows(
    latent: np.ndarray,
    seed: int,
) -> Dict:
    rng = np.random.default_rng(seed)

    length = len(latent)

    sensor_a = np.zeros(
        length,
        dtype=np.float32,
    )

    sensor_b = np.zeros(
        length,
        dtype=np.float32,
    )

    prediction = np.zeros(
        length,
        dtype=np.float32,
    )

    outcome = np.zeros(
        length,
        dtype=np.float32,
    )

    signed_error = np.zeros(
        length,
        dtype=np.float32,
    )

    absolute_error = np.zeros(
        length,
        dtype=np.float32,
    )

    prompts = []

    for time_index in range(length):
        seasonal = (
            0.22
            * np.sin(
                2.0
                * np.pi
                * time_index
                / 17.0
            )
        )

        sensor_a[time_index] = (
            0.45 * seasonal
            + rng.normal(0.0, 0.45)
        )

        sensor_b[time_index] = (
            -0.20 * seasonal
            + rng.normal(0.0, 0.48)
        )

        prediction[time_index] = (
            0.55 * sensor_a[time_index]
            - 0.30 * sensor_b[time_index]
            + 0.10 * seasonal
            + rng.normal(0.0, 0.04)
        )

        sensitivity = (
            0.72
            + 0.10
            * np.tanh(
                sensor_a[time_index]
                - sensor_b[time_index]
            )
        )

        outcome[time_index] = (
            prediction[time_index]
            + sensitivity
            * latent[time_index]
            + rng.normal(0.0, 0.055)
        )

        signed_error[time_index] = (
            outcome[time_index]
            - prediction[time_index]
        )

        absolute_error[time_index] = abs(
            signed_error[time_index]
        )

        # The exact same template is used for every mechanism.
        prompt = (
            "Feedback telemetry record.\n"
            f"time_index: {time_index}\n"
            f"sensor_a: {sensor_a[time_index]:+.6f}\n"
            f"sensor_b: {sensor_b[time_index]:+.6f}\n"
            f"prediction: {prediction[time_index]:+.6f}\n"
            f"outcome: {outcome[time_index]:+.6f}\n"
            f"signed_error: {signed_error[time_index]:+.6f}\n"
            f"absolute_error: {absolute_error[time_index]:+.6f}\n"
            "Assess the internal feedback state represented by this record."
        )

        prompts.append(prompt)

    raw_features = np.stack([
        sensor_a,
        sensor_b,
        prediction,
        outcome,
        signed_error,
        absolute_error,
    ], axis=1).astype(np.float32)

    return {
        "raw_features": raw_features,
        "prompts": prompts,
    }


def generate_group(
    group_name: str,
    number_of_streams: int,
    families: Tuple[str, ...],
    seed_offset: int,
    parameter_mode: str,
    novelty_label: int = 0,
) -> Dict:
    streams = []

    for stream_index in range(
        number_of_streams
    ):
        family = families[
            stream_index % len(families)
        ]

        stream_seed = (
            CFG.seed
            + seed_offset
            + stream_index * 1009
        )

        latent = generate_latent_stream(
            family=family,
            length=CFG.stream_length,
            seed=stream_seed,
            parameter_mode=parameter_mode,
        )

        observable = generate_observable_rows(
            latent=latent,
            seed=stream_seed + 101,
        )

        streams.append({
            "group_name": group_name,
            "stream_id": stream_index,
            "family": family,
            "novelty_label": novelty_label,
            "latent": latent,
            "raw_features": observable[
                "raw_features"
            ],
            "prompts": observable["prompts"],
            "activations": {},
        })

    return {
        "name": group_name,
        "streams": streams,
    }


# =================================================================================================
# 6. INDEPENDENT DATA GROUPS
# =================================================================================================

train_group = generate_group(
    group_name="TRAIN_MIXED_KNOWN",
    number_of_streams=CFG.train_streams,
    families=TRAIN_FAMILIES,
    seed_offset=10_000,
    parameter_mode="train",
)

calibration_selection_group = generate_group(
    group_name="CALIBRATION_SELECTION",
    number_of_streams=(
        CFG.calibration_selection_streams
    ),
    families=TRAIN_FAMILIES,
    seed_offset=20_000,
    parameter_mode="train",
)

calibration_probability_group = generate_group(
    group_name="CALIBRATION_PROBABILITY",
    number_of_streams=(
        CFG.calibration_probability_streams
    ),
    families=TRAIN_FAMILIES,
    seed_offset=30_000,
    parameter_mode="train",
)

calibration_policy_group = generate_group(
    group_name="CALIBRATION_POLICY",
    number_of_streams=(
        CFG.calibration_policy_streams
    ),
    families=TRAIN_FAMILIES,
    seed_offset=40_000,
    parameter_mode="train",
)

validation_group = generate_group(
    group_name="VALIDATION",
    number_of_streams=CFG.validation_streams,
    families=TRAIN_FAMILIES,
    seed_offset=50_000,
    parameter_mode="train",
)

test_groups = {
    "AR2_TRANSFER": generate_group(
        group_name="AR2_TRANSFER",
        number_of_streams=(
            CFG.test_streams_per_regime
        ),
        families=("AR2",),
        seed_offset=60_000,
        parameter_mode="transfer",
    ),

    "DELAYED_TRANSFER": generate_group(
        group_name="DELAYED_TRANSFER",
        number_of_streams=(
            CFG.test_streams_per_regime
        ),
        families=("DELAYED",),
        seed_offset=70_000,
        parameter_mode="transfer",
    ),

    "OSCILLATORY_TRANSFER": generate_group(
        group_name="OSCILLATORY_TRANSFER",
        number_of_streams=(
            CFG.test_streams_per_regime
        ),
        families=("OSCILLATORY",),
        seed_offset=80_000,
        parameter_mode="transfer",
    ),

    "MEAN_REVERTING_TRANSFER": generate_group(
        group_name="MEAN_REVERTING_TRANSFER",
        number_of_streams=(
            CFG.test_streams_per_regime
        ),
        families=("MEAN_REVERTING",),
        seed_offset=90_000,
        parameter_mode="transfer",
    ),

    "ABRUPT_UNKNOWN": generate_group(
        group_name="ABRUPT_UNKNOWN",
        number_of_streams=(
            CFG.test_streams_per_regime
        ),
        families=("ABRUPT_UNKNOWN",),
        seed_offset=100_000,
        parameter_mode="transfer",
        novelty_label=1,
    ),

    "MECHANISM_NOVEL": generate_group(
        group_name="MECHANISM_NOVEL",
        number_of_streams=(
            CFG.test_streams_per_regime
        ),
        families=("MECHANISM_NOVEL",),
        seed_offset=110_000,
        parameter_mode="transfer",
        novelty_label=1,
    ),
}

all_groups = [
    train_group,
    calibration_selection_group,
    calibration_probability_group,
    calibration_policy_group,
    validation_group,
] + list(
    test_groups.values()
)

print_header(
    "🔬 MILESTONE 20.6.3.1 — BENCHMARK REPAIR AND DISENTANGLED TEMPORAL VALIDATION"
)

print(f"Device                       : {DEVICE.upper()}")
print(f"Model                        : {CFG.model_name}")
print(f"History length               : {CFG.history_length}")
print(f"Forecast horizon             : {CFG.forecast_horizon}")
print(f"Latent threshold             : {CFG.latent_event_threshold}")
print(f"Train streams                : {CFG.train_streams}")
print(f"Validation streams           : {CFG.validation_streams}")
print(f"Test streams per regime      : {CFG.test_streams_per_regime}")
print(f"Global seed                  : {CFG.seed}")
print("=" * 120)
print("✅ Independent groups generated.")


# =================================================================================================
# 7. LOAD TRANSFORMERLENS MODEL
# =================================================================================================

print("\n⏳ Loading TransformerLens model...")

model = HookedTransformer.from_pretrained(
    CFG.model_name,
    device=DEVICE,
    dtype=TORCH_DTYPE,
)

model.eval()

if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = (
        model.tokenizer.eos_token
    )

model.tokenizer.padding_side = "left"

number_of_layers = int(
    model.cfg.n_layers
)

valid_layers = sorted(
    set(
        np.linspace(
            0,
            number_of_layers - 1,
            8,
            dtype=int,
        ).tolist()
    )
)

print(
    f"✅ Model loaded. Valid layers: "
    f"{valid_layers}"
)


# =================================================================================================
# 8. ACTIVATION EXTRACTION
# =================================================================================================

@torch.inference_mode()
def extract_prompt_activations(
    prompts: List[str],
    layers: List[int],
) -> Dict[int, np.ndarray]:
    output = {
        layer: []
        for layer in layers
    }

    for start in tqdm(
        range(
            0,
            len(prompts),
            CFG.extraction_batch_size,
        ),
        leave=False,
    ):
        batch_prompts = prompts[
            start:
            start + CFG.extraction_batch_size
        ]

        tokens = model.to_tokens(
            batch_prompts,
            prepend_bos=True,
        ).to(DEVICE)

        hook_names = {
            f"blocks.{layer}.hook_resid_post"
            for layer in layers
        }

        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: (
                name in hook_names
            ),
            return_type=None,
        )

        for layer in layers:
            hook_name = (
                f"blocks.{layer}.hook_resid_post"
            )

            activation = cache[
                hook_name
            ][:, -1, :]

            output[layer].append(
                activation.detach()
                .float()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

        del tokens
        del cache

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return {
        layer: np.concatenate(
            batches,
            axis=0,
        )
        for layer, batches in output.items()
    }


print("\n🔬 Extracting fixed-format feedback activations...")

for group in all_groups:
    print(f"  Extracting {group['name']}...")

    for stream in group["streams"]:
        extracted = extract_prompt_activations(
            prompts=stream["prompts"],
            layers=valid_layers,
        )

        stream["activations"] = extracted


# =================================================================================================
# 9. FLATTEN GROUP ROWS
# =================================================================================================

def flatten_group_raw(
    group: Dict,
) -> np.ndarray:
    return np.concatenate([
        stream["raw_features"]
        for stream in group["streams"]
    ], axis=0)


def flatten_group_latent(
    group: Dict,
) -> np.ndarray:
    return np.concatenate([
        stream["latent"]
        for stream in group["streams"]
    ], axis=0)


def flatten_group_activation(
    group: Dict,
    layer: int,
) -> np.ndarray:
    return np.concatenate([
        stream["activations"][layer]
        for stream in group["streams"]
    ], axis=0)


# =================================================================================================
# 10. TEST A — RAW / ACTIVATION / COMBINED OBSERVABILITY
# =================================================================================================

def observability_pipeline(
    alpha: float = 10.0,
) -> Pipeline:
    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(alpha=alpha),
        ),
    ])


train_raw_rows = flatten_group_raw(
    train_group
)

selection_raw_rows = flatten_group_raw(
    calibration_selection_group
)

train_latent_rows = flatten_group_latent(
    train_group
)

selection_latent_rows = flatten_group_latent(
    calibration_selection_group
)

raw_observer = observability_pipeline()

raw_observer.fit(
    train_raw_rows,
    train_latent_rows,
)

selection_raw_prediction = (
    raw_observer.predict(
        selection_raw_rows
    )
)

raw_observability_metrics = evaluate_point(
    selection_latent_rows,
    selection_raw_prediction,
    np.zeros(
        len(selection_latent_rows),
        dtype=np.int64,
    ),
)

layer_observability_results = {}

print_header(
    "🧪 TEST A — INCREMENTAL POST-FEEDBACK OBSERVABILITY"
)

print(
    f"RAW ONLY   | "
    f"rho={raw_observability_metrics['rho']: .4f} | "
    f"R²={raw_observability_metrics['r2']: .4f} | "
    f"MAE={raw_observability_metrics['mae']: .4f}"
)

for layer in valid_layers:
    train_activation = flatten_group_activation(
        train_group,
        layer,
    )

    selection_activation = (
        flatten_group_activation(
            calibration_selection_group,
            layer,
        )
    )

    activation_observer = (
        observability_pipeline()
    )

    activation_observer.fit(
        train_activation,
        train_latent_rows,
    )

    activation_prediction = (
        activation_observer.predict(
            selection_activation
        )
    )

    activation_metrics = evaluate_point(
        selection_latent_rows,
        activation_prediction,
        np.zeros(
            len(selection_latent_rows),
            dtype=np.int64,
        ),
    )

    combined_observer = (
        observability_pipeline()
    )

    combined_observer.fit(
        np.concatenate([
            train_raw_rows,
            train_activation,
        ], axis=1),
        train_latent_rows,
    )

    combined_prediction = (
        combined_observer.predict(
            np.concatenate([
                selection_raw_rows,
                selection_activation,
            ], axis=1)
        )
    )

    combined_metrics = evaluate_point(
        selection_latent_rows,
        combined_prediction,
        np.zeros(
            len(selection_latent_rows),
            dtype=np.int64,
        ),
    )

    delta_r2 = float(
        combined_metrics["r2"]
        - raw_observability_metrics["r2"]
    )

    layer_observability_results[layer] = {
        "activation_only": activation_metrics,
        "combined": combined_metrics,
        "delta_r2_over_raw": delta_r2,
    }

    print(
        f"Layer {layer:<3} | "
        f"Act rho={activation_metrics['rho']: .4f} | "
        f"Act R²={activation_metrics['r2']: .4f} | "
        f"Combined R²={combined_metrics['r2']: .4f} | "
        f"ΔR²={delta_r2:+.4f}"
    )


BEST_FEEDBACK_LAYER = max(
    valid_layers,
    key=lambda layer: (
        layer_observability_results[
            layer
        ]["combined"]["r2"]
    ),
)

print(
    f"\n🏆 Selected feedback layer: "
    f"{BEST_FEEDBACK_LAYER}"
)


# =================================================================================================
# 11. FIT RAW SCALER AND ACTIVATION PCA ONLY ON TRAIN
# =================================================================================================

raw_scaler = StandardScaler()

raw_scaler.fit(
    train_raw_rows
)

train_selected_activation = (
    flatten_group_activation(
        train_group,
        BEST_FEEDBACK_LAYER,
    )
)

activation_scaler = StandardScaler()

train_activation_scaled = (
    activation_scaler.fit_transform(
        train_selected_activation
    )
)

pca_dimension = min(
    CFG.activation_pca_dimension,
    train_activation_scaled.shape[1],
    train_activation_scaled.shape[0] - 1,
)

activation_pca = PCA(
    n_components=pca_dimension,
    random_state=CFG.seed,
)

activation_pca.fit(
    train_activation_scaled
)


def attach_transformed_features(
    group: Dict,
) -> None:
    for stream in group["streams"]:
        stream["raw_scaled"] = (
            raw_scaler.transform(
                stream["raw_features"]
            ).astype(np.float32)
        )

        selected_activation = (
            stream["activations"][
                BEST_FEEDBACK_LAYER
            ]
        )

        scaled_activation = (
            activation_scaler.transform(
                selected_activation
            )
        )

        stream["activation_pca"] = (
            activation_pca.transform(
                scaled_activation
            ).astype(np.float32)
        )


for group in all_groups:
    attach_transformed_features(group)


# =================================================================================================
# 12. TEMPORAL DATASET CONSTRUCTION
# =================================================================================================

def build_temporal_dataset(
    group: Dict,
) -> Dict:
    raw_sequences = []
    activation_sequences = []
    latent_histories = []

    point_targets = []
    hazard_targets = []

    hazard_eligible = []

    stream_ids = []
    times = []
    families = []
    novelty_labels = []

    current_latent = []
    future_latent = []

    for stream_id, stream in enumerate(
        group["streams"]
    ):
        latent = stream["latent"]
        raw = stream["raw_scaled"]
        activation = stream["activation_pca"]

        first_time = (
            CFG.history_length - 1
        )

        last_time = (
            len(latent)
            - CFG.forecast_horizon
            - 1
        )

        for time_index in range(
            first_time,
            last_time + 1,
        ):
            history_start = (
                time_index
                - CFG.history_length
                + 1
            )

            future = latent[
                time_index + 1:
                time_index
                + CFG.forecast_horizon
                + 1
            ]

            current_is_safe = bool(
                abs(latent[time_index])
                <= CFG.latent_event_threshold
            )

            crossing = bool(
                current_is_safe
                and np.any(
                    np.abs(future)
                    > CFG.latent_event_threshold
                )
            )

            raw_sequences.append(
                raw[
                    history_start:
                    time_index + 1
                ]
            )

            activation_sequences.append(
                activation[
                    history_start:
                    time_index + 1
                ]
            )

            latent_histories.append(
                latent[
                    history_start:
                    time_index + 1
                ]
            )

            point_targets.append(
                latent[
                    time_index
                    + CFG.forecast_horizon
                ]
            )

            hazard_targets.append(
                int(crossing)
            )

            hazard_eligible.append(
                current_is_safe
            )

            stream_ids.append(stream_id)
            times.append(time_index)
            families.append(stream["family"])

            novelty_labels.append(
                stream["novelty_label"]
            )

            current_latent.append(
                latent[time_index]
            )

            future_latent.append(future)

    return {
        "X_raw": np.asarray(
            raw_sequences,
            dtype=np.float32,
        ),
        "X_activation": np.asarray(
            activation_sequences,
            dtype=np.float32,
        ),
        "latent_history": np.asarray(
            latent_histories,
            dtype=np.float32,
        ),
        "y_point": np.asarray(
            point_targets,
            dtype=np.float32,
        ),
        "y_hazard": np.asarray(
            hazard_targets,
            dtype=np.int64,
        ),
        "hazard_eligible": np.asarray(
            hazard_eligible,
            dtype=bool,
        ),
        "stream_ids": np.asarray(
            stream_ids,
            dtype=np.int64,
        ),
        "times": np.asarray(
            times,
            dtype=np.int64,
        ),
        "families": np.asarray(
            families,
            dtype=object,
        ),
        "novelty_labels": np.asarray(
            novelty_labels,
            dtype=np.int64,
        ),
        "current_latent": np.asarray(
            current_latent,
            dtype=np.float32,
        ),
        "future_latent": np.asarray(
            future_latent,
            dtype=np.float32,
        ),
    }


train_dataset = build_temporal_dataset(
    train_group
)

calibration_selection_dataset = (
    build_temporal_dataset(
        calibration_selection_group
    )
)

calibration_probability_dataset = (
    build_temporal_dataset(
        calibration_probability_group
    )
)

calibration_policy_dataset = (
    build_temporal_dataset(
        calibration_policy_group
    )
)

validation_dataset = build_temporal_dataset(
    validation_group
)

test_datasets = {
    name: build_temporal_dataset(group)
    for name, group in test_groups.items()
}

print("\nTemporal shapes:")
print(
    f"  Train raw        : "
    f"{train_dataset['X_raw'].shape}"
)
print(
    f"  Train activation : "
    f"{train_dataset['X_activation'].shape}"
)
print(
    f"  Validation raw   : "
    f"{validation_dataset['X_raw'].shape}"
)


# =================================================================================================
# 13. FEATURE VIEWS
# =================================================================================================

def bag_features(
    sequence: np.ndarray,
) -> np.ndarray:
    return np.concatenate([
        np.mean(
            sequence,
            axis=1,
        ),
        np.std(
            sequence,
            axis=1,
        ),
        np.min(
            sequence,
            axis=1,
        ),
        np.max(
            sequence,
            axis=1,
        ),
    ], axis=1)


def slope_features(
    sequence: np.ndarray,
) -> np.ndarray:
    slope = (
        sequence[:, -1, :]
        - sequence[:, 0, :]
    ) / max(
        sequence.shape[1] - 1,
        1,
    )

    return np.concatenate([
        sequence[:, -1, :],
        slope,
    ], axis=1)


def feature_view(
    dataset: Dict,
    name: str,
) -> np.ndarray:
    raw = dataset["X_raw"]
    activation = dataset[
        "X_activation"
    ]

    if name == "RAW_LATEST":
        return raw[:, -1, :]

    if name == "RAW_BAG":
        return bag_features(raw)

    if name == "RAW_SLOPE":
        return slope_features(raw)

    if name == "RAW_ORDERED":
        return raw.reshape(
            len(raw),
            -1,
        )

    if name == "ACTIVATION_ORDERED":
        return activation.reshape(
            len(activation),
            -1,
        )

    if name == "COMBINED_ORDERED":
        combined = np.concatenate([
            raw,
            activation,
        ], axis=2)

        return combined.reshape(
            len(combined),
            -1,
        )

    raise ValueError(
        f"Unknown feature view: {name}"
    )


CAUSAL_MODEL_NAMES = (
    "RAW_LATEST",
    "RAW_BAG",
    "RAW_SLOPE",
    "RAW_ORDERED",
)

DIAGNOSTIC_MODEL_NAMES = (
    "ACTIVATION_ORDERED",
    "COMBINED_ORDERED",
)


# =================================================================================================
# 14. TRUE LATENT-HISTORY ORACLES
# =================================================================================================

def latent_history_features(
    dataset: Dict,
) -> np.ndarray:
    history = dataset[
        "latent_history"
    ]

    first_difference = np.diff(
        history,
        axis=1,
    )

    return np.concatenate([
        history,
        first_difference,
        history ** 2,
    ], axis=1).reshape(
        len(history),
        -1,
    )


point_oracle = Pipeline([
    (
        "scale",
        StandardScaler(),
    ),
    (
        "ridge",
        Ridge(
            alpha=2.0
        ),
    ),
])

point_oracle.fit(
    latent_history_features(
        train_dataset
    ),
    train_dataset["y_point"],
)

train_hazard_mask = train_dataset[
    "hazard_eligible"
]

hazard_oracle = Pipeline([
    (
        "scale",
        StandardScaler(),
    ),
    (
        "logistic",
        LogisticRegression(
            C=0.5,
            max_iter=3000,
            class_weight="balanced",
            random_state=CFG.seed,
        ),
    ),
])

hazard_oracle.fit(
    latent_history_features(
        train_dataset
    )[train_hazard_mask],
    train_dataset[
        "y_hazard"
    ][train_hazard_mask],
)


# =================================================================================================
# 15. MODEL TRAINING HELPERS
# =================================================================================================

def make_point_model() -> Pipeline:
    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(
                alpha=CFG.ridge_alpha
            ),
        ),
    ])


def make_hazard_model() -> Pipeline:
    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "logistic",
            LogisticRegression(
                C=CFG.logistic_c,
                max_iter=4000,
                class_weight="balanced",
                random_state=CFG.seed,
            ),
        ),
    ])


def decision_score(
    model: Pipeline,
    X: np.ndarray,
) -> np.ndarray:
    if hasattr(
        model,
        "decision_function",
    ):
        score = model.decision_function(X)
    else:
        score = model.predict_proba(
            X
        )[:, 1]

    return np.asarray(
        score,
        dtype=np.float64,
    )


class PlattCalibrator:
    def __init__(self):
        self.model = LogisticRegression(
            C=1000.0,
            max_iter=2000,
            class_weight=None,
            random_state=CFG.seed,
        )

    def fit(
        self,
        score: np.ndarray,
        y_true: np.ndarray,
    ) -> "PlattCalibrator":
        self.model.fit(
            np.asarray(score).reshape(
                -1,
                1,
            ),
            y_true,
        )

        return self

    def predict(
        self,
        score: np.ndarray,
    ) -> np.ndarray:
        return self.model.predict_proba(
            np.asarray(score).reshape(
                -1,
                1,
            )
        )[:, 1]


# =================================================================================================
# 16. CALIBRATION-ONLY MODEL SELECTION
# =================================================================================================

point_models = {}
point_selection_results = {}

for model_name in (
    CAUSAL_MODEL_NAMES
    + DIAGNOSTIC_MODEL_NAMES
):
    model_instance = make_point_model()

    model_instance.fit(
        feature_view(
            train_dataset,
            model_name,
        ),
        train_dataset["y_point"],
    )

    prediction = model_instance.predict(
        feature_view(
            calibration_selection_dataset,
            model_name,
        )
    )

    metrics = evaluate_point(
        calibration_selection_dataset[
            "y_point"
        ],
        prediction,
        calibration_selection_dataset[
            "stream_ids"
        ],
    )

    point_models[model_name] = (
        model_instance
    )

    point_selection_results[
        model_name
    ] = metrics


hazard_models = {}
hazard_selection_results = {}

selection_hazard_mask = (
    calibration_selection_dataset[
        "hazard_eligible"
    ]
)

for model_name in (
    CAUSAL_MODEL_NAMES
    + DIAGNOSTIC_MODEL_NAMES
):
    model_instance = make_hazard_model()

    model_instance.fit(
        feature_view(
            train_dataset,
            model_name,
        )[train_hazard_mask],
        train_dataset[
            "y_hazard"
        ][train_hazard_mask],
    )

    score = decision_score(
        model_instance,
        feature_view(
            calibration_selection_dataset,
            model_name,
        )[selection_hazard_mask],
    )

    probability = 1.0 / (
        1.0 + np.exp(-score)
    )

    metrics = evaluate_hazard(
        calibration_selection_dataset[
            "y_hazard"
        ][selection_hazard_mask],
        probability,
        threshold=0.5,
    )

    hazard_models[model_name] = (
        model_instance
    )

    hazard_selection_results[
        model_name
    ] = metrics


BEST_POINT_MODEL_NAME = max(
    CAUSAL_MODEL_NAMES,
    key=lambda name: (
        point_selection_results[name][
            "rho"
        ]
        + 0.35
        * point_selection_results[name][
            "macro_stream_rho"
        ]
        + 0.20
        * point_selection_results[name][
            "r2"
        ]
    ),
)

BEST_HAZARD_MODEL_NAME = max(
    CAUSAL_MODEL_NAMES,
    key=lambda name: (
        (
            hazard_selection_results[
                name
            ]["auroc"]
            or 0.0
        )
        + 0.50
        * (
            hazard_selection_results[
                name
            ]["auprc"]
            or 0.0
        )
    ),
)


print_header(
    "🧪 TEST B — DISENTANGLED MODEL SELECTION"
)

print("\nPoint models:")

for name, metrics in (
    point_selection_results.items()
):
    channel = (
        "CAUSAL"
        if name in CAUSAL_MODEL_NAMES
        else "DIAGNOSTIC"
    )

    print(
        f"  {name:<20} [{channel:<10}] | "
        f"rho={metrics['rho']: .4f} | "
        f"Macro={metrics['macro_stream_rho']: .4f} | "
        f"R²={metrics['r2']: .4f}"
    )

print("\nHazard models:")

for name, metrics in (
    hazard_selection_results.items()
):
    channel = (
        "CAUSAL"
        if name in CAUSAL_MODEL_NAMES
        else "DIAGNOSTIC"
    )

    print(
        f"  {name:<20} [{channel:<10}] | "
        f"AUC={(metrics['auroc'] or 0.0): .4f} | "
        f"AP={(metrics['auprc'] or 0.0): .4f}"
    )

print(
    f"\n🏆 Selected causal point model : "
    f"{BEST_POINT_MODEL_NAME}"
)

print(
    f"🏆 Selected causal hazard model: "
    f"{BEST_HAZARD_MODEL_NAME}"
)


# =================================================================================================
# 17. INDEPENDENT PROBABILITY CALIBRATION
# =================================================================================================

probability_hazard_mask = (
    calibration_probability_dataset[
        "hazard_eligible"
    ]
)

selected_hazard_model = (
    hazard_models[
        BEST_HAZARD_MODEL_NAME
    ]
)

probability_calibration_score = (
    decision_score(
        selected_hazard_model,
        feature_view(
            calibration_probability_dataset,
            BEST_HAZARD_MODEL_NAME,
        )[probability_hazard_mask],
    )
)

hazard_calibrator = PlattCalibrator()

hazard_calibrator.fit(
    probability_calibration_score,
    calibration_probability_dataset[
        "y_hazard"
    ][probability_hazard_mask],
)


def predict_hazard_probability(
    dataset: Dict,
) -> np.ndarray:
    mask = dataset[
        "hazard_eligible"
    ]

    score = decision_score(
        selected_hazard_model,
        feature_view(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )[mask],
    )

    return hazard_calibrator.predict(
        score
    )


# =================================================================================================
# 18. INDEPENDENT POLICY THRESHOLD SELECTION
# =================================================================================================

policy_hazard_mask = (
    calibration_policy_dataset[
        "hazard_eligible"
    ]
)

policy_probability = (
    predict_hazard_probability(
        calibration_policy_dataset
    )
)

policy_labels = calibration_policy_dataset[
    "y_hazard"
][policy_hazard_mask]


def choose_operational_threshold(
    y_true: np.ndarray,
    probability: np.ndarray,
    maximum_fpr: float,
) -> Dict:
    candidate_thresholds = np.unique(
        np.concatenate([
            np.linspace(
                0.01,
                0.99,
                199,
            ),
            probability,
        ])
    )

    feasible = []

    all_results = []

    for threshold in candidate_thresholds:
        metrics = evaluate_hazard(
            y_true,
            probability,
            float(threshold),
        )

        record = {
            "threshold": float(
                threshold
            ),
            **metrics,
        }

        all_results.append(record)

        if metrics["fpr"] <= maximum_fpr:
            feasible.append(record)

    if feasible:
        selected = max(
            feasible,
            key=lambda record: (
                record["recall"],
                record["precision"],
                -record["fpr"],
            ),
        )
    else:
        selected = min(
            all_results,
            key=lambda record: (
                record["fpr"],
                -record["recall"],
            ),
        )

    return selected


threshold_selection = (
    choose_operational_threshold(
        y_true=policy_labels,
        probability=policy_probability,
        maximum_fpr=CFG.maximum_policy_fpr,
    )
)

HAZARD_DECISION_THRESHOLD = float(
    threshold_selection["threshold"]
)

print("\nIndependent probability/policy calibration:")
print(
    f"  Operational threshold : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)
print(
    f"  Policy FPR            : "
    f"{threshold_selection['fpr']:.4f}"
)
print(
    f"  Policy recall         : "
    f"{threshold_selection['recall']:.4f}"
)
print(
    f"  Policy precision      : "
    f"{threshold_selection['precision']:.4f}"
)


# =================================================================================================
# 19. UNTOUCHED VALIDATION
# =================================================================================================

selected_point_model = point_models[
    BEST_POINT_MODEL_NAME
]

validation_point_prediction = (
    selected_point_model.predict(
        feature_view(
            validation_dataset,
            BEST_POINT_MODEL_NAME,
        )
    )
)

validation_point_metrics = evaluate_point(
    validation_dataset["y_point"],
    validation_point_prediction,
    validation_dataset["stream_ids"],
)

validation_oracle_prediction = (
    point_oracle.predict(
        latent_history_features(
            validation_dataset
        )
    )
)

validation_oracle_metrics = evaluate_point(
    validation_dataset["y_point"],
    validation_oracle_prediction,
    validation_dataset["stream_ids"],
)

validation_persistence_prediction = (
    validation_dataset[
        "current_latent"
    ]
)

validation_persistence_metrics = evaluate_point(
    validation_dataset["y_point"],
    validation_persistence_prediction,
    validation_dataset["stream_ids"],
)

validation_hazard_mask = (
    validation_dataset[
        "hazard_eligible"
    ]
)

validation_hazard_probability = (
    predict_hazard_probability(
        validation_dataset
    )
)

validation_hazard_labels = (
    validation_dataset[
        "y_hazard"
    ][validation_hazard_mask]
)

validation_hazard_metrics = evaluate_hazard(
    validation_hazard_labels,
    validation_hazard_probability,
    HAZARD_DECISION_THRESHOLD,
)

validation_hazard_oracle_probability = (
    hazard_oracle.predict_proba(
        latent_history_features(
            validation_dataset
        )[validation_hazard_mask]
    )[:, 1]
)

validation_hazard_oracle_metrics = (
    evaluate_hazard(
        validation_hazard_labels,
        validation_hazard_oracle_probability,
        threshold=0.5,
    )
)

constant_probability = np.full(
    len(validation_hazard_labels),
    np.mean(
        train_dataset[
            "y_hazard"
        ][train_hazard_mask]
    ),
    dtype=np.float64,
)

constant_brier = float(
    brier_score_loss(
        validation_hazard_labels,
        constant_probability,
    )
)


print_header(
    "🧪 TEST C — UNTOUCHED VALIDATION"
)

print("\nSelected causal point forecaster:")
print(
    f"  Model      : {BEST_POINT_MODEL_NAME}"
)
print(
    f"  rho        : {validation_point_metrics['rho']:.4f}"
)
print(
    f"  Macro rho  : {validation_point_metrics['macro_stream_rho']:.4f}"
)
print(
    f"  R²         : {validation_point_metrics['r2']:.4f}"
)
print(
    f"  MAE        : {validation_point_metrics['mae']:.4f}"
)

print("\nTrue latent-history oracle:")
print(
    f"  rho        : {validation_oracle_metrics['rho']:.4f}"
)
print(
    f"  Macro rho  : {validation_oracle_metrics['macro_stream_rho']:.4f}"
)
print(
    f"  R²         : {validation_oracle_metrics['r2']:.4f}"
)

print("\nPersistence baseline:")
print(
    f"  rho        : {validation_persistence_metrics['rho']:.4f}"
)
print(
    f"  R²         : {validation_persistence_metrics['r2']:.4f}"
)

print("\nCalibrated hazard forecaster:")
print(
    f"  Model      : {BEST_HAZARD_MODEL_NAME}"
)
print(
    f"  AUC        : {(validation_hazard_metrics['auroc'] or 0.0):.4f}"
)
print(
    f"  AP         : {(validation_hazard_metrics['auprc'] or 0.0):.4f}"
)
print(
    f"  Prevalence : {validation_hazard_metrics['prevalence']:.4f}"
)
print(
    f"  Recall     : {validation_hazard_metrics['recall']:.4f}"
)
print(
    f"  Precision  : {validation_hazard_metrics['precision']:.4f}"
)
print(
    f"  FPR        : {validation_hazard_metrics['fpr']:.4f}"
)
print(
    f"  Brier      : {validation_hazard_metrics['brier']:.4f}"
)
print(
    f"  Const Brier: {constant_brier:.4f}"
)

print("\nHazard latent-history oracle:")
print(
    f"  AUC        : "
    f"{(validation_hazard_oracle_metrics['auroc'] or 0.0):.4f}"
)
print(
    f"  AP         : "
    f"{(validation_hazard_oracle_metrics['auprc'] or 0.0):.4f}"
)


# =================================================================================================
# 20. ORDER ABLATIONS INDEPENDENT OF MODEL SELECTION
# =================================================================================================

def shuffle_sequence_rows(
    sequence: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    shuffled = np.empty_like(
        sequence
    )

    for sample_index in range(
        len(sequence)
    ):
        permutation = rng.permutation(
            sequence.shape[1]
        )

        shuffled[sample_index] = (
            sequence[
                sample_index,
                permutation,
            ]
        )

    return shuffled


def ordered_raw_flat(
    sequence: np.ndarray,
) -> np.ndarray:
    return sequence.reshape(
        len(sequence),
        -1,
    )


ordered_point_model = make_point_model()

ordered_point_model.fit(
    ordered_raw_flat(
        train_dataset["X_raw"]
    ),
    train_dataset["y_point"],
)

ordered_point_prediction = (
    ordered_point_model.predict(
        ordered_raw_flat(
            validation_dataset["X_raw"]
        )
    )
)

ordered_point_rho = safe_spearman(
    validation_dataset["y_point"],
    ordered_point_prediction,
)

reversed_point_prediction = (
    ordered_point_model.predict(
        ordered_raw_flat(
            validation_dataset[
                "X_raw"
            ][:, ::-1, :]
        )
    )
)

reversed_point_rho = safe_spearman(
    validation_dataset["y_point"],
    reversed_point_prediction,
)

shuffle_point_rhos = []
shuffle_hazard_aucs = []

for repetition in range(
    CFG.order_shuffle_repetitions
):
    rng = np.random.default_rng(
        CFG.seed
        + 200_000
        + repetition
    )

    shuffled_train_raw = (
        shuffle_sequence_rows(
            train_dataset["X_raw"],
            rng,
        )
    )

    shuffled_validation_raw = (
        shuffle_sequence_rows(
            validation_dataset["X_raw"],
            rng,
        )
    )

    shuffled_point_model = (
        make_point_model()
    )

    shuffled_point_model.fit(
        ordered_raw_flat(
            shuffled_train_raw
        ),
        train_dataset["y_point"],
    )

    shuffled_point_prediction = (
        shuffled_point_model.predict(
            ordered_raw_flat(
                shuffled_validation_raw
            )
        )
    )

    shuffle_point_rhos.append(
        safe_spearman(
            validation_dataset[
                "y_point"
            ],
            shuffled_point_prediction,
        )
    )

    shuffled_hazard_model = (
        make_hazard_model()
    )

    shuffled_hazard_model.fit(
        ordered_raw_flat(
            shuffled_train_raw
        )[train_hazard_mask],
        train_dataset[
            "y_hazard"
        ][train_hazard_mask],
    )

    shuffled_hazard_score = (
        decision_score(
            shuffled_hazard_model,
            ordered_raw_flat(
                shuffled_validation_raw
            )[validation_hazard_mask],
        )
    )

    shuffled_auc = safe_auroc(
        validation_hazard_labels,
        shuffled_hazard_score,
    )

    if shuffled_auc is not None:
        shuffle_hazard_aucs.append(
            shuffled_auc
        )


ordered_hazard_model = make_hazard_model()

ordered_hazard_model.fit(
    ordered_raw_flat(
        train_dataset["X_raw"]
    )[train_hazard_mask],
    train_dataset[
        "y_hazard"
    ][train_hazard_mask],
)

ordered_hazard_score = decision_score(
    ordered_hazard_model,
    ordered_raw_flat(
        validation_dataset["X_raw"]
    )[validation_hazard_mask],
)

ordered_hazard_auc = safe_auroc(
    validation_hazard_labels,
    ordered_hazard_score,
) or 0.0

reversed_hazard_score = decision_score(
    ordered_hazard_model,
    ordered_raw_flat(
        validation_dataset[
            "X_raw"
        ][:, ::-1, :]
    )[validation_hazard_mask],
)

reversed_hazard_auc = safe_auroc(
    validation_hazard_labels,
    reversed_hazard_score,
) or 0.0

shuffle_point_p95 = float(
    np.quantile(
        shuffle_point_rhos,
        0.95,
    )
)

shuffle_hazard_p95 = float(
    np.quantile(
        shuffle_hazard_aucs,
        0.95,
    )
) if shuffle_hazard_aucs else 0.0

point_order_advantage = float(
    ordered_point_rho
    - shuffle_point_p95
)

hazard_order_advantage = float(
    ordered_hazard_auc
    - shuffle_hazard_p95
)

order_ablation = {
    "point": {
        "ordered_rho": ordered_point_rho,
        "reversed_evaluation_rho": (
            reversed_point_rho
        ),
        "shuffle_retrained_mean": float(
            np.mean(
                shuffle_point_rhos
            )
        ),
        "shuffle_retrained_p95": (
            shuffle_point_p95
        ),
        "advantage_over_shuffle_p95": (
            point_order_advantage
        ),
    },
    "hazard": {
        "ordered_auc": ordered_hazard_auc,
        "reversed_evaluation_auc": (
            reversed_hazard_auc
        ),
        "shuffle_retrained_mean": float(
            np.mean(
                shuffle_hazard_aucs
            )
        ) if shuffle_hazard_aucs else None,
        "shuffle_retrained_p95": (
            shuffle_hazard_p95
        ),
        "advantage_over_shuffle_p95": (
            hazard_order_advantage
        ),
    },
}


print_header(
    "🧪 TEST D — ORDER-IDENTIFIABLE TEMPORAL ABLATIONS"
)

print(
    f"Point  | Ordered={ordered_point_rho:.4f} | "
    f"Reversed evaluation={reversed_point_rho:.4f} | "
    f"Shuffle p95={shuffle_point_p95:.4f} | "
    f"Advantage={point_order_advantage:+.4f}"
)

print(
    f"Hazard | Ordered={ordered_hazard_auc:.4f} | "
    f"Reversed evaluation={reversed_hazard_auc:.4f} | "
    f"Shuffle p95={shuffle_hazard_p95:.4f} | "
    f"Advantage={hazard_order_advantage:+.4f}"
)


# =================================================================================================
# 21. EVENT-ONSET AND FALSE-ALERT EPISODE AUDIT
# =================================================================================================

def event_episode_audit(
    dataset: Dict,
    eligible_probability: np.ndarray,
    threshold: float,
) -> Dict:
    eligible_indices = np.flatnonzero(
        dataset["hazard_eligible"]
    )

    probability_map = {}

    for index, probability in zip(
        eligible_indices,
        eligible_probability,
    ):
        key = (
            int(
                dataset[
                    "stream_ids"
                ][index]
            ),
            int(
                dataset["times"][index]
            ),
        )

        probability_map[key] = float(
            probability
        )

    total_events = 0
    detected_events = 0

    lead_times = []

    total_alert_samples = 0
    false_alert_samples = 0
    false_alert_episodes = 0

    total_eligible_samples = len(
        eligible_indices
    )

    for stream_id in np.unique(
        dataset["stream_ids"]
    ):
        stream_mask = (
            dataset["stream_ids"]
            == stream_id
        )

        indices = np.flatnonzero(
            stream_mask
        )

        ordering = np.argsort(
            dataset["times"][indices]
        )

        indices = indices[ordering]

        latent_by_time = {}

        for index in indices:
            time_index = int(
                dataset["times"][index]
            )

            latent_by_time[time_index] = float(
                dataset[
                    "current_latent"
                ][index]
            )

            for offset, value in enumerate(
                dataset[
                    "future_latent"
                ][index],
                start=1,
            ):
                latent_by_time[
                    time_index + offset
                ] = float(value)

        sorted_times = sorted(
            latent_by_time.keys()
        )

        onset_times = []

        for previous_time, current_time in zip(
            sorted_times[:-1],
            sorted_times[1:],
        ):
            if (
                current_time
                != previous_time + 1
            ):
                continue

            was_safe = (
                abs(
                    latent_by_time[
                        previous_time
                    ]
                )
                <= CFG.latent_event_threshold
            )

            is_degraded = (
                abs(
                    latent_by_time[
                        current_time
                    ]
                )
                > CFG.latent_event_threshold
            )

            if was_safe and is_degraded:
                onset_times.append(
                    current_time
                )

        for onset_time in onset_times:
            total_events += 1

            candidate_leads = []

            for warning_time in range(
                onset_time
                - CFG.forecast_horizon,
                onset_time,
            ):
                probability = probability_map.get(
                    (
                        int(stream_id),
                        int(warning_time),
                    )
                )

                if probability is None:
                    continue

                if probability >= threshold:
                    candidate_leads.append(
                        onset_time
                        - warning_time
                    )

            if candidate_leads:
                detected_events += 1
                lead_times.append(
                    max(candidate_leads)
                )

        stream_alert_records = []

        for index in indices:
            time_index = int(
                dataset["times"][index]
            )

            probability = probability_map.get(
                (
                    int(stream_id),
                    time_index,
                )
            )

            if probability is None:
                continue

            alert = bool(
                probability >= threshold
            )

            true_window = bool(
                dataset[
                    "y_hazard"
                ][index] == 1
            )

            if alert:
                total_alert_samples += 1

                if not true_window:
                    false_alert_samples += 1

            stream_alert_records.append((
                time_index,
                alert and not true_window,
            ))

        previous_false_alert = False
        previous_time = None

        for time_index, false_alert in (
            stream_alert_records
            ):
            contiguous = (
                previous_time is not None
                and time_index == previous_time + 1
            )

            if (
                false_alert
                and (
                    not previous_false_alert
                    or not contiguous
                )
            ):
                false_alert_episodes += 1

            previous_false_alert = false_alert
            previous_time = time_index

    event_recall = (
        detected_events
        / max(total_events, 1)
    )

    false_alert_fraction = (
        false_alert_samples
        / max(total_alert_samples, 1)
    )

    false_alert_episodes_per_100 = (
        100.0
        * false_alert_episodes
        / max(total_eligible_samples, 1)
    )

    return {
        "total_unique_event_onsets": int(
            total_events
        ),
        "detected_unique_event_onsets": int(
            detected_events
        ),
        "event_recall": float(
            event_recall
        ),
        "mean_detected_lead_time": (
            float(np.mean(lead_times))
            if lead_times
            else None
        ),
        "median_detected_lead_time": (
            float(np.median(lead_times))
            if lead_times
            else None
        ),
        "minimum_detected_lead_time": (
            int(np.min(lead_times))
            if lead_times
            else None
        ),
        "maximum_detected_lead_time": (
            int(np.max(lead_times))
            if lead_times
            else None
        ),
        "total_alert_samples": int(
            total_alert_samples
        ),
        "false_alert_samples": int(
            false_alert_samples
        ),
        "false_alert_fraction": float(
            false_alert_fraction
        ),
        "false_alert_episodes": int(
            false_alert_episodes
        ),
        "false_alert_episodes_per_100_eligible_samples": float(
            false_alert_episodes_per_100
        ),
        "total_eligible_samples": int(
            total_eligible_samples
        ),
    }


validation_event_audit = event_episode_audit(
    dataset=validation_dataset,
    eligible_probability=(
        validation_hazard_probability
    ),
    threshold=HAZARD_DECISION_THRESHOLD,
)


print_header(
    "🧪 TEST E — EVENT-ONSET AND FALSE-ALERT EPISODE AUDIT"
)

for key, value in (
    validation_event_audit.items()
):
    print(f"  {key:<52}: {value}")


# =================================================================================================
# 22. CROSS-REGIME TRANSFER
# =================================================================================================

def relative_oracle_recovery(
    model_value: float,
    baseline_value: float,
    oracle_value: float,
) -> float:
    denominator = (
        oracle_value
        - baseline_value
    )

    if abs(denominator) < 1e-8:
        if model_value >= oracle_value:
            return 1.0

        return 0.0

    return float(
        (model_value - baseline_value)
        / denominator
    )


def evaluate_transfer_regime(
    dataset: Dict,
) -> Dict:
    point_prediction = (
        selected_point_model.predict(
            feature_view(
                dataset,
                BEST_POINT_MODEL_NAME,
            )
        )
    )

    point_metrics = evaluate_point(
        dataset["y_point"],
        point_prediction,
        dataset["stream_ids"],
    )

    persistence_prediction = (
        dataset["current_latent"]
    )

    persistence_metrics = evaluate_point(
        dataset["y_point"],
        persistence_prediction,
        dataset["stream_ids"],
    )

    oracle_prediction = (
        point_oracle.predict(
            latent_history_features(
                dataset
            )
        )
    )

    oracle_metrics = evaluate_point(
        dataset["y_point"],
        oracle_prediction,
        dataset["stream_ids"],
    )

    persistence_rho = float(
        persistence_metrics["rho"]
    )

    oracle_rho = float(
        oracle_metrics["rho"]
    )

    model_rho = float(
        point_metrics["rho"]
    )

    oracle_recovery = (
        relative_oracle_recovery(
            model_value=model_rho,
            baseline_value=persistence_rho,
            oracle_value=oracle_rho,
        )
    )

    oracle_is_forecastable = bool(
        oracle_rho
        >= CFG.minimum_point_rho
        and oracle_metrics["r2"]
        >= CFG.minimum_point_r2
    )

    # A transfer passes if the selected model has useful absolute
    # performance and either recovers most of the oracle advantage
    # or is already within a small tolerance of the oracle.
    point_close_to_oracle = bool(
        model_rho
        >= oracle_rho - 0.08
    )

    point_recovers_oracle = bool(
        oracle_recovery >= 0.60
    )

    point_gate = bool(
        oracle_is_forecastable
        and point_metrics["rho"]
        >= CFG.minimum_point_rho
        and point_metrics[
            "macro_stream_rho"
        ] >= CFG.minimum_point_macro_rho
        and (
            point_close_to_oracle
            or point_recovers_oracle
        )
    )

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    hazard_labels = dataset[
        "y_hazard"
    ][hazard_mask]

    hazard_probability = (
        predict_hazard_probability(
            dataset
        )
    )

    hazard_metrics = evaluate_hazard(
        hazard_labels,
        hazard_probability,
        HAZARD_DECISION_THRESHOLD,
    )

    hazard_oracle_probability = (
        hazard_oracle.predict_proba(
            latent_history_features(
                dataset
            )[hazard_mask]
        )[:, 1]
    )

    hazard_oracle_metrics = (
        evaluate_hazard(
            hazard_labels,
            hazard_oracle_probability,
            threshold=0.5,
        )
    )

    prevalence = float(
        np.mean(hazard_labels)
    )

    hazard_ap = (
        hazard_metrics["auprc"]
        if hazard_metrics["auprc"]
        is not None
        else 0.0
    )

    ap_gain = float(
        hazard_ap - prevalence
    )

    oracle_auc = (
        hazard_oracle_metrics["auroc"]
        if hazard_oracle_metrics["auroc"]
        is not None
        else 0.0
    )

    oracle_ap = (
        hazard_oracle_metrics["auprc"]
        if hazard_oracle_metrics["auprc"]
        is not None
        else 0.0
    )

    hazard_oracle_forecastable = bool(
        oracle_auc >= CFG.minimum_hazard_auc
        and (
            oracle_ap - prevalence
        ) >= CFG.minimum_hazard_ap_gain
    )

    hazard_auc = (
        hazard_metrics["auroc"]
        if hazard_metrics["auroc"]
        is not None
        else 0.0
    )

    hazard_gate = bool(
        hazard_oracle_forecastable
        and hazard_auc
        >= CFG.minimum_hazard_auc
        and ap_gain
        >= CFG.minimum_hazard_ap_gain
        and hazard_metrics["fpr"]
        <= CFG.maximum_policy_fpr + 0.07
    )

    event_audit = event_episode_audit(
        dataset=dataset,
        eligible_probability=(
            hazard_probability
        ),
        threshold=HAZARD_DECISION_THRESHOLD,
    )

    return {
        "point": {
            **point_metrics,
            "persistence": (
                persistence_metrics
            ),
            "oracle": oracle_metrics,
            "oracle_recovery": (
                oracle_recovery
            ),
            "oracle_forecastable": (
                oracle_is_forecastable
            ),
            "gate": point_gate,
        },
        "hazard": {
            **hazard_metrics,
            "ap_gain_over_prevalence": (
                ap_gain
            ),
            "oracle": (
                hazard_oracle_metrics
            ),
            "oracle_forecastable": (
                hazard_oracle_forecastable
            ),
            "gate": hazard_gate,
        },
        "event_audit": event_audit,
    }


transfer_results = {}

print_header(
    "🧪 TEST F — CROSS-REGIME TRANSFER"
)

for regime_name, dataset in (
    test_datasets.items()
):
    result = evaluate_transfer_regime(
        dataset
    )

    transfer_results[
        regime_name
    ] = result

    point = result["point"]
    hazard = result["hazard"]

    print(f"\n{regime_name}")

    print(
        f"  Point  | "
        f"rho={point['rho']: .4f} | "
        f"Macro={point['macro_stream_rho']: .4f} | "
        f"R²={point['r2']: .4f} | "
        f"Persist={point['persistence']['rho']: .4f} | "
        f"Oracle={point['oracle']['rho']: .4f} | "
        f"Recovery={point['oracle_recovery']: .4f} | "
        f"Gate={'PASS' if point['gate'] else 'FAIL'}"
    )

    print(
        f"  Hazard | "
        f"AUC={(hazard['auroc'] or 0.0): .4f} | "
        f"AP={(hazard['auprc'] or 0.0): .4f} | "
        f"Prev={hazard['prevalence']: .4f} | "
        f"ΔAP={hazard['ap_gain_over_prevalence']: .4f} | "
        f"FPR={hazard['fpr']: .4f} | "
        f"Gate={'PASS' if hazard['gate'] else 'FAIL'}"
    )


# =================================================================================================
# 23. MECHANISM NOVELTY FEATURES
# =================================================================================================

def mechanism_novelty_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]
    activation = dataset[
        "X_activation"
    ]

    signed_error = raw[:, :, 4]
    absolute_error = raw[:, :, 5]

    error_difference = np.diff(
        signed_error,
        axis=1,
    )

    if signed_error.shape[1] >= 3:
        lag_one_product = (
            signed_error[:, 1:]
            * signed_error[:, :-1]
        )

        lag_two_product = (
            signed_error[:, 2:]
            * signed_error[:, :-2]
        )

        lag_one_mean = np.mean(
            lag_one_product,
            axis=1,
            keepdims=True,
        )

        lag_two_mean = np.mean(
            lag_two_product,
            axis=1,
            keepdims=True,
        )
    else:
        lag_one_mean = np.zeros(
            (len(raw), 1),
            dtype=np.float32,
        )

        lag_two_mean = np.zeros(
            (len(raw), 1),
            dtype=np.float32,
        )

    raw_summary = np.concatenate([
        np.mean(
            raw,
            axis=1,
        ),
        np.std(
            raw,
            axis=1,
        ),
        raw[:, -1, :],
        raw[:, -1, :]
        - raw[:, 0, :],
    ], axis=1)

    activation_summary = np.concatenate([
        np.mean(
            activation,
            axis=1,
        ),
        np.std(
            activation,
            axis=1,
        ),
        activation[:, -1, :],
        activation[:, -1, :]
        - activation[:, 0, :],
    ], axis=1)

    dynamic_error_features = np.concatenate([
        signed_error,
        absolute_error,
        error_difference,
        lag_one_mean,
        lag_two_mean,
    ], axis=1)

    return np.concatenate([
        raw_summary,
        activation_summary,
        dynamic_error_features,
    ], axis=1).astype(np.float32)


novelty_scaler = StandardScaler()

train_novelty_features = (
    mechanism_novelty_features(
        train_dataset
    )
)

train_novelty_scaled = (
    novelty_scaler.fit_transform(
        train_novelty_features
    )
)

novelty_detector = IsolationForest(
    n_estimators=350,
    contamination="auto",
    max_samples="auto",
    random_state=CFG.seed,
    n_jobs=-1,
)

novelty_detector.fit(
    train_novelty_scaled
)


def novelty_score(
    dataset: Dict,
) -> np.ndarray:
    features = mechanism_novelty_features(
        dataset
    )

    scaled = novelty_scaler.transform(
        features
    )

    # Larger score means more novel.
    return -novelty_detector.score_samples(
        scaled
    )


novelty_calibration_scores = (
    novelty_score(
        calibration_selection_dataset
    )
)

NOVELTY_THRESHOLD = float(
    np.quantile(
        novelty_calibration_scores,
        1.0 - CFG.maximum_novelty_fpr,
    )
)

validation_novelty_scores = (
    novelty_score(
        validation_dataset
    )
)

validation_novelty_fpr = float(
    np.mean(
        validation_novelty_scores
        >= NOVELTY_THRESHOLD
    )
)

known_transfer_names = (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
)

unknown_regime_names = (
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
)

known_transfer_scores = (
    np.concatenate([
        novelty_score(
            test_datasets[name]
        )
        for name in known_transfer_names
    ])
)

unknown_scores_by_regime = {
    name: novelty_score(
        test_datasets[name]
    )
    for name in unknown_regime_names
}

unknown_scores = np.concatenate(
    list(
        unknown_scores_by_regime.values()
    )
)

novelty_evaluation_scores = (
    np.concatenate([
        validation_novelty_scores,
        known_transfer_scores,
        unknown_scores,
    ])
)

novelty_evaluation_labels = (
    np.concatenate([
        np.zeros(
            len(validation_novelty_scores),
            dtype=np.int64,
        ),
        np.zeros(
            len(known_transfer_scores),
            dtype=np.int64,
        ),
        np.ones(
            len(unknown_scores),
            dtype=np.int64,
        ),
    ])
)

novelty_auc = (
    safe_auroc(
        novelty_evaluation_labels,
        novelty_evaluation_scores,
    )
    or 0.0
)

known_transfer_novelty_fpr = float(
    np.mean(
        known_transfer_scores
        >= NOVELTY_THRESHOLD
    )
)

unknown_novelty_tpr = float(
    np.mean(
        unknown_scores
        >= NOVELTY_THRESHOLD
    )
)

novelty_regime_results = {}

for name in unknown_regime_names:
    scores = unknown_scores_by_regime[
        name
    ]

    novelty_regime_results[name] = {
        "mean_score": float(
            np.mean(scores)
        ),
        "tpr": float(
            np.mean(
                scores
                >= NOVELTY_THRESHOLD
            )
        ),
    }


# =================================================================================================
# 24. NOVELTY / DEGRADATION INDEPENDENCE AND CONTAMINATION AUDIT
# =================================================================================================

def phi_coefficient(
    binary_a: np.ndarray,
    binary_b: np.ndarray,
) -> float:
    binary_a = np.asarray(
        binary_a,
        dtype=np.int64,
    )

    binary_b = np.asarray(
        binary_b,
        dtype=np.int64,
    )

    if (
        len(binary_a) == 0
        or np.std(binary_a) < 1e-12
        or np.std(binary_b) < 1e-12
    ):
        return 0.0

    value = np.corrcoef(
        binary_a,
        binary_b,
    )[0, 1]

    if not np.isfinite(value):
        return 0.0

    return float(value)


factorial_novelty_labels = []
factorial_degradation_labels = []
factorial_detected_novelty = []

factorial_datasets = {
    "VALIDATION_KNOWN": validation_dataset,
    **{
        name: test_datasets[name]
        for name in known_transfer_names
    },
    **{
        name: test_datasets[name]
        for name in unknown_regime_names
    },
}

for name, dataset in (
    factorial_datasets.items()
):
    scores = novelty_score(
        dataset
    )

    true_novelty = (
        dataset["novelty_labels"]
    ).astype(np.int64)

    degradation = (
        np.abs(
            dataset["current_latent"]
        )
        > CFG.latent_event_threshold
    ).astype(np.int64)

    detected_novelty = (
        scores
        >= NOVELTY_THRESHOLD
    ).astype(np.int64)

    factorial_novelty_labels.append(
        true_novelty
    )

    factorial_degradation_labels.append(
        degradation
    )

    factorial_detected_novelty.append(
        detected_novelty
    )

factorial_novelty_labels = (
    np.concatenate(
        factorial_novelty_labels
    )
)

factorial_degradation_labels = (
    np.concatenate(
        factorial_degradation_labels
    )
)

factorial_detected_novelty = (
    np.concatenate(
        factorial_detected_novelty
    )
)

generator_novelty_degradation_phi = (
    phi_coefficient(
        factorial_novelty_labels,
        factorial_degradation_labels,
    )
)

detector_degradation_phi = (
    phi_coefficient(
        factorial_detected_novelty,
        factorial_degradation_labels,
    )
)


def evaluate_hazard_channel(
    dataset: Dict,
    model_name: str,
    model_instance: Pipeline,
    use_calibrator: bool,
) -> Dict:
    mask = dataset[
        "hazard_eligible"
    ]

    score = decision_score(
        model_instance,
        feature_view(
            dataset,
            model_name,
        )[mask],
    )

    if use_calibrator:
        probability = (
            hazard_calibrator.predict(
                score
            )
        )
    else:
        probability = (
            1.0
            / (
                1.0
                + np.exp(-score)
            )
        )

    return evaluate_hazard(
        dataset["y_hazard"][mask],
        probability,
        HAZARD_DECISION_THRESHOLD,
    )


combined_hazard_model = (
    hazard_models[
        "COMBINED_ORDERED"
    ]
)

novel_hazard_contamination = {}

for name in unknown_regime_names:
    dataset = test_datasets[name]

    raw_metrics = evaluate_hazard_channel(
        dataset=dataset,
        model_name=(
            BEST_HAZARD_MODEL_NAME
        ),
        model_instance=(
            selected_hazard_model
        ),
        use_calibrator=True,
    )

    combined_metrics = (
        evaluate_hazard_channel(
            dataset=dataset,
            model_name=(
                "COMBINED_ORDERED"
            ),
            model_instance=(
                combined_hazard_model
            ),
            use_calibrator=False,
        )
    )

    novel_hazard_contamination[
        name
    ] = {
        "selected_raw_causal": (
            raw_metrics
        ),
        "combined_activation_channel": (
            combined_metrics
        ),
        "combined_minus_raw_fpr": float(
            combined_metrics["fpr"]
            - raw_metrics["fpr"]
        ),
        "combined_minus_raw_brier": float(
            combined_metrics["brier"]
            - raw_metrics["brier"]
        ),
    }


print_header(
    "🧪 TEST G — FIXED-FORMAT MECHANISM NOVELTY"
)

print(
    f"Novelty AUROC                 : "
    f"{novelty_auc:.4f}"
)

print(
    f"Validation known FPR          : "
    f"{validation_novelty_fpr:.4f}"
)

print(
    f"Known-parameter-transfer FPR  : "
    f"{known_transfer_novelty_fpr:.4f}"
)

print(
    f"Unknown-mechanism TPR         : "
    f"{unknown_novelty_tpr:.4f}"
)

print(
    f"Novelty threshold             : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

for name, result in (
    novelty_regime_results.items()
):
    print(
        f"  {name:<22} | "
        f"mean score={result['mean_score']:.4f} | "
        f"TPR={result['tpr']:.4f}"
    )


print_header(
    "🧪 TEST H — NOVELTY / DEGRADATION DISENTANGLEMENT"
)

print(
    f"Generator novelty/degradation phi : "
    f"{generator_novelty_degradation_phi:+.4f}"
)

print(
    f"Detector/degradation phi           : "
    f"{detector_degradation_phi:+.4f}"
)

for name, result in (
    novel_hazard_contamination.items()
):
    raw_result = result[
        "selected_raw_causal"
    ]

    combined_result = result[
        "combined_activation_channel"
    ]

    print(f"\n{name}")

    print(
        f"  Raw causal | "
        f"AUC={(raw_result['auroc'] or 0.0):.4f} | "
        f"AP={(raw_result['auprc'] or 0.0):.4f} | "
        f"FPR={raw_result['fpr']:.4f} | "
        f"Brier={raw_result['brier']:.4f}"
    )

    print(
        f"  Combined   | "
        f"AUC={(combined_result['auroc'] or 0.0):.4f} | "
        f"AP={(combined_result['auprc'] or 0.0):.4f} | "
        f"FPR={combined_result['fpr']:.4f} | "
        f"Brier={combined_result['brier']:.4f}"
    )

    print(
        f"  ΔFPR combined−raw   : "
        f"{result['combined_minus_raw_fpr']:+.4f}"
    )

    print(
        f"  ΔBrier combined−raw : "
        f"{result['combined_minus_raw_brier']:+.4f}"
    )


# =================================================================================================
# 25. STREAM-LEVEL BOOTSTRAP INTERVALS
# =================================================================================================

def percentile_interval(
    values: List[float],
    confidence: float = 0.95,
) -> List[Optional[float]]:
    finite_values = np.asarray([
        value
        for value in values
        if value is not None
        and np.isfinite(value)
    ], dtype=np.float64)

    if len(finite_values) == 0:
        return [None, None]

    alpha = (
        1.0 - confidence
    ) / 2.0

    return [
        float(
            np.quantile(
                finite_values,
                alpha,
            )
        ),
        float(
            np.quantile(
                finite_values,
                1.0 - alpha,
            )
        ),
    ]


def stream_level_bootstrap(
    dataset: Dict,
    point_prediction: np.ndarray,
    hazard_probability: np.ndarray,
    iterations: int,
    seed: int,
) -> Dict:
    rng = np.random.default_rng(seed)

    unique_streams = np.unique(
        dataset["stream_ids"]
    )

    eligible_indices = np.flatnonzero(
        dataset["hazard_eligible"]
    )

    eligible_stream_ids = dataset[
        "stream_ids"
    ][eligible_indices]

    hazard_labels = dataset[
        "y_hazard"
    ][eligible_indices]

    point_rhos = []
    point_macro_rhos = []
    point_r2_values = []

    hazard_aucs = []
    hazard_aps = []
    hazard_briers = []

    for _ in range(iterations):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        point_true_parts = []
        point_prediction_parts = []
        bootstrap_stream_ids = []

        hazard_true_parts = []
        hazard_probability_parts = []

        for bootstrap_id, stream_id in enumerate(
            sampled_streams
        ):
            point_mask = (
                dataset["stream_ids"]
                == stream_id
            )

            point_true_parts.append(
                dataset[
                    "y_point"
                ][point_mask]
            )

            point_prediction_parts.append(
                point_prediction[
                    point_mask
                ]
            )

            bootstrap_stream_ids.append(
                np.full(
                    np.sum(point_mask),
                    bootstrap_id,
                    dtype=np.int64,
                )
            )

            hazard_mask = (
                eligible_stream_ids
                == stream_id
            )

            hazard_true_parts.append(
                hazard_labels[
                    hazard_mask
                ]
            )

            hazard_probability_parts.append(
                hazard_probability[
                    hazard_mask
                ]
            )

        boot_point_true = (
            np.concatenate(
                point_true_parts
            )
        )

        boot_point_prediction = (
            np.concatenate(
                point_prediction_parts
            )
        )

        boot_stream_ids = (
            np.concatenate(
                bootstrap_stream_ids
            )
        )

        point_rhos.append(
            safe_spearman(
                boot_point_true,
                boot_point_prediction,
            )
        )

        point_macro_rhos.append(
            macro_stream_rho(
                boot_point_true,
                boot_point_prediction,
                boot_stream_ids,
            )
        )

        point_r2_values.append(
            float(
                r2_score(
                    boot_point_true,
                    boot_point_prediction,
                )
            )
        )

        boot_hazard_true = (
            np.concatenate(
                hazard_true_parts
            )
        )

        boot_hazard_probability = (
            np.concatenate(
                hazard_probability_parts
            )
        )

        auc = safe_auroc(
            boot_hazard_true,
            boot_hazard_probability,
        )

        ap = safe_auprc(
            boot_hazard_true,
            boot_hazard_probability,
        )

        if auc is not None:
            hazard_aucs.append(auc)

        if ap is not None:
            hazard_aps.append(ap)

        if len(
            np.unique(
                boot_hazard_true
            )
        ) >= 2:
            hazard_briers.append(
                float(
                    brier_score_loss(
                        boot_hazard_true,
                        boot_hazard_probability,
                    )
                )
            )

    return {
        "point_rho_95_ci": (
            percentile_interval(
                point_rhos
            )
        ),
        "point_macro_rho_95_ci": (
            percentile_interval(
                point_macro_rhos
            )
        ),
        "point_r2_95_ci": (
            percentile_interval(
                point_r2_values
            )
        ),
        "hazard_auc_95_ci": (
            percentile_interval(
                hazard_aucs
            )
        ),
        "hazard_ap_95_ci": (
            percentile_interval(
                hazard_aps
            )
        ),
        "hazard_brier_95_ci": (
            percentile_interval(
                hazard_briers
            )
        ),
    }


bootstrap_results = stream_level_bootstrap(
    dataset=validation_dataset,
    point_prediction=(
        validation_point_prediction
    ),
    hazard_probability=(
        validation_hazard_probability
    ),
    iterations=CFG.bootstrap_iterations,
    seed=CFG.seed + 300_000,
)


print_header(
    "🧪 TEST I — STREAM-LEVEL BOOTSTRAP INTERVALS"
)

print(
    f"Point rho 95% CI       : "
    f"{bootstrap_results['point_rho_95_ci']}"
)

print(
    f"Point macro rho 95% CI : "
    f"{bootstrap_results['point_macro_rho_95_ci']}"
)

print(
    f"Point R² 95% CI        : "
    f"{bootstrap_results['point_r2_95_ci']}"
)

print(
    f"Hazard AUC 95% CI      : "
    f"{bootstrap_results['hazard_auc_95_ci']}"
)

print(
    f"Hazard AP 95% CI       : "
    f"{bootstrap_results['hazard_ap_95_ci']}"
)

print(
    f"Hazard Brier 95% CI    : "
    f"{bootstrap_results['hazard_brier_95_ci']}"
)


# =================================================================================================
# 26. SCIENTIFIC GATES
# =================================================================================================

selected_layer_result = (
    layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]
)

feedback_observability_valid = bool(
    selected_layer_result[
        "activation_only"
    ]["rho"]
    >= CFG.minimum_observability_rho
    and selected_layer_result[
        "activation_only"
    ]["r2"]
    >= CFG.minimum_observability_r2
)

activation_incremental_value_valid = bool(
    selected_layer_result[
        "delta_r2_over_raw"
    ] >= 0.01
)

point_oracle_forecastable = bool(
    validation_oracle_metrics["rho"]
    >= CFG.minimum_point_rho
    and validation_oracle_metrics["r2"]
    >= CFG.minimum_point_r2
)

hazard_oracle_auc = (
    validation_hazard_oracle_metrics[
        "auroc"
    ]
    or 0.0
)

hazard_oracle_ap = (
    validation_hazard_oracle_metrics[
        "auprc"
    ]
    or 0.0
)

hazard_oracle_ap_gain = float(
    hazard_oracle_ap
    - validation_hazard_oracle_metrics[
        "prevalence"
    ]
)

hazard_oracle_forecastable = bool(
    hazard_oracle_auc
    >= CFG.minimum_hazard_auc
    and hazard_oracle_ap_gain
    >= CFG.minimum_hazard_ap_gain
)

selected_point_forecaster_valid = bool(
    validation_point_metrics["rho"]
    >= CFG.minimum_point_rho
    and validation_point_metrics[
        "macro_stream_rho"
    ] >= CFG.minimum_point_macro_rho
    and validation_point_metrics["r2"]
    >= CFG.minimum_point_r2
)

validation_hazard_auc = (
    validation_hazard_metrics[
        "auroc"
    ]
    or 0.0
)

validation_hazard_ap = (
    validation_hazard_metrics[
        "auprc"
    ]
    or 0.0
)

validation_hazard_ap_gain = float(
    validation_hazard_ap
    - validation_hazard_metrics[
        "prevalence"
    ]
)

selected_hazard_forecaster_valid = bool(
    validation_hazard_auc
    >= CFG.minimum_hazard_auc
    and validation_hazard_ap_gain
    >= CFG.minimum_hazard_ap_gain
)

point_temporal_order_established = bool(
    point_order_advantage
    >= CFG.minimum_point_order_advantage
    and ordered_point_rho
    > reversed_point_rho
)

hazard_temporal_order_established = bool(
    hazard_order_advantage
    >= CFG.minimum_hazard_order_advantage
    and ordered_hazard_auc
    > reversed_hazard_auc
)

probability_calibration_valid = bool(
    validation_hazard_metrics[
        "brier"
    ] < constant_brier
)

operational_warning_policy_valid = bool(
    validation_event_audit[
        "event_recall"
    ] >= CFG.minimum_event_recall
    and validation_event_audit[
        "false_alert_episodes_per_100_eligible_samples"
    ] <= CFG.maximum_false_alert_episodes_per_100
)

novelty_sensor_valid = bool(
    novelty_auc
    >= CFG.minimum_novelty_auc
    and unknown_novelty_tpr
    >= CFG.minimum_novelty_tpr
    and validation_novelty_fpr
    <= CFG.maximum_novelty_fpr + 0.03
)

factorial_novelty_independence_valid = bool(
    abs(
        generator_novelty_degradation_phi
    ) <= 0.20
    and abs(
        detector_degradation_phi
    ) <= 0.25
)

known_point_transfer_gates = {
    name: bool(
        transfer_results[
            name
        ]["point"]["gate"]
    )
    for name in known_transfer_names
}

known_hazard_transfer_gates = {
    name: bool(
        transfer_results[
            name
        ]["hazard"]["gate"]
    )
    for name in known_transfer_names
}

point_transfer_count = int(
    sum(
        known_point_transfer_gates.values()
    )
)

hazard_transfer_count = int(
    sum(
        known_hazard_transfer_gates.values()
    )
)

required_known_transfer_passes = (
    len(known_transfer_names) // 2
    + 1
)

point_transfer_strict_majority = bool(
    point_transfer_count
    >= required_known_transfer_passes
)

hazard_transfer_strict_majority = bool(
    hazard_transfer_count
    >= required_known_transfer_passes
)

# The causal forecaster is considered insulated from activation
# novelty if the raw channel is not systematically worse than the
# combined channel on both unknown regimes.
raw_channel_insulation_votes = []

for name in unknown_regime_names:
    result = novel_hazard_contamination[
        name
    ]

    raw_metrics = result[
        "selected_raw_causal"
    ]

    combined_metrics = result[
        "combined_activation_channel"
    ]

    raw_is_insulated = bool(
        raw_metrics["fpr"]
        <= combined_metrics["fpr"] + 0.05
        or raw_metrics["brier"]
        <= combined_metrics["brier"] + 0.03
    )

    raw_channel_insulation_votes.append(
        raw_is_insulated
    )

causal_novelty_channel_separation_valid = bool(
    all(
        raw_channel_insulation_votes
    )
)

benchmark_construction_valid = bool(
    point_oracle_forecastable
    and hazard_oracle_forecastable
    and (
        point_temporal_order_established
        or hazard_temporal_order_established
    )
)

observable_feedback_self_model = bool(
    feedback_observability_valid
)

temporal_order_learning = bool(
    point_temporal_order_established
    or hazard_temporal_order_established
)

hazard_event_onset_system = bool(
    selected_hazard_forecaster_valid
    and probability_calibration_valid
    and operational_warning_policy_valid
)

cross_regime_transfer = bool(
    point_transfer_strict_majority
    and hazard_transfer_strict_majority
)

novelty_detection = bool(
    novelty_sensor_valid
)

disentangled_architecture = bool(
    causal_novelty_channel_separation_valid
)

overall_status = bool(
    benchmark_construction_valid
    and observable_feedback_self_model
    and temporal_order_learning
    and hazard_event_onset_system
    and cross_regime_transfer
    and novelty_detection
    and disentangled_architecture
)


scientific_gates = {
    "feedback_observability_valid": (
        feedback_observability_valid
    ),
    "activation_incremental_value_valid": (
        activation_incremental_value_valid
    ),
    "point_oracle_forecastable": (
        point_oracle_forecastable
    ),
    "hazard_oracle_forecastable": (
        hazard_oracle_forecastable
    ),
    "selected_point_forecaster_valid": (
        selected_point_forecaster_valid
    ),
    "selected_hazard_forecaster_valid": (
        selected_hazard_forecaster_valid
    ),
    "point_temporal_order_established": (
        point_temporal_order_established
    ),
    "hazard_temporal_order_established": (
        hazard_temporal_order_established
    ),
    "probability_calibration_valid": (
        probability_calibration_valid
    ),
    "operational_warning_policy_valid": (
        operational_warning_policy_valid
    ),
    "novelty_sensor_valid": (
        novelty_sensor_valid
    ),
    "factorial_novelty_independence_valid": (
        factorial_novelty_independence_valid
    ),
    "causal_novelty_channel_separation_valid": (
        causal_novelty_channel_separation_valid
    ),
    "point_transfer_strict_majority": (
        point_transfer_strict_majority
    ),
    "hazard_transfer_strict_majority": (
        hazard_transfer_strict_majority
    ),
}


scientific_statuses = {
    "benchmark_construction": (
        benchmark_construction_valid
    ),
    "observable_feedback_self_model": (
        observable_feedback_self_model
    ),
    "temporal_order_learning": (
        temporal_order_learning
    ),
    "hazard_event_onset_system": (
        hazard_event_onset_system
    ),
    "cross_regime_transfer": (
        cross_regime_transfer
    ),
    "novelty_detection": (
        novelty_detection
    ),
    "disentangled_architecture": (
        disentangled_architecture
    ),
    "overall": overall_status,
}


# =================================================================================================
# 27. AUTOMATED DIAGNOSIS
# =================================================================================================

diagnostic_codes = []

if not feedback_observability_valid:
    diagnostic_codes.append(
        "POST_FEEDBACK_LATENT_NOT_OBSERVABLE"
    )

if (
    feedback_observability_valid
    and not activation_incremental_value_valid
):
    diagnostic_codes.append(
        "ACTIVATION_ENCODES_FEEDBACK_WITHOUT_INCREMENTAL_VALUE"
    )

if not point_oracle_forecastable:
    diagnostic_codes.append(
        "POINT_TARGET_NOT_FORECASTABLE_BY_TRUE_HISTORY_ORACLE"
    )

if not hazard_oracle_forecastable:
    diagnostic_codes.append(
        "HAZARD_TARGET_NOT_FORECASTABLE_BY_TRUE_HISTORY_ORACLE"
    )

if not selected_point_forecaster_valid:
    diagnostic_codes.append(
        "SELECTED_POINT_FORECASTER_INVALID"
    )

if not selected_hazard_forecaster_valid:
    diagnostic_codes.append(
        "SELECTED_HAZARD_FORECASTER_INVALID"
    )

if not point_temporal_order_established:
    diagnostic_codes.append(
        "POINT_TEMPORAL_ORDER_NOT_ESTABLISHED"
    )

if not hazard_temporal_order_established:
    diagnostic_codes.append(
        "HAZARD_TEMPORAL_ORDER_NOT_ESTABLISHED"
    )

if not probability_calibration_valid:
    diagnostic_codes.append(
        "HAZARD_PROBABILITY_CALIBRATION_FAILED"
    )

if not operational_warning_policy_valid:
    diagnostic_codes.append(
        "OPERATIONAL_WARNING_POLICY_FAILED"
    )

if not point_transfer_strict_majority:
    diagnostic_codes.append(
        "POINT_KNOWN_FAMILY_TRANSFER_FAILED"
    )

if not hazard_transfer_strict_majority:
    diagnostic_codes.append(
        "HAZARD_KNOWN_FAMILY_TRANSFER_FAILED"
    )

if not novelty_sensor_valid:
    diagnostic_codes.append(
        "MECHANISM_NOVELTY_DETECTION_FAILED"
    )

if not factorial_novelty_independence_valid:
    diagnostic_codes.append(
        "NOVELTY_DEGRADATION_STATISTICAL_CONFOUNDING"
    )

if not causal_novelty_channel_separation_valid:
    diagnostic_codes.append(
        "ACTIVATION_NOVELTY_CONTAMINATES_CAUSAL_CHANNEL"
    )

if not diagnostic_codes:
    diagnostic_codes.append(
        "ALL_SCIENTIFIC_GATES_PASSED"
    )


# =================================================================================================
# 28. PRINT AUTOMATED SUMMARY
# =================================================================================================

print_header(
    "🧠 AUTOMATED DIAGNOSTIC SUMMARY"
)

for gate_name, gate_value in (
    scientific_gates.items()
):
    symbol = "✅" if gate_value else "❌"

    print(
        f"{symbol} {gate_name}"
    )


print("\nKnown-family point transfer gates:")

for name, gate in (
    known_point_transfer_gates.items()
):
    symbol = "✅" if gate else "❌"

    print(
        f"  {symbol} {name}"
    )

print(
    f"\nPoint transfer count: "
    f"{point_transfer_count}/"
    f"{len(known_transfer_names)} "
    f"(strict majority requires "
    f"{required_known_transfer_passes})"
)


print("\nKnown-family hazard transfer gates:")

for name, gate in (
    known_hazard_transfer_gates.items()
):
    symbol = "✅" if gate else "❌"

    print(
        f"  {symbol} {name}"
    )

print(
    f"\nHazard transfer count: "
    f"{hazard_transfer_count}/"
    f"{len(known_transfer_names)} "
    f"(strict majority requires "
    f"{required_known_transfer_passes})"
)


print("\nSelected architecture:")

print(
    f"  Feedback layer        : "
    f"{BEST_FEEDBACK_LAYER}"
)

print(
    f"  Activation PCA dim    : "
    f"{pca_dimension}"
)

print(
    f"  Causal point model    : "
    f"{BEST_POINT_MODEL_NAME}"
)

print(
    f"  Causal hazard model   : "
    f"{BEST_HAZARD_MODEL_NAME}"
)

print(
    f"  Hazard threshold      : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)

print(
    f"  Novelty threshold     : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

print(
    "  Architecture policy  : "
    "RAW causal forecasting + separated "
    "activation-assisted novelty sensing"
)


print("\nOverall scientific statuses:")

for status_name, status_value in (
    scientific_statuses.items()
):
    text = (
        "PASS"
        if status_value
        else "FAIL"
    )

    print(
        f"  {status_name:<38}: "
        f"{text}"
    )


print("\nFinal diagnosis:")

print(
    " | ".join(
        diagnostic_codes
    )
)


# =================================================================================================
# 29. TELEMETRY
# =================================================================================================

telemetry = {
    "milestone": CFG.milestone,
    "title": CFG.title,
    "configuration": asdict(CFG),
    "runtime": {
        "device": DEVICE,
        "torch_dtype": str(
            TORCH_DTYPE
        ),
        "model_name": CFG.model_name,
        "number_of_model_layers": (
            number_of_layers
        ),
        "evaluated_layers": valid_layers,
    },
    "selected_architecture": {
        "feedback_layer": (
            BEST_FEEDBACK_LAYER
        ),
        "activation_pca_dimension": (
            pca_dimension
        ),
        "point_model": (
            BEST_POINT_MODEL_NAME
        ),
        "hazard_model": (
            BEST_HAZARD_MODEL_NAME
        ),
        "hazard_decision_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "novelty_threshold": (
            NOVELTY_THRESHOLD
        ),
        "causal_channel": (
            "raw_observable_feedback_only"
        ),
        "novelty_channel": (
            "raw_plus_activation_temporal_signature"
        ),
    },
    "dataset_shapes": {
        "train_raw": (
            train_dataset[
                "X_raw"
            ].shape
        ),
        "train_activation": (
            train_dataset[
                "X_activation"
            ].shape
        ),
        "calibration_selection_raw": (
            calibration_selection_dataset[
                "X_raw"
            ].shape
        ),
        "calibration_probability_raw": (
            calibration_probability_dataset[
                "X_raw"
            ].shape
        ),
        "calibration_policy_raw": (
            calibration_policy_dataset[
                "X_raw"
            ].shape
        ),
        "validation_raw": (
            validation_dataset[
                "X_raw"
            ].shape
        ),
    },
    "hazard_prevalence": {
        "train": float(
            np.mean(
                train_dataset[
                    "y_hazard"
                ][train_hazard_mask]
            )
        ),
        "calibration_selection": float(
            np.mean(
                calibration_selection_dataset[
                    "y_hazard"
                ][selection_hazard_mask]
            )
        ),
        "calibration_probability": float(
            np.mean(
                calibration_probability_dataset[
                    "y_hazard"
                ][probability_hazard_mask]
            )
        ),
        "calibration_policy": float(
            np.mean(
                calibration_policy_dataset[
                    "y_hazard"
                ][policy_hazard_mask]
            )
        ),
        "validation": float(
            np.mean(
                validation_hazard_labels
            )
        ),
    },
    "observability": {
        "raw_only": (
            raw_observability_metrics
        ),
        "layers": (
            layer_observability_results
        ),
        "selected_layer": (
            BEST_FEEDBACK_LAYER
        ),
    },
    "model_selection": {
        "point": (
            point_selection_results
        ),
        "hazard": (
            hazard_selection_results
        ),
        "selected_point": (
            BEST_POINT_MODEL_NAME
        ),
        "selected_hazard": (
            BEST_HAZARD_MODEL_NAME
        ),
    },
    "probability_calibration": {
        "method": (
            "independent_platt_scaling"
        ),
        "policy_threshold_selection": (
            threshold_selection
        ),
        "validation_model_brier": (
            validation_hazard_metrics[
                "brier"
            ]
        ),
        "validation_constant_brier": (
            constant_brier
        ),
    },
    "validation": {
        "selected_point": (
            validation_point_metrics
        ),
        "point_oracle": (
            validation_oracle_metrics
        ),
        "persistence": (
            validation_persistence_metrics
        ),
        "selected_hazard": (
            validation_hazard_metrics
        ),
        "hazard_oracle": (
            validation_hazard_oracle_metrics
        ),
        "event_audit": (
            validation_event_audit
        ),
    },
    "order_ablation": (
        order_ablation
    ),
    "cross_regime_transfer": (
        transfer_results
    ),
    "novelty": {
        "auc": novelty_auc,
        "validation_known_fpr": (
            validation_novelty_fpr
        ),
        "known_transfer_fpr": (
            known_transfer_novelty_fpr
        ),
        "unknown_tpr": (
            unknown_novelty_tpr
        ),
        "threshold": (
            NOVELTY_THRESHOLD
        ),
        "regime_results": (
            novelty_regime_results
        ),
    },
    "disentanglement": {
        "generator_novelty_degradation_phi": (
            generator_novelty_degradation_phi
        ),
        "detector_degradation_phi": (
            detector_degradation_phi
        ),
        "hazard_channel_contamination": (
            novel_hazard_contamination
        ),
    },
    "bootstrap": (
        bootstrap_results
    ),
    "scientific_gates": (
        scientific_gates
    ),
    "scientific_statuses": (
        scientific_statuses
    ),
    "diagnostic_codes": (
        diagnostic_codes
    ),
}


with open(
    CFG.output_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_builtin(telemetry),
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    f"\n💾 Telemetry written to: "
    f"{CFG.output_file}"
)


# =================================================================================================
# 30. CLEANUP
# =================================================================================================

try:
    del model
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("=" * 120)

print(
    "✅ M20.6.3.1 execution completed."
)

print("=" * 120)


🔬 MILESTONE 20.6.3.1 — BENCHMARK REPAIR AND DISENTANGLED TEMPORAL VALIDATION
Device                       : CUDA
Model                        : Qwen/Qwen2.5-0.5B
History length               : 8
Forecast horizon             : 6
Latent threshold             : 0.72
Train streams                : 36
Validation streams           : 12
Test streams per regime      : 12
Global seed                  : 206311
✅ Independent groups generated.

⏳ Loading TransformerLens model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Model loaded. Valid layers: [0, 3, 6, 9, 13, 16, 19, 23]

🔬 Extracting fixed-format feedback activations...
  Extracting TRAIN_MIXED_KNOWN...


  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

...

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]


🧪 TEST A — INCREMENTAL POST-FEEDBACK OBSERVABILITY
RAW ONLY   | rho= 0.9342 | R²= 0.8946 | MAE= 0.0578
Layer 0   | Act rho= 0.7462 | Act R²= 0.5283 | Combined R²= 0.8845 | ΔR²=-0.0101
Layer 3   | Act rho= 0.8409 | Act R²= 0.6641 | Combined R²= 0.8854 | ΔR²=-0.0092
Layer 6   | Act rho= 0.8265 | Act R²= 0.6223 | Combined R²= 0.8857 | ΔR²=-0.0089
Layer 9   | Act rho= 0.8272 | Act R²= 0.6245 | Combined R²= 0.8836 | ΔR²=-0.0109
Layer 13  | Act rho= 0.8458 | Act R²= 0.6438 | Combined R²= 0.8842 | ΔR²=-0.0103
Layer 16  | Act rho= 0.8559 | Act R²= 0.6784 | Combined R²= 0.8881 | ΔR²=-0.0065
Layer 19  | Act rho= 0.8552 | Act R²= 0.6753 | Combined R²= 0.8876 | ΔR²=-0.0070
Layer 23  | Act rho= 0.8737 | Act R²= 0.7193 | Combined R²= 0.8825 | ΔR²=-0.0120

🏆 Selected feedback layer: 16

Temporal shapes:
  Train raw        : (3276, 8, 6)
  Train activation : (3276, 8, 12)
  Validation raw   : (1092, 8, 6)

🧪 TEST B — DISENTANGLED MODEL SELECTION

Point models:
  RAW_LATEST           [CAUSAL    ] | rh

In [ ]:
# @title Default title text
# =================================================================================================
# MILESTONE 20.6.3.2
# STABLE-MECHANISM, STRUCTURAL-ORACLE, EVENT-BALANCED TEMPORAL BENCHMARK
# =================================================================================================

from __future__ import annotations

import gc
import json
import math
import random
import warnings

from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch

from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    IsolationForest,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from transformer_lens import HookedTransformer


warnings.filterwarnings("ignore")


# =================================================================================================
# 1. CONFIGURATION
# =================================================================================================

@dataclass
class Config:
    milestone: str = "M20.6.3.2"

    title: str = (
        "Stable-Mechanism, Structural-Oracle, "
        "Event-Balanced Temporal Benchmark"
    )

    model_name: str = "Qwen/Qwen2.5-0.5B"

    seed: int = 206322

    history_length: int = 8

    forecast_horizons: Tuple[int, ...] = (
        1,
        2,
        3,
        6,
    )

    primary_horizon: int = 3

    stream_length: int = 112

    burn_in: int = 48

    train_streams: int = 40

    calibration_selection_streams: int = 14

    calibration_probability_streams: int = 14

    calibration_policy_streams: int = 14

    validation_streams: int = 18

    test_streams_per_regime: int = 16

    latent_event_threshold: float = 0.72

    observation_noise: float = 0.025

    process_noise: float = 0.018

    activation_batch_size: int = 32

    activation_pca_dimension: int = 12

    oracle_rollouts_preflight: int = 192

    oracle_rollouts_evaluation: int = 256

    order_shuffle_trials: int = 100

    bootstrap_iterations: int = 300

    maximum_policy_fpr: float = 0.08

    minimum_event_recall: float = 0.50

    maximum_false_alert_episodes_per_100: float = 8.0

    minimum_preflight_point_rho: float = 0.40

    minimum_preflight_point_r2: float = 0.10

    minimum_preflight_hazard_auc: float = 0.70

    minimum_point_rho: float = 0.20

    minimum_point_macro_rho: float = 0.12

    minimum_point_r2: float = 0.02

    minimum_hazard_auc: float = 0.65

    minimum_hazard_ap_gain: float = 0.03

    minimum_order_advantage: float = 0.02

    minimum_observability_rho: float = 0.55

    minimum_observability_r2: float = 0.25

    minimum_novelty_auc: float = 0.75

    minimum_novelty_tpr: float = 0.55

    maximum_novelty_fpr: float = 0.15

    output_file: str = (
        "m20_6_3_2_stable_mechanism_structural_oracle.json"
    )


CFG = Config()


KNOWN_FAMILIES = (
    "AR2",
    "DELAYED",
    "OSCILLATORY",
    "MEAN_REVERTING",
)

UNKNOWN_FAMILIES = (
    "ABRUPT_SWITCH",
    "NONLINEAR_NOVEL",
)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

TORCH_DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)


# =================================================================================================
# 2. REPRODUCIBILITY AND UTILITIES
# =================================================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            False
        )
    except Exception:
        pass


seed_everything(CFG.seed)


def print_header(title: str) -> None:
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


def to_builtin(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    if isinstance(value, bool):
        return bool(value)

    if isinstance(value, float):
        if not np.isfinite(value):
            return None

        return float(value)

    return value


def safe_spearman(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
) -> float:
    y_true = np.asarray(y_true)
    y_prediction = np.asarray(y_prediction)

    if (
        len(y_true) < 3
        or np.std(y_true) < 1e-12
        or np.std(y_prediction) < 1e-12
    ):
        return 0.0

    value = spearmanr(
        y_true,
        y_prediction,
    ).statistic

    if not np.isfinite(value):
        return 0.0

    return float(value)


def safe_auroc(
    labels: np.ndarray,
    scores: np.ndarray,
) -> Optional[float]:
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    if len(np.unique(labels)) < 2:
        return None

    return float(
        roc_auc_score(
            labels,
            scores,
        )
    )


def safe_auprc(
    labels: np.ndarray,
    scores: np.ndarray,
) -> Optional[float]:
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    if len(np.unique(labels)) < 2:
        return None

    return float(
        average_precision_score(
            labels,
            scores,
        )
    )


def macro_stream_rho(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
    stream_ids: np.ndarray,
) -> float:
    values = []

    for stream_id in np.unique(
        stream_ids
    ):
        mask = (
            stream_ids
            == stream_id
        )

        if np.sum(mask) < 4:
            continue

        values.append(
            safe_spearman(
                y_true[mask],
                y_prediction[mask],
            )
        )

    if not values:
        return 0.0

    return float(
        np.mean(values)
    )


def evaluate_point(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
    stream_ids: np.ndarray,
) -> Dict:
    return {
        "rho": safe_spearman(
            y_true,
            y_prediction,
        ),
        "macro_stream_rho": (
            macro_stream_rho(
                y_true,
                y_prediction,
                stream_ids,
            )
        ),
        "r2": float(
            r2_score(
                y_true,
                y_prediction,
            )
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_prediction,
            )
        ),
    }


def evaluate_hazard(
    labels: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> Dict:
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    prediction = (
        probability
        >= threshold
    ).astype(np.int64)

    tp = int(
        np.sum(
            (prediction == 1)
            & (labels == 1)
        )
    )

    fp = int(
        np.sum(
            (prediction == 1)
            & (labels == 0)
        )
    )

    tn = int(
        np.sum(
            (prediction == 0)
            & (labels == 0)
        )
    )

    fn = int(
        np.sum(
            (prediction == 0)
            & (labels == 1)
        )
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    precision = (
        tp / max(tp + fp, 1)
    )

    fpr = (
        fp / max(fp + tn, 1)
    )

    return {
        "auroc": safe_auroc(
            labels,
            probability,
        ),
        "auprc": safe_auprc(
            labels,
            probability,
        ),
        "prevalence": float(
            np.mean(labels)
        ),
        "recall": float(recall),
        "precision": float(precision),
        "fpr": float(fpr),
        "brier": float(
            brier_score_loss(
                labels,
                probability,
            )
        ),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


def sigmoid(
    value: np.ndarray,
) -> np.ndarray:
    value = np.clip(
        value,
        -30.0,
        30.0,
    )

    return (
        1.0
        / (
            1.0
            + np.exp(-value)
        )
    )


# =================================================================================================
# 3. STABLE STREAM-LEVEL MECHANISMS
# =================================================================================================

def sample_stream_parameters(
    family: str,
    rng: np.random.Generator,
) -> Dict[str, float]:
    common = {
        "forcing_amplitude": float(
            rng.uniform(
                0.055,
                0.105,
            )
        ),
        "forcing_frequency": float(
            rng.uniform(
                0.105,
                0.205,
            )
        ),
        "forcing_phase": float(
            rng.uniform(
                -np.pi,
                np.pi,
            )
        ),
        "bias": float(
            rng.uniform(
                -0.012,
                0.012,
            )
        ),
        "scale": 1.0,
    }

    if family == "AR2":
        common.update({
            "a1": float(
                rng.uniform(
                    0.78,
                    0.94,
                )
            ),
            "a2": float(
                rng.uniform(
                    -0.24,
                    -0.08,
                )
            ),
        })

    elif family == "DELAYED":
        common.update({
            "a1": float(
                rng.uniform(
                    0.52,
                    0.68,
                )
            ),
            "delay_weight": float(
                rng.uniform(
                    0.20,
                    0.34,
                )
            ),
            "delay": int(
                rng.choice([2, 3])
            ),
        })

    elif family == "OSCILLATORY":
        common.update({
            "a1": float(
                rng.uniform(
                    1.20,
                    1.38,
                )
            ),
            "a2": float(
                rng.uniform(
                    -0.82,
                    -0.68,
                )
            ),
        })

    elif family == "MEAN_REVERTING":
        common.update({
            "kappa": float(
                rng.uniform(
                    0.18,
                    0.34,
                )
            ),
            "mean": float(
                rng.uniform(
                    -0.10,
                    0.10,
                )
            ),
            "momentum": float(
                rng.uniform(
                    0.04,
                    0.14,
                )
            ),
        })

    elif family == "ABRUPT_SWITCH":
        common.update({
            "pre_a1": float(
                rng.uniform(
                    0.78,
                    0.92,
                )
            ),
            "pre_a2": float(
                rng.uniform(
                    -0.22,
                    -0.10,
                )
            ),
            "post_a1": float(
                rng.uniform(
                    1.10,
                    1.28,
                )
            ),
            "post_a2": float(
                rng.uniform(
                    -0.76,
                    -0.58,
                )
            ),
            "switch_time": int(
                rng.integers(
                    CFG.burn_in + 30,
                    CFG.burn_in + 65,
                )
            ),
        })

    elif family == "NONLINEAR_NOVEL":
        common.update({
            "linear": float(
                rng.uniform(
                    0.52,
                    0.68,
                )
            ),
            "nonlinear": float(
                rng.uniform(
                    0.24,
                    0.38,
                )
            ),
            "delay_weight": float(
                rng.uniform(
                    -0.18,
                    -0.08,
                )
            ),
        })

    else:
        raise ValueError(
            f"Unknown family: {family}"
        )

    return common


def forcing_term(
    time_index: int,
    parameters: Dict[str, float],
) -> float:
    return float(
        parameters["forcing_amplitude"]
        * np.sin(
            parameters["forcing_frequency"]
            * time_index
            + parameters["forcing_phase"]
        )
        + parameters["bias"]
    )


def base_dynamic_mean(
    family: str,
    history: np.ndarray,
    time_index: int,
    parameters: Dict[str, float],
) -> np.ndarray:
    history = np.asarray(
        history,
        dtype=np.float64,
    )

    current = history[..., -1]

    previous = (
        history[..., -2]
        if history.shape[-1] >= 2
        else current
    )

    drive = forcing_term(
        time_index,
        parameters,
    )

    if family == "AR2":
        value = (
            parameters["a1"]
            * current
            + parameters["a2"]
            * previous
            + drive
        )

    elif family == "DELAYED":
        delay = int(
            parameters["delay"]
        )

        delayed = (
            history[..., -1 - delay]
            if history.shape[-1] > delay
            else history[..., 0]
        )

        value = (
            parameters["a1"]
            * current
            + parameters["delay_weight"]
            * delayed
            + drive
        )

    elif family == "OSCILLATORY":
        value = (
            parameters["a1"]
            * current
            + parameters["a2"]
            * previous
            + drive
        )

    elif family == "MEAN_REVERTING":
        value = (
            current
            + parameters["kappa"]
            * (
                parameters["mean"]
                - current
            )
            + parameters["momentum"]
            * (
                current
                - previous
            )
            + drive
        )

    elif family == "ABRUPT_SWITCH":
        if (
            time_index
            < parameters["switch_time"]
        ):
            a1 = parameters["pre_a1"]
            a2 = parameters["pre_a2"]
        else:
            a1 = parameters["post_a1"]
            a2 = parameters["post_a2"]

        value = (
            a1 * current
            + a2 * previous
            + drive
        )

    elif family == "NONLINEAR_NOVEL":
        delayed = (
            history[..., -3]
            if history.shape[-1] >= 3
            else history[..., 0]
        )

        value = (
            parameters["linear"]
            * current
            + parameters["nonlinear"]
            * np.tanh(
                2.4 * current
            )
            + parameters["delay_weight"]
            * delayed
            + drive
        )

    else:
        raise ValueError(family)

    return np.clip(
        value,
        -2.5,
        2.5,
    )


def scaled_dynamic_step(
    family: str,
    scaled_history: np.ndarray,
    time_index: int,
    parameters: Dict[str, float],
    rng: np.random.Generator,
    process_noise: float,
) -> np.ndarray:
    scale = float(
        parameters["scale"]
    )

    base_history = (
        np.asarray(
            scaled_history,
            dtype=np.float64,
        )
        / max(scale, 1e-8)
    )

    base_mean = base_dynamic_mean(
        family,
        base_history,
        time_index,
        parameters,
    )

    noise = rng.normal(
        0.0,
        process_noise,
        size=np.shape(base_mean),
    )

    return (
        scale
        * (
            base_mean
            + noise
        )
    )


# =================================================================================================
# 4. STREAM GENERATION
# =================================================================================================

@dataclass
class StreamRecord:
    stream_id: int

    group_name: str

    family: str

    novel: bool

    parameters: Dict[str, float]

    latent: np.ndarray

    raw: np.ndarray

    activation: Optional[np.ndarray] = None


def generate_base_latent(
    family: str,
    parameters: Dict[str, float],
    total_length: int,
    rng: np.random.Generator,
) -> np.ndarray:
    latent = np.zeros(
        total_length,
        dtype=np.float64,
    )

    latent[:4] = rng.normal(
        0.0,
        0.20,
        size=4,
    )

    for time_index in range(
        4,
        total_length,
    ):
        mean_value = base_dynamic_mean(
            family,
            latent[
                max(
                    0,
                    time_index - 8,
                ):
                time_index
            ],
            time_index,
            parameters,
        )

        latent[time_index] = (
            mean_value
            + rng.normal(
                0.0,
                CFG.process_noise,
            )
        )

    return latent


def amplitude_match_latent(
    latent: np.ndarray,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, float]:
    active = latent[
        CFG.burn_in:
    ]

    reference = float(
        np.quantile(
            np.abs(active),
            0.88,
        )
    )

    target = float(
        rng.uniform(
            0.76,
            0.90,
        )
    )

    scale = (
        target
        / max(reference, 0.08)
    )

    scale = float(
        np.clip(
            scale,
            0.35,
            5.0,
        )
    )

    return (
        latent * scale,
        scale,
    )


def generate_raw_feedback(
    latent: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    total_length = len(latent)

    raw = np.zeros(
        (
            total_length,
            6,
        ),
        dtype=np.float32,
    )

    previous_prediction = 0.0

    for time_index in range(
        total_length
    ):
        previous_latent = (
            latent[time_index - 1]
            if time_index > 0
            else 0.0
        )

        trend = (
            latent[time_index - 1]
            - latent[time_index - 2]
            if time_index > 1
            else 0.0
        )

        prediction = (
            0.68 * previous_latent
            + 0.12 * trend
            + rng.normal(
                0.0,
                CFG.observation_noise,
            )
        )

        sensitivity = float(
            rng.uniform(
                0.74,
                0.90,
            )
        )

        signed_error = (
            sensitivity
            * latent[time_index]
            + rng.normal(
                0.0,
                CFG.observation_noise,
            )
        )

        outcome = (
            prediction
            + signed_error
        )

        confidence = float(
            np.clip(
                0.90
                - 0.34
                * abs(signed_error)
                + rng.normal(
                    0.0,
                    0.025,
                ),
                0.02,
                0.98,
            )
        )

        prediction_change = (
            prediction
            - previous_prediction
        )

        raw[time_index] = np.asarray([
            prediction,
            outcome,
            confidence,
            prediction_change,
            signed_error,
            abs(signed_error),
        ], dtype=np.float32)

        previous_prediction = prediction

    return raw


def generate_group(
    group_name: str,
    number_of_streams: int,
    families: Sequence[str],
    seed: int,
) -> List[StreamRecord]:
    rng = np.random.default_rng(seed)

    total_length = (
        CFG.burn_in
        + CFG.stream_length
        + max(
            CFG.forecast_horizons
        )
        + 2
    )

    streams = []

    for stream_index in range(
        number_of_streams
    ):
        family = families[
            stream_index
            % len(families)
        ]

        parameters = (
            sample_stream_parameters(
                family,
                rng,
            )
        )

        base_latent = (
            generate_base_latent(
                family,
                parameters,
                total_length,
                rng,
            )
        )

        latent, scale = (
            amplitude_match_latent(
                base_latent,
                rng,
            )
        )

        parameters["scale"] = scale

        raw = generate_raw_feedback(
            latent,
            rng,
        )

        streams.append(
            StreamRecord(
                stream_id=stream_index,
                group_name=group_name,
                family=family,
                novel=(
                    family
                    in UNKNOWN_FAMILIES
                ),
                parameters=parameters,
                latent=latent.astype(
                    np.float32
                ),
                raw=raw,
            )
        )

    return streams


# =================================================================================================
# 5. INDEPENDENT GROUPS
# =================================================================================================

GROUPS: Dict[str, List[StreamRecord]] = {
    "TRAIN_MIXED_KNOWN": generate_group(
        "TRAIN_MIXED_KNOWN",
        CFG.train_streams,
        KNOWN_FAMILIES,
        CFG.seed + 1_000,
    ),
    "CALIBRATION_SELECTION": generate_group(
        "CALIBRATION_SELECTION",
        CFG.calibration_selection_streams,
        KNOWN_FAMILIES,
        CFG.seed + 2_000,
    ),
    "CALIBRATION_PROBABILITY": generate_group(
        "CALIBRATION_PROBABILITY",
        CFG.calibration_probability_streams,
        KNOWN_FAMILIES,
        CFG.seed + 3_000,
    ),
    "CALIBRATION_POLICY": generate_group(
        "CALIBRATION_POLICY",
        CFG.calibration_policy_streams,
        KNOWN_FAMILIES,
        CFG.seed + 4_000,
    ),
    "VALIDATION": generate_group(
        "VALIDATION",
        CFG.validation_streams,
        KNOWN_FAMILIES,
        CFG.seed + 5_000,
    ),
    "AR2_TRANSFER": generate_group(
        "AR2_TRANSFER",
        CFG.test_streams_per_regime,
        ("AR2",),
        CFG.seed + 6_000,
    ),
    "DELAYED_TRANSFER": generate_group(
        "DELAYED_TRANSFER",
        CFG.test_streams_per_regime,
        ("DELAYED",),
        CFG.seed + 7_000,
    ),
    "OSCILLATORY_TRANSFER": generate_group(
        "OSCILLATORY_TRANSFER",
        CFG.test_streams_per_regime,
        ("OSCILLATORY",),
        CFG.seed + 8_000,
    ),
    "MEAN_REVERTING_TRANSFER": generate_group(
        "MEAN_REVERTING_TRANSFER",
        CFG.test_streams_per_regime,
        ("MEAN_REVERTING",),
        CFG.seed + 9_000,
    ),
    "ABRUPT_UNKNOWN": generate_group(
        "ABRUPT_UNKNOWN",
        CFG.test_streams_per_regime,
        ("ABRUPT_SWITCH",),
        CFG.seed + 10_000,
    ),
    "MECHANISM_NOVEL": generate_group(
        "MECHANISM_NOVEL",
        CFG.test_streams_per_regime,
        ("NONLINEAR_NOVEL",),
        CFG.seed + 11_000,
    ),
}


# =================================================================================================
# 6. TEMPORAL DATASETS
# =================================================================================================

def build_temporal_dataset(
    streams: Sequence[StreamRecord],
    horizon: int,
) -> Dict:
    raw_windows = []
    activation_windows = []

    y_point = []
    y_hazard = []
    hazard_eligible = []

    stream_ids = []
    family_names = []
    novelty_labels = []

    row_times = []
    stream_references = []

    current_latent = []
    current_proxy = []

    threshold = (
        CFG.latent_event_threshold
    )

    for global_stream_id, stream in enumerate(
        streams
    ):
        start = (
            CFG.burn_in
            + CFG.history_length
            - 1
        )

        stop = (
            CFG.burn_in
            + CFG.stream_length
        )

        for time_index in range(
            start,
            stop
        ):
            if (
                time_index + horizon
                >= len(stream.latent)
            ):
                continue

            history_slice = slice(
                time_index
                - CFG.history_length
                + 1,
                time_index + 1,
            )

            future = stream.latent[
                time_index + 1:
                time_index + horizon + 1
            ]

            raw_windows.append(
                stream.raw[
                    history_slice
                ]
            )

            if (
                stream.activation
                is not None
            ):
                activation_windows.append(
                    stream.activation[
                        history_slice
                    ]
                )

            current = float(
                stream.latent[
                    time_index
                ]
            )

            eligible = bool(
                abs(current)
                < threshold
            )

            event = bool(
                eligible
                and np.any(
                    np.abs(future)
                    >= threshold
                )
            )

            y_point.append(
                float(
                    stream.latent[
                        time_index + horizon
                    ]
                )
            )

            y_hazard.append(
                int(event)
            )

            hazard_eligible.append(
                eligible
            )

            stream_ids.append(
                global_stream_id
            )

            family_names.append(
                stream.family
            )

            novelty_labels.append(
                int(stream.novel)
            )

            row_times.append(
                time_index
            )

            stream_references.append(
                stream
            )

            current_latent.append(
                current
            )

            current_proxy.append(
                float(
                    stream.raw[
                        time_index,
                        4,
                    ]
                )
            )

    dataset = {
        "X_raw": np.asarray(
            raw_windows,
            dtype=np.float32,
        ),
        "y_point": np.asarray(
            y_point,
            dtype=np.float32,
        ),
        "y_hazard": np.asarray(
            y_hazard,
            dtype=np.int64,
        ),
        "hazard_eligible": np.asarray(
            hazard_eligible,
            dtype=bool,
        ),
        "stream_ids": np.asarray(
            stream_ids,
            dtype=np.int64,
        ),
        "family_names": np.asarray(
            family_names,
            dtype=object,
        ),
        "novelty_labels": np.asarray(
            novelty_labels,
            dtype=np.int64,
        ),
        "row_times": np.asarray(
            row_times,
            dtype=np.int64,
        ),
        "stream_references": (
            stream_references
        ),
        "current_latent": np.asarray(
            current_latent,
            dtype=np.float32,
        ),
        "current_proxy": np.asarray(
            current_proxy,
            dtype=np.float32,
        ),
        "horizon": int(horizon),
    }

    if activation_windows:
        dataset["X_activation"] = (
            np.asarray(
                activation_windows,
                dtype=np.float32,
            )
        )

    return dataset


RAW_DATASETS_BY_HORIZON = {
    horizon: {
        name: build_temporal_dataset(
            streams,
            horizon,
        )
        for name, streams in GROUPS.items()
    }
    for horizon in CFG.forecast_horizons
}


# =================================================================================================
# 7. STRUCTURAL MONTE CARLO ORACLE
# =================================================================================================

def structural_oracle(
    dataset: Dict,
    rollouts: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)

    number_of_rows = len(
        dataset["y_point"]
    )

    horizon = int(
        dataset["horizon"]
    )

    point_prediction = np.zeros(
        number_of_rows,
        dtype=np.float64,
    )

    hazard_probability = np.zeros(
        number_of_rows,
        dtype=np.float64,
    )

    threshold = (
        CFG.latent_event_threshold
    )

    for row_index in range(
        number_of_rows
    ):
        stream = dataset[
            "stream_references"
        ][row_index]

        time_index = int(
            dataset["row_times"][
                row_index
            ]
        )

        history = stream.latent[
            time_index
            - CFG.history_length
            + 1:
            time_index + 1
        ].astype(np.float64)

        rollout_history = np.repeat(
            history[None, :],
            rollouts,
            axis=0,
        )

        crossed = np.zeros(
            rollouts,
            dtype=bool,
        )

        for lead in range(
            1,
            horizon + 1,
        ):
            next_value = (
                scaled_dynamic_step(
                    stream.family,
                    rollout_history,
                    time_index + lead,
                    stream.parameters,
                    rng,
                    CFG.process_noise,
                )
            )

            crossed |= (
                np.abs(next_value)
                >= threshold
            )

            rollout_history = (
                np.concatenate([
                    rollout_history[:, 1:],
                    next_value[:, None],
                ], axis=1)
            )

        point_prediction[
            row_index
        ] = float(
            np.mean(
                rollout_history[:, -1]
            )
        )

        if dataset[
            "hazard_eligible"
        ][row_index]:
            hazard_probability[
                row_index
            ] = float(
                np.mean(crossed)
            )
        else:
            hazard_probability[
                row_index
            ] = 0.0

    return (
        point_prediction.astype(
            np.float32
        ),
        hazard_probability.astype(
            np.float32
        ),
    )


# =================================================================================================
# 8. RAW-ONLY PREFLIGHT
# =================================================================================================

print_header(
    "🔬 MILESTONE 20.6.3.2 — "
    "STABLE-MECHANISM STRUCTURAL-ORACLE PREFLIGHT"
)

print(f"Device                 : {DEVICE.upper()}")
print(f"Model                  : {CFG.model_name}")
print(f"History length         : {CFG.history_length}")
print(f"Forecast horizons      : {CFG.forecast_horizons}")
print(f"Primary horizon        : {CFG.primary_horizon}")
print(f"Latent threshold       : {CFG.latent_event_threshold}")
print(f"Global seed            : {CFG.seed}")


preflight_results = {}

print_header(
    "🧪 PREFLIGHT A — STRUCTURAL FORECASTABILITY"
)

for horizon in CFG.forecast_horizons:
    dataset = RAW_DATASETS_BY_HORIZON[
        horizon
    ]["VALIDATION"]

    point_oracle_prediction, hazard_oracle_probability = (
        structural_oracle(
            dataset,
            CFG.oracle_rollouts_preflight,
            CFG.seed + 20_000 + horizon,
        )
    )

    point_metrics = evaluate_point(
        dataset["y_point"],
        point_oracle_prediction,
        dataset["stream_ids"],
    )

    mask = dataset[
        "hazard_eligible"
    ]

    hazard_metrics = evaluate_hazard(
        dataset["y_hazard"][mask],
        hazard_oracle_probability[mask],
        threshold=0.5,
    )

    hazard_auc = (
        hazard_metrics["auroc"]
        if hazard_metrics["auroc"]
        is not None
        else 0.0
    )

    passed = bool(
        point_metrics["rho"]
        >= CFG.minimum_preflight_point_rho
        and point_metrics["r2"]
        >= CFG.minimum_preflight_point_r2
        and hazard_auc
        >= CFG.minimum_preflight_hazard_auc
    )

    preflight_results[horizon] = {
        "point": point_metrics,
        "hazard": hazard_metrics,
        "pass": passed,
    }

    print(
        f"H={horizon:<2} | "
        f"Point rho={point_metrics['rho']:.4f} | "
        f"Macro={point_metrics['macro_stream_rho']:.4f} | "
        f"R²={point_metrics['r2']:.4f} | "
        f"Hazard AUC={hazard_auc:.4f} | "
        f"AP={(hazard_metrics['auprc'] or 0.0):.4f} | "
        f"Prev={hazard_metrics['prevalence']:.4f} | "
        f"{'PASS' if passed else 'FAIL'}"
    )


primary_preflight_passed = bool(
    preflight_results[
        CFG.primary_horizon
    ]["pass"]
)

if not primary_preflight_passed:
    failure_telemetry = {
        "milestone": CFG.milestone,
        "title": CFG.title,
        "configuration": asdict(CFG),
        "status": (
            "PREFLIGHT_FAILED_BEFORE_ACTIVATION_EXTRACTION"
        ),
        "preflight": preflight_results,
    }

    with open(
        CFG.output_file,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            to_builtin(
                failure_telemetry
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    raise RuntimeError(
        "Primary structural-oracle preflight failed. "
        "Activation extraction was intentionally skipped."
    )


print(
    "\n✅ Primary benchmark is structurally "
    "forecastable. Activation extraction may proceed."
)


# =================================================================================================
# 9. ACTIVATION EXTRACTION
# =================================================================================================

def feedback_prompt(
    raw_row: np.ndarray,
) -> str:
    return (
        "Feedback record\n"
        f"prediction={raw_row[0]:+08.4f}\n"
        f"outcome={raw_row[1]:+08.4f}\n"
        f"confidence={raw_row[2]:+08.4f}\n"
        f"prediction_change={raw_row[3]:+08.4f}\n"
        f"signed_error={raw_row[4]:+08.4f}\n"
        f"absolute_error={raw_row[5]:+08.4f}\n"
        "Assess the internal feedback state:"
    )


print_header(
    "⏳ LOADING TRANSFORMERLENS MODEL"
)

model = HookedTransformer.from_pretrained(
    CFG.model_name,
    device=DEVICE,
    dtype=TORCH_DTYPE,
)

model.eval()

number_of_layers = int(
    model.cfg.n_layers
)

valid_layers = sorted(
    set(
        np.linspace(
            0,
            number_of_layers - 1,
            8,
            dtype=int,
        ).tolist()
    )
)

try:
    model.tokenizer.padding_side = "left"

    if (
        model.tokenizer.pad_token_id
        is None
    ):
        model.tokenizer.pad_token = (
            model.tokenizer.eos_token
        )
except Exception:
    pass


def extract_stream_activations(
    streams: Sequence[StreamRecord],
    layers: Sequence[int],
) -> Dict[int, List[np.ndarray]]:
    layer_outputs = {
        layer: []
        for layer in layers
    }

    for stream in streams:
        prompts = [
            feedback_prompt(row)
            for row in stream.raw
        ]

        collected = {
            layer: []
            for layer in layers
        }

        for start in range(
            0,
            len(prompts),
            CFG.activation_batch_size,
        ):
            batch_prompts = prompts[
                start:
                start
                + CFG.activation_batch_size
            ]

            tokens = model.to_tokens(
                batch_prompts,
                prepend_bos=True,
            )

            hook_names = {
                f"blocks.{layer}.hook_resid_post"
                for layer in layers
            }

            with torch.inference_mode():
                _, cache = model.run_with_cache(
                    tokens,
                    names_filter=lambda name: (
                        name in hook_names
                    ),
                )

            for layer in layers:
                value = cache[
                    f"blocks.{layer}.hook_resid_post"
                ][:, -1, :]

                collected[layer].append(
                    value.detach()
                    .float()
                    .cpu()
                    .numpy()
                )

            del cache
            del tokens

        for layer in layers:
            layer_outputs[layer].append(
                np.concatenate(
                    collected[layer],
                    axis=0,
                ).astype(np.float32)
            )

    return layer_outputs


print(
    f"✅ Model loaded. "
    f"Valid layers: {valid_layers}"
)

print(
    "\n🔬 Extracting train/selection "
    "activations for layer selection..."
)

train_layer_activations = (
    extract_stream_activations(
        GROUPS["TRAIN_MIXED_KNOWN"],
        valid_layers,
    )
)

selection_layer_activations = (
    extract_stream_activations(
        GROUPS[
            "CALIBRATION_SELECTION"
        ],
        valid_layers,
    )
)


# =================================================================================================
# 10. LAYER SELECTION USING ACTIVATION-ONLY OBSERVABILITY
# =================================================================================================

def activation_rows_and_targets(
    streams: Sequence[StreamRecord],
    activation_list: Sequence[np.ndarray],
) -> Tuple[np.ndarray, np.ndarray]:
    rows = []
    targets = []

    for stream, activation in zip(
        streams,
        activation_list,
    ):
        start = CFG.burn_in

        stop = (
            CFG.burn_in
            + CFG.stream_length
        )

        rows.append(
            activation[
                start:stop
            ]
        )

        targets.append(
            stream.latent[
                start:stop
            ]
        )

    return (
        np.concatenate(
            rows,
            axis=0,
        ),
        np.concatenate(
            targets,
            axis=0,
        ),
    )


layer_observability_results = {}
layer_transformers = {}

print_header(
    "🧪 TEST A — ACTIVATION-ONLY FEEDBACK OBSERVABILITY"
)

for layer in valid_layers:
    train_x, train_y = (
        activation_rows_and_targets(
            GROUPS[
                "TRAIN_MIXED_KNOWN"
            ],
            train_layer_activations[
                layer
            ],
        )
    )

    selection_x, selection_y = (
        activation_rows_and_targets(
            GROUPS[
                "CALIBRATION_SELECTION"
            ],
            selection_layer_activations[
                layer
            ],
        )
    )

    pca_dimension = min(
        CFG.activation_pca_dimension,
        train_x.shape[1],
        train_x.shape[0] - 1,
    )

    pca = PCA(
        n_components=pca_dimension,
        random_state=CFG.seed,
    )

    train_reduced = (
        pca.fit_transform(
            train_x
        )
    )

    selection_reduced = (
        pca.transform(
            selection_x
        )
    )

    decoder = Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(alpha=2.0),
        ),
    ])

    decoder.fit(
        train_reduced,
        train_y,
    )

    prediction = decoder.predict(
        selection_reduced
    )

    metrics = {
        "rho": safe_spearman(
            selection_y,
            prediction,
        ),
        "r2": float(
            r2_score(
                selection_y,
                prediction,
            )
        ),
        "mae": float(
            mean_absolute_error(
                selection_y,
                prediction,
            )
        ),
    }

    layer_observability_results[
        layer
    ] = metrics

    layer_transformers[
        layer
    ] = {
        "pca": pca,
        "decoder": decoder,
    }

    print(
        f"Layer {layer:<2} | "
        f"rho={metrics['rho']:.4f} | "
        f"R²={metrics['r2']:.4f} | "
        f"MAE={metrics['mae']:.4f}"
    )


BEST_FEEDBACK_LAYER = max(
    valid_layers,
    key=lambda layer: (
        layer_observability_results[
            layer
        ]["r2"]
    ),
)

activation_pca = layer_transformers[
    BEST_FEEDBACK_LAYER
]["pca"]

print(
    f"\n🏆 Selected activation-only "
    f"feedback layer: {BEST_FEEDBACK_LAYER}"
)


# =================================================================================================
# 11. EXTRACT SELECTED LAYER FOR REMAINING GROUPS
# =================================================================================================

for group_name, streams in GROUPS.items():
    if group_name == "TRAIN_MIXED_KNOWN":
        raw_activation_list = (
            train_layer_activations[
                BEST_FEEDBACK_LAYER
            ]
        )

    elif group_name == "CALIBRATION_SELECTION":
        raw_activation_list = (
            selection_layer_activations[
                BEST_FEEDBACK_LAYER
            ]
        )

    else:
        print(
            f"  Extracting {group_name}..."
        )

        raw_activation_list = (
            extract_stream_activations(
                streams,
                (BEST_FEEDBACK_LAYER,),
            )[BEST_FEEDBACK_LAYER]
        )

    for stream, raw_activation in zip(
        streams,
        raw_activation_list,
    ):
        stream.activation = (
            activation_pca.transform(
                raw_activation
            ).astype(np.float32)
        )


del train_layer_activations
del selection_layer_activations

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# =================================================================================================
# 12. REBUILD DATASETS WITH ACTIVATIONS
# =================================================================================================

DATASETS_BY_HORIZON = {
    horizon: {
        name: build_temporal_dataset(
            streams,
            horizon,
        )
        for name, streams in GROUPS.items()
    }
    for horizon in CFG.forecast_horizons
}

DATASETS = DATASETS_BY_HORIZON[
    CFG.primary_horizon
]

train_dataset = DATASETS[
    "TRAIN_MIXED_KNOWN"
]

selection_dataset = DATASETS[
    "CALIBRATION_SELECTION"
]

probability_dataset = DATASETS[
    "CALIBRATION_PROBABILITY"
]

policy_dataset = DATASETS[
    "CALIBRATION_POLICY"
]

validation_dataset = DATASETS[
    "VALIDATION"
]


# =================================================================================================
# 13. FEATURE VIEWS
# =================================================================================================

def ordered_raw_features(
    dataset: Dict,
) -> np.ndarray:
    return dataset[
        "X_raw"
    ].reshape(
        len(
            dataset["X_raw"]
        ),
        -1,
    )


def latest_raw_features(
    dataset: Dict,
) -> np.ndarray:
    return dataset[
        "X_raw"
    ][:, -1, :]


def bag_raw_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]

    return np.concatenate([
        np.mean(
            raw,
            axis=1,
        ),
        np.std(
            raw,
            axis=1,
        ),
        raw[:, -1, :],
    ], axis=1)


def slope_raw_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]

    time_axis = np.arange(
        raw.shape[1],
        dtype=np.float32,
    )

    centered_time = (
        time_axis
        - np.mean(time_axis)
    )

    denominator = float(
        np.sum(
            centered_time ** 2
        )
    )

    slope = np.sum(
        raw
        * centered_time[
            None,
            :,
            None,
        ],
        axis=1,
    ) / max(
        denominator,
        1e-8,
    )

    return np.concatenate([
        raw[:, -1, :],
        slope,
        raw[:, -1, :]
        - raw[:, 0, :],
    ], axis=1)


def activation_ordered_features(
    dataset: Dict,
) -> np.ndarray:
    activation = dataset[
        "X_activation"
    ]

    return activation.reshape(
        len(activation),
        -1,
    )


def combined_ordered_features(
    dataset: Dict,
) -> np.ndarray:
    return np.concatenate([
        ordered_raw_features(
            dataset
        ),
        activation_ordered_features(
            dataset
        ),
    ], axis=1)


def feature_view(
    dataset: Dict,
    model_name: str,
) -> np.ndarray:
    if model_name == "RAW_LATEST":
        return latest_raw_features(
            dataset
        )

    if model_name == "RAW_BAG":
        return bag_raw_features(
            dataset
        )

    if model_name == "RAW_SLOPE":
        return slope_raw_features(
            dataset
        )

    if model_name == "RAW_ORDERED":
        return ordered_raw_features(
            dataset
        )

    if model_name == "ACTIVATION_ORDERED":
        return activation_ordered_features(
            dataset
        )

    if model_name == "COMBINED_ORDERED":
        return combined_ordered_features(
            dataset
        )

    raise ValueError(
        model_name
    )


# =================================================================================================
# 14. POINT AND HAZARD MODEL SELECTION
# =================================================================================================

MODEL_NAMES = (
    "RAW_LATEST",
    "RAW_BAG",
    "RAW_SLOPE",
    "RAW_ORDERED",
    "ACTIVATION_ORDERED",
    "COMBINED_ORDERED",
)

CAUSAL_MODEL_NAMES = (
    "RAW_LATEST",
    "RAW_BAG",
    "RAW_SLOPE",
    "RAW_ORDERED",
)


def make_point_model(
    model_name: str,
):
    if model_name in (
        "RAW_ORDERED",
        "COMBINED_ORDERED",
        "ACTIVATION_ORDERED",
    ):
        return ExtraTreesRegressor(
            n_estimators=320,
            min_samples_leaf=5,
            max_features=0.75,
            random_state=CFG.seed,
            n_jobs=-1,
        )

    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(alpha=3.0),
        ),
    ])


def make_hazard_model(
    model_name: str,
):
    if model_name in (
        "RAW_ORDERED",
        "COMBINED_ORDERED",
        "ACTIVATION_ORDERED",
    ):
        return ExtraTreesClassifier(
            n_estimators=360,
            min_samples_leaf=5,
            class_weight="balanced",
            max_features=0.75,
            random_state=CFG.seed,
            n_jobs=-1,
        )

    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "logistic",
            LogisticRegression(
                C=0.7,
                class_weight="balanced",
                max_iter=3000,
                random_state=CFG.seed,
            ),
        ),
    ])


def model_score(
    model_instance,
    features: np.ndarray,
) -> np.ndarray:
    if hasattr(
        model_instance,
        "predict_proba",
    ):
        probability = (
            model_instance.predict_proba(
                features
            )[:, 1]
        )

        probability = np.clip(
            probability,
            1e-6,
            1.0 - 1e-6,
        )

        return np.log(
            probability
            / (
                1.0
                - probability
            )
        )

    if hasattr(
        model_instance,
        "decision_function",
    ):
        return np.asarray(
            model_instance.decision_function(
                features
            )
        )

    return np.asarray(
        model_instance.predict(
            features
        )
    )


point_models = {}
hazard_models = {}

point_selection_results = {}
hazard_selection_results = {}

print_header(
    "🧪 TEST B — DISENTANGLED MODEL SELECTION"
)

print("\nPoint models:")

for model_name in MODEL_NAMES:
    point_model = make_point_model(
        model_name
    )

    point_model.fit(
        feature_view(
            train_dataset,
            model_name,
        ),
        train_dataset["y_point"],
    )

    prediction = point_model.predict(
        feature_view(
            selection_dataset,
            model_name,
        )
    )

    metrics = evaluate_point(
        selection_dataset["y_point"],
        prediction,
        selection_dataset["stream_ids"],
    )

    point_models[model_name] = (
        point_model
    )

    point_selection_results[
        model_name
    ] = metrics

    channel = (
        "CAUSAL"
        if model_name
        in CAUSAL_MODEL_NAMES
        else "DIAGNOSTIC"
    )

    print(
        f"  {model_name:<20} "
        f"[{channel:<10}] | "
        f"rho={metrics['rho']:.4f} | "
        f"Macro={metrics['macro_stream_rho']:.4f} | "
        f"R²={metrics['r2']:.4f}"
    )


train_hazard_mask = train_dataset[
    "hazard_eligible"
]

selection_hazard_mask = selection_dataset[
    "hazard_eligible"
]

print("\nHazard models:")

for model_name in MODEL_NAMES:
    hazard_model = make_hazard_model(
        model_name
    )

    hazard_model.fit(
        feature_view(
            train_dataset,
            model_name,
        )[train_hazard_mask],
        train_dataset[
            "y_hazard"
        ][train_hazard_mask],
    )

    score = model_score(
        hazard_model,
        feature_view(
            selection_dataset,
            model_name,
        )[selection_hazard_mask],
    )

    probability = sigmoid(score)

    metrics = evaluate_hazard(
        selection_dataset[
            "y_hazard"
        ][selection_hazard_mask],
        probability,
        threshold=0.5,
    )

    hazard_models[
        model_name
    ] = hazard_model

    hazard_selection_results[
        model_name
    ] = metrics

    channel = (
        "CAUSAL"
        if model_name
        in CAUSAL_MODEL_NAMES
        else "DIAGNOSTIC"
    )

    print(
        f"  {model_name:<20} "
        f"[{channel:<10}] | "
        f"AUC={(metrics['auroc'] or 0.0):.4f} | "
        f"AP={(metrics['auprc'] or 0.0):.4f}"
    )


BEST_POINT_MODEL_NAME = max(
    CAUSAL_MODEL_NAMES,
    key=lambda name: (
        point_selection_results[
            name
        ]["rho"]
        + 0.40
        * point_selection_results[
            name
        ]["macro_stream_rho"]
    ),
)

BEST_HAZARD_MODEL_NAME = max(
    CAUSAL_MODEL_NAMES,
    key=lambda name: (
        hazard_selection_results[
            name
        ]["auroc"]
        or 0.0
    ),
)

selected_point_model = (
    point_models[
        BEST_POINT_MODEL_NAME
    ]
)

selected_hazard_model = (
    hazard_models[
        BEST_HAZARD_MODEL_NAME
    ]
)

print(
    f"\n🏆 Selected causal point model : "
    f"{BEST_POINT_MODEL_NAME}"
)

print(
    f"🏆 Selected causal hazard model: "
    f"{BEST_HAZARD_MODEL_NAME}"
)


# =================================================================================================
# 15. INDEPENDENT CHANNEL CALIBRATION
# =================================================================================================

class PlattCalibrator:
    def __init__(self):
        self.model = LogisticRegression(
            C=1.0,
            max_iter=3000,
            random_state=CFG.seed,
        )

    def fit(
        self,
        score: np.ndarray,
        labels: np.ndarray,
    ) -> "PlattCalibrator":
        self.model.fit(
            np.asarray(score).reshape(
                -1,
                1,
            ),
            labels,
        )

        return self

    def predict(
        self,
        score: np.ndarray,
    ) -> np.ndarray:
        return self.model.predict_proba(
            np.asarray(score).reshape(
                -1,
                1,
            )
        )[:, 1]


probability_mask = probability_dataset[
    "hazard_eligible"
]

channel_calibrators = {}

for model_name in MODEL_NAMES:
    score = model_score(
        hazard_models[model_name],
        feature_view(
            probability_dataset,
            model_name,
        )[probability_mask],
    )

    calibrator = PlattCalibrator()

    calibrator.fit(
        score,
        probability_dataset[
            "y_hazard"
        ][probability_mask],
    )

    channel_calibrators[
        model_name
    ] = calibrator


def calibrated_hazard_probability(
    dataset: Dict,
    model_name: str,
) -> np.ndarray:
    mask = dataset[
        "hazard_eligible"
    ]

    probability = np.zeros(
        len(
            dataset["y_hazard"]
        ),
        dtype=np.float64,
    )

    score = model_score(
        hazard_models[
            model_name
        ],
        feature_view(
            dataset,
            model_name,
        )[mask],
    )

    probability[mask] = (
        channel_calibrators[
            model_name
        ].predict(score)
    )

    return probability


# =================================================================================================
# 16. EVENT-LEVEL AUDIT AND POLICY SELECTION
# =================================================================================================

def event_episode_audit(
    dataset: Dict,
    eligible_probability: np.ndarray,
    threshold: float,
) -> Dict:
    alert = (
        eligible_probability
        >= threshold
    ) & dataset[
        "hazard_eligible"
    ]

    total_events = 0
    detected_events = 0

    lead_times = []

    false_alert_samples = 0
    total_alert_samples = int(
        np.sum(alert)
    )

    false_alert_episodes = 0

    for stream_id in np.unique(
        dataset["stream_ids"]
    ):
        stream_mask = (
            dataset["stream_ids"]
            == stream_id
        )

        indices = np.flatnonzero(
            stream_mask
        )

        order = np.argsort(
            dataset[
                "row_times"
            ][indices]
        )

        indices = indices[order]

        stream_alert = alert[indices]
        stream_event = dataset[
            "y_hazard"
        ][indices].astype(bool)

        stream_times = dataset[
            "row_times"
        ][indices]

        event_indices = np.flatnonzero(
            stream_event
        )

        previous_event_time = None

        for local_event_index in event_indices:
            event_time = int(
                stream_times[
                    local_event_index
                ]
            )

            if (
                previous_event_time is not None
                and event_time
                <= previous_event_time
                + dataset["horizon"]
            ):
                continue

            total_events += 1

            window_start = max(
                0,
                local_event_index
                - dataset["horizon"]
                + 1,
            )

            candidate = np.flatnonzero(
                stream_alert[
                    window_start:
                    local_event_index + 1
                ]
            )

            if len(candidate) > 0:
                detected_events += 1

                alert_local_index = (
                    window_start
                    + candidate[0]
                )

                lead_times.append(
                    int(
                        event_time
                        - stream_times[
                            alert_local_index
                        ]
                        + 1
                    )
                )

            previous_event_time = event_time

        false_alert = (
            stream_alert
            & ~stream_event
        )

        false_alert_samples += int(
            np.sum(false_alert)
        )

        previous_false = False
        previous_time = None

        for time_value, is_false in zip(
            stream_times,
            false_alert,
        ):
            contiguous = bool(
                previous_time is not None
                and time_value
                == previous_time + 1
            )

            if (
                is_false
                and (
                    not previous_false
                    or not contiguous
                )
            ):
                false_alert_episodes += 1

            previous_false = bool(
                is_false
            )

            previous_time = int(
                time_value
            )

    eligible_count = int(
        np.sum(
            dataset[
                "hazard_eligible"
            ]
        )
    )

    return {
        "total_unique_event_onsets": int(
            total_events
        ),
        "detected_unique_event_onsets": int(
            detected_events
        ),
        "event_recall": float(
            detected_events
            / max(total_events, 1)
        ),
        "mean_detected_lead_time": (
            float(
                np.mean(lead_times)
            )
            if lead_times
            else None
        ),
        "median_detected_lead_time": (
            float(
                np.median(lead_times)
            )
            if lead_times
            else None
        ),
        "total_alert_samples": int(
            total_alert_samples
        ),
        "false_alert_samples": int(
            false_alert_samples
        ),
        "false_alert_fraction": float(
            false_alert_samples
            / max(
                total_alert_samples,
                1,
            )
        ),
        "false_alert_episodes": int(
            false_alert_episodes
        ),
        "false_alert_episodes_per_100_eligible_samples": float(
            100.0
            * false_alert_episodes
            / max(
                eligible_count,
                1,
            )
        ),
        "total_eligible_samples": int(
            eligible_count
        ),
    }


def select_event_policy_threshold(
    dataset: Dict,
    probability: np.ndarray,
) -> Dict:
    positive_probability = probability[
        dataset["hazard_eligible"]
    ]

    quantiles = np.linspace(
        0.02,
        0.98,
        120,
    )

    candidates = np.unique(
        np.concatenate([
            np.quantile(
                positive_probability,
                quantiles,
            ),
            np.asarray([
                0.01,
                0.025,
                0.05,
                0.10,
                0.20,
                0.50,
            ]),
        ])
    )

    best = None

    for threshold in candidates:
        audit = event_episode_audit(
            dataset,
            probability,
            float(threshold),
        )

        sample_metrics = evaluate_hazard(
            dataset[
                "y_hazard"
            ][
                dataset[
                    "hazard_eligible"
                ]
            ],
            probability[
                dataset[
                    "hazard_eligible"
                ]
            ],
            float(threshold),
        )

        utility = (
            2.0
            * audit["event_recall"]
            + 0.40
            * sample_metrics["precision"]
            - 0.06
            * audit[
                "false_alert_episodes_per_100_eligible_samples"
            ]
            - 0.20
            * sample_metrics["fpr"]
        )

        feasible = bool(
            sample_metrics["fpr"]
            <= CFG.maximum_policy_fpr
            and audit[
                "false_alert_episodes_per_100_eligible_samples"
            ]
            <= CFG.maximum_false_alert_episodes_per_100
        )

        result = {
            "threshold": float(
                threshold
            ),
            "utility": float(
                utility
            ),
            "feasible": feasible,
            "sample_metrics": (
                sample_metrics
            ),
            "event_audit": audit,
        }

        if best is None:
            best = result
            continue

        if (
            result["feasible"]
            and not best["feasible"]
        ):
            best = result
            continue

        if (
            result["feasible"]
            == best["feasible"]
            and result["utility"]
            > best["utility"]
        ):
            best = result

    return best


policy_probability = (
    calibrated_hazard_probability(
        policy_dataset,
        BEST_HAZARD_MODEL_NAME,
    )
)

threshold_selection = (
    select_event_policy_threshold(
        policy_dataset,
        policy_probability,
    )
)

HAZARD_DECISION_THRESHOLD = float(
    threshold_selection[
        "threshold"
    ]
)

print(
    "\nIndependent event-policy calibration:"
)

print(
    f"  Threshold             : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)

print(
    f"  Policy event recall   : "
    f"{threshold_selection['event_audit']['event_recall']:.4f}"
)

print(
    f"  False episodes / 100  : "
    f"{threshold_selection['event_audit']['false_alert_episodes_per_100_eligible_samples']:.4f}"
)

print(
    f"  Policy sample FPR     : "
    f"{threshold_selection['sample_metrics']['fpr']:.4f}"
)


# =================================================================================================
# 17. UNTOUCHED VALIDATION
# =================================================================================================

validation_point_prediction = (
    selected_point_model.predict(
        feature_view(
            validation_dataset,
            BEST_POINT_MODEL_NAME,
        )
    )
)

validation_point_metrics = (
    evaluate_point(
        validation_dataset[
            "y_point"
        ],
        validation_point_prediction,
        validation_dataset[
            "stream_ids"
        ],
    )
)

validation_hazard_probability = (
    calibrated_hazard_probability(
        validation_dataset,
        BEST_HAZARD_MODEL_NAME,
    )
)

validation_hazard_mask = (
    validation_dataset[
        "hazard_eligible"
    ]
)

validation_hazard_metrics = (
    evaluate_hazard(
        validation_dataset[
            "y_hazard"
        ][validation_hazard_mask],
        validation_hazard_probability[
            validation_hazard_mask
        ],
        HAZARD_DECISION_THRESHOLD,
    )
)

validation_oracle_point, validation_oracle_hazard = (
    structural_oracle(
        validation_dataset,
        CFG.oracle_rollouts_evaluation,
        CFG.seed + 30_000,
    )
)

validation_oracle_point_metrics = (
    evaluate_point(
        validation_dataset["y_point"],
        validation_oracle_point,
        validation_dataset["stream_ids"],
    )
)

validation_oracle_hazard_metrics = (
    evaluate_hazard(
        validation_dataset["y_hazard"][
            validation_hazard_mask
        ],
        validation_oracle_hazard[
            validation_hazard_mask
        ],
        threshold=0.5,
    )
)

validation_persistence_prediction = (
    validation_dataset["current_proxy"]
)

validation_persistence_metrics = (
    evaluate_point(
        validation_dataset["y_point"],
        validation_persistence_prediction,
        validation_dataset["stream_ids"],
    )
)

validation_constant_probability = np.full(
    np.sum(validation_hazard_mask),
    np.mean(
        validation_dataset["y_hazard"][
            validation_hazard_mask
        ]
    ),
    dtype=np.float64,
)

validation_constant_brier = float(
    brier_score_loss(
        validation_dataset["y_hazard"][
            validation_hazard_mask
        ],
        validation_constant_probability,
    )
)

validation_event_audit = (
    event_episode_audit(
        validation_dataset,
        validation_hazard_probability,
        HAZARD_DECISION_THRESHOLD,
    )
)

print_header(
    "🧪 TEST C — UNTOUCHED VALIDATION"
)

print("\nSelected causal point forecaster:")

print(
    f"  Model      : {BEST_POINT_MODEL_NAME}"
)

print(
    f"  rho        : "
    f"{validation_point_metrics['rho']:.4f}"
)

print(
    f"  Macro rho  : "
    f"{validation_point_metrics['macro_stream_rho']:.4f}"
)

print(
    f"  R²         : "
    f"{validation_point_metrics['r2']:.4f}"
)

print(
    f"  MAE        : "
    f"{validation_point_metrics['mae']:.4f}"
)

print("\nStructural Monte Carlo oracle:")

print(
    f"  rho        : "
    f"{validation_oracle_point_metrics['rho']:.4f}"
)

print(
    f"  Macro rho  : "
    f"{validation_oracle_point_metrics['macro_stream_rho']:.4f}"
)

print(
    f"  R²         : "
    f"{validation_oracle_point_metrics['r2']:.4f}"
)

print("\nObservable persistence baseline:")

print(
    f"  rho        : "
    f"{validation_persistence_metrics['rho']:.4f}"
)

print(
    f"  R²         : "
    f"{validation_persistence_metrics['r2']:.4f}"
)

print("\nCalibrated hazard forecaster:")

print(
    f"  Model      : {BEST_HAZARD_MODEL_NAME}"
)

print(
    f"  AUC        : "
    f"{(validation_hazard_metrics['auroc'] or 0.0):.4f}"
)

print(
    f"  AP         : "
    f"{(validation_hazard_metrics['auprc'] or 0.0):.4f}"
)

print(
    f"  Prevalence : "
    f"{validation_hazard_metrics['prevalence']:.4f}"
)

print(
    f"  Recall     : "
    f"{validation_hazard_metrics['recall']:.4f}"
)

print(
    f"  Precision  : "
    f"{validation_hazard_metrics['precision']:.4f}"
)

print(
    f"  FPR        : "
    f"{validation_hazard_metrics['fpr']:.4f}"
)

print(
    f"  Brier      : "
    f"{validation_hazard_metrics['brier']:.4f}"
)

print(
    f"  Const Brier: "
    f"{validation_constant_brier:.4f}"
)

print("\nStructural hazard oracle:")

print(
    f"  AUC        : "
    f"{(validation_oracle_hazard_metrics['auroc'] or 0.0):.4f}"
)

print(
    f"  AP         : "
    f"{(validation_oracle_hazard_metrics['auprc'] or 0.0):.4f}"
)

print(
    f"  Brier      : "
    f"{validation_oracle_hazard_metrics['brier']:.4f}"
)


# =================================================================================================
# 18. HORIZON-SPECIFIC STRUCTURAL AND LEARNED EVALUATION
# =================================================================================================

def fit_horizon_raw_ordered_models(
    horizon: int,
) -> Dict[str, Any]:
    horizon_datasets = (
        DATASETS_BY_HORIZON[horizon]
    )

    train = horizon_datasets[
        "TRAIN_MIXED_KNOWN"
    ]

    selection = horizon_datasets[
        "CALIBRATION_SELECTION"
    ]

    probability = horizon_datasets[
        "CALIBRATION_PROBABILITY"
    ]

    validation = horizon_datasets[
        "VALIDATION"
    ]

    point_model = ExtraTreesRegressor(
        n_estimators=320,
        min_samples_leaf=5,
        max_features=0.75,
        random_state=CFG.seed + horizon,
        n_jobs=-1,
    )

    point_model.fit(
        ordered_raw_features(train),
        train["y_point"],
    )

    selection_prediction = (
        point_model.predict(
            ordered_raw_features(
                selection
            )
        )
    )

    selection_metrics = (
        evaluate_point(
            selection["y_point"],
            selection_prediction,
            selection["stream_ids"],
        )
    )

    validation_prediction = (
        point_model.predict(
            ordered_raw_features(
                validation
            )
        )
    )

    validation_metrics = (
        evaluate_point(
            validation["y_point"],
            validation_prediction,
            validation["stream_ids"],
        )
    )

    train_mask = train[
        "hazard_eligible"
    ]

    probability_mask = probability[
        "hazard_eligible"
    ]

    validation_mask = validation[
        "hazard_eligible"
    ]

    hazard_model = ExtraTreesClassifier(
        n_estimators=360,
        min_samples_leaf=5,
        class_weight="balanced",
        max_features=0.75,
        random_state=CFG.seed + 100 + horizon,
        n_jobs=-1,
    )

    hazard_model.fit(
        ordered_raw_features(train)[
            train_mask
        ],
        train["y_hazard"][
            train_mask
        ],
    )

    probability_score = model_score(
        hazard_model,
        ordered_raw_features(
            probability
        )[probability_mask],
    )

    calibrator = PlattCalibrator()

    calibrator.fit(
        probability_score,
        probability["y_hazard"][
            probability_mask
        ],
    )

    validation_score = model_score(
        hazard_model,
        ordered_raw_features(
            validation
        )[validation_mask],
    )

    validation_probability = (
        calibrator.predict(
            validation_score
        )
    )

    validation_hazard_metrics_local = (
        evaluate_hazard(
            validation["y_hazard"][
                validation_mask
            ],
            validation_probability,
            threshold=0.5,
        )
    )

    oracle_point, oracle_hazard = (
        structural_oracle(
            validation,
            CFG.oracle_rollouts_evaluation,
            CFG.seed + 40_000 + horizon,
        )
    )

    oracle_point_metrics = (
        evaluate_point(
            validation["y_point"],
            oracle_point,
            validation["stream_ids"],
        )
    )

    oracle_hazard_metrics = (
        evaluate_hazard(
            validation["y_hazard"][
                validation_mask
            ],
            oracle_hazard[
                validation_mask
            ],
            threshold=0.5,
        )
    )

    return {
        "point_model": point_model,
        "hazard_model": hazard_model,
        "calibrator": calibrator,
        "selection_point": (
            selection_metrics
        ),
        "validation_point": (
            validation_metrics
        ),
        "validation_hazard": (
            validation_hazard_metrics_local
        ),
        "oracle_point": (
            oracle_point_metrics
        ),
        "oracle_hazard": (
            oracle_hazard_metrics
        ),
    }


print_header(
    "🧪 TEST D — HORIZON-SPECIFIC TEMPORAL VALIDATION"
)

horizon_results = {}

for horizon in CFG.forecast_horizons:
    result = (
        fit_horizon_raw_ordered_models(
            horizon
        )
    )

    horizon_results[
        horizon
    ] = {
        key: value
        for key, value in result.items()
        if key not in {
            "point_model",
            "hazard_model",
            "calibrator",
        }
    }

    learned_point = result[
        "validation_point"
    ]

    learned_hazard = result[
        "validation_hazard"
    ]

    oracle_point = result[
        "oracle_point"
    ]

    oracle_hazard = result[
        "oracle_hazard"
    ]

    print(
        f"H={horizon:<2} | "
        f"Learned point rho={learned_point['rho']:.4f} | "
        f"R²={learned_point['r2']:.4f} | "
        f"Oracle rho={oracle_point['rho']:.4f} | "
        f"Oracle R²={oracle_point['r2']:.4f} | "
        f"Learned hazard AUC="
        f"{(learned_hazard['auroc'] or 0.0):.4f} | "
        f"Oracle hazard AUC="
        f"{(oracle_hazard['auroc'] or 0.0):.4f}"
    )


# =================================================================================================
# 19. TEMPORAL ORDER ABLATIONS
# =================================================================================================

def transformed_temporal_dataset(
    dataset: Dict,
    mode: str,
    rng: Optional[
        np.random.Generator
    ] = None,
) -> Dict:
    transformed = dict(dataset)

    raw = np.array(
        dataset["X_raw"],
        copy=True,
    )

    activation = (
        np.array(
            dataset["X_activation"],
            copy=True,
        )
        if "X_activation" in dataset
        else None
    )

    if mode == "REVERSED":
        raw = raw[:, ::-1, :]

        if activation is not None:
            activation = (
                activation[:, ::-1, :]
            )

    elif mode == "SHUFFLED":
        if rng is None:
            raise ValueError(
                "SHUFFLED mode requires rng."
            )

        for row_index in range(
            len(raw)
        ):
            permutation = rng.permutation(
                raw.shape[1]
            )

            raw[row_index] = (
                raw[
                    row_index,
                    permutation,
                    :
                ]
            )

            if activation is not None:
                activation[row_index] = (
                    activation[
                        row_index,
                        permutation,
                        :
                    ]
                )

    elif mode == "CIRCULAR":
        if rng is None:
            raise ValueError(
                "CIRCULAR mode requires rng."
            )

        for row_index in range(
            len(raw)
        ):
            shift = int(
                rng.integers(
                    1,
                    raw.shape[1],
                )
            )

            raw[row_index] = np.roll(
                raw[row_index],
                shift,
                axis=0,
            )

            if activation is not None:
                activation[row_index] = (
                    np.roll(
                        activation[
                            row_index
                        ],
                        shift,
                        axis=0,
                    )
                )

    else:
        raise ValueError(
            f"Unknown transformation: {mode}"
        )

    transformed["X_raw"] = raw

    if activation is not None:
        transformed[
            "X_activation"
        ] = activation

    return transformed


def evaluate_order_variant(
    dataset: Dict,
) -> Tuple[float, float]:
    point_prediction = (
        selected_point_model.predict(
            feature_view(
                dataset,
                BEST_POINT_MODEL_NAME,
            )
        )
    )

    point_rho = safe_spearman(
        dataset["y_point"],
        point_prediction,
    )

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    hazard_score = model_score(
        selected_hazard_model,
        feature_view(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )[hazard_mask],
    )

    hazard_probability = (
        channel_calibrators[
            BEST_HAZARD_MODEL_NAME
        ].predict(
            hazard_score
        )
    )

    hazard_auc = safe_auroc(
        dataset["y_hazard"][
            hazard_mask
        ],
        hazard_probability,
    )

    return (
        float(point_rho),
        float(
            hazard_auc
            if hazard_auc is not None
            else 0.5
        ),
    )


print_header(
    "🧪 TEST E — ORDER-IDENTIFIABLE TEMPORAL ABLATIONS"
)

ordered_point_rho, ordered_hazard_auc = (
    evaluate_order_variant(
        validation_dataset
    )
)

reversed_dataset = (
    transformed_temporal_dataset(
        validation_dataset,
        "REVERSED",
    )
)

reversed_point_rho, reversed_hazard_auc = (
    evaluate_order_variant(
        reversed_dataset
    )
)

shuffle_point_values = []
shuffle_hazard_values = []

circular_point_values = []
circular_hazard_values = []

order_rng = np.random.default_rng(
    CFG.seed + 50_000
)

for trial_index in range(
    CFG.order_shuffle_trials
):
    shuffled_dataset = (
        transformed_temporal_dataset(
            validation_dataset,
            "SHUFFLED",
            order_rng,
        )
    )

    shuffled_point, shuffled_hazard = (
        evaluate_order_variant(
            shuffled_dataset
        )
    )

    shuffle_point_values.append(
        shuffled_point
    )

    shuffle_hazard_values.append(
        shuffled_hazard
    )

    circular_dataset = (
        transformed_temporal_dataset(
            validation_dataset,
            "CIRCULAR",
            order_rng,
        )
    )

    circular_point, circular_hazard = (
        evaluate_order_variant(
            circular_dataset
        )
    )

    circular_point_values.append(
        circular_point
    )

    circular_hazard_values.append(
        circular_hazard
    )


point_shuffle_p95 = float(
    np.quantile(
        shuffle_point_values,
        0.95,
    )
)

hazard_shuffle_p95 = float(
    np.quantile(
        shuffle_hazard_values,
        0.95,
    )
)

point_circular_p95 = float(
    np.quantile(
        circular_point_values,
        0.95,
    )
)

hazard_circular_p95 = float(
    np.quantile(
        circular_hazard_values,
        0.95,
    )
)

point_order_null = max(
    reversed_point_rho,
    point_shuffle_p95,
    point_circular_p95,
)

hazard_order_null = max(
    reversed_hazard_auc,
    hazard_shuffle_p95,
    hazard_circular_p95,
)

point_order_advantage = float(
    ordered_point_rho
    - point_order_null
)

hazard_order_advantage = float(
    ordered_hazard_auc
    - hazard_order_null
)

print(
    f"Point  | Ordered={ordered_point_rho:.4f} | "
    f"Reversed={reversed_point_rho:.4f} | "
    f"Shuffle p95={point_shuffle_p95:.4f} | "
    f"Circular p95={point_circular_p95:.4f} | "
    f"Advantage={point_order_advantage:+.4f}"
)

print(
    f"Hazard | Ordered={ordered_hazard_auc:.4f} | "
    f"Reversed={reversed_hazard_auc:.4f} | "
    f"Shuffle p95={hazard_shuffle_p95:.4f} | "
    f"Circular p95={hazard_circular_p95:.4f} | "
    f"Advantage={hazard_order_advantage:+.4f}"
)


# =================================================================================================
# 20. EVENT-ONSET AND FALSE-ALERT AUDIT
# =================================================================================================

print_header(
    "🧪 TEST F — EVENT-ONSET AND FALSE-ALERT EPISODE AUDIT"
)

for key, value in (
    validation_event_audit.items()
):
    print(
        f"  {key:<55}: {value}"
    )


# =================================================================================================
# 21. CROSS-REGIME TRANSFER
# =================================================================================================

TRANSFER_GROUPS = (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
)


def safe_recovery_fraction(
    model_score_value: float,
    persistence_score_value: float,
    oracle_score_value: float,
) -> float:
    denominator = (
        oracle_score_value
        - persistence_score_value
    )

    if abs(denominator) < 1e-8:
        return 0.0

    return float(
        (
            model_score_value
            - persistence_score_value
        )
        / denominator
    )


print_header(
    "🧪 TEST G — CROSS-REGIME TRANSFER"
)

transfer_results = {}

known_point_passes = 0
known_hazard_passes = 0

for group_offset, group_name in enumerate(
    TRANSFER_GROUPS
):
    dataset = DATASETS[
        group_name
    ]

    point_prediction = (
        selected_point_model.predict(
            feature_view(
                dataset,
                BEST_POINT_MODEL_NAME,
            )
        )
    )

    point_metrics = evaluate_point(
        dataset["y_point"],
        point_prediction,
        dataset["stream_ids"],
    )

    persistence_metrics = evaluate_point(
        dataset["y_point"],
        dataset["current_proxy"],
        dataset["stream_ids"],
    )

    oracle_point, oracle_hazard = (
        structural_oracle(
            dataset,
            CFG.oracle_rollouts_evaluation,
            CFG.seed
            + 60_000
            + group_offset,
        )
    )

    oracle_point_metrics = (
        evaluate_point(
            dataset["y_point"],
            oracle_point,
            dataset["stream_ids"],
        )
    )

    point_recovery = (
        safe_recovery_fraction(
            point_metrics["rho"],
            persistence_metrics["rho"],
            oracle_point_metrics["rho"],
        )
    )

    hazard_probability = (
        calibrated_hazard_probability(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )
    )

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    hazard_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            hazard_probability[
                hazard_mask
            ],
            HAZARD_DECISION_THRESHOLD,
        )
    )

    oracle_hazard_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            oracle_hazard[
                hazard_mask
            ],
            threshold=0.5,
        )
    )

    hazard_auc = (
        hazard_metrics["auroc"]
        or 0.0
    )

    hazard_ap = (
        hazard_metrics["auprc"]
        or 0.0
    )

    hazard_ap_gain = float(
        hazard_ap
        - hazard_metrics[
            "prevalence"
        ]
    )

    known_family = (
        group_name
        in {
            "AR2_TRANSFER",
            "DELAYED_TRANSFER",
            "OSCILLATORY_TRANSFER",
            "MEAN_REVERTING_TRANSFER",
        }
    )

    oracle_forecastable = bool(
        oracle_point_metrics["rho"]
        >= CFG.minimum_preflight_point_rho
        and oracle_point_metrics["r2"]
        >= CFG.minimum_preflight_point_r2
    )

    point_gate = bool(
        oracle_forecastable
        and point_metrics["rho"]
        >= CFG.minimum_point_rho
        and point_metrics[
            "macro_stream_rho"
        ]
        >= CFG.minimum_point_macro_rho
        and point_metrics["r2"]
        >= CFG.minimum_point_r2
        and point_metrics["rho"]
        > persistence_metrics["rho"]
    )

    hazard_oracle_auc = (
        oracle_hazard_metrics[
            "auroc"
        ]
        or 0.0
    )

    hazard_oracle_forecastable = bool(
        hazard_oracle_auc
        >= CFG.minimum_preflight_hazard_auc
    )

    hazard_gate = bool(
        hazard_oracle_forecastable
        and hazard_auc
        >= CFG.minimum_hazard_auc
        and hazard_ap_gain
        >= CFG.minimum_hazard_ap_gain
        and hazard_metrics["fpr"]
        <= CFG.maximum_policy_fpr
    )

    if known_family:
        known_point_passes += int(
            point_gate
        )

        known_hazard_passes += int(
            hazard_gate
        )

    transfer_results[
        group_name
    ] = {
        "point": point_metrics,
        "persistence": (
            persistence_metrics
        ),
        "point_oracle": (
            oracle_point_metrics
        ),
        "point_recovery": (
            point_recovery
        ),
        "point_oracle_forecastable": (
            oracle_forecastable
        ),
        "point_gate": point_gate,
        "hazard": hazard_metrics,
        "hazard_oracle": (
            oracle_hazard_metrics
        ),
        "hazard_ap_gain": (
            hazard_ap_gain
        ),
        "hazard_oracle_forecastable": (
            hazard_oracle_forecastable
        ),
        "hazard_gate": hazard_gate,
    }

    print(f"\n{group_name}")

    print(
        f"  Point  | "
        f"rho={point_metrics['rho']:.4f} | "
        f"Macro={point_metrics['macro_stream_rho']:.4f} | "
        f"R²={point_metrics['r2']:.4f} | "
        f"Persist={persistence_metrics['rho']:.4f} | "
        f"Oracle={oracle_point_metrics['rho']:.4f} | "
        f"Recovery={point_recovery:.4f} | "
        f"Gate={'PASS' if point_gate else 'FAIL'}"
    )

    print(
        f"  Hazard | "
        f"AUC={hazard_auc:.4f} | "
        f"AP={hazard_ap:.4f} | "
        f"Prev={hazard_metrics['prevalence']:.4f} | "
        f"ΔAP={hazard_ap_gain:+.4f} | "
        f"FPR={hazard_metrics['fpr']:.4f} | "
        f"Oracle AUC={hazard_oracle_auc:.4f} | "
        f"Gate={'PASS' if hazard_gate else 'FAIL'}"
    )


# =================================================================================================
# 22. TRANSITION-RESIDUAL NOVELTY SENSOR
# =================================================================================================

def transition_prediction_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]
    activation = dataset[
        "X_activation"
    ]

    raw_past = raw[
        :,
        :-1,
        :
    ].reshape(
        len(raw),
        -1,
    )

    latest_context = raw[
        :,
        -2,
        :
    ]

    activation_past = activation[
        :,
        :-1,
        :
    ]

    activation_summary = (
        np.concatenate([
            np.mean(
                activation_past,
                axis=1,
            ),
            np.std(
                activation_past,
                axis=1,
            ),
            activation_past[
                :,
                -1,
                :
            ],
        ], axis=1)
    )

    return np.concatenate([
        raw_past,
        latest_context,
        activation_summary,
    ], axis=1)


def transition_target(
    dataset: Dict,
) -> np.ndarray:
    return dataset[
        "X_raw"
    ][:, -1, 4]


transition_model = (
    ExtraTreesRegressor(
        n_estimators=420,
        min_samples_leaf=5,
        max_features=0.72,
        random_state=CFG.seed + 70_000,
        n_jobs=-1,
    )
)

transition_model.fit(
    transition_prediction_features(
        train_dataset
    ),
    transition_target(
        train_dataset
    ),
)


def raw_transition_residual_score(
    dataset: Dict,
) -> np.ndarray:
    prediction = (
        transition_model.predict(
            transition_prediction_features(
                dataset
            )
        )
    )

    target = transition_target(
        dataset
    )

    return np.abs(
        target
        - prediction
    )


selection_transition_score = (
    raw_transition_residual_score(
        selection_dataset
    )
)

transition_center = float(
    np.median(
        selection_transition_score
    )
)

transition_scale = float(
    1.4826
    * np.median(
        np.abs(
            selection_transition_score
            - transition_center
        )
    )
    + 1e-6
)


def activation_novelty_features(
    dataset: Dict,
) -> np.ndarray:
    activation = dataset[
        "X_activation"
    ]

    return np.concatenate([
        activation[:, -1, :],
        activation[:, -1, :]
        - activation[:, -2, :],
        np.mean(
            activation,
            axis=1,
        ),
        np.std(
            activation,
            axis=1,
        ),
    ], axis=1)


activation_novelty_model = (
    IsolationForest(
        n_estimators=360,
        max_samples="auto",
        contamination="auto",
        random_state=CFG.seed + 71_000,
        n_jobs=-1,
    )
)

activation_novelty_model.fit(
    activation_novelty_features(
        train_dataset
    )
)

selection_activation_raw_score = (
    -activation_novelty_model.score_samples(
        activation_novelty_features(
            selection_dataset
        )
    )
)

activation_score_center = float(
    np.median(
        selection_activation_raw_score
    )
)

activation_score_scale = float(
    1.4826
    * np.median(
        np.abs(
            selection_activation_raw_score
            - activation_score_center
        )
    )
    + 1e-6
)


def mechanism_novelty_score(
    dataset: Dict,
) -> np.ndarray:
    transition_score = (
        raw_transition_residual_score(
            dataset
        )
    )

    transition_z = (
        transition_score
        - transition_center
    ) / transition_scale

    activation_score = (
        -activation_novelty_model.score_samples(
            activation_novelty_features(
                dataset
            )
        )
    )

    activation_z = (
        activation_score
        - activation_score_center
    ) / activation_score_scale

    combined_score = (
        0.75 * transition_z
        + 0.25 * activation_z
    )

    return combined_score.astype(
        np.float64
    )


novelty_policy_score = (
    mechanism_novelty_score(
        policy_dataset
    )
)

NOVELTY_THRESHOLD = float(
    np.quantile(
        novelty_policy_score,
        1.0
        - CFG.maximum_novelty_fpr,
    )
)

validation_novelty_score = (
    mechanism_novelty_score(
        validation_dataset
    )
)

validation_novelty_fpr = float(
    np.mean(
        validation_novelty_score
        >= NOVELTY_THRESHOLD
    )
)

known_transfer_scores = []
known_transfer_labels = []

for group_name in (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
):
    group_score = (
        mechanism_novelty_score(
            DATASETS[group_name]
        )
    )

    known_transfer_scores.append(
        group_score
    )

    known_transfer_labels.append(
        np.zeros(
            len(group_score),
            dtype=np.int64,
        )
    )

known_transfer_score = (
    np.concatenate(
        known_transfer_scores
    )
)

known_transfer_novelty_fpr = float(
    np.mean(
        known_transfer_score
        >= NOVELTY_THRESHOLD
    )
)

unknown_group_scores = {}

for group_name in (
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
):
    unknown_group_scores[
        group_name
    ] = mechanism_novelty_score(
        DATASETS[group_name]
    )

unknown_score = np.concatenate([
    unknown_group_scores[
        "ABRUPT_UNKNOWN"
    ],
    unknown_group_scores[
        "MECHANISM_NOVEL"
    ],
])

unknown_novelty_tpr = float(
    np.mean(
        unknown_score
        >= NOVELTY_THRESHOLD
    )
)

novelty_evaluation_labels = (
    np.concatenate([
        np.zeros(
            len(known_transfer_score),
            dtype=np.int64,
        ),
        np.ones(
            len(unknown_score),
            dtype=np.int64,
        ),
    ])
)

novelty_evaluation_scores = (
    np.concatenate([
        known_transfer_score,
        unknown_score,
    ])
)

novelty_auc = safe_auroc(
    novelty_evaluation_labels,
    novelty_evaluation_scores,
)

novelty_ap = safe_auprc(
    novelty_evaluation_labels,
    novelty_evaluation_scores,
)

print_header(
    "🧪 TEST H — MATCHED TRANSITION-RESIDUAL MECHANISM NOVELTY"
)

print(
    f"Novelty AUROC                 : "
    f"{(novelty_auc or 0.0):.4f}"
)

print(
    f"Novelty AUPRC                 : "
    f"{(novelty_ap or 0.0):.4f}"
)

print(
    f"Validation known FPR          : "
    f"{validation_novelty_fpr:.4f}"
)

print(
    f"Known-parameter-transfer FPR  : "
    f"{known_transfer_novelty_fpr:.4f}"
)

print(
    f"Unknown-mechanism TPR         : "
    f"{unknown_novelty_tpr:.4f}"
)

print(
    f"Novelty threshold             : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

for group_name, score in (
    unknown_group_scores.items()
):
    print(
        f"  {group_name:<22} | "
        f"mean score={np.mean(score):.4f} | "
        f"TPR={np.mean(score >= NOVELTY_THRESHOLD):.4f}"
    )


# =================================================================================================
# 23. NOVELTY / DEGRADATION CONFOUNDING AUDIT
# =================================================================================================

def binary_phi(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    first = np.asarray(
        first,
        dtype=np.int64,
    )

    second = np.asarray(
        second,
        dtype=np.int64,
    )

    if (
        len(first) < 2
        or len(np.unique(first)) < 2
        or len(np.unique(second)) < 2
    ):
        return 0.0

    correlation = np.corrcoef(
        first,
        second,
    )[0, 1]

    if not np.isfinite(
        correlation
    ):
        return 0.0

    return float(
        correlation
    )


known_for_confounding = np.concatenate([
    DATASETS["AR2_TRANSFER"][
        "y_point"
    ],
    DATASETS["DELAYED_TRANSFER"][
        "y_point"
    ],
    DATASETS["OSCILLATORY_TRANSFER"][
        "y_point"
    ],
    DATASETS["MEAN_REVERTING_TRANSFER"][
        "y_point"
    ],
])

unknown_for_confounding = np.concatenate([
    DATASETS["ABRUPT_UNKNOWN"][
        "y_point"
    ],
    DATASETS["MECHANISM_NOVEL"][
        "y_point"
    ],
])

generator_novelty_labels = np.concatenate([
    np.zeros(
        len(known_for_confounding),
        dtype=np.int64,
    ),
    np.ones(
        len(unknown_for_confounding),
        dtype=np.int64,
    ),
])

generator_degradation_labels = (
    np.abs(
        np.concatenate([
            known_for_confounding,
            unknown_for_confounding,
        ])
    )
    >= CFG.latent_event_threshold
).astype(np.int64)

generator_novelty_degradation_phi = (
    binary_phi(
        generator_novelty_labels,
        generator_degradation_labels,
    )
)

detector_labels = (
    novelty_evaluation_scores
    >= NOVELTY_THRESHOLD
).astype(np.int64)

detector_degradation_labels = (
    np.abs(
        np.concatenate([
            known_for_confounding,
            unknown_for_confounding,
        ])
    )
    >= CFG.latent_event_threshold
).astype(np.int64)

minimum_length = min(
    len(detector_labels),
    len(detector_degradation_labels),
)

detector_degradation_phi = (
    binary_phi(
        detector_labels[
            :minimum_length
        ],
        detector_degradation_labels[
            :minimum_length
        ],
    )
)

print_header(
    "🧪 TEST I — NOVELTY / DEGRADATION DISENTANGLEMENT"
)

print(
    "Generator novelty/degradation phi : "
    f"{generator_novelty_degradation_phi:+.4f}"
)

print(
    "Detector/degradation phi           : "
    f"{detector_degradation_phi:+.4f}"
)


# =================================================================================================
# 24. INDEPENDENT RAW VS ACTIVATION-COMBINED HAZARD AUDIT
# =================================================================================================

combined_policy_probability = (
    calibrated_hazard_probability(
        policy_dataset,
        "COMBINED_ORDERED",
    )
)

combined_threshold_selection = (
    select_event_policy_threshold(
        policy_dataset,
        combined_policy_probability,
    )
)

COMBINED_HAZARD_THRESHOLD = float(
    combined_threshold_selection[
        "threshold"
    ]
)

channel_separation_results = {}

for group_name in (
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
):
    dataset = DATASETS[
        group_name
    ]

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    raw_probability = (
        calibrated_hazard_probability(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )
    )

    combined_probability = (
        calibrated_hazard_probability(
            dataset,
            "COMBINED_ORDERED",
        )
    )

    raw_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            raw_probability[
                hazard_mask
            ],
            HAZARD_DECISION_THRESHOLD,
        )
    )

    combined_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            combined_probability[
                hazard_mask
            ],
            COMBINED_HAZARD_THRESHOLD,
        )
    )

    channel_separation_results[
        group_name
    ] = {
        "raw_causal": raw_metrics,
        "combined_diagnostic": (
            combined_metrics
        ),
        "raw_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "combined_threshold": (
            COMBINED_HAZARD_THRESHOLD
        ),
        "delta_fpr": float(
            combined_metrics["fpr"]
            - raw_metrics["fpr"]
        ),
        "delta_brier": float(
            combined_metrics["brier"]
            - raw_metrics["brier"]
        ),
    }

    print(f"\n{group_name}")

    print(
        f"  Raw causal | "
        f"AUC={(raw_metrics['auroc'] or 0.0):.4f} | "
        f"AP={(raw_metrics['auprc'] or 0.0):.4f} | "
        f"FPR={raw_metrics['fpr']:.4f} | "
        f"Brier={raw_metrics['brier']:.4f}"
    )

    print(
        f"  Combined   | "
        f"AUC={(combined_metrics['auroc'] or 0.0):.4f} | "
        f"AP={(combined_metrics['auprc'] or 0.0):.4f} | "
        f"FPR={combined_metrics['fpr']:.4f} | "
        f"Brier={combined_metrics['brier']:.4f}"
    )

    print(
        f"  ΔFPR combined−raw   : "
        f"{combined_metrics['fpr'] - raw_metrics['fpr']:+.4f}"
    )

    print(
        f"  ΔBrier combined−raw : "
        f"{combined_metrics['brier'] - raw_metrics['brier']:+.4f}"
    )


# =================================================================================================
# 25. STREAM-LEVEL BOOTSTRAP INTERVALS
# =================================================================================================

def percentile_interval(
    values: Sequence[float],
    lower: float = 0.025,
    upper: float = 0.975,
) -> List[float]:
    valid = np.asarray([
        value
        for value in values
        if value is not None
        and np.isfinite(value)
    ], dtype=np.float64)

    if len(valid) == 0:
        return [
            float("nan"),
            float("nan"),
        ]

    return [
        float(
            np.quantile(
                valid,
                lower,
            )
        ),
        float(
            np.quantile(
                valid,
                upper,
            )
        ),
    ]


def stream_bootstrap_intervals(
    dataset: Dict,
    point_prediction: np.ndarray,
    hazard_probability: np.ndarray,
    iterations: int,
    seed: int,
) -> Dict:
    rng = np.random.default_rng(
        seed
    )

    unique_streams = np.unique(
        dataset["stream_ids"]
    )

    point_rhos = []
    point_macro_rhos = []
    point_r2_values = []

    hazard_aucs = []
    hazard_aps = []
    hazard_briers = []

    for _ in range(
        iterations
    ):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        sampled_indices = []
        resampled_ids = []

        for bootstrap_stream_id, original_stream_id in enumerate(
            sampled_streams
        ):
            indices = np.flatnonzero(
                dataset["stream_ids"]
                == original_stream_id
            )

            sampled_indices.append(
                indices
            )

            resampled_ids.append(
                np.full(
                    len(indices),
                    bootstrap_stream_id,
                    dtype=np.int64,
                )
            )

        sampled_indices = (
            np.concatenate(
                sampled_indices
            )
        )

        resampled_ids = (
            np.concatenate(
                resampled_ids
            )
        )

        point_metrics = evaluate_point(
            dataset["y_point"][
                sampled_indices
            ],
            point_prediction[
                sampled_indices
            ],
            resampled_ids,
        )

        point_rhos.append(
            point_metrics["rho"]
        )

        point_macro_rhos.append(
            point_metrics[
                "macro_stream_rho"
            ]
        )

        point_r2_values.append(
            point_metrics["r2"]
        )

        eligible = dataset[
            "hazard_eligible"
        ][sampled_indices]

        labels = dataset[
            "y_hazard"
        ][sampled_indices][eligible]

        probability = hazard_probability[
            sampled_indices
        ][eligible]

        if len(np.unique(labels)) < 2:
            continue

        hazard_aucs.append(
            roc_auc_score(
                labels,
                probability,
            )
        )

        hazard_aps.append(
            average_precision_score(
                labels,
                probability,
            )
        )

        hazard_briers.append(
            brier_score_loss(
                labels,
                probability,
            )
        )

    return {
        "point_rho_95_ci": (
            percentile_interval(
                point_rhos
            )
        ),
        "point_macro_rho_95_ci": (
            percentile_interval(
                point_macro_rhos
            )
        ),
        "point_r2_95_ci": (
            percentile_interval(
                point_r2_values
            )
        ),
        "hazard_auc_95_ci": (
            percentile_interval(
                hazard_aucs
            )
        ),
        "hazard_ap_95_ci": (
            percentile_interval(
                hazard_aps
            )
        ),
        "hazard_brier_95_ci": (
            percentile_interval(
                hazard_briers
            )
        ),
    }


bootstrap_intervals = (
    stream_bootstrap_intervals(
        validation_dataset,
        validation_point_prediction,
        validation_hazard_probability,
        CFG.bootstrap_iterations,
        CFG.seed + 80_000,
    )
)

print_header(
    "🧪 TEST J — STREAM-LEVEL BOOTSTRAP INTERVALS"
)

print(
    "Point rho 95% CI       : "
    f"{bootstrap_intervals['point_rho_95_ci']}"
)

print(
    "Point macro rho 95% CI : "
    f"{bootstrap_intervals['point_macro_rho_95_ci']}"
)

print(
    "Point R² 95% CI        : "
    f"{bootstrap_intervals['point_r2_95_ci']}"
)

print(
    "Hazard AUC 95% CI      : "
    f"{bootstrap_intervals['hazard_auc_95_ci']}"
)

print(
    "Hazard AP 95% CI       : "
    f"{bootstrap_intervals['hazard_ap_95_ci']}"
)

print(
    "Hazard Brier 95% CI    : "
    f"{bootstrap_intervals['hazard_brier_95_ci']}"
)


# =================================================================================================
# 26. AUTOMATED SCIENTIFIC GATES
# =================================================================================================

feedback_observability_valid = bool(
    layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]["rho"]
    >= CFG.minimum_observability_rho
    and layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]["r2"]
    >= CFG.minimum_observability_r2
)

point_oracle_forecastable = bool(
    validation_oracle_point_metrics[
        "rho"
    ]
    >= CFG.minimum_preflight_point_rho
    and validation_oracle_point_metrics[
        "r2"
    ]
    >= CFG.minimum_preflight_point_r2
)

hazard_oracle_forecastable = bool(
    (
        validation_oracle_hazard_metrics[
            "auroc"
        ]
        or 0.0
    )
    >= CFG.minimum_preflight_hazard_auc
)

selected_point_forecaster_valid = bool(
    validation_point_metrics["rho"]
    >= CFG.minimum_point_rho
    and validation_point_metrics[
        "macro_stream_rho"
    ]
    >= CFG.minimum_point_macro_rho
    and validation_point_metrics["r2"]
    >= CFG.minimum_point_r2
    and validation_point_metrics["rho"]
    > validation_persistence_metrics[
        "rho"
    ]
)

validation_hazard_ap_gain = float(
    (
        validation_hazard_metrics[
            "auprc"
        ]
        or 0.0
    )
    - validation_hazard_metrics[
        "prevalence"
    ]
)

selected_hazard_forecaster_valid = bool(
    (
        validation_hazard_metrics[
            "auroc"
        ]
        or 0.0
    )
    >= CFG.minimum_hazard_auc
    and validation_hazard_ap_gain
    >= CFG.minimum_hazard_ap_gain
    and validation_hazard_metrics[
        "brier"
    ]
    < validation_constant_brier
)

point_temporal_order_established = bool(
    BEST_POINT_MODEL_NAME
    in {
        "RAW_ORDERED",
    }
    and point_order_advantage
    >= CFG.minimum_order_advantage
)

hazard_temporal_order_established = bool(
    BEST_HAZARD_MODEL_NAME
    in {
        "RAW_ORDERED",
    }
    and hazard_order_advantage
    >= CFG.minimum_order_advantage
)

probability_calibration_valid = bool(
    validation_hazard_metrics[
        "brier"
    ]
    < validation_constant_brier
)

operational_warning_policy_valid = bool(
    validation_event_audit[
        "event_recall"
    ]
    >= CFG.minimum_event_recall
    and validation_event_audit[
        "false_alert_episodes_per_100_eligible_samples"
    ]
    <= CFG.maximum_false_alert_episodes_per_100
    and validation_hazard_metrics[
        "fpr"
    ]
    <= CFG.maximum_policy_fpr
)

novelty_sensor_valid = bool(
    (
        novelty_auc
        or 0.0
    )
    >= CFG.minimum_novelty_auc
    and unknown_novelty_tpr
    >= CFG.minimum_novelty_tpr
    and validation_novelty_fpr
    <= CFG.maximum_novelty_fpr
    and known_transfer_novelty_fpr
    <= CFG.maximum_novelty_fpr
)

factorial_novelty_independence_valid = bool(
    abs(
        generator_novelty_degradation_phi
    )
    <= 0.20
)

detector_novelty_independence_valid = bool(
    abs(
        detector_degradation_phi
    )
    <= 0.25
)

causal_novelty_channel_separation_valid = bool(
    BEST_POINT_MODEL_NAME
    in CAUSAL_MODEL_NAMES
    and BEST_HAZARD_MODEL_NAME
    in CAUSAL_MODEL_NAMES
)

point_transfer_strict_majority = bool(
    known_point_passes
    >= 3
)

hazard_transfer_strict_majority = bool(
    known_hazard_passes
    >= 3
)

benchmark_construction_valid = bool(
    primary_preflight_passed
    and point_oracle_forecastable
    and hazard_oracle_forecastable
    and factorial_novelty_independence_valid
)

observable_feedback_self_model_valid = (
    feedback_observability_valid
)

temporal_order_learning_valid = bool(
    selected_point_forecaster_valid
    and selected_hazard_forecaster_valid
    and point_temporal_order_established
    and hazard_temporal_order_established
)

hazard_event_onset_system_valid = bool(
    selected_hazard_forecaster_valid
    and probability_calibration_valid
    and operational_warning_policy_valid
)

cross_regime_transfer_valid = bool(
    point_transfer_strict_majority
    and hazard_transfer_strict_majority
)

disentangled_architecture_valid = bool(
    novelty_sensor_valid
    and factorial_novelty_independence_valid
    and detector_novelty_independence_valid
    and causal_novelty_channel_separation_valid
)

overall_valid = bool(
    benchmark_construction_valid
    and observable_feedback_self_model_valid
    and temporal_order_learning_valid
    and hazard_event_onset_system_valid
    and cross_regime_transfer_valid
    and novelty_sensor_valid
    and disentangled_architecture_valid
)


scientific_gates = {
    "feedback_observability_valid": (
        feedback_observability_valid
    ),
    "point_oracle_forecastable": (
        point_oracle_forecastable
    ),
    "hazard_oracle_forecastable": (
        hazard_oracle_forecastable
    ),
    "selected_point_forecaster_valid": (
        selected_point_forecaster_valid
    ),
    "selected_hazard_forecaster_valid": (
        selected_hazard_forecaster_valid
    ),
    "point_temporal_order_established": (
        point_temporal_order_established
    ),
    "hazard_temporal_order_established": (
        hazard_temporal_order_established
    ),
    "probability_calibration_valid": (
        probability_calibration_valid
    ),
    "operational_warning_policy_valid": (
        operational_warning_policy_valid
    ),
    "novelty_sensor_valid": (
        novelty_sensor_valid
    ),
    "factorial_novelty_independence_valid": (
        factorial_novelty_independence_valid
    ),
    "detector_novelty_independence_valid": (
        detector_novelty_independence_valid
    ),
    "causal_novelty_channel_separation_valid": (
        causal_novelty_channel_separation_valid
    ),
    "point_transfer_strict_majority": (
        point_transfer_strict_majority
    ),
    "hazard_transfer_strict_majority": (
        hazard_transfer_strict_majority
    ),
}


overall_statuses = {
    "benchmark_construction": (
        benchmark_construction_valid
    ),
    "observable_feedback_self_model": (
        observable_feedback_self_model_valid
    ),
    "temporal_order_learning": (
        temporal_order_learning_valid
    ),
    "hazard_event_onset_system": (
        hazard_event_onset_system_valid
    ),
    "cross_regime_transfer": (
        cross_regime_transfer_valid
    ),
    "novelty_detection": (
        novelty_sensor_valid
    ),
    "disentangled_architecture": (
        disentangled_architecture_valid
    ),
    "overall": overall_valid,
}


diagnostic_messages = []

diagnostic_map = {
    "feedback_observability_valid": (
        "ACTIVATION_FEEDBACK_OBSERVABILITY_FAILED"
    ),
    "point_oracle_forecastable": (
        "POINT_TARGET_NOT_FORECASTABLE_BY_STRUCTURAL_ORACLE"
    ),
    "hazard_oracle_forecastable": (
        "HAZARD_TARGET_NOT_FORECASTABLE_BY_STRUCTURAL_ORACLE"
    ),
    "selected_point_forecaster_valid": (
        "SELECTED_POINT_FORECASTER_INVALID"
    ),
    "selected_hazard_forecaster_valid": (
        "SELECTED_HAZARD_FORECASTER_INVALID"
    ),
    "point_temporal_order_established": (
        "POINT_TEMPORAL_ORDER_NOT_ESTABLISHED"
    ),
    "hazard_temporal_order_established": (
        "HAZARD_TEMPORAL_ORDER_NOT_ESTABLISHED"
    ),
    "probability_calibration_valid": (
        "HAZARD_PROBABILITY_CALIBRATION_FAILED"
    ),
    "operational_warning_policy_valid": (
        "OPERATIONAL_WARNING_POLICY_FAILED"
    ),
    "novelty_sensor_valid": (
        "MECHANISM_NOVELTY_SENSOR_FAILED"
    ),
    "factorial_novelty_independence_valid": (
        "GENERATOR_NOVELTY_DEGRADATION_CONFOUNDING"
    ),
    "detector_novelty_independence_valid": (
        "DETECTOR_NOVELTY_DEGRADATION_CONFOUNDING"
    ),
    "causal_novelty_channel_separation_valid": (
        "ACTIVATION_CONTAMINATED_CAUSAL_CHANNEL_SELECTION"
    ),
    "point_transfer_strict_majority": (
        "POINT_KNOWN_FAMILY_TRANSFER_FAILED"
    ),
    "hazard_transfer_strict_majority": (
        "HAZARD_KNOWN_FAMILY_TRANSFER_FAILED"
    ),
}

for gate_name, gate_value in (
    scientific_gates.items()
):
    if not gate_value:
        diagnostic_messages.append(
            diagnostic_map[
                gate_name
            ]
        )

if not diagnostic_messages:
    diagnostic_messages.append(
        "ALL_PRIMARY_SCIENTIFIC_GATES_PASSED"
    )


# =================================================================================================
# 27. SUMMARY
# =================================================================================================

print_header(
    "🧠 AUTOMATED DIAGNOSTIC SUMMARY"
)

for gate_name, gate_value in (
    scientific_gates.items()
):
    print(
        f"{'✅' if gate_value else '❌'} "
        f"{gate_name}"
    )

print(
    "\nKnown-family point transfer gates:"
)

for group_name in (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
):
    passed = transfer_results[
        group_name
    ]["point_gate"]

    print(
        f"  {'✅' if passed else '❌'} "
        f"{group_name}"
    )

print(
    f"\nPoint transfer count: "
    f"{known_point_passes}/4 "
    f"(strict majority requires 3)"
)

print(
    "\nKnown-family hazard transfer gates:"
)

for group_name in (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
):
    passed = transfer_results[
        group_name
    ]["hazard_gate"]

    print(
        f"  {'✅' if passed else '❌'} "
        f"{group_name}"
    )

print(
    f"\nHazard transfer count: "
    f"{known_hazard_passes}/4 "
    f"(strict majority requires 3)"
)

print("\nSelected architecture:")

print(
    f"  Feedback layer        : "
    f"{BEST_FEEDBACK_LAYER}"
)

print(
    f"  Activation PCA dim    : "
    f"{CFG.activation_pca_dimension}"
)

print(
    f"  Causal point model    : "
    f"{BEST_POINT_MODEL_NAME}"
)

print(
    f"  Causal hazard model   : "
    f"{BEST_HAZARD_MODEL_NAME}"
)

print(
    f"  Hazard threshold      : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)

print(
    f"  Combined threshold    : "
    f"{COMBINED_HAZARD_THRESHOLD:.4f}"
)

print(
    f"  Novelty threshold     : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

print(
    "  Architecture policy  : "
    "RAW causal forecasting + "
    "transition-residual/activation novelty sensing"
)

print("\nOverall scientific statuses:")

for status_name, status_value in (
    overall_statuses.items()
):
    print(
        f"  {status_name:<38}: "
        f"{'PASS' if status_value else 'FAIL'}"
    )

print("\nFinal diagnosis:")

print(
    " | ".join(
        diagnostic_messages
    )
)


# =================================================================================================
# 28. TELEMETRY SERIALIZATION
# =================================================================================================

telemetry = {
    "milestone": CFG.milestone,
    "title": CFG.title,
    "execution_status": "COMPLETE",
    "configuration": asdict(CFG),
    "device": DEVICE,
    "model": CFG.model_name,
    "valid_layers": valid_layers,
    "selected_architecture": {
        "feedback_layer": (
            BEST_FEEDBACK_LAYER
        ),
        "activation_pca_dimension": (
            CFG.activation_pca_dimension
        ),
        "causal_point_model": (
            BEST_POINT_MODEL_NAME
        ),
        "causal_hazard_model": (
            BEST_HAZARD_MODEL_NAME
        ),
        "hazard_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "combined_hazard_threshold": (
            COMBINED_HAZARD_THRESHOLD
        ),
        "novelty_threshold": (
            NOVELTY_THRESHOLD
        ),
        "policy": (
            "RAW causal forecasting + "
            "separated transition-residual/"
            "activation-assisted novelty sensing"
        ),
    },
    "preflight": (
        preflight_results
    ),
    "activation_observability": {
        "layer_results": (
            layer_observability_results
        ),
        "selected_layer": (
            BEST_FEEDBACK_LAYER
        ),
    },
    "model_selection": {
        "point": (
            point_selection_results
        ),
        "hazard": (
            hazard_selection_results
        ),
    },
    "probability_and_policy_calibration": {
        "selected_hazard_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "selected_policy": (
            threshold_selection
        ),
        "combined_hazard_threshold": (
            COMBINED_HAZARD_THRESHOLD
        ),
        "combined_policy": (
            combined_threshold_selection
        ),
    },
    "untouched_validation": {
        "point": (
            validation_point_metrics
        ),
        "point_oracle": (
            validation_oracle_point_metrics
        ),
        "persistence": (
            validation_persistence_metrics
        ),
        "hazard": (
            validation_hazard_metrics
        ),
        "hazard_ap_gain": (
            validation_hazard_ap_gain
        ),
        "hazard_oracle": (
            validation_oracle_hazard_metrics
        ),
        "constant_brier": (
            validation_constant_brier
        ),
        "event_audit": (
            validation_event_audit
        ),
    },
    "horizon_specific_results": (
        horizon_results
    ),
    "temporal_order_ablation": {
        "point": {
            "ordered": (
                ordered_point_rho
            ),
            "reversed": (
                reversed_point_rho
            ),
            "shuffle_p95": (
                point_shuffle_p95
            ),
            "circular_p95": (
                point_circular_p95
            ),
            "advantage": (
                point_order_advantage
            ),
        },
        "hazard": {
            "ordered": (
                ordered_hazard_auc
            ),
            "reversed": (
                reversed_hazard_auc
            ),
            "shuffle_p95": (
                hazard_shuffle_p95
            ),
            "circular_p95": (
                hazard_circular_p95
            ),
            "advantage": (
                hazard_order_advantage
            ),
        },
    },
    "cross_regime_transfer": (
        transfer_results
    ),
    "novelty": {
        "auroc": novelty_auc,
        "auprc": novelty_ap,
        "validation_known_fpr": (
            validation_novelty_fpr
        ),
        "known_transfer_fpr": (
            known_transfer_novelty_fpr
        ),
        "unknown_tpr": (
            unknown_novelty_tpr
        ),
        "threshold": (
            NOVELTY_THRESHOLD
        ),
        "group_results": {
            group_name: {
                "mean_score": float(
                    np.mean(score)
                ),
                "tpr": float(
                    np.mean(
                        score
                        >= NOVELTY_THRESHOLD
                    )
                ),
            }
            for group_name, score in (
                unknown_group_scores.items()
            )
        },
    },
    "novelty_degradation_disentanglement": {
        "generator_phi": (
            generator_novelty_degradation_phi
        ),
        "detector_phi": (
            detector_degradation_phi
        ),
    },
    "channel_separation": (
        channel_separation_results
    ),
    "bootstrap_intervals": (
        bootstrap_intervals
    ),
    "scientific_gates": (
        scientific_gates
    ),
    "overall_statuses": (
        overall_statuses
    ),
    "diagnostic_messages": (
        diagnostic_messages
    ),
}

with open(
    CFG.output_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_builtin(
            telemetry
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


# =================================================================================================
# 29. CLEANUP
# =================================================================================================

try:
    del model
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    f"\n💾 Telemetry written to: "
    f"{CFG.output_file}"
)

print_header(
    "✅ M20.6.3.2 EXECUTION COMPLETED"
)

print(
    f"Final scientific status: "
    f"{'PASS' if overall_valid else 'FAIL'}"
)


🔬 MILESTONE 20.6.3.2 — STABLE-MECHANISM STRUCTURAL-ORACLE PREFLIGHT
Device                 : CUDA
Model                  : Qwen/Qwen2.5-0.5B
History length         : 8
Forecast horizons      : (1, 2, 3, 6)
Primary horizon        : 3
Latent threshold       : 0.72
Global seed            : 206322

🧪 PREFLIGHT A — STRUCTURAL FORECASTABILITY
H=1  | Point rho=0.9946 | Macro=0.9915 | R²=0.9901 | Hazard AUC=0.9765 | AP=0.7796 | Prev=0.0708 | PASS
H=2  | Point rho=0.9894 | Macro=0.9853 | R²=0.9807 | Hazard AUC=0.9684 | AP=0.8462 | Prev=0.1350 | PASS
H=3  | Point rho=0.9867 | Macro=0.9822 | R²=0.9758 | Hazard AUC=0.9690 | AP=0.8931 | Prev=0.1978 | PASS
H=6  | Point rho=0.9832 | Macro=0.9777 | R²=0.9699 | Hazard AUC=0.9717 | AP=0.9556 | Prev=0.3796 | PASS

✅ Primary benchmark is structurally forecastable. Activation extraction may proceed.

⏳ LOADING TRANSFORMERLENS MODEL


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Model loaded. Valid layers: [0, 3, 6, 9, 13, 16, 19, 23]

🔬 Extracting train/selection activations for layer selection...

🧪 TEST A — ACTIVATION-ONLY FEEDBACK OBSERVABILITY
Layer 0  | rho=0.8109 | R²=0.7923 | MAE=0.2278
Layer 3  | rho=0.8239 | R²=0.8048 | MAE=0.2194
Layer 6  | rho=0.9208 | R²=0.8729 | MAE=0.1697
Layer 9  | rho=0.9268 | R²=0.8779 | MAE=0.1652
Layer 13 | rho=0.9343 | R²=0.8878 | MAE=0.1595
Layer 16 | rho=0.9322 | R²=0.8857 | MAE=0.1618
Layer 19 | rho=0.9322 | R²=0.8859 | MAE=0.1596
Layer 23 | rho=0.9271 | R²=0.8787 | MAE=0.1642

🏆 Selected activation-only feedback layer: 13
  Extracting CALIBRATION_PROBABILITY...
  Extracting CALIBRATION_POLICY...
  Extracting VALIDATION...
  Extracting AR2_TRANSFER...
  Extracting DELAYED_TRANSFER...
  Extracting OSCILLATORY_TRANSFER...
  Extracting MEAN_REVERTING_TRANSFER...
  Extracting ABRUPT_UNKNOWN...
  Extracting MECHANISM_NOVEL...

🧪 TEST B — DISENTANGLED MODEL SE